In [ ]:
#new integration approaches 27032026

In [1]:
import scanpy as sc 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd
import muon as mu

/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
import os
os.getcwd()

'/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026'

In [3]:
import os
os.chdir('/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_18122025')

In [4]:
dcm1 = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/NPC_diff_plate1_plate2_WG.csv', index_col=0)
rna1 =pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool30_31_RNA/NPC/NPC_counts.csv',index_col=0)
meta1 = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/metadata_NPC_DNA_RNA_wholegene.csv', index_col=0)
meta1['DCM_matrixID'] = 'X' + meta1['Barcode'].astype(str) + '_' + meta1['Plate.x']
meta1.rename(columns={'Plate.x': 'Plate'}, inplace=True)
meta1.rename(columns={'Sample': 'Sample_RNA'}, inplace=True)

In [6]:
root ='/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026'

In [24]:
##############################################
# INTEGRATION: HARMONY
# Fully self-contained — no shared state with other scripts
##############################################

import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
import harmonypy as hm
import os

METHOD_NAME = 'Harmony'

root   = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
outdir = os.path.join(root, 'integration', 'method_comparison', METHOD_NAME)
os.makedirs(outdir, exist_ok=True)

sc.set_figure_params(dpi=300, dpi_save=600, fontsize=6)
sc.settings.figdir = outdir

N_TOP_GENES = 2000
N_PCS       = 30
RESOLUTION  = 0.5
SAMPLE_SIZE = 5000

# ─────────────────────────────────────────────
# STEP 0: LOAD & QC REFERENCE DATASETS
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 0: LOADING REFERENCE DATASETS")
print("=" * 60)

import muon as mu

adata_ref_D2 = sc.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/NPC_D2/sample_filtered_feature_bc_matrix.h5")
adata_ref_D4 = sc.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/NPC_D4/sample_filtered_feature_bc_matrix.h5")

mdata           = mu.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/filtered_feature_bc_matrix.h5")
D2_D4_multi_RNA = mdata.mod['rna']

human_barcodes  = pd.read_csv("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/gem_classification.csv")
mouse_barcodes  = human_barcodes[human_barcodes["call"] == "mm10"]
D2_D4_multi_RNA = D2_D4_multi_RNA[D2_D4_multi_RNA.obs.index.isin(mouse_barcodes['barcode'])].copy()

print(f"  ref_D2    : {adata_ref_D2.n_obs} cells")
print(f"  ref_D4    : {adata_ref_D4.n_obs} cells")
print(f"  multi_RNA : {D2_D4_multi_RNA.n_obs} cells (mouse only)")

for ref in [adata_ref_D2, adata_ref_D4, D2_D4_multi_RNA]:
    ref.var_names_make_unique()
    ref.var['mt'] = ref.var_names.str.startswith('mt-')
    sc.pp.calculate_qc_metrics(ref, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

adata_ref_D2    = adata_ref_D2[
    (adata_ref_D2.obs['n_genes_by_counts'] >= 200) &
    (adata_ref_D2.obs['pct_counts_mt'] < 25)].copy()
adata_ref_D4    = adata_ref_D4[
    (adata_ref_D4.obs['n_genes_by_counts'] >= 200) &
    (adata_ref_D4.obs['pct_counts_mt'] < 25)].copy()
D2_D4_multi_RNA = D2_D4_multi_RNA[
    (D2_D4_multi_RNA.obs['n_genes_by_counts'] >= 200) &
    (D2_D4_multi_RNA.obs['pct_counts_mt'] < 25)].copy()

print(f"\nAfter QC:")
print(f"  ref_D2    : {adata_ref_D2.n_obs} cells")
print(f"  ref_D4    : {adata_ref_D4.n_obs} cells")
print(f"  multi_RNA : {D2_D4_multi_RNA.n_obs} cells")

# ─────────────────────────────────────────────
# STEP 1: LOAD scNMT DATA
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING scNMT DATA")
print("=" * 60)

dcm1  = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/NPC_diff_plate1_plate2_WG.csv', index_col=0).T
rna1  = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool30_31_RNA/NPC/NPC_counts.csv', index_col=0).T
meta1 = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/metadata_NPC_DNA_RNA_wholegene.csv', index_col=0)

meta1['DCM_matrixID'] = 'X' + meta1['Barcode'].astype(str) + '_' + meta1['Plate.x']
meta1.rename(columns={'Plate.x': 'Plate', 'Sample': 'Sample_RNA'}, inplace=True)

def prepare(meta, pool_name):
    m = meta.copy()
    m = m[m["Plate"].notna() & m["Well"].notna()].copy()
    m["pool"]       = pool_name
    m["plate_well"] = m["Plate"].astype(str) + "_" + m["Well"].astype(str)
    return m

m1 = prepare(meta1, "pool1")

dcm1 = dcm1.reindex(columns=sorted(set(dcm1.columns)), fill_value=0)
rna1 = rna1.reindex(columns=sorted(set(rna1.columns)), fill_value=0)

adata_dcm1 = sc.AnnData(X=dcm1)
adata_rna1 = sc.AnnData(X=rna1)

sc.pp.filter_cells(adata_dcm1, min_genes=200)
sc.pp.filter_cells(adata_rna1, min_genes=200)

def attach_meta(adata, meta, id_col):
    m    = meta.set_index(id_col)
    keep = m.index.intersection(adata.obs_names)
    adata = adata[keep].copy()
    adata.obs = adata.obs.join(m.loc[keep])
    return adata

adata_dcm1 = attach_meta(adata_dcm1, m1, "DCM_matrixID")
adata_rna1 = attach_meta(adata_rna1, m1, "Sample_RNA")

adata_dcm1.obs['batch']      = 'scNMT_DCM'
adata_rna1.obs['batch']      = 'scNMT_RNA'
adata_dcm1.obs['orig.ident'] = 'scNMT_' + adata_dcm1.obs['pool'].astype(str) + '_DCM'
adata_rna1.obs['orig.ident'] = 'scNMT_' + adata_rna1.obs['pool'].astype(str) + '_RNA'

adata_dcm1 = adata_dcm1[adata_dcm1.obs['Dox'] != 'Minus'].copy()
adata_rna1 = adata_rna1[adata_rna1.obs['Dox'] != 'Minus'].copy()



# ─────────────────────────────────────────────
# STEP 2: PREPARE REFERENCE DATASETS
# ─────────────────────────────────────────────
adata_ref_D2.obs['batch']        = 'ref_D2'
adata_ref_D2.obs['orig.ident']   = 'ref_D2'
adata_ref_D4.obs['batch']        = 'ref_D4'
adata_ref_D4.obs['orig.ident']   = 'ref_D4'
D2_D4_multi_RNA.obs['batch']     = 'multi_RNA'
D2_D4_multi_RNA.obs['orig.ident']= 'multi_RNA'

# ─────────────────────────────────────────────
# STEP 3: COMBINE & PREPROCESS
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 3: COMBINING & PREPROCESSING")
print("=" * 60)

all_datasets = [adata_dcm1, adata_rna1, adata_ref_D2, adata_ref_D4, D2_D4_multi_RNA]
all_obs_cols  = list(set().union(*[a.obs.columns for a in all_datasets]))

for a in all_datasets:
    for col in all_obs_cols:
        if col not in a.obs.columns:
            a.obs[col] = 'Unknown'

adata = sc.concat(all_datasets, join='inner', merge='unique', index_unique=None)

for col in ['Dox', 'orig.ident', 'batch']:
    if col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].fillna('Unknown').astype(str)

print(f"Combined: {adata.n_obs} cells, {adata.n_vars} genes")
print(adata.obs['batch'].value_counts())

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers['lognorm'] = adata.X.copy()

sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, batch_key='orig.ident')
print(f"HVGs: {adata.var['highly_variable'].sum()}")

sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver='arpack')
print("PCA done.")

# Set .raw from lognorm layer
adata_raw   = adata.copy()
adata_raw.X = adata_raw.layers['lognorm']
adata.raw   = adata_raw

# ─────────────────────────────────────────────
# STEP 4: HARMONY INTEGRATION
#
# Key fix: harmonypy receives a plain numpy array and a plain
# pandas DataFrame — both freshly extracted here so there is
# zero risk of stale index or dtype issues from earlier steps.
# The corrected embedding is stored as X_pca_harmony; neighbors
# and UMAP are built exclusively from that key.
# ─────────────────────────────────────────────
print("=" * 60)
print(f"INTEGRATION: {METHOD_NAME}")
print("=" * 60)

# Extract PCA matrix as a plain numpy array
pca_matrix = adata.obsm['X_pca'][:, :N_PCS].copy()   # (n_cells, N_PCS)

# Build a fresh DataFrame with a clean integer index — critical to avoid
# the index mismatch that causes ho.Z_corr to collapse to shape (N_PCS,)
batch_df = pd.DataFrame(
    {'orig.ident': adata.obs['orig.ident'].values},   # .values strips the AnnData index
    index=np.arange(adata.n_obs)                       # clean 0..n_cells integer index
)

print(f"  PCA matrix  : {pca_matrix.shape}")
print(f"  Batch labels: {batch_df['orig.ident'].value_counts().to_dict()}")

ho = hm.run_harmony(
    pca_matrix,
    batch_df,
    vars_use='orig.ident',
    random_state=42,
    max_iter_harmony=20,
    verbose=True
)

# ho.Z_corr orientation depends on the harmonypy version:
#   older versions : (N_PCS, n_cells)  → needs .T
#   newer versions : (n_cells, N_PCS)  → use directly
_raw = ho.Z_corr
if _raw.shape == (adata.n_obs, N_PCS):
    harmony_embedding = _raw          # already (n_cells, N_PCS)
elif _raw.shape == (N_PCS, adata.n_obs):
    harmony_embedding = _raw.T        # transpose to (n_cells, N_PCS)
else:
    raise ValueError(
        f"Unexpected ho.Z_corr shape {_raw.shape}. "
        f"Expected ({adata.n_obs}, {N_PCS}) or ({N_PCS}, {adata.n_obs})."
    )
print(f"  Harmony raw output shape : {_raw.shape}  →  embedding: {harmony_embedding.shape}")

adata.obsm['X_pca_harmony'] = harmony_embedding
print(f"  Harmony embedding stored: {adata.obsm['X_pca_harmony'].shape}")

# Build neighbors and UMAP exclusively from the harmony embedding
sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=15, n_pcs=N_PCS)
print("  Neighbors built from X_pca_harmony.")

# ─────────────────────────────────────────────
# STEP 5: UMAP + LEIDEN
# ─────────────────────────────────────────────
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=RESOLUTION, key_added='leiden', random_state=0)
n_clusters = len(adata.obs['leiden'].unique())
print(f"Clusters: {n_clusters}")

annots = {}
for cl in adata.obs['leiden'].astype(str).unique():
    cells  = adata.obs[adata.obs['leiden'].astype(str) == cl]
    counts = cells['batch'].value_counts()
    annots[cl] = counts.idxmax() if len(counts) > 0 else 'Unknown'
adata.obs['leiden_annot'] = adata.obs['leiden'].astype(str).map(annots).fillna('Unknown')
adata.obs['leiden_annot'] = adata.obs['leiden'].astype(str) + '_' + adata.obs['leiden_annot']

# ─────────────────────────────────────────────
# STEP 6: SILHOUETTE SCORES (use X_pca_harmony)
# ─────────────────────────────────────────────
sample_idx  = np.random.choice(adata.n_obs, min(SAMPLE_SIZE, adata.n_obs), replace=False)
emb         = adata.obsm['X_pca_harmony'][sample_idx, :N_PCS]
sil_batch   = silhouette_score(emb, adata.obs['orig.ident'].values[sample_idx])
sil_cluster = silhouette_score(emb, adata.obs['leiden'].values[sample_idx]) if n_clusters > 1 else np.nan
print(f"Silhouette batch   (lower=better) : {sil_batch:.4f}")
print(f"Silhouette cluster (higher=better): {sil_cluster:.4f}")

pd.DataFrame([{
    'method': METHOD_NAME, 'silhouette_batch': sil_batch,
    'silhouette_cluster': sil_cluster, 'n_clusters': n_clusters
}]).to_csv(os.path.join(outdir, f'{METHOD_NAME}_silhouette.csv'), index=False)

# ─────────────────────────────────────────────
# STEP 7: PLOTS
# ─────────────────────────────────────────────

# Plot 1: UMAP overview
plot_cols = [c for c in ['Dox', 'batch', 'orig.ident', 'leiden_annot', 'leiden'] if c in adata.obs.columns]
sc.pl.umap(adata, color=plot_cols, frameon=False, legend_loc='on data', show=False,
           save=f'_{METHOD_NAME}_umap_overview.pdf')
print("Saved: umap_overview")

# Plot 2: Per-batch highlight
batches = adata.obs['orig.ident'].unique()
fig, axes = plt.subplots(1, len(batches), figsize=(6 * len(batches), 5))
if len(batches) == 1:
    axes = [axes]
for ax, b in zip(axes, batches):
    adata.obs['_hl'] = (adata.obs['orig.ident'] == b).astype(str)
    sc.pl.umap(adata, color='_hl', ax=ax, show=False, title=str(b),
               palette={'True': '#e74c3c', 'False': '#d3d3d3'})
del adata.obs['_hl']
plt.suptitle(f'{METHOD_NAME} — Per-batch highlight', fontsize=10, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(outdir, f'{METHOD_NAME}_umap_per_batch.pdf'), dpi=200, bbox_inches='tight')
plt.close()
print("Saved: umap_per_batch")

# Plot 3: Marker genes + dotplot
try:
    sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon', use_raw=True)
    sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False, show=False,
                            save=f'_{METHOD_NAME}_rank_genes.pdf')
    top_genes = []
    for cl in sorted(adata.obs['leiden'].unique(), key=lambda x: int(x) if str(x).isdigit() else x):
        top_genes.extend(sc.get.rank_genes_groups_df(adata, group=str(cl)).head(5)['names'].tolist())
    top_genes = list(dict.fromkeys(top_genes))
    sc.pl.dotplot(adata, var_names=top_genes, groupby='leiden', show=False, use_raw=True,
                  save=f'_{METHOD_NAME}_dotplot_top5.pdf')
    print("Saved: rank_genes + dotplot")
except Exception as e:
    print(f"Marker gene plots failed: {e}")



print(f"\nDone! All outputs in: {outdir}")

STEP 0: LOADING REFERENCE DATASETS


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/hom

Added `interval` annotation for features from /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/filtered_feature_bc_matrix.h5


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/mudata/_core/mudata.py:915: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  warnings.warn(
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/pyt

  ref_D2    : 2548 cells
  ref_D4    : 2347 cells
  multi_RNA : 10305 cells (mouse only)

After QC:
  ref_D2    : 2545 cells
  ref_D4    : 2335 cells
  multi_RNA : 9912 cells
STEP 1: LOADING scNMT DATA
STEP 3: COMBINING & PREPROCESSING
Combined: 15920 cells, 14384 genes
batch
multi_RNA    9912
ref_D2       2545
ref_D4       2335
scNMT_RNA     723
scNMT_DCM     405
Name: count, dtype: int64
HVGs: 2000


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


PCA done.


2026-03-27 17:49:53,345 - harmonypy - INFO - Running Harmony (PyTorch on cpu)
2026-03-27 17:49:53,346 - harmonypy - INFO -   Parameters:
2026-03-27 17:49:53,346 - harmonypy - INFO -     max_iter_harmony: 20
2026-03-27 17:49:53,347 - harmonypy - INFO -     max_iter_kmeans: 20
2026-03-27 17:49:53,347 - harmonypy - INFO -     epsilon_cluster: 1e-05
2026-03-27 17:49:53,347 - harmonypy - INFO -     epsilon_harmony: 0.0001
2026-03-27 17:49:53,347 - harmonypy - INFO -     nclust: 100
2026-03-27 17:49:53,348 - harmonypy - INFO -     block_size: 0.05
2026-03-27 17:49:53,348 - harmonypy - INFO -     lamb: [1. 1. 1. 1. 1.]
2026-03-27 17:49:53,349 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2.]
2026-03-27 17:49:53,350 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-03-27 17:49:53,350 - harmonypy - INFO -     verbose: True
2026-03-27 17:49:53,351 - harmonypy - INFO -     random_state: 42
2026-03-27 17:49:53,352 - harmonypy - INFO -   Data: 30 PCs × 15920 cells
2026-03-27 17:49:53,352

INTEGRATION: Harmony
  PCA matrix  : (15920, 30)
  Batch labels: {'multi_RNA': 9912, 'ref_D2': 2545, 'ref_D4': 2335, 'scNMT_pool1_RNA': 723, 'scNMT_pool1_DCM': 405}


2026-03-27 17:49:53,603 - harmonypy - INFO - KMeans initialization complete.
2026-03-27 17:49:53,771 - harmonypy - INFO - Iteration 1 of 20
2026-03-27 17:50:09,698 - harmonypy - INFO - Iteration 2 of 20
2026-03-27 17:50:24,465 - harmonypy - INFO - Iteration 3 of 20
2026-03-27 17:50:39,690 - harmonypy - INFO - Iteration 4 of 20
2026-03-27 17:50:54,379 - harmonypy - INFO - Iteration 5 of 20
2026-03-27 17:51:07,995 - harmonypy - INFO - Iteration 6 of 20
2026-03-27 17:51:22,770 - harmonypy - INFO - Iteration 7 of 20
2026-03-27 17:51:38,655 - harmonypy - INFO - Iteration 8 of 20
2026-03-27 17:51:53,116 - harmonypy - INFO - Iteration 9 of 20
2026-03-27 17:52:14,152 - harmonypy - INFO - Iteration 10 of 20
2026-03-27 17:52:34,310 - harmonypy - INFO - Iteration 11 of 20
2026-03-27 17:52:50,326 - harmonypy - INFO - Iteration 12 of 20
2026-03-27 17:53:06,301 - harmonypy - INFO - Iteration 13 of 20
2026-03-27 17:53:20,623 - harmonypy - INFO - Iteration 14 of 20
2026-03-27 17:53:34,398 - harmonypy 

  Harmony raw output shape : (15920, 30)  →  embedding: (15920, 30)
  Harmony embedding stored: (15920, 30)
  Neighbors built from X_pca_harmony.
Clusters: 11
Silhouette batch   (lower=better) : -0.0622
Silhouette cluster (higher=better): 0.1480
Saved: umap_overview
Saved: umap_per_batch
Saved: rank_genes + dotplot

Done! All outputs in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony


In [7]:
meta1 = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/metadata_NPC_DNA_RNA_wholegene.csv', index_col=0)

meta1

,Sample,Plate.x,PE_raw,PE_aligned,Assigned,UMI,Unique_Gene_Count,Aligned_percentage,Assigned_percentage,Unique_percentage,...,CpG_unique,DCM_unique,DCM_UMI_genebodies,Genes,DCM_UMI_wholegene,Genes_wholegene,Aligned_P,CpG_unique_P,DCM_unique_P,DCM.CpG
1,NPC_plate1_A10,Plate_1,11074,3402,2194,1034,786,30.720607,64.491476,47.128532,...,694,40,19,14,23,17,53.146294,4.374961,0.00350390948701,0.057637
2,NPC_plate1_A11,Plate_1,427680,184963,130340,10113,3597,43.247989,70.468148,7.758938,...,47396,4670,1548,1067,1731,1146,91.567894,6.122707,7.22161225973062,0.098532
3,NPC_plate1_A12,Plate_1,390172,285007,193931,10529,3676,73.046503,68.044294,5.429251,...,829,35,16,14,21,19,40.843262,11.804072,0.004561288733487,0.042220
4,NPC_plate1_A13,Plate_1,1570682,764331,465566,12125,4032,48.662365,60.911568,2.604357,...,27124,987,437,290,463,307,93.741647,7.253260,5.1980198019802,0.036388
5,NPC_plate1_A14,Plate_1,10580,6733,5031,2068,1265,63.638941,74.721521,41.105148,...,410,18,7,4,8,5,46.193818,11.585194,0.003126959778611,0.043902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729,NPC_plate2_P5,Plate_2,86872,44752,29263,10354,4292,51.514872,65.389256,35.382565,...,96993,6187,2327,1685,2755,1794,96.184770,12.258354,46.3584594635097,0.063788
730,NPC_plate2_P6,Plate_2,71396,31748,21989,7775,3513,44.467477,69.261056,35.358588,...,38839,2269,947,694,1078,741,95.336176,12.891245,72.284166932144,0.058421
731,NPC_plate2_P7,Plate_2,515910,349038,221657,14881,4298,67.654824,63.505120,6.713526,...,467,46,26,14,36,20,85.502360,17.709518,8.37887067395264,0.098501
732,NPC_plate2_P8,Plate_2,93382,37169,28292,10703,4021,39.803174,76.117194,37.830482,...,227,34,13,13,16,15,80.369128,24.148936,40.9638554216867,0.149780


In [8]:
dcm1  = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/NPC_diff_plate1_plate2_WG.csv', index_col=0).T
rna1  = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool30_31_RNA/NPC/NPC_counts.csv', index_col=0).T

In [10]:
rna1

,0610005C13Rik,0610009E02Rik,0610009L18Rik,0610010K14Rik,0610012D04Rik,0610030E20Rik,0610038B21Rik,0610039K10Rik,0610040B10Rik,0610040F04Rik,...,mt-Tl1,mt-Tn,mt-Tp,mt-Tq,mt-Tt,mt-Tv,mt-Tw,n-R5s173,n-R5s25,n-R5s88
NPC_plate1_A1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NPC_plate1_A2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NPC_plate1_A3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NPC_plate1_A4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NPC_plate1_A5,0,0,0,0,0,1,0,0,0,0,...,0,0,0,2,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NPC_plate2_P20,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NPC_plate2_P21,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
NPC_plate2_P22,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NPC_plate2_P23,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
##############################################
# HARMONY PARAMETER SWEEP
# Evaluates all combinations of:
#   N_TOP_GENES, N_PCS, n_neighbors, resolution
# Validates with:
#   1. Silhouette scores (batch mixing + cluster separation)
#   2. DCM→RNA transition stability per Dox condition
#   3. Label consistency across resolutions (DCM cells only)
# Outputs ranked CSV + heatmap + best-config final run
##############################################

import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score
import harmonypy as hm
import os
import itertools
from copy import deepcopy

# ─────────────────────────────────────────────
# PATHS & OUTPUT
# ─────────────────────────────────────────────
root   = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_sweep')
os.makedirs(outdir, exist_ok=True)

sc.set_figure_params(dpi=150, dpi_save=300, fontsize=6)
sc.settings.figdir = outdir

# How many cells to sample for silhouette (speed vs accuracy trade-off)
SAMPLE_SIZE = 5000

# ─────────────────────────────────────────────
# PARAMETER GRID — edit to taste
# ─────────────────────────────────────────────
PARAM_GRID = {
    'N_TOP_GENES': [3000],
    'N_PCS':       [20, 30, 50],
    'n_neighbors': [10, 20, 30],
    'resolution':  [0.2],
}

# ─────────────────────────────────────────────
# STEP 0: LOAD & QC  (identical to original script)
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 0: LOADING & QC")
print("=" * 60)

import muon as mu

adata_ref_D2 = sc.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/NPC_D2/sample_filtered_feature_bc_matrix.h5")
adata_ref_D4 = sc.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/NPC_D4/sample_filtered_feature_bc_matrix.h5")

mdata           = mu.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/filtered_feature_bc_matrix.h5")
D2_D4_multi_RNA = mdata.mod['rna']

human_barcodes  = pd.read_csv("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/gem_classification.csv")
mouse_barcodes  = human_barcodes[human_barcodes["call"] == "mm10"]
D2_D4_multi_RNA = D2_D4_multi_RNA[D2_D4_multi_RNA.obs.index.isin(mouse_barcodes['barcode'])].copy()

for ref in [adata_ref_D2, adata_ref_D4, D2_D4_multi_RNA]:
    ref.var_names_make_unique()
    ref.var['mt'] = ref.var_names.str.startswith('mt-')
    sc.pp.calculate_qc_metrics(ref, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

adata_ref_D2    = adata_ref_D2[(adata_ref_D2.obs['n_genes_by_counts'] >= 200) & (adata_ref_D2.obs['pct_counts_mt'] < 25)].copy()
adata_ref_D4    = adata_ref_D4[(adata_ref_D4.obs['n_genes_by_counts'] >= 200) & (adata_ref_D4.obs['pct_counts_mt'] < 25)].copy()
D2_D4_multi_RNA = D2_D4_multi_RNA[(D2_D4_multi_RNA.obs['n_genes_by_counts'] >= 200) & (D2_D4_multi_RNA.obs['pct_counts_mt'] < 25)].copy()

# scNMT
dcm1  = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/NPC_diff_plate1_plate2_WG.csv', index_col=0).T
rna1  = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool30_31_RNA/NPC/NPC_counts.csv', index_col=0).T
meta1 = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/metadata_NPC_DNA_RNA_wholegene.csv', index_col=0)

meta1['DCM_matrixID'] = 'X' + meta1['Barcode'].astype(str) + '_' + meta1['Plate.x']
meta1.rename(columns={'Plate.x': 'Plate', 'Sample': 'Sample_RNA'}, inplace=True)

def prepare(meta, pool_name):
    m = meta.copy()
    m = m[m["Plate"].notna() & m["Well"].notna()].copy()
    m["pool"]       = pool_name
    m["plate_well"] = m["Plate"].astype(str) + "_" + m["Well"].astype(str)
    return m

m1 = prepare(meta1, "pool1")
dcm1 = dcm1.reindex(columns=sorted(set(dcm1.columns)), fill_value=0)
rna1 = rna1.reindex(columns=sorted(set(rna1.columns)), fill_value=0)

adata_dcm1 = sc.AnnData(X=dcm1)
adata_rna1 = sc.AnnData(X=rna1)
sc.pp.filter_cells(adata_dcm1, min_genes=200)
sc.pp.filter_cells(adata_rna1, min_genes=200)

def attach_meta(adata, meta, id_col):
    m    = meta.set_index(id_col)
    keep = m.index.intersection(adata.obs_names)
    adata = adata[keep].copy()
    adata.obs = adata.obs.join(m.loc[keep])
    return adata

adata_dcm1 = attach_meta(adata_dcm1, m1, "DCM_matrixID")
adata_rna1 = attach_meta(adata_rna1, m1, "Sample_RNA")

adata_dcm1.obs['batch']      = 'scNMT_DCM'
adata_rna1.obs['batch']      = 'scNMT_RNA'
adata_dcm1.obs['orig.ident'] = 'scNMT_pool1_DCM'
adata_rna1.obs['orig.ident'] = 'scNMT_pool1_RNA'
adata_dcm1 = adata_dcm1[adata_dcm1.obs['Dox'] != 'Minus'].copy()
adata_rna1 = adata_rna1[adata_rna1.obs['Dox'] != 'Minus'].copy()

adata_ref_D2.obs['batch']         = 'ref_D2';    adata_ref_D2.obs['orig.ident']    = 'ref_D2'
adata_ref_D4.obs['batch']         = 'ref_D4';    adata_ref_D4.obs['orig.ident']    = 'ref_D4'
D2_D4_multi_RNA.obs['batch']      = 'multi_RNA'; D2_D4_multi_RNA.obs['orig.ident'] = 'multi_RNA'

all_datasets = [adata_dcm1, adata_rna1, adata_ref_D2, adata_ref_D4, D2_D4_multi_RNA]
all_obs_cols  = list(set().union(*[a.obs.columns for a in all_datasets]))
for a in all_datasets:
    for col in all_obs_cols:
        if col not in a.obs.columns:
            a.obs[col] = 'Unknown'

# ─────────────────────────────────────────────
# RAW CONCATENATED OBJECT (before normalisation)
# We'll normalise inside each sweep iteration so
# HVG selection is affected by N_TOP_GENES.
# ─────────────────────────────────────────────
adata_raw_concat = sc.concat(all_datasets, join='inner', merge='unique', index_unique=None)
for col in ['Dox', 'orig.ident', 'batch']:
    if col in adata_raw_concat.obs.columns:
        adata_raw_concat.obs[col] = adata_raw_concat.obs[col].fillna('Unknown').astype(str)

print(f"Raw concat: {adata_raw_concat.n_obs} cells × {adata_raw_concat.n_vars} genes")
print(adata_raw_concat.obs['batch'].value_counts())

# ─────────────────────────────────────────────
# HELPER: run one Harmony config
# ─────────────────────────────────────────────
def run_harmony_config(adata_in, N_TOP_GENES, N_PCS, n_neighbors, resolution):
    """
    Full preprocessing + Harmony + UMAP + Leiden for one parameter set.
    Returns the annotated AnnData (no plots saved here).
    """
    adata = adata_in.copy()

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers['lognorm'] = adata.X.copy()

    sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, batch_key='orig.ident')
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=50, svd_solver='arpack')

    # Store .raw from lognorm
    adata_raw_obj   = adata.copy()
    adata_raw_obj.X = adata_raw_obj.layers['lognorm']
    adata.raw       = adata_raw_obj

    # Harmony
    pca_matrix = adata.obsm['X_pca'][:, :N_PCS].copy()
    batch_df   = pd.DataFrame(
        {'orig.ident': adata.obs['orig.ident'].values},
        index=np.arange(adata.n_obs)
    )
    ho   = hm.run_harmony(pca_matrix, batch_df, vars_use='orig.ident',
                          random_state=42, max_iter_harmony=20, verbose=False)
    _raw = ho.Z_corr
    if _raw.shape == (adata.n_obs, N_PCS):
        harmony_emb = _raw
    elif _raw.shape == (N_PCS, adata.n_obs):
        harmony_emb = _raw.T
    else:
        raise ValueError(f"Unexpected ho.Z_corr shape {_raw.shape}")

    adata.obsm['X_pca_harmony'] = harmony_emb

    sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=n_neighbors, n_pcs=N_PCS)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=resolution, key_added='leiden', random_state=0)

    # Annotate clusters by majority batch
    annots = {}
    for cl in adata.obs['leiden'].astype(str).unique():
        cells  = adata.obs[adata.obs['leiden'].astype(str) == cl]
        counts = cells['batch'].value_counts()
        annots[cl] = counts.idxmax() if len(counts) > 0 else 'Unknown'
    adata.obs['leiden_annot'] = adata.obs['leiden'].astype(str).map(annots).fillna('Unknown')
    adata.obs['leiden_annot'] = adata.obs['leiden'].astype(str) + '_' + adata.obs['leiden_annot']

    return adata

# ─────────────────────────────────────────────
# METRIC 1: Silhouette
# ─────────────────────────────────────────────
def compute_silhouette(adata, N_PCS, sample_size=SAMPLE_SIZE):
    n        = min(sample_size, adata.n_obs)
    idx      = np.random.choice(adata.n_obs, n, replace=False)
    emb      = adata.obsm['X_pca_harmony'][idx, :N_PCS]
    batch_lb = adata.obs['orig.ident'].values[idx]
    clust_lb = adata.obs['leiden'].values[idx]
    n_cl     = len(np.unique(clust_lb))
    sil_batch   = silhouette_score(emb, batch_lb)
    sil_cluster = silhouette_score(emb, clust_lb) if n_cl > 1 else np.nan
    return sil_batch, sil_cluster

# ─────────────────────────────────────────────
# METRIC 2: DCM→RNA transition stability
# Matching via pool + plate_well (spatial coordinate), mirroring
# the reference matching script exactly.  Returns mean stable
# fraction across Dox groups.
#
# Index name asymmetry (from attach_meta in the loading step):
#   DCM obs_names  = DCM_matrixID  (format  X{Barcode}_{Plate})
#   RNA obs_names  = Sample_RNA    (the RNA sample identifier)
# After sc.concat both live in one .obs but the obs_names are
# still the original modality-specific IDs.  We never rely on
# the obs_names for matching — we use pool + plate_well instead,
# which is present in .obs for both modalities because attach_meta
# pulled it from the shared metadata table.
# ─────────────────────────────────────────────
def compute_transition_stability(adata):
    """
    1. Split adata.obs into DCM and RNA subsets by batch label.
    2. Add the original modality-specific ID back as a plain column:
         DCM rows → 'DCM_matrixID'  (= obs_names for that subset)
         RNA rows → 'Sample_RNA'    (= obs_names for that subset)
    3. Inner-join on pool + plate_well, same as the reference script.
    4. Compare leiden_annot_clean on both sides per Dox group.
    """
    obs = adata.obs.copy()
    obs['leiden_annot_clean'] = obs['leiden_annot'].str.replace(r'^\d+_', '', regex=True)

    dcm_mask = obs['batch'].str.contains('DCM', case=False, na=False)
    rna_mask = obs['batch'].str.contains('RNA', case=False, na=False)
    dcm_obs  = obs[dcm_mask].copy()
    rna_obs  = obs[rna_mask].copy()

    if dcm_obs.empty or rna_obs.empty:
        return np.nan, pd.DataFrame()

    required = {'pool', 'plate_well'}
    if not required.issubset(dcm_obs.columns) or not required.issubset(rna_obs.columns):
        print("  [stability] pool/plate_well columns missing — skipping metric")
        return np.nan, pd.DataFrame()

    # Expose the modality-specific ID as an explicit column.
    # For DCM cells the obs_names ARE the DCM_matrixID values
    # (set by attach_meta(..., id_col='DCM_matrixID')).
    # For RNA cells the obs_names ARE the Sample_RNA values
    # (set by attach_meta(..., id_col='Sample_RNA')).
    dcm_obs = dcm_obs.copy()
    dcm_obs['DCM_matrixID'] = dcm_obs.index   # X{Barcode}_{Plate}

    rna_obs = rna_obs.copy()
    rna_obs['Sample_RNA'] = rna_obs.index      # RNA sample identifier

    # Build per-modality tables with only the columns needed for the join,
    # mirroring valid_dcm / valid_rna in the reference script.
    dcm_tbl = dcm_obs[['DCM_matrixID', 'pool', 'plate_well', 'Dox',
                        'leiden_annot_clean']].reset_index(drop=True)

    rna_tbl = rna_obs[['Sample_RNA', 'pool', 'plate_well',
                        'leiden_annot_clean']].reset_index(drop=True)

    # Inner join on pool + plate_well — identical to the reference script's
    # matched = valid_dcm.merge(valid_rna, on=['pool','plate_well'], how='inner')
    matched = dcm_tbl.merge(
        rna_tbl,
        on=['pool', 'plate_well'],
        suffixes=('_dcm', '_rna'),
        how='inner'
    )

    if matched.empty:
        print("  [stability] no pool/plate_well matches found")
        return np.nan, pd.DataFrame()

    print(f"  [stability] matched {len(matched)} DCM–RNA pairs "
          f"(DCM_matrixID ↔ Sample_RNA via pool+plate_well)")

    results = []
    for dox in matched['Dox'].unique():
        sub        = matched[matched['Dox'] == dox]
        stable_pct = (sub['leiden_annot_clean_dcm'] == sub['leiden_annot_clean_rna']).mean() * 100
        results.append({'Dox': dox, 'stable_pct': stable_pct, 'n_cells': len(sub)})

    df = pd.DataFrame(results)
    return df['stable_pct'].mean() if not df.empty else np.nan, df

# ─────────────────────────────────────────────
# METRIC 3: Label consistency across resolutions
# (DCM cells: how consistently do they get the same cluster label
#  when resolution changes within this config's resolution range?)
# Here we re-cluster at the adjacent resolution points and measure
# cross-resolution agreement for DCM cells only.
# ─────────────────────────────────────────────
def compute_label_consistency(adata, resolutions, N_PCS):
    """
    For each pair of resolutions, compute fraction of DCM cells
    assigned the same majority cluster type.
    Returns mean pairwise consistency.
    """
    obs   = adata.obs.copy()
    dcm_m = obs['batch'].str.contains('DCM', case=False, na=False)
    dcm_cells = adata[dcm_m].obs_names

    if len(dcm_cells) == 0:
        return np.nan

    label_store = {}
    for res in resolutions:
        sc.tl.leiden(adata, resolution=res, key_added=f'_tmp_leiden_{res}', random_state=0)
        # Strip number prefix so we compare cell-type names
        adata.obs[f'_tmp_leiden_{res}'] = adata.obs[f'_tmp_leiden_{res}'].astype(str)
        label_store[res] = adata.obs.loc[dcm_cells, f'_tmp_leiden_{res}'].values

    pairs = list(itertools.combinations(resolutions, 2))
    agreements = []
    for r1, r2 in pairs:
        agree = (label_store[r1] == label_store[r2]).mean()
        agreements.append(agree)

    # Clean up temp columns
    for res in resolutions:
        if f'_tmp_leiden_{res}' in adata.obs.columns:
            del adata.obs[f'_tmp_leiden_{res}']

    return float(np.mean(agreements)) if agreements else np.nan

# ─────────────────────────────────────────────
# SWEEP
# ─────────────────────────────────────────────
print("=" * 60)
print("PARAMETER SWEEP")
print("=" * 60)

keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*[PARAM_GRID[k] for k in keys]))
print(f"Total configurations: {len(combos)}")

all_results = []
np.random.seed(42)

for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    cfg_id = f"cfg_{i:03d}"
    print(f"\n[{i+1}/{len(combos)}] {params}")

    try:
        adata_cfg = run_harmony_config(
            adata_raw_concat,
            N_TOP_GENES = params['N_TOP_GENES'],
            N_PCS       = params['N_PCS'],
            n_neighbors = params['n_neighbors'],
            resolution  = params['resolution'],
        )

        n_clusters              = adata_cfg.obs['leiden'].nunique()
        sil_batch, sil_cluster  = compute_silhouette(adata_cfg, params['N_PCS'])
        mean_stable, stability_df = compute_transition_stability(adata_cfg)

        # Consistency: test at ±0.2 of the chosen resolution
        res_range = sorted(set([
            max(0.1, params['resolution'] - 0.2),
            params['resolution'],
            params['resolution'] + 0.2,
        ]))
        consistency = compute_label_consistency(adata_cfg, res_range, params['N_PCS'])

        # Composite: reward cluster separation and stability, penalise batch mixing
        # (sil_batch lower is better → we subtract it)
        composite = (sil_cluster - sil_batch) + mean_stable / 100 + consistency

        row = {
            'cfg_id':        cfg_id,
            **params,
            'n_clusters':    n_clusters,
            'sil_batch':     round(sil_batch,    4),
            'sil_cluster':   round(sil_cluster,  4),
            'mean_stable_pct': round(mean_stable, 2) if not np.isnan(mean_stable) else np.nan,
            'consistency':   round(consistency,  4) if not np.isnan(consistency) else np.nan,
            'composite':     round(composite,    4),
        }
        all_results.append(row)
        print(f"  → n_clusters={n_clusters}, sil_batch={sil_batch:.3f}, "
              f"sil_cluster={sil_cluster:.3f}, stable={mean_stable:.1f}%, "
              f"consistency={consistency:.3f}, composite={composite:.3f}")

        # Save the individual config UMAP (lightweight)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        sc.pl.umap(adata_cfg, color='batch',         ax=axes[0], show=False, frameon=False, legend_loc='on data')
        sc.pl.umap(adata_cfg, color='leiden',        ax=axes[1], show=False, frameon=False, legend_loc='on data')
        sc.pl.umap(adata_cfg, color='leiden_annot',  ax=axes[2], show=False, frameon=False, legend_loc='on data')
        title = (f"cfg{i+1}: genes={params['N_TOP_GENES']} pcs={params['N_PCS']} "
                 f"k={params['n_neighbors']} res={params['resolution']}")
        fig.suptitle(title, fontsize=9)
        plt.tight_layout()
        plt.savefig(os.path.join(outdir, f'{cfg_id}_umap.png'), dpi=100, bbox_inches='tight')
        plt.close()

    except Exception as e:
        print(f"  ERROR: {e}")
        all_results.append({'cfg_id': cfg_id, **params, 'error': str(e)})

# ─────────────────────────────────────────────
# RESULTS TABLE
# ─────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('composite', ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(outdir, 'sweep_results.csv'), index=False)

print("\n" + "=" * 60)
print("TOP 10 CONFIGURATIONS")
print("=" * 60)
cols_show = ['cfg_id', 'N_TOP_GENES', 'N_PCS', 'n_neighbors', 'resolution',
             'n_clusters', 'sil_batch', 'sil_cluster', 'mean_stable_pct', 'consistency', 'composite']
print(results_df[cols_show].head(10).to_string(index=False))

# ─────────────────────────────────────────────
# HEATMAP: composite score across param pairs
# ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
metric_pairs = [
    ('N_TOP_GENES', 'N_PCS'),
    ('N_TOP_GENES', 'n_neighbors'),
    ('N_TOP_GENES', 'resolution'),
    ('N_PCS',       'n_neighbors'),
    ('N_PCS',       'resolution'),
    ('n_neighbors', 'resolution'),
]
for ax, (p1, p2) in zip(axes.flat, metric_pairs):
    pivot = results_df.pivot_table(values='composite', index=p1, columns=p2, aggfunc='mean')
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis', ax=ax,
                cbar_kws={'label': 'Composite (higher=better)'})
    ax.set_title(f'{p1} vs {p2}')

plt.suptitle('Composite score (sil_cluster − sil_batch + stable/100 + consistency)\nMean over other params',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'sweep_heatmap_composite.png'), dpi=200, bbox_inches='tight')
plt.close()
print("Saved: sweep_heatmap_composite.png")

# ─────────────────────────────────────────────
# HEATMAP: individual metrics
# ─────────────────────────────────────────────
for metric in ['sil_batch', 'sil_cluster', 'mean_stable_pct', 'consistency']:
    pivot = results_df.pivot_table(values=metric, index='N_PCS', columns='n_neighbors', aggfunc='mean')
    plt.figure(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='coolwarm')
    plt.title(f'{metric}: N_PCS vs n_neighbors (mean over other params)')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'sweep_heatmap_{metric}.png'), dpi=150, bbox_inches='tight')
    plt.close()
print("Saved: per-metric heatmaps")

# ─────────────────────────────────────────────
# FULL RUN WITH BEST CONFIG
# ─────────────────────────────────────────────
best = results_df.iloc[0]
print(f"\n{'='*60}")
print("RUNNING BEST CONFIGURATION")
print(f"  N_TOP_GENES={int(best['N_TOP_GENES'])}, N_PCS={int(best['N_PCS'])}, "
      f"n_neighbors={int(best['n_neighbors'])}, resolution={best['resolution']}")
print(f"  Composite score: {best['composite']}")
print(f"{'='*60}")

METHOD_NAME = 'Harmony_best'
best_outdir = os.path.join(root, 'integration', 'method_comparison', METHOD_NAME)
os.makedirs(best_outdir, exist_ok=True)
sc.settings.figdir = best_outdir

N_TOP_GENES = int(best['N_TOP_GENES'])
N_PCS       = int(best['N_PCS'])
n_neighbors = int(best['n_neighbors'])
RESOLUTION  = float(best['resolution'])

adata_best = run_harmony_config(
    adata_raw_concat,
    N_TOP_GENES = N_TOP_GENES,
    N_PCS       = N_PCS,
    n_neighbors = n_neighbors,
    resolution  = RESOLUTION,
)

# ── UMAP overview ──
plot_cols = [c for c in ['Dox', 'batch', 'orig.ident', 'leiden_annot', 'leiden']
             if c in adata_best.obs.columns]
sc.pl.umap(adata_best, color=plot_cols, frameon=False, legend_loc='on data', show=False,
           save=f'_{METHOD_NAME}_umap_overview.pdf')

# ── Per-batch highlight ──
batches = adata_best.obs['orig.ident'].unique()
fig, axes = plt.subplots(1, len(batches), figsize=(6 * len(batches), 5))
if len(batches) == 1:
    axes = [axes]
for ax, b in zip(axes, batches):
    adata_best.obs['_hl'] = (adata_best.obs['orig.ident'] == b).astype(str)
    sc.pl.umap(adata_best, color='_hl', ax=ax, show=False, title=str(b),
               palette={'True': '#e74c3c', 'False': '#d3d3d3'})
del adata_best.obs['_hl']
plt.suptitle(f'{METHOD_NAME} — Per-batch highlight', fontsize=10, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(best_outdir, f'{METHOD_NAME}_umap_per_batch.pdf'), dpi=200, bbox_inches='tight')
plt.close()

# ── Marker genes + dotplot ──
try:
    sc.tl.rank_genes_groups(adata_best, 'leiden', method='wilcoxon', use_raw=True)
    sc.pl.rank_genes_groups(adata_best, n_genes=5, sharey=False, show=False,
                            save=f'_{METHOD_NAME}_rank_genes.pdf')
    top_genes = []
    for cl in sorted(adata_best.obs['leiden'].unique(), key=lambda x: int(x) if str(x).isdigit() else x):
        top_genes.extend(sc.get.rank_genes_groups_df(adata_best, group=str(cl)).head(5)['names'].tolist())
    top_genes = list(dict.fromkeys(top_genes))
    sc.pl.dotplot(adata_best, var_names=top_genes, groupby='leiden', show=False, use_raw=True,
                  save=f'_{METHOD_NAME}_dotplot_top5.pdf')
    print("Saved: rank_genes + dotplot")
except Exception as e:
    print(f"Marker gene plots failed: {e}")

# ── Silhouette of best ──
sil_batch_best, sil_cluster_best = compute_silhouette(adata_best, N_PCS)
print(f"\nBest config silhouette:")
print(f"  Batch   (lower=better) : {sil_batch_best:.4f}")
print(f"  Cluster (higher=better): {sil_cluster_best:.4f}")

pd.DataFrame([{
    'method': METHOD_NAME,
    'N_TOP_GENES': N_TOP_GENES, 'N_PCS': N_PCS,
    'n_neighbors': n_neighbors, 'resolution': RESOLUTION,
    'silhouette_batch': sil_batch_best,
    'silhouette_cluster': sil_cluster_best,
    'n_clusters': adata_best.obs['leiden'].nunique(),
}]).to_csv(os.path.join(best_outdir, f'{METHOD_NAME}_silhouette.csv'), index=False)

# ── Transition stability heatmap (best config) ──
mean_stable_best, stability_df_best = compute_transition_stability(adata_best)
if not stability_df_best.empty:
    print(f"\nTransition stability (best config): {mean_stable_best:.1f}% mean stable across Dox")
    print(stability_df_best.to_string(index=False))
    stability_df_best.to_csv(os.path.join(best_outdir, 'best_transition_stability.csv'), index=False)

# ── DCM label consistency histogram (best config) ──
obs_best = adata_best.obs.copy()
obs_best['leiden_clean'] = obs_best['leiden_annot'].str.replace(r'^\d+_', '', regex=True)
dcm_mask_best = obs_best['batch'].str.contains('DCM', case=False, na=False)
dcm_obs_best  = obs_best[dcm_mask_best]

if not dcm_obs_best.empty and 'Dox' in dcm_obs_best.columns:
    # Consistency at the chosen resolution (compare within Dox groups)
    plt.figure(figsize=(8, 4))
    vc = dcm_obs_best['leiden_clean'].value_counts()
    vc_pct = (vc / vc.sum() * 100).reset_index()
    vc_pct.columns = ['Cell type', 'Percentage']
    sns.barplot(data=vc_pct, x='Cell type', y='Percentage', palette='tab10')
    plt.title('DCM cell-type distribution (best config)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(best_outdir, 'best_dcm_celltype_distribution.png'), dpi=150, bbox_inches='tight')
    plt.close()

# ── Save best adata ──
adata_best.write_h5ad(os.path.join(best_outdir, 'adata_harmony_best.h5ad'))

print(f"\n{'='*60}")
print(f"DONE — All outputs in: {outdir}")
print(f"Best config outputs in: {best_outdir}")
print(f"{'='*60}")
print(f"\nBest parameters:")
print(f"  N_TOP_GENES : {N_TOP_GENES}")
print(f"  N_PCS       : {N_PCS}")
print(f"  n_neighbors : {n_neighbors}")
print(f"  resolution  : {RESOLUTION}")

STEP 0: LOADING & QC


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/hom

Added `interval` annotation for features from /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/filtered_feature_bc_matrix.h5


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/mudata/_core/mudata.py:915: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  warnings.warn(
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/pyt

Raw concat: 15920 cells × 14384 genes
batch
multi_RNA    9912
ref_D2       2545
ref_D4       2335
scNMT_RNA     723
scNMT_DCM     405
Name: count, dtype: int64
PARAMETER SWEEP
Total configurations: 9

[1/9] {'N_TOP_GENES': 3000, 'N_PCS': 20, 'n_neighbors': 10, 'resolution': 0.2}


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-04-01 09:14:15,963 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-04-01 09:14:16,141 - harmonypy - INFO - KMeans initialization complete.


... (17 outputs left out)

/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-04-01 10:11:18,590 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-04-01 10:11:18,745 - harmonypy - INFO - KMeans initialization complete.


Saved: rank_genes + dotplot

Best config silhouette:
  Batch   (lower=better) : -0.0761
  Cluster (higher=better): 0.1881
  [stability] matched 399 DCM–RNA pairs (DCM_matrixID ↔ Sample_RNA via pool+plate_well)

Transition stability (best config): 100.0% mean stable across Dox
         Dox  stable_pct  n_cells
16h_24h_dtag       100.0      282
         16h       100.0      117


/tmp/ipykernel_236665/3134958911.py:559: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=vc_pct, x='Cell type', y='Percentage', palette='tab10')


TypeError: Can't implicitly convert non-string objects to strings

In [47]:
##############################################
# HARMONY PARAMETER SWEEP
# Evaluates all combinations of:
#   N_TOP_GENES, N_PCS, n_neighbors, resolution
# Validates with:
#   1. Silhouette scores (batch mixing + cluster separation)
#   2. DCM→RNA transition stability per Dox condition
#   3. Label consistency across resolutions (DCM cells only)
# Outputs ranked CSV + heatmap + best-config final run
##############################################

import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score
import harmonypy as hm
import os
import itertools
from copy import deepcopy

# ─────────────────────────────────────────────
# PATHS & OUTPUT
# ─────────────────────────────────────────────
root   = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_sweep')
os.makedirs(outdir, exist_ok=True)

sc.set_figure_params(dpi=150, dpi_save=300, fontsize=6)
sc.settings.figdir = outdir

# How many cells to sample for silhouette (speed vs accuracy trade-off)
SAMPLE_SIZE = 5000

# ─────────────────────────────────────────────
# PARAMETER GRID — edit to taste
# ─────────────────────────────────────────────
PARAM_GRID = {
    'N_TOP_GENES': [1500, 2000, 3000],
    'N_PCS':       [20, 30, 50],
    'n_neighbors': [10, 15, 30],
    'resolution':  [0.3, 0.5, 0.8],
}

# ─────────────────────────────────────────────
# STEP 0: LOAD & QC  (identical to original script)
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 0: LOADING & QC")
print("=" * 60)

import muon as mu

adata_ref_D2 = sc.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/NPC_D2/sample_filtered_feature_bc_matrix.h5")
adata_ref_D4 = sc.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/NPC_D4/sample_filtered_feature_bc_matrix.h5")

mdata           = mu.read_10x_h5("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/filtered_feature_bc_matrix.h5")
D2_D4_multi_RNA = mdata.mod['rna']

human_barcodes  = pd.read_csv("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/gem_classification.csv")
mouse_barcodes  = human_barcodes[human_barcodes["call"] == "mm10"]
D2_D4_multi_RNA = D2_D4_multi_RNA[D2_D4_multi_RNA.obs.index.isin(mouse_barcodes['barcode'])].copy()

for ref in [adata_ref_D2, adata_ref_D4, D2_D4_multi_RNA]:
    ref.var_names_make_unique()
    ref.var['mt'] = ref.var_names.str.startswith('mt-')
    sc.pp.calculate_qc_metrics(ref, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

adata_ref_D2    = adata_ref_D2[(adata_ref_D2.obs['n_genes_by_counts'] >= 200) & (adata_ref_D2.obs['pct_counts_mt'] < 25)].copy()
adata_ref_D4    = adata_ref_D4[(adata_ref_D4.obs['n_genes_by_counts'] >= 200) & (adata_ref_D4.obs['pct_counts_mt'] < 25)].copy()
D2_D4_multi_RNA = D2_D4_multi_RNA[(D2_D4_multi_RNA.obs['n_genes_by_counts'] >= 200) & (D2_D4_multi_RNA.obs['pct_counts_mt'] < 25)].copy()

# scNMT
dcm1  = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/NPC_diff_plate1_plate2_WG.csv', index_col=0).T
rna1  = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool30_31_RNA/NPC/NPC_counts.csv', index_col=0).T
meta1 = pd.read_csv('/media/erlantz/71DE167A75E99BB4/Pool_30_31_DNA/deepseq/240818_A00383_0709_AHLJ5WDRX2/Creyghton/Analysis_R-Py/NPC/metadata_NPC_DNA_RNA_wholegene.csv', index_col=0)

meta1['DCM_matrixID'] = 'X' + meta1['Barcode'].astype(str) + '_' + meta1['Plate.x']
meta1.rename(columns={'Plate.x': 'Plate', 'Sample': 'Sample_RNA'}, inplace=True)

def prepare(meta, pool_name):
    m = meta.copy()
    m = m[m["Plate"].notna() & m["Well"].notna()].copy()
    m["pool"]       = pool_name
    m["plate_well"] = m["Plate"].astype(str) + "_" + m["Well"].astype(str)
    return m

m1 = prepare(meta1, "pool1")
dcm1 = dcm1.reindex(columns=sorted(set(dcm1.columns)), fill_value=0)
rna1 = rna1.reindex(columns=sorted(set(rna1.columns)), fill_value=0)

adata_dcm1 = sc.AnnData(X=dcm1)
adata_rna1 = sc.AnnData(X=rna1)
sc.pp.filter_cells(adata_dcm1, min_genes=200)
sc.pp.filter_cells(adata_rna1, min_genes=200)

def attach_meta(adata, meta, id_col):
    m    = meta.set_index(id_col)
    keep = m.index.intersection(adata.obs_names)
    adata = adata[keep].copy()
    adata.obs = adata.obs.join(m.loc[keep])
    return adata

adata_dcm1 = attach_meta(adata_dcm1, m1, "DCM_matrixID")
adata_rna1 = attach_meta(adata_rna1, m1, "Sample_RNA")

adata_dcm1.obs['batch']      = 'scNMT_DCM'
adata_rna1.obs['batch']      = 'scNMT_RNA'
adata_dcm1.obs['orig.ident'] = 'scNMT_pool1_DCM'
adata_rna1.obs['orig.ident'] = 'scNMT_pool1_RNA'
adata_dcm1 = adata_dcm1[adata_dcm1.obs['Dox'] != 'Minus'].copy()
adata_rna1 = adata_rna1[adata_rna1.obs['Dox'] != 'Minus'].copy()

adata_ref_D2.obs['batch']         = 'ref_D2';    adata_ref_D2.obs['orig.ident']    = 'ref_D2'
adata_ref_D4.obs['batch']         = 'ref_D4';    adata_ref_D4.obs['orig.ident']    = 'ref_D4'
D2_D4_multi_RNA.obs['batch']      = 'multi_RNA'; D2_D4_multi_RNA.obs['orig.ident'] = 'multi_RNA'

all_datasets = [adata_dcm1, adata_rna1, adata_ref_D2, adata_ref_D4, D2_D4_multi_RNA]
all_obs_cols  = list(set().union(*[a.obs.columns for a in all_datasets]))
for a in all_datasets:
    for col in all_obs_cols:
        if col not in a.obs.columns:
            a.obs[col] = 'Unknown'

# ─────────────────────────────────────────────
# RAW CONCATENATED OBJECT (before normalisation)
# We'll normalise inside each sweep iteration so
# HVG selection is affected by N_TOP_GENES.
# ─────────────────────────────────────────────
adata_raw_concat = sc.concat(all_datasets, join='inner', merge='unique', index_unique=None)
for col in ['Dox', 'orig.ident', 'batch']:
    if col in adata_raw_concat.obs.columns:
        adata_raw_concat.obs[col] = adata_raw_concat.obs[col].fillna('Unknown').astype(str)

print(f"Raw concat: {adata_raw_concat.n_obs} cells × {adata_raw_concat.n_vars} genes")
print(adata_raw_concat.obs['batch'].value_counts())

# ─────────────────────────────────────────────
# HELPER: run one Harmony config
# ─────────────────────────────────────────────
def run_harmony_config(adata_in, N_TOP_GENES, N_PCS, n_neighbors, resolution):
    """
    Full preprocessing + Harmony + UMAP + Leiden for one parameter set.
    Returns the annotated AnnData (no plots saved here).
    """
    adata = adata_in.copy()

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers['lognorm'] = adata.X.copy()

    sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, batch_key='orig.ident')
    sc.pp.scale(adata, max_value=10)
    adata.layers['lognorm_scaled'] = adata.X.copy()
    sc.tl.pca(adata, n_comps=50, svd_solver='arpack')

    # Store .raw from lognorm
    adata_raw_obj   = adata.copy()
    adata_raw_obj.X = adata_raw_obj.layers['lognorm']
    adata.raw       = adata_raw_obj

    # Harmony
    pca_matrix = adata.obsm['X_pca'][:, :N_PCS].copy()
    batch_df   = pd.DataFrame(
        {'orig.ident': adata.obs['orig.ident'].values},
        index=np.arange(adata.n_obs)
    )
    ho   = hm.run_harmony(pca_matrix, batch_df, vars_use='orig.ident',
                          random_state=42, max_iter_harmony=20, verbose=False)
    _raw = ho.Z_corr
    if _raw.shape == (adata.n_obs, N_PCS):
        harmony_emb = _raw
    elif _raw.shape == (N_PCS, adata.n_obs):
        harmony_emb = _raw.T
    else:
        raise ValueError(f"Unexpected ho.Z_corr shape {_raw.shape}")

    adata.obsm['X_pca_harmony'] = harmony_emb

    sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=n_neighbors, n_pcs=N_PCS)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=resolution, key_added='leiden', random_state=0)

    # Annotate clusters by majority batch
    annots = {}
    for cl in adata.obs['leiden'].astype(str).unique():
        cells  = adata.obs[adata.obs['leiden'].astype(str) == cl]
        counts = cells['batch'].value_counts()
        annots[cl] = counts.idxmax() if len(counts) > 0 else 'Unknown'
    adata.obs['leiden_annot'] = adata.obs['leiden'].astype(str).map(annots).fillna('Unknown')
    adata.obs['leiden_annot'] = adata.obs['leiden'].astype(str) + '_' + adata.obs['leiden_annot']

    return adata

# ─────────────────────────────────────────────
# METRIC 1: Silhouette
# ─────────────────────────────────────────────
def compute_silhouette(adata, N_PCS, sample_size=SAMPLE_SIZE):
    n        = min(sample_size, adata.n_obs)
    idx      = np.random.choice(adata.n_obs, n, replace=False)
    emb      = adata.obsm['X_pca_harmony'][idx, :N_PCS]
    batch_lb = adata.obs['orig.ident'].values[idx]
    clust_lb = adata.obs['leiden'].values[idx]
    n_cl     = len(np.unique(clust_lb))
    sil_batch   = silhouette_score(emb, batch_lb)
    sil_cluster = silhouette_score(emb, clust_lb) if n_cl > 1 else np.nan
    return sil_batch, sil_cluster

# ─────────────────────────────────────────────
# METRIC 2: DCM→RNA transition stability
# Matching via pool + plate_well (spatial coordinate), mirroring
# the reference matching script exactly.  Returns mean stable
# fraction across Dox groups.
#
# Index name asymmetry (from attach_meta in the loading step):
#   DCM obs_names  = DCM_matrixID  (format  X{Barcode}_{Plate})
#   RNA obs_names  = Sample_RNA    (the RNA sample identifier)
# After sc.concat both live in one .obs but the obs_names are
# still the original modality-specific IDs.  We never rely on
# the obs_names for matching — we use pool + plate_well instead,
# which is present in .obs for both modalities because attach_meta
# pulled it from the shared metadata table.
# ─────────────────────────────────────────────
def compute_transition_stability(adata):
    """
    1. Split adata.obs into DCM and RNA subsets by batch label.
    2. Add the original modality-specific ID back as a plain column:
         DCM rows → 'DCM_matrixID'  (= obs_names for that subset)
         RNA rows → 'Sample_RNA'    (= obs_names for that subset)
    3. Inner-join on pool + plate_well, same as the reference script.
    4. Compare the raw leiden cluster number on both sides per Dox group.

    WHY leiden and not leiden_annot:
    leiden_annot is assigned by majority batch within each cluster, so
    in a well-mixed embedding DCM-dominated clusters get label 'scNMT_DCM'
    and RNA-dominated clusters get 'scNMT_RNA' / 'multi_RNA'.  Comparing
    those labels across modalities is therefore always False by construction
    (DCM cell annot = 'scNMT_DCM', matched RNA cell annot = 'scNMT_RNA'),
    making stable_pct artifically 0% or 100% depending on the cluster
    structure — neither is meaningful.

    The correct question is: does a matched DCM–RNA pair land in the
    *same cluster* in the shared UMAP space?  That is directly answered
    by comparing the integer leiden cluster IDs, which are assigned once
    for the whole integrated object and are therefore comparable across
    modalities.
    """
    obs = adata.obs.copy()

    # Use the raw cluster number, not the batch-annotated label
    obs['cluster'] = obs['leiden'].astype(str)

    dcm_mask = obs['batch'].str.contains('DCM', case=False, na=False)
    rna_mask = obs['batch'].str.contains('RNA', case=False, na=False)

    # Restrict RNA side to scNMT_RNA only (not ref / multi_RNA which have
    # no plate_well metadata and would produce spurious many-to-one joins)
    scnmt_rna_mask = obs['batch'].str.contains('scNMT', case=False, na=False) & rna_mask

    dcm_obs = obs[dcm_mask].copy()
    rna_obs = obs[scnmt_rna_mask].copy()

    if dcm_obs.empty or rna_obs.empty:
        return np.nan, pd.DataFrame()

    required = {'pool', 'plate_well'}
    if not required.issubset(dcm_obs.columns) or not required.issubset(rna_obs.columns):
        print("  [stability] pool/plate_well columns missing — skipping metric")
        return np.nan, pd.DataFrame()

    dcm_obs = dcm_obs.copy()
    dcm_obs['DCM_matrixID'] = dcm_obs.index   # X{Barcode}_{Plate}

    rna_obs = rna_obs.copy()
    rna_obs['Sample_RNA'] = rna_obs.index      # RNA sample identifier

    dcm_tbl = dcm_obs[['DCM_matrixID', 'pool', 'plate_well', 'Dox',
                        'cluster']].reset_index(drop=True)

    rna_tbl = rna_obs[['Sample_RNA', 'pool', 'plate_well',
                        'cluster']].reset_index(drop=True)

    # Inner join on pool + plate_well — identical to the reference script's
    # matched = valid_dcm.merge(valid_rna, on=['pool','plate_well'], how='inner')
    matched = dcm_tbl.merge(
        rna_tbl,
        on=['pool', 'plate_well'],
        suffixes=('_dcm', '_rna'),
        how='inner'
    )

    if matched.empty:
        print("  [stability] no pool/plate_well matches found")
        return np.nan, pd.DataFrame()

    print(f"  [stability] matched {len(matched)} DCM–RNA pairs "
          f"(DCM_matrixID ↔ Sample_RNA via pool+plate_well)")

    results = []
    for dox in matched['Dox'].unique():
        sub        = matched[matched['Dox'] == dox]
        stable_pct = (sub['cluster_dcm'] == sub['cluster_rna']).mean() * 100
        results.append({'Dox': dox, 'stable_pct': stable_pct, 'n_cells': len(sub)})

    df = pd.DataFrame(results)
    return df['stable_pct'].mean() if not df.empty else np.nan, df

# ─────────────────────────────────────────────
# METRIC 3: Label consistency across resolutions
# (DCM cells: how consistently do they get the same cluster label
#  when resolution changes within this config's resolution range?)
# Here we re-cluster at the adjacent resolution points and measure
# cross-resolution agreement for DCM cells only.
# ─────────────────────────────────────────────
def compute_label_consistency(adata, resolutions, N_PCS):
    """
    For each pair of resolutions, compute fraction of DCM cells
    assigned the same majority cluster type.
    Returns mean pairwise consistency.
    """
    obs   = adata.obs.copy()
    dcm_m = obs['batch'].str.contains('DCM', case=False, na=False)
    dcm_cells = adata[dcm_m].obs_names

    if len(dcm_cells) == 0:
        return np.nan

    label_store = {}
    for res in resolutions:
        sc.tl.leiden(adata, resolution=res, key_added=f'_tmp_leiden_{res}', random_state=0)
        # Strip number prefix so we compare cell-type names
        adata.obs[f'_tmp_leiden_{res}'] = adata.obs[f'_tmp_leiden_{res}'].astype(str)
        label_store[res] = adata.obs.loc[dcm_cells, f'_tmp_leiden_{res}'].values

    pairs = list(itertools.combinations(resolutions, 2))
    agreements = []
    for r1, r2 in pairs:
        agree = (label_store[r1] == label_store[r2]).mean()
        agreements.append(agree)

    # Clean up temp columns
    for res in resolutions:
        if f'_tmp_leiden_{res}' in adata.obs.columns:
            del adata.obs[f'_tmp_leiden_{res}']

    return float(np.mean(agreements)) if agreements else np.nan
# ─────────────────────────────────────────────
# BUILD assay_day LABEL ON THE RAW CONCAT
# ─────────────────────────────────────────────
if 'Day' in adata_raw_concat.obs.columns:
    adata_raw_concat.obs['Day'] = adata_raw_concat.obs['Day'].astype(str)
    _scnmt_m = adata_raw_concat.obs['batch'].str.contains('scNMT', case=False, na=False)
    adata_raw_concat.obs['assay_day'] = adata_raw_concat.obs['batch'].copy()
    adata_raw_concat.obs.loc[_scnmt_m, 'assay_day'] = (
        adata_raw_concat.obs.loc[_scnmt_m, 'batch'] + '_Day' +
        adata_raw_concat.obs.loc[_scnmt_m, 'Day'].str.replace('Unknown', 'NA')
    )
    print("assay_day groups:", sorted(adata_raw_concat.obs['assay_day'].unique()))
else:
    adata_raw_concat.obs['assay_day'] = adata_raw_concat.obs['batch'].copy()
    print("'Day' column not found -- assay_day == batch")

# ─────────────────────────────────────────────
# METRIC 4: Proportion similarity
# Expected alignment:
#   scNMT_DCM_DayD2          -> ref_D2
#   scNMT_DCM_DayD2_D4_chase -> ref_D4
#   scNMT_DCM_DayD4          -> ref_D4
#   scNMT_RNA_DayD2          -> ref_D2
#   scNMT_RNA_DayD2_D4_chase -> ref_D4
#   scNMT_RNA_DayD4          -> ref_D4
# Score = 1 - mean_absolute_difference (higher = more similar, max 1.0)
# ─────────────────────────────────────────────
PROPORTION_PAIRS = [
    ('DCM_DayD2',          'ref_D2'),
    ('DCM_DayD2_D4_chase', 'ref_D4'),
    ('DCM_DayD4',          'ref_D4'),
    ('RNA_DayD2',          'ref_D2'),
    ('RNA_DayD2_D4_chase', 'ref_D4'),
    ('RNA_DayD4',          'ref_D4'),
]

def compute_proportion_similarity(adata):
    obs = adata.obs.copy()
    obs['leiden']    = obs['leiden'].astype(str)
    obs['assay_day'] = obs['assay_day'].astype(str)
    obs['batch']     = obs['batch'].astype(str)
    clusters = sorted(obs['leiden'].unique(),
                      key=lambda x: int(x) if x.isdigit() else x)
    def cluster_props(mask):
        sub = obs[mask]
        if len(sub) == 0:
            return None
        return sub['leiden'].value_counts().reindex(clusters, fill_value=0) / len(sub)
    scores, rows = [], []
    for scnmt_sub, ref_batch in PROPORTION_PAIRS:
        sm = obs['assay_day'].str.contains(scnmt_sub, case=False, na=False)
        rm = obs['batch'] == ref_batch
        ps, pr = cluster_props(sm), cluster_props(rm)
        if ps is None or pr is None:
            rows.append({'scnmt_group': scnmt_sub, 'ref_group': ref_batch,
                         'n_scnmt': int(sm.sum()), 'n_ref': int(rm.sum()),
                         'similarity': np.nan})
            continue
        sim = float(1.0 - (ps - pr).abs().mean())
        scores.append(sim)
        rows.append({'scnmt_group': scnmt_sub, 'ref_group': ref_batch,
                     'n_scnmt': int(sm.sum()), 'n_ref': int(rm.sum()),
                     'similarity': round(sim, 4)})
    return (float(np.nanmean(scores)) if scores else np.nan), pd.DataFrame(rows)


# ─────────────────────────────────────────────
# HELPER: distribution plots for one config
# ─────────────────────────────────────────────
def save_distribution_plots(adata_cfg, target_dir):
    obs = adata_cfg.obs.copy()
    obs['leiden']    = obs['leiden'].astype(str)
    obs['assay_day'] = obs['assay_day'].astype(str)
    clusters_s = sorted(obs['leiden'].unique(), key=lambda x: int(x) if x.isdigit() else x)
    assays_s   = sorted(obs['assay_day'].unique())
    count_df = (
        obs.groupby(['leiden', 'assay_day'], observed=True)
        .size().unstack(fill_value=0)
        .reindex(index=clusters_s, fill_value=0)
        .reindex(columns=assays_s, fill_value=0)
    )
    prop_cl = count_df.div(count_df.sum(axis=1), axis=0) * 100
    prop_as = count_df.div(count_df.sum(axis=0), axis=1) * 100
    count_df.to_csv(os.path.join(target_dir, 'dist_counts.csv'))
    prop_cl.to_csv(os.path.join(target_dir, 'dist_prop_within_cluster.csv'))
    prop_as.to_csv(os.path.join(target_dir, 'dist_prop_within_assay.csv'))
    fw = max(len(assays_s) * 1.4 + 2, 8)
    fh = max(len(clusters_s) * 0.5 + 2, 4)
    for data, fname, title, fmt, vmax in [
        (count_df, 'dist_heatmap_counts.pdf',      'Cell counts',             'd',   None),
        (prop_as,  'dist_heatmap_pct_assay.pdf',   '% of assay (cols=100%)',  '.1f', 100),
        (prop_cl,  'dist_heatmap_pct_cluster.pdf', '% of cluster (rows=100%)','.1f', 100),
    ]:
        fig, ax = plt.subplots(figsize=(fw, fh))
        kw = dict(ax=ax, cmap='YlOrRd', annot=True, fmt=fmt,
                  linewidths=0.3, linecolor='white',
                  cbar_kws={'label': title, 'shrink': 0.6})
        if vmax is not None:
            kw['vmin'] = 0; kw['vmax'] = vmax
        sns.heatmap(data, **kw)
        ax.set_title(title, fontsize=9)
        ax.tick_params(axis='x', rotation=35, labelsize=6)
        ax.tick_params(axis='y', rotation=0,  labelsize=7)
        plt.tight_layout()
        plt.savefig(os.path.join(target_dir, fname), dpi=150, bbox_inches='tight')
        plt.close()
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(clusters_s) * 0.5 + 2)))
    count_df.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab10',
                  edgecolor='white', linewidth=0.3)
    axes[0].set_title('Absolute counts')
    axes[0].legend(title='Assay/day', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=6)
    axes[0].tick_params(axis='x', rotation=0, labelsize=7)
    prop_cl.plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10',
                 edgecolor='white', linewidth=0.3)
    axes[1].set_title('Proportion per cluster (%)')
    axes[1].set_ylim(0, 100)
    axes[1].legend(title='Assay/day', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=6)
    axes[1].tick_params(axis='x', rotation=0, labelsize=7)
    plt.suptitle('Cell distribution by assay/day', fontsize=10, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, 'dist_barplot_stacked.pdf'), dpi=150, bbox_inches='tight')
    plt.close()


# ─────────────────────────────────────────────
# SWEEP
# ─────────────────────────────────────────────
print("=" * 60)
def _clean_obs_for_h5ad(adata_obj):
    """
    Coerce every .obs column to either float64 or plain str so h5py
    can serialise it.  Handles CategoricalDtype, mixed object columns,
    and sentinel strings ('Unknown', 'nan', 'None', '<NA>').
    """
    for col in adata_obj.obs.columns:
        s = adata_obj.obs[col]
        # Strip CategoricalDtype before any replace/fillna
        if hasattr(s, 'cat'):
            s = s.astype(str)
        # Normalise all sentinel strings to NaN on a plain str series
        s = s.astype(str).replace(
            {'Unknown': float('nan'), 'nan': float('nan'),
             'None': float('nan'), '<NA>': float('nan')}
        )
        s_num = pd.to_numeric(s, errors='coerce')
        if s_num.isna().all():
            # Genuinely string column
            adata_obj.obs[col] = s.fillna('Unknown').astype(str)
        else:
            # Numeric column — float64 is always safe for h5py
            adata_obj.obs[col] = s_num

print("PARAMETER SWEEP")
print("=" * 60)

keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*[PARAM_GRID[k] for k in keys]))
print(f"Total configurations: {len(combos)}")

all_results = []
np.random.seed(42)

for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    cfg_id = f"cfg_{i:03d}"
    print(f"\n[{i+1}/{len(combos)}] {params}")

    try:
        adata_cfg = run_harmony_config(
            adata_raw_concat,
            N_TOP_GENES = params['N_TOP_GENES'],
            N_PCS       = params['N_PCS'],
            n_neighbors = params['n_neighbors'],
            resolution  = params['resolution'],
        )

        # ── per-config output directory ──
        cfg_outdir = os.path.join(outdir, cfg_id)
        os.makedirs(cfg_outdir, exist_ok=True)
        sc.settings.figdir = cfg_outdir
        adata_cfg.obs['assay_day'] = adata_raw_concat.obs['assay_day'].values

        n_clusters              = adata_cfg.obs['leiden'].nunique()
        sil_batch, sil_cluster  = compute_silhouette(adata_cfg, params['N_PCS'])
        mean_stable, stability_df = compute_transition_stability(adata_cfg)

        # Consistency: test at +-0.2 of the chosen resolution
        res_range = sorted(set([
            max(0.1, params['resolution'] - 0.2),
            params['resolution'],
            params['resolution'] + 0.2,
        ]))
        consistency = compute_label_consistency(adata_cfg, res_range, params['N_PCS'])

        # Metric 4: proportion similarity
        mean_prop_sim, prop_sim_df = compute_proportion_similarity(adata_cfg)
        if not prop_sim_df.empty:
            prop_sim_df.to_csv(os.path.join(cfg_outdir, 'proportion_similarity.csv'), index=False)
        prop_sim_val = mean_prop_sim if not np.isnan(mean_prop_sim) else 0.0
        composite = (sil_cluster - sil_batch) + mean_stable / 100 + consistency + prop_sim_val

        row = {
            'cfg_id':          cfg_id,
            **params,
            'n_clusters':      n_clusters,
            'sil_batch':       round(sil_batch,     4),
            'sil_cluster':     round(sil_cluster,   4),
            'mean_stable_pct': round(mean_stable,   2) if not np.isnan(mean_stable)   else np.nan,
            'consistency':     round(consistency,   4) if not np.isnan(consistency)   else np.nan,
            'prop_similarity': round(mean_prop_sim, 4) if not np.isnan(mean_prop_sim) else np.nan,
            'composite':       round(composite,     4),
        }
        all_results.append(row)
        print(f"  -> n_clusters={n_clusters}, sil_batch={sil_batch:.3f}, "
              f"sil_cluster={sil_cluster:.3f}, stable={mean_stable:.1f}%, "
              f"consistency={consistency:.3f}, prop_sim={prop_sim_val:.3f}, "
              f"composite={composite:.3f}")

        # ── UMAP ──
        title = (f"cfg{i+1}: genes={params['N_TOP_GENES']} pcs={params['N_PCS']} "
                 f"k={params['n_neighbors']} res={params['resolution']}")
        fig, axes = plt.subplots(1, 4, figsize=(28, 5))
        sc.pl.umap(adata_cfg, color='batch',        ax=axes[0], show=False, frameon=False, legend_loc='right margin', legend_fontsize=7)
        sc.pl.umap(adata_cfg, color='assay_day',    ax=axes[1], show=False, frameon=False, legend_loc='right margin', legend_fontsize=7)
        sc.pl.umap(adata_cfg, color='leiden',       ax=axes[2], show=False, frameon=False, legend_loc='right margin', legend_fontsize=7)
        sc.pl.umap(adata_cfg, color='leiden_annot', ax=axes[3], show=False, frameon=False, legend_loc='right margin', legend_fontsize=7)
        fig.suptitle(title, fontsize=9)
        plt.tight_layout()
        plt.savefig(os.path.join(cfg_outdir, 'umap.png'), dpi=120, bbox_inches='tight')
        plt.close()

        # ── distribution plots ──
        save_distribution_plots(adata_cfg, cfg_outdir)

        # ── save h5ad ──
        _clean_obs_for_h5ad(adata_cfg)
        adata_cfg.write_h5ad(os.path.join(cfg_outdir, 'adata_cfg.h5ad'))

    except Exception as e:
        print(f"  ERROR: {e}")
        all_results.append({'cfg_id': cfg_id, **params, 'error': str(e)})

# ─────────────────────────────────────────────
# RESULTS TABLE
# ─────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('composite', ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(outdir, 'sweep_results.csv'), index=False)

print("\n" + "=" * 60)
print("TOP 10 CONFIGURATIONS")
print("=" * 60)
cols_show = ['cfg_id', 'N_TOP_GENES', 'N_PCS', 'n_neighbors', 'resolution',
             'n_clusters', 'sil_batch', 'sil_cluster', 'mean_stable_pct',
             'consistency', 'prop_similarity', 'composite']

# ─────────────────────────────────────────────
# HEATMAP: composite score across param pairs
# ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
metric_pairs = [
    ('N_TOP_GENES', 'N_PCS'),
    ('N_TOP_GENES', 'n_neighbors'),
    ('N_TOP_GENES', 'resolution'),
    ('N_PCS',       'n_neighbors'),
    ('N_PCS',       'resolution'),
    ('n_neighbors', 'resolution'),
]
for ax, (p1, p2) in zip(axes.flat, metric_pairs):
    pivot = results_df.pivot_table(values='composite', index=p1, columns=p2, aggfunc='mean')
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis', ax=ax,
                cbar_kws={'label': 'Composite (higher=better)'})
    ax.set_title(f'{p1} vs {p2}')

plt.suptitle('Composite score (sil_cluster − sil_batch + stable/100 + consistency)\nMean over other params',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'sweep_heatmap_composite.png'), dpi=200, bbox_inches='tight')
plt.close()
print("Saved: sweep_heatmap_composite.png")

# ─────────────────────────────────────────────
# HEATMAP: individual metrics
# ─────────────────────────────────────────────
for metric in ['sil_batch', 'sil_cluster', 'mean_stable_pct', 'consistency']:
    pivot = results_df.pivot_table(values=metric, index='N_PCS', columns='n_neighbors', aggfunc='mean')
    plt.figure(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='coolwarm')
    plt.title(f'{metric}: N_PCS vs n_neighbors (mean over other params)')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'sweep_heatmap_{metric}.png'), dpi=150, bbox_inches='tight')
    plt.close()
print("Saved: per-metric heatmaps")

# ─────────────────────────────────────────────
# FULL RUN WITH BEST CONFIG
# ─────────────────────────────────────────────
best = results_df.iloc[0]
print(f"\n{'='*60}")
print("RUNNING BEST CONFIGURATION")
print(f"  N_TOP_GENES={int(best['N_TOP_GENES'])}, N_PCS={int(best['N_PCS'])}, "
      f"n_neighbors={int(best['n_neighbors'])}, resolution={best['resolution']}")
print(f"  Composite score: {best['composite']}")
print(f"{'='*60}")

METHOD_NAME = 'Harmony_best'
best_outdir = os.path.join(root, 'integration', 'method_comparison', METHOD_NAME)
os.makedirs(best_outdir, exist_ok=True)
sc.settings.figdir = best_outdir

N_TOP_GENES = int(best['N_TOP_GENES'])
N_PCS       = int(best['N_PCS'])
n_neighbors = int(best['n_neighbors'])
RESOLUTION  = float(best['resolution'])

adata_best = run_harmony_config(
    adata_raw_concat,
    N_TOP_GENES = N_TOP_GENES,
    N_PCS       = N_PCS,
    n_neighbors = n_neighbors,
    resolution  = RESOLUTION,
)
adata_best.obs['assay_day'] = adata_raw_concat.obs['assay_day'].values

# ── Store leiden clustering per N_PCS dimension ──
# For each N_PCS value tested in the grid, rebuild the Harmony embedding
# with that many PCs and re-run leiden at the best resolution.
# Stored as leiden_dim_{N_PCS} so the full annotation across all
# dimensions is available in one object.
print("\nStoring leiden per N_PCS dimension ...")
_pca_full = adata_best.obsm['X_pca']   # shape (n_cells, 50)
for _n_pcs in sorted(set(PARAM_GRID['N_PCS'])):
    _col = f'leiden_dim_{_n_pcs}'
    if _n_pcs == N_PCS:
        # Best dimension already clustered — just alias it
        adata_best.obs[_col] = adata_best.obs['leiden'].astype(str)
    else:
        _pca_trunc = _pca_full[:, :_n_pcs].copy()
        _batch_df  = pd.DataFrame(
            {'orig.ident': adata_best.obs['orig.ident'].values},
            index=np.arange(adata_best.n_obs)
        )
        _ho = hm.run_harmony(
            _pca_trunc, _batch_df, vars_use='orig.ident',
            random_state=42, max_iter_harmony=20, verbose=False
        )
        _raw = _ho.Z_corr
        _emb = _raw.T if _raw.shape == (_n_pcs, adata_best.n_obs) else _raw
        _tmp_key = f'_X_harmony_{_n_pcs}'
        adata_best.obsm[_tmp_key] = _emb
        sc.pp.neighbors(adata_best, use_rep=_tmp_key,
                        n_neighbors=n_neighbors, n_pcs=_n_pcs)
        sc.tl.leiden(adata_best, resolution=RESOLUTION,
                     key_added=_col, random_state=0)
        adata_best.obs[_col] = adata_best.obs[_col].astype(str)
        del adata_best.obsm[_tmp_key]
    n_cl = adata_best.obs[_col].nunique()
    print(f"  {_col}: {n_cl} clusters")

# Restore best-config neighbors graph for downstream steps
sc.pp.neighbors(adata_best, use_rep='X_pca_harmony',
                n_neighbors=n_neighbors, n_pcs=N_PCS)

# ── UMAP overview ──
plot_cols = [c for c in ['Dox', 'batch', 'assay_day', 'orig.ident', 'leiden_annot', 'leiden']
             if c in adata_best.obs.columns]
sc.pl.umap(adata_best, color=plot_cols, frameon=False, legend_loc='right margin', legend_fontsize=7, show=False,
           save=f'_{METHOD_NAME}_umap_overview.pdf')

# ── Per-batch highlight ──
batches = adata_best.obs['orig.ident'].unique()
fig, axes = plt.subplots(1, len(batches), figsize=(6 * len(batches), 5))
if len(batches) == 1:
    axes = [axes]
for ax, b in zip(axes, batches):
    adata_best.obs['_hl'] = (adata_best.obs['orig.ident'] == b).astype(str)
    sc.pl.umap(adata_best, color='_hl', ax=ax, show=False, title=str(b),
               palette={'True': '#e74c3c', 'False': '#d3d3d3'})
del adata_best.obs['_hl']
plt.suptitle(f'{METHOD_NAME} — Per-batch highlight', fontsize=10, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(best_outdir, f'{METHOD_NAME}_umap_per_batch.pdf'), dpi=200, bbox_inches='tight')
plt.close()

# ── Distribution plots (best config) ──
save_distribution_plots(adata_best, best_outdir)
mean_prop_sim_best, prop_sim_df_best = compute_proportion_similarity(adata_best)
if not prop_sim_df_best.empty:
    prop_sim_df_best.to_csv(os.path.join(best_outdir, 'proportion_similarity.csv'), index=False)
    print(f'Proportion similarity (best config): {mean_prop_sim_best:.4f}')
    print(prop_sim_df_best.to_string(index=False))

# ── Marker genes: DE + visualizations for cluster labelling ──
try:
    N_GENES_RANK  = 10   # genes shown in the rank_genes_groups panel plot
    N_GENES_PLOTS = 5    # genes per cluster used in dotplot / matrixplot / heatmap

    sc.tl.rank_genes_groups(adata_best, 'leiden', method='wilcoxon',
                            use_raw=True, key_added='rank_genes_leiden')

    clusters_sorted = sorted(
        adata_best.obs['leiden'].unique(),
        key=lambda x: int(x) if str(x).isdigit() else x
    )

    # ── 1. Ranked gene panel (score + logFC per cluster) ──
    sc.pl.rank_genes_groups(
        adata_best, n_genes=N_GENES_RANK, sharey=False, show=False,
        key='rank_genes_leiden',
        save=f'_{METHOD_NAME}_rank_genes_panel.pdf'
    )
    print("Saved: rank_genes_panel")

    # ── Build top-gene list (deduplicated, order preserved) ──
    top_genes_per_cluster = {}
    for cl in clusters_sorted:
        df_cl = sc.get.rank_genes_groups_df(
            adata_best, group=str(cl), key='rank_genes_leiden'
        )
        # Keep only positively enriched, significant genes
        df_cl = df_cl[(df_cl['logfoldchanges'] > 0) & (df_cl['pvals_adj'] < 0.05)]
        top_genes_per_cluster[str(cl)] = df_cl.head(N_GENES_PLOTS)['names'].tolist()

    # Flat deduplicated list for multi-cluster plots
    top_genes_flat = list(dict.fromkeys(
        g for genes in top_genes_per_cluster.values() for g in genes
    ))

    # Save the full marker table as CSV (useful for manual labelling)
    marker_rows = []
    for cl, genes in top_genes_per_cluster.items():
        df_cl = sc.get.rank_genes_groups_df(
            adata_best, group=cl, key='rank_genes_leiden'
        )
        df_cl = df_cl[(df_cl['logfoldchanges'] > 0) & (df_cl['pvals_adj'] < 0.05)].copy()
        df_cl.insert(0, 'cluster', cl)
        marker_rows.append(df_cl)
    if marker_rows:
        pd.concat(marker_rows, ignore_index=True).to_csv(
            os.path.join(best_outdir, f'{METHOD_NAME}_marker_genes.csv'), index=False
        )
        print("Saved: marker_genes.csv")

    if top_genes_flat:
        # ── 2. Dotplot — size = fraction expressing, colour = mean expression ──
        sc.pl.dotplot(
            adata_best, var_names=top_genes_per_cluster, groupby='leiden',
            show=False, use_raw=True, standard_scale='var',
            save=f'_{METHOD_NAME}_dotplot_top{N_GENES_PLOTS}.pdf'
        )
        print("Saved: dotplot")

        # ── 3. Matrixplot — mean expression per cluster, good for many clusters ──
        sc.pl.matrixplot(
            adata_best, var_names=top_genes_per_cluster, groupby='leiden',
            show=False, use_raw=True, standard_scale='var', cmap='RdBu_r',
            save=f'_{METHOD_NAME}_matrixplot_top{N_GENES_PLOTS}.pdf'
        )
        print("Saved: matrixplot")

        # ── 4. Heatmap — single-cell resolution, rows = cells grouped by cluster ──
        sc.pl.heatmap(
            adata_best, var_names=top_genes_per_cluster, groupby='leiden',
            show=False, use_raw=True, standard_scale='var',
            cmap='viridis', swap_axes=False,
            save=f'_{METHOD_NAME}_heatmap_top{N_GENES_PLOTS}.pdf'
        )
        print("Saved: heatmap")

        # ── 5. Stacked violin — distribution of each marker per cluster ──
        sc.pl.stacked_violin(
            adata_best, var_names=top_genes_per_cluster, groupby='leiden',
            show=False, use_raw=True, standard_scale='var',
            save=f'_{METHOD_NAME}_stacked_violin_top{N_GENES_PLOTS}.pdf'
        )
        print("Saved: stacked_violin")

    else:
        print("  No significant marker genes found after filtering — "
              "try relaxing pvals_adj or logfoldchanges thresholds.")

except Exception as e:
    print(f"Marker gene plots failed: {e}")

# ── Silhouette of best ──
sil_batch_best, sil_cluster_best = compute_silhouette(adata_best, N_PCS)
print(f"\nBest config silhouette:")
print(f"  Batch   (lower=better) : {sil_batch_best:.4f}")
print(f"  Cluster (higher=better): {sil_cluster_best:.4f}")

pd.DataFrame([{
    'method': METHOD_NAME,
    'N_TOP_GENES': N_TOP_GENES, 'N_PCS': N_PCS,
    'n_neighbors': n_neighbors, 'resolution': RESOLUTION,
    'silhouette_batch': sil_batch_best,
    'silhouette_cluster': sil_cluster_best,
    'n_clusters': adata_best.obs['leiden'].nunique(),
}]).to_csv(os.path.join(best_outdir, f'{METHOD_NAME}_silhouette.csv'), index=False)

# ── Transition stability heatmap (best config) ──
mean_stable_best, stability_df_best = compute_transition_stability(adata_best)
if not stability_df_best.empty:
    print(f"\nTransition stability (best config): {mean_stable_best:.1f}% mean stable across Dox")
    print(stability_df_best.to_string(index=False))
    stability_df_best.to_csv(os.path.join(best_outdir, 'best_transition_stability.csv'), index=False)

# ── DCM label consistency histogram (best config) ──
obs_best = adata_best.obs.copy()
obs_best['leiden_clean'] = obs_best['leiden_annot'].str.replace(r'^\d+_', '', regex=True)
dcm_mask_best = obs_best['batch'].str.contains('DCM', case=False, na=False)
dcm_obs_best  = obs_best[dcm_mask_best]

if not dcm_obs_best.empty and 'Dox' in dcm_obs_best.columns:
    # Consistency at the chosen resolution (compare within Dox groups)
    plt.figure(figsize=(8, 4))
    vc = dcm_obs_best['leiden_clean'].value_counts()
    vc_pct = (vc / vc.sum() * 100).reset_index()
    vc_pct.columns = ['Cell type', 'Percentage']
    sns.barplot(data=vc_pct, x='Cell type', y='Percentage',
                hue='Cell type', palette='tab10', legend=False)
    plt.title('DCM cell-type distribution (best config)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(best_outdir, 'best_dcm_celltype_distribution.png'), dpi=150, bbox_inches='tight')
    plt.close()

# ── Fill n_genes Unknowns from n_genes_by_counts (and vice versa) ──
# The scNMT datasets have n_genes set by sc.pp.filter_cells while the
# 10x reference datasets compute n_genes_by_counts via calculate_qc_metrics.
# After sc.concat the padding loop fills missing columns with 'Unknown',
# so each modality ends up with 'Unknown' in the other's column.
# We reconcile both directions before dtype coercion.
if 'n_genes' in adata_best.obs.columns and 'n_genes_by_counts' in adata_best.obs.columns:
    # Replace 'Unknown' strings with NaN so we can work numerically
    for _col in ['n_genes', 'n_genes_by_counts']:
        adata_best.obs[_col] = adata_best.obs[_col].replace('Unknown', np.nan)
        adata_best.obs[_col] = pd.to_numeric(adata_best.obs[_col], errors='coerce')

    # Fill NaN in n_genes from n_genes_by_counts and vice versa
    n_filled_ng  = adata_best.obs['n_genes'].isna().sum()
    n_filled_nbc = adata_best.obs['n_genes_by_counts'].isna().sum()
    adata_best.obs['n_genes'] = (
        adata_best.obs['n_genes']
        .fillna(adata_best.obs['n_genes_by_counts'])
    )
    adata_best.obs['n_genes_by_counts'] = (
        adata_best.obs['n_genes_by_counts']
        .fillna(adata_best.obs['n_genes'])
    )
    print(f"  Filled {n_filled_ng} NaN in n_genes from n_genes_by_counts")
    print(f"  Filled {n_filled_nbc} NaN in n_genes_by_counts from n_genes")

# ── Clean .obs and save adata_best ──
_clean_obs_for_h5ad(adata_best)
adata_best.write_h5ad(os.path.join(best_outdir, 'adata_harmony_best.h5ad'))

print(f"\n{'='*60}")
print(f"DONE — All outputs in: {outdir}")
print(f"Best config outputs in: {best_outdir}")
print(f"{'='*60}")
print(f"\nBest parameters:")
print(f"  N_TOP_GENES : {N_TOP_GENES}")
print(f"  N_PCS       : {N_PCS}")
print(f"  n_neighbors : {n_neighbors}")
print(f"  resolution  : {RESOLUTION}")

STEP 0: LOADING & QC


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/hom

Added `interval` annotation for features from /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Multiome_10x/filtered_feature_bc_matrix.h5


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/mudata/_core/mudata.py:915: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  warnings.warn(
/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/pyt

Raw concat: 15920 cells × 14384 genes
batch
multi_RNA    9912
ref_D2       2545
ref_D4       2335
scNMT_RNA     723
scNMT_DCM     405
Name: count, dtype: int64
assay_day groups: ['multi_RNA', 'ref_D2', 'ref_D4', 'scNMT_DCM_DayD2', 'scNMT_DCM_DayD2_D4_chase', 'scNMT_DCM_DayD4', 'scNMT_RNA_DayD2', 'scNMT_RNA_DayD2_D4_chase', 'scNMT_RNA_DayD4']
PARAMETER SWEEP
Total configurations: 4

[1/4] {'N_TOP_GENES': 3000, 'N_PCS': 20, 'n_neighbors': 30, 'resolution': 0.15}


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-04-01 16:50:05,559 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-04-01 16:50:05,707 - harmonypy - INFO - KMeans initialization complete.


... (7 outputs left out)

/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-04-01 17:18:16,422 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-04-01 17:18:16,563 - harmonypy - INFO - KMeans initialization complete.


Proportion similarity (best config): 0.8259
       scnmt_group ref_group  n_scnmt  n_ref  similarity
         DCM_DayD2    ref_D2      325   2545      0.8051
DCM_DayD2_D4_chase    ref_D4      282   2335      0.7602
         DCM_DayD4    ref_D4       80   2335      0.7263
         RNA_DayD2    ref_D2      602   2545      0.8680
RNA_DayD2_D4_chase    ref_D4      490   2335      0.8881
         RNA_DayD4    ref_D4      121   2335      0.9074
Saved: rank_genes_panel
Saved: marker_genes.csv
Saved: dotplot
Saved: matrixplot
Saved: heatmap
Saved: stacked_violin

Best config silhouette:
  Batch   (lower=better) : -0.0761
  Cluster (higher=better): 0.2357
  [stability] matched 399 DCM–RNA pairs (DCM_matrixID ↔ Sample_RNA via pool+plate_well)

Transition stability (best config): 23.4% mean stable across Dox
         Dox  stable_pct  n_cells
16h_24h_dtag   20.212766      282
         16h   26.495726      117


/tmp/ipykernel_236665/820659008.py:844: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata_best.obs[_col] = adata_best.obs[_col].replace('Unknown', np.nan)


  Filled 14792 NaN in n_genes from n_genes_by_counts
  Filled 1128 NaN in n_genes_by_counts from n_genes

DONE — All outputs in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_sweep
Best config outputs in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best

Best parameters:
  N_TOP_GENES : 3000
  N_PCS       : 20
  n_neighbors : 30
  resolution  : 0.15


In [5]:
import os
# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root        = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
best_outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best')
outdir      = os.path.join(best_outdir, 'de_global_per_batch')
os.makedirs(outdir, exist_ok=True)

# ─────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────
print("Loading adata_harmony_best.h5ad ...")
#adata = sc.read_h5ad(os.path.join(best_outdir, 'adata_harmony_best2.h5ad'))
adata = sc.read_h5ad(os.path.join(best_outdir, 'adata_harmony_best_with_gmm_scores.h5ad'))

Loading adata_harmony_best.h5ad ...


In [6]:
adata

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [5]:
##############################################
# STABILITY ANALYSIS — BEST HARMONY CONFIG
# Reproduces the stability plots (stable vs switch % per Dox)
# using the best integrated configuration.
#
# "Dimension" here = Leiden resolution tested.
# For each resolution we re-cluster the saved adata_best,
# match DCM↔RNA cells via pool+plate_well, compare cluster
# labels, and plot stable vs switch % per Dox condition.
##############################################

import os
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root        = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
best_outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best')
outdir      = os.path.join(best_outdir, 'stability_analysis')
os.makedirs(outdir, exist_ok=True)

# ─────────────────────────────────────────────
# RESOLUTIONS TO TEST
# These become the "dimensions" on the x-axis.
# Add or remove values to taste.
# ─────────────────────────────────────────────
RESOLUTIONS = [0.1]

# ─────────────────────────────────────────────
# LOAD BEST ADATA
# ─────────────────────────────────────────────
print("Loading adata_harmony_best.h5ad ...")
#adata = sc.read_h5ad(os.path.join(best_outdir, 'adata_harmony_best.h5ad'))
print(f"  {adata.n_obs} cells × {adata.n_vars} genes")
print(f"  Batches: {adata.obs['batch'].value_counts().to_dict()}")

adata.obs['batch'] = adata.obs['batch'].astype(str)

# ─────────────────────────────────────────────
# RE-CLUSTER AT EACH RESOLUTION
# Leiden IDs are stored as leiden_annot_{res} columns
# ─────────────────────────────────────────────
print("\nRe-clustering at each resolution ...")

for res in RESOLUTIONS:
    col = f'leiden_annot_{res}'
    sc.tl.leiden(adata, resolution=res, key_added=col, random_state=0)
    adata.obs[col] = adata.obs[col].astype(str)
    print(f"  res={res}: {adata.obs[col].nunique()} clusters")

# ─────────────────────────────────────────────
# SPLIT DCM AND SCNMT_RNA CELLS
# Match via pool + plate_well (same logic as the sweep script)
# ─────────────────────────────────────────────
print("\nMatching DCM↔RNA cells via pool + plate_well ...")

dcm_mask      = adata.obs['batch'].str.contains('DCM',   case=False, na=False)
scnmt_rna_mask = (adata.obs['batch'].str.contains('scNMT', case=False, na=False) &
                   adata.obs['batch'].str.contains('RNA',   case=False, na=False))

dcm_obs = adata.obs[dcm_mask].copy()
rna_obs = adata.obs[scnmt_rna_mask].copy()

# Restore modality-specific IDs as plain columns
dcm_obs['DCM_matrixID'] = dcm_obs.index
rna_obs['Sample_RNA']   = rna_obs.index

# Columns needed for the join
leiden_cols_all = [f'leiden_annot_{res}' for res in RESOLUTIONS]
required_cols   = ['pool', 'plate_well', 'Dox'] + leiden_cols_all

# Check required metadata columns exist
missing_dcm = [c for c in required_cols if c not in dcm_obs.columns]
missing_rna = [c for c in required_cols if c not in rna_obs.columns]
if missing_dcm or missing_rna:
    raise ValueError(
        f"Missing columns — DCM: {missing_dcm}, RNA: {missing_rna}\n"
        "Make sure pool and plate_well were saved in adata.obs."
    )

dcm_tbl = dcm_obs[['DCM_matrixID', 'pool', 'plate_well', 'Dox'] + leiden_cols_all].reset_index(drop=True)
rna_tbl = rna_obs[['Sample_RNA',   'pool', 'plate_well']         + leiden_cols_all].reset_index(drop=True)
# Note: Dox is only in dcm_tbl so after the merge it stays as plain 'Dox' (no suffix added)

matched_df = dcm_tbl.merge(
    rna_tbl,
    on=['pool', 'plate_well'],
    suffixes=('_DCM', '_RNA'),
    how='inner'
)

print(f"  Matched DCM–RNA pairs: {len(matched_df)}")
print(f"  Dox conditions: {sorted(matched_df['Dox'].unique())}")

if matched_df.empty:
    raise ValueError("No matched cells found — check pool/plate_well columns in adata.obs.")

matched_df.to_csv(os.path.join(outdir, 'matched_dcm_rna_cells.csv'), index=False)

# ─────────────────────────────────────────────
# STABILITY CALCULATION
# Mirrors calculate_stability() from the reference code exactly.
# ─────────────────────────────────────────────

def calculate_stability(df, leiden_col, dox_col='Dox'):
    dcm_col = f'{leiden_col}_DCM'
    rna_col = f'{leiden_col}_RNA'

    results = []
    for dox in df[dox_col].unique():
        dox_df     = df[df[dox_col] == dox]
        total_cells = len(dox_df)
        if total_cells == 0:
            continue

        stable_mask  = dox_df[dcm_col] == dox_df[rna_col]
        stable_count = stable_mask.sum()
        switch_count = total_cells - stable_count

        results.append({
            'Dox':         dox,
            'leiden_dim':  leiden_col,
            'resolution':  float(leiden_col.replace('leiden_annot_', '')),
            'stable_count': stable_count,
            'switch_count': switch_count,
            'total_cells':  total_cells,
            'stable_pct':   stable_count / total_cells * 100,
            'switch_pct':   switch_count / total_cells * 100,
        })
    return pd.DataFrame(results)


stability_results = []
for res in RESOLUTIONS:
    leiden_col = f'leiden_annot_{res}'
    print(f"  Processing resolution {res} ...")
    result = calculate_stability(matched_df, leiden_col)
    stability_results.append(result)

stability_df = pd.concat(stability_results, ignore_index=True)
stability_df = stability_df.sort_values(['Dox', 'resolution']).reset_index(drop=True)

print(f"\nStability dataframe: {stability_df.shape}")
print(stability_df.to_string(index=False))
stability_df.to_csv(os.path.join(outdir, 'stability_percentages.csv'), index=False)

# ─────────────────────────────────────────────
# RESHAPE FOR PLOTTING
# ─────────────────────────────────────────────
plot_df = stability_df.melt(
    id_vars=['Dox', 'leiden_dim', 'resolution', 'total_cells'],
    value_vars=['stable_pct', 'switch_pct'],
    var_name='transition_type',
    value_name='percentage'
)
plot_df['transition_type'] = plot_df['transition_type'].map({
    'stable_pct': 'Stable',
    'switch_pct': 'Switch'
})
plot_df.to_csv(os.path.join(outdir, 'stability_percentages_long.csv'), index=False)

dox_conditions = sorted(stability_df['Dox'].unique())
n_dox          = len(dox_conditions)
colors         = {'Stable': '#2ecc71', 'Switch': '#e74c3c'}

# ─────────────────────────────────────────────
# PLOT 1: GROUPED BARPLOT PER RESOLUTION, SEPARATED BY DOX
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, n_dox, figsize=(5 * n_dox, 6), sharey=True)
if n_dox == 1:
    axes = [axes]

for ax, dox in zip(axes, dox_conditions):
    dox_data = plot_df[plot_df['Dox'] == dox].sort_values('resolution')
    sns.barplot(
        data=dox_data, x='resolution', y='percentage',
        hue='transition_type', palette=colors, ax=ax
    )
    ax.set_title(f'Dox = {dox}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Resolution', fontsize=12)
    ax.set_ylabel('Percentage (%)' if ax == axes[0] else '', fontsize=12)
    ax.set_ylim(0, 100)
    ax.legend(title='Transition', loc='upper right')
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Cell Type Stability: DCM vs RNA', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'stability_barplot_by_dox.pdf'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: stability_barplot_by_dox.pdf")

# ─────────────────────────────────────────────
# PLOT 2: STACKED BARPLOT
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, n_dox, figsize=(5 * n_dox, 6), sharey=True)
if n_dox == 1:
    axes = [axes]

for ax, dox in zip(axes, dox_conditions):
    dox_data   = stability_df[stability_df['Dox'] == dox].sort_values('resolution')
    resolutions = dox_data['resolution'].values
    stable      = dox_data['stable_pct'].values
    switch      = dox_data['switch_pct'].values
    x           = np.arange(len(resolutions))

    ax.bar(x, stable, label='Stable', color='#2ecc71')
    ax.bar(x, switch, bottom=stable, label='Switch', color='#e74c3c')
    ax.set_title(f'Dox = {dox}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Resolution', fontsize=12)
    ax.set_ylabel('Percentage (%)' if ax == axes[0] else '', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(resolutions)
    ax.set_ylim(0, 100)
    ax.legend(title='Transition', loc='upper right')

plt.suptitle('Cell Type Stability: DCM vs RNA (Stacked)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'stability_stacked_barplot_by_dox.pdf'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: stability_stacked_barplot_by_dox.pdf")

# ─────────────────────────────────────────────
# PLOT 3: FACETED BY RESOLUTION
# ─────────────────────────────────────────────
g = sns.catplot(
    data=plot_df,
    x='Dox', y='percentage',
    hue='transition_type', col='resolution',
    kind='bar', palette=colors,
    col_wrap=3, height=4, aspect=1.2
)
g.set_axis_labels('Dox Condition', 'Percentage (%)')
g.set_titles('Resolution {col_name}')
g.add_legend(title='Transition')
for ax in g.axes.flat:
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylim(0, 100)

plt.suptitle('Cell Type Stability Across Resolutions and Dox Conditions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'stability_faceted_by_resolution.pdf'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: stability_faceted_by_resolution.pdf")

# ─────────────────────────────────────────────
# PLOT 4: LINE PLOT — STABILITY TREND ACROSS RESOLUTIONS
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for dox in dox_conditions:
    dox_data = stability_df[stability_df['Dox'] == dox].sort_values('resolution')
    ax.plot(dox_data['resolution'], dox_data['stable_pct'],
            marker='o', label=f'Dox {dox}', linewidth=2)

ax.set_xlabel('Resolution', fontsize=12)
ax.set_ylabel('Stable Percentage (%)', fontsize=12)
ax.set_title('Cell Type Stability Across Resolutions by Dox Condition',
             fontsize=14, fontweight='bold')
ax.legend(title='Dox Condition')
ax.set_ylim(0, 100)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'stability_lineplot_by_dox.pdf'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: stability_lineplot_by_dox.pdf")

print(f"\n{'='*60}")
print(f"DONE — all outputs in: {outdir}")
print(f"{'='*60}")

Loading adata_harmony_best.h5ad ...
  15920 cells × 14384 genes
  Batches: {'multi_RNA': 9912, 'ref_D2': 2545, 'ref_D4': 2335, 'scNMT_RNA': 723, 'scNMT_DCM': 405}

Re-clustering at each resolution ...


/tmp/ipykernel_485362/3945647689.py:52: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=res, key_added=col, random_state=0)


  res=0.1: 4 clusters

Matching DCM↔RNA cells via pool + plate_well ...
  Matched DCM–RNA pairs: 399
  Dox conditions: ['16h', '16h_24h_dtag']
  Processing resolution 0.1 ...

Stability dataframe: (2, 8)
         Dox       leiden_dim  resolution  stable_count  switch_count  total_cells  stable_pct  switch_pct
         16h leiden_annot_0.1         0.1            32            85          117   27.350427   72.649573
16h_24h_dtag leiden_annot_0.1         0.1            57           225          282   20.212766   79.787234
Saved: stability_barplot_by_dox.pdf
Saved: stability_stacked_barplot_by_dox.pdf
Saved: stability_faceted_by_resolution.pdf
Saved: stability_lineplot_by_dox.pdf

DONE — all outputs in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/stability_analysis


In [52]:
adata.obs['leiden_annot_0.1'].unique()

array(['1', '0', '3', '2'], dtype=object)

In [53]:
##############################################
# DCM → RNA TRANSITION ANALYSIS
# Single dimension: leiden_annot_0.1
##############################################

import os
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root        = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
best_outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best')
outdir      = os.path.join(best_outdir, 'transitions_leiden_annot_0.1')
os.makedirs(outdir, exist_ok=True)

# ─────────────────────────────────────────────
# COLUMN TO ANALYSE — change here if needed
# ─────────────────────────────────────────────
LEIDEN_COL = 'leiden_annot_0.1'

# ─────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────
print("Loading adata_harmony_best.h5ad ...")
#adata = sc.read_h5ad(os.path.join(best_outdir, 'adata_harmony_best.h5ad'))
print(f"  {adata.n_obs} cells × {adata.n_vars} genes")
print(f"  Available columns matching 'leiden': "
      f"{[c for c in adata.obs.columns if 'leiden' in c.lower()]}")

if LEIDEN_COL not in adata.obs.columns:
    raise KeyError(
        f"Column '{LEIDEN_COL}' not found in adata.obs.\n"
        f"Available leiden columns: "
        f"{[c for c in adata.obs.columns if 'leiden' in c.lower()]}"
    )

adata.obs['batch']    = adata.obs['batch'].astype(str)
adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype(str)

# ─────────────────────────────────────────────
# SEPARATE DCM AND scNMT_RNA CELLS
# ─────────────────────────────────────────────
dcm_mask       = adata.obs['batch'].str.contains('DCM',   case=False, na=False)
scnmt_rna_mask = (adata.obs['batch'].str.contains('scNMT', case=False, na=False) &
                   adata.obs['batch'].str.contains('RNA',   case=False, na=False))

adata_dcm = adata[dcm_mask].copy()
adata_rna = adata[scnmt_rna_mask].copy()

print(f"\n  DCM cells : {adata_dcm.n_obs}")
print(f"  scNMT RNA : {adata_rna.n_obs}")

# ─────────────────────────────────────────────
# BUILD DCM AND RNA TABLES
# ─────────────────────────────────────────────
for label, obs in [('DCM', adata_dcm.obs), ('RNA', adata_rna.obs)]:
    missing = {'pool', 'plate_well', 'Dox'} - set(obs.columns)
    if missing:
        raise ValueError(f"Missing columns in {label} obs: {missing}")

dcm_df = adata_dcm.obs[['pool', 'plate_well', 'Dox', LEIDEN_COL]].copy()
dcm_df['DCM_matrixID'] = dcm_df.index

rna_df = adata_rna.obs[['pool', 'plate_well', LEIDEN_COL]].copy()
rna_df['Sample_RNA'] = rna_df.index

print(f"\nDCM obs_names examples : {dcm_df.index[:3].tolist()}")
print(f"RNA obs_names examples : {rna_df.index[:3].tolist()}")

# ─────────────────────────────────────────────
# MATCH VIA POOL + PLATE_WELL
# ─────────────────────────────────────────────
matched_df = dcm_df.reset_index(drop=True).merge(
    rna_df.reset_index(drop=True),
    on=['pool', 'plate_well'],
    suffixes=('_DCM', '_RNA'),
    how='inner'
)

print(f"\nMatched DCM–RNA pairs: {len(matched_df)}")
print(f"Dox conditions       : {sorted(matched_df['Dox'].unique())}")

if matched_df.empty:
    raise ValueError("No matched cells — check pool/plate_well in adata.obs.")

matched_df.to_csv(os.path.join(outdir, 'matched_dcm_rna_cells.csv'), index=False)

# Column names after merge
DCM_COL = f'{LEIDEN_COL}_DCM'
RNA_COL = f'{LEIDEN_COL}_RNA'

# ─────────────────────────────────────────────
# CALCULATE TRANSITIONS PER DOX
# ─────────────────────────────────────────────
dox_conditions = sorted(matched_df['Dox'].unique())
all_results    = []

for dox in dox_conditions:
    dox_df = matched_df[matched_df['Dox'] == dox]

    transition_counts = (
        dox_df.groupby([DCM_COL, RNA_COL])
        .size()
        .unstack(fill_value=0)
    )
    transition_pct = transition_counts.div(
        transition_counts.sum(axis=1), axis=0
    ) * 100

    for dcm_state in transition_pct.index:
        for rna_state in transition_pct.columns:
            all_results.append({
                'Dox':          dox,
                'DCM_celltype': dcm_state,
                'RNA_celltype': rna_state,
                'count':        int(transition_counts.loc[dcm_state, rna_state]),
                'percentage':   transition_pct.loc[dcm_state, rna_state],
            })

transitions_df = pd.DataFrame(all_results)
transitions_df.to_csv(os.path.join(outdir, 'cell_type_transitions.csv'), index=False)

# ─────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"TRANSITIONS FOR {LEIDEN_COL}")
print('=' * 60)
for dox in dox_conditions:
    print(f"\n--- Dox condition: {dox} ---")
    dox_subset = transitions_df[transitions_df['Dox'] == dox]
    pivot = dox_subset.pivot(
        index='DCM_celltype', columns='RNA_celltype', values='percentage'
    )
    print(pivot.round(1).to_string())

# ─────────────────────────────────────────────
# PLOT: TRANSITION HEATMAP PER DOX
# ─────────────────────────────────────────────
print("\nPlotting transition heatmaps ...")

for dox in dox_conditions:
    subset = transitions_df[transitions_df['Dox'] == dox]
    if subset.empty:
        continue

    pivot  = subset.pivot(
        index='DCM_celltype', columns='RNA_celltype', values='percentage'
    )
    n_rows, n_cols = pivot.shape
    fig_w = max(6, n_cols * 0.9 + 2)
    fig_h = max(5, n_rows * 0.7 + 2)

    plt.figure(figsize=(fig_w, fig_h))
    sns.heatmap(
        pivot, annot=True, fmt='.1f', cmap='Blues',
        linewidths=0.3, linecolor='white',
        cbar_kws={'label': '% of DCM cluster → RNA cluster'},
        vmin=0, vmax=100,
    )
    plt.title(f'Transitions: {LEIDEN_COL}  |  Dox = {dox}', fontsize=11)
    plt.xlabel('RNA cluster')
    plt.ylabel('DCM cluster')
    plt.tight_layout()

    fname = f'transition_heatmap_dox{dox}.pdf'
    plt.savefig(os.path.join(outdir, fname), dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fname}")

# ─────────────────────────────────────────────
# STABLE % SUMMARY
# ─────────────────────────────────────────────
print("\nStable (same cluster) % per Dox:")
for dox in dox_conditions:
    sub        = matched_df[matched_df['Dox'] == dox]
    stable_pct = (sub[DCM_COL] == sub[RNA_COL]).mean() * 100
    print(f"  Dox {dox}: {stable_pct:.1f}%  ({len(sub)} pairs)")

print(f"\n{'='*60}")
print(f"DONE — outputs in: {outdir}")
print(f"{'='*60}")

Loading adata_harmony_best.h5ad ...
  15920 cells × 14384 genes
  Available columns matching 'leiden': ['leiden', 'leiden_annot', 'leiden_annot_0.1', 'leiden_annot_0.2', 'leiden_annot_0.3', 'leiden_annot_0.5', 'leiden_annot_0.8', 'leiden_annot_1.0']

  DCM cells : 405
  scNMT RNA : 723

DCM obs_names examples : ['X100_Plate_1', 'X154_Plate_1', 'X82_Plate_1']
RNA obs_names examples : ['NPC_plate1_A10', 'NPC_plate1_A11', 'NPC_plate1_A12']

Matched DCM–RNA pairs: 399
Dox conditions       : ['16h', '16h_24h_dtag']

TRANSITIONS FOR leiden_annot_0.1

--- Dox condition: 16h ---
RNA_celltype     0     1     2     3
DCM_celltype                        
0             68.2  13.6  18.2   0.0
1             51.1  18.2  18.2  12.5
3             42.9  14.3  28.6  14.3

--- Dox condition: 16h_24h_dtag ---
RNA_celltype     0     1     2     3
DCM_celltype                        
0             52.4   4.8  25.4  17.5
1             52.9  11.0  17.6  18.6
3             66.7  11.1  11.1  11.1

Plotting trans

In [1]:
import os
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root        = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
best_outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best')
outdir      = os.path.join(best_outdir, 'de_reference_only')
os.makedirs(outdir, exist_ok=True)
adata = sc.read_h5ad(os.path.join(best_outdir, 'adata_harmony_best.h5ad'))


In [4]:
adata.obs['leiden'].unique()

array([1, 0, 3, 2])

In [58]:
#marker genes reference

array(['scNMT_DCM', 'scNMT_RNA', 'ref_D2', 'ref_D4', 'multi_RNA'],
      dtype=object)

In [59]:
##############################################
# DE ANALYSIS — REFERENCE DATA ONLY
# Subsets to ref_D2, ref_D4, multi_RNA batches,
# runs logreg + t-test rank_genes_groups with leiden
# as groupby, then plots KDE and scatter comparisons.
##############################################



# ─────────────────────────────────────────────
# LOAD & SUBSET TO REFERENCE BATCHES
# ─────────────────────────────────────────────
print("Loading adata_harmony_best.h5ad ...")
print(f"  {adata.n_obs} cells × {adata.n_vars} genes")
print(f"  Batches: {adata.obs['batch'].value_counts().to_dict()}")

adata.obs['batch']  = adata.obs['batch'].astype(str)
adata.obs['leiden_annot_0.1'] = adata.obs['leiden_annot_0.1'].astype(str)

REF_BATCHES = ['ref_D2', 'ref_D4', 'multi_RNA']
ref_mask    = adata.obs['batch'].isin(REF_BATCHES)
adata_ref   = adata[ref_mask].copy()

print(f"\nReference subset: {adata_ref.n_obs} cells")
print(adata_ref.obs['batch'].value_counts().to_string())
print(f"\nLeiden clusters: {sorted(adata_ref.obs['leiden_annot_0.1'].unique(), key=lambda x: int(x) if x.isdigit() else x)}")

# Use lognorm layer — same as the reference code's adata_ref.X = adata_ref.layers['lognorm']
if 'lognorm' in adata_ref.layers:
    adata_ref.X = adata_ref.layers['lognorm']
    print("\nUsing layers['lognorm'] for expression")
else:
    raise KeyError("layers['lognorm'] not found. Make sure adata was produced by run_harmony_config.")

# Drop clusters with fewer than 3 cells (logreg requirement)
cluster_counts = adata_ref.obs['leiden_annot_0.1'].value_counts()
valid_clusters  = cluster_counts[cluster_counts >= 3].index.tolist()
if len(valid_clusters) < len(cluster_counts):
    print(f"\nDropping clusters with <3 cells: "
          f"{cluster_counts[cluster_counts < 3].index.tolist()}")
    adata_ref = adata_ref[adata_ref.obs['leiden_annot_0.1'].isin(valid_clusters)].copy()

print(f"\nFinal reference subset: {adata_ref.n_obs} cells, "
      f"{adata_ref.obs['leiden_annot_0.1'].nunique()} clusters")

# ─────────────────────────────────────────────
# RANK GENES — LOGREG
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("RUNNING logreg rank_genes_groups ...")
print("=" * 60)

sc.tl.rank_genes_groups(
    adata_ref, groupby='leiden_annot_0.1',
    method='logreg',
    use_raw=False,
    key_added='rank_genes_logreg',
)
logreg_res = adata_ref.uns['rank_genes_logreg']

# ─────────────────────────────────────────────
# RANK GENES — T-TEST
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("RUNNING t-test rank_genes_groups ...")
print("=" * 60)

sc.tl.rank_genes_groups(
    adata_ref, groupby='leiden_annot_0.1',
    method='t-test',
    use_raw=False,
    key_added='rank_genes_ttest',
)
ttest_res = adata_ref.uns['rank_genes_ttest']

# ─────────────────────────────────────────────
# EXTRACT RESULTS
# Mirrors the reference extract_results() helper exactly.
# ─────────────────────────────────────────────
def extract_results(res, method):
    groups   = res['names'].dtype.names
    all_genes = []
    for ct in groups:
        df = pd.DataFrame({
            'gene':        res['names'][ct],
            'score_logreg': res['scores'][ct]         if method == 'logreg' else None,
            'logfc_ttest':  res['logfoldchanges'][ct]  if method == 't-test' else None,
            'pvals_adj':    res['pvals_adj'][ct]       if method == 't-test' else None,
            'cell_type':    ct,
        })
        all_genes.append(df)
    return pd.concat(all_genes, ignore_index=True)


logreg_df = extract_results(logreg_res, 'logreg')
ttest_df  = extract_results(ttest_res,  't-test')

# ─────────────────────────────────────────────
# MERGE AND FILTER
# ─────────────────────────────────────────────
compare_df = pd.merge(
    logreg_df[['gene', 'cell_type', 'score_logreg']],
    ttest_df[['gene',  'cell_type', 'logfc_ttest', 'pvals_adj']],
    on=['gene', 'cell_type'],
)

compare_df['significant_logreg'] = compare_df['score_logreg'] > 0.5
compare_df['significant_ttest']  = compare_df['pvals_adj']    < 0.05

sig_df = compare_df[
    compare_df['significant_logreg'] | compare_df['significant_ttest']
].copy()

# Also apply the strict filter from the global DE script
ttest_df_sig = compare_df[
    (compare_df['pvals_adj']   < 0.0001) &
    (compare_df['logfc_ttest'] > 1.0)
].copy()

print(f"\nGenes significant in ≥1 method : {len(sig_df)}")
print(f"Genes passing strict filter     : {len(ttest_df_sig)}  "
      f"(pvals_adj<0.0001 & logfc>1)")

compare_df.to_csv(os.path.join(outdir, 'de_compare_logreg_ttest.csv'),    index=False)
sig_df.to_csv(    os.path.join(outdir, 'de_sig_at_least_one_method.csv'), index=False)
ttest_df_sig.to_csv(os.path.join(outdir, 'de_sig_strict.csv'),            index=False)

# ─────────────────────────────────────────────
# PLOT 1: KDE — logreg score vs t-test logFC
# (mirrors reference KDE plot exactly)
# ─────────────────────────────────────────────
clusters = sorted(sig_df['cell_type'].unique(), key=lambda x: int(x) if x.isdigit() else x)
palette  = dict(zip(clusters, sns.color_palette('tab10', len(clusters))))

plt.figure(figsize=(9, 6))
sns.kdeplot(
    data=sig_df, x='score_logreg',
    hue='cell_type', hue_order=clusters,
    palette=palette,
    common_norm=False, fill=True, alpha=0.3,
    label='logreg score'
)
sns.kdeplot(
    data=sig_df, x='logfc_ttest',
    hue='cell_type', hue_order=clusters,
    palette=palette,
    common_norm=False, fill=False, linestyle='--', alpha=0.7,
    label='t-test logFC'
)
plt.xlabel('Marker score')
plt.title('Distribution of logreg vs t-test marker scores\n'
          '(significant genes, reference cells only)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'kde_logreg_vs_ttest.pdf'), dpi=200, bbox_inches='tight')
plt.close()
print("Saved: kde_logreg_vs_ttest.pdf")

# ─────────────────────────────────────────────
# PLOT 2: Scatter — logreg score vs t-test logFC
# (mirrors reference scatter plot exactly)
# ─────────────────────────────────────────────
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=sig_df, x='score_logreg', y='logfc_ttest',
    hue='cell_type', hue_order=clusters,
    palette=palette, alpha=0.7, s=20,
)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axhline(1,    color='#e74c3c', linestyle=':', linewidth=0.8, label='logFC=1')
plt.axvline(0.5,  color='#3498db', linestyle=':', linewidth=0.8, label='logreg=0.5')
plt.xlabel('logreg score')
plt.ylabel('t-test log fold change')
plt.title('Comparison of logreg vs t-test marker scores\n'
          '(reference cells only)', fontsize=11)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'scatter_logreg_vs_ttest.pdf'), dpi=200, bbox_inches='tight')
plt.close()
print("Saved: scatter_logreg_vs_ttest.pdf")

# ─────────────────────────────────────────────
# PLOT 3: Top markers per cluster — dotplot
# ─────────────────────────────────────────────
try:
    top_per_cluster = {}
    for cl in clusters:
        sub  = ttest_df_sig[ttest_df_sig['cell_type'] == cl]
        sub  = sub.sort_values('logfc_ttest', ascending=False)
        top_per_cluster[cl] = sub.head(5)['gene'].tolist()

    top_per_cluster = {k: v for k, v in top_per_cluster.items() if v}

    if top_per_cluster:
        sc.pl.dotplot(
            adata_ref, var_names=top_per_cluster, groupby='leiden_annot_0.1',
            show=False, use_raw=False, standard_scale='var',
            save=None
        )
        plt.savefig(os.path.join(outdir, 'dotplot_top_markers_ref.pdf'),
                    dpi=200, bbox_inches='tight')
        plt.close()
        print("Saved: dotplot_top_markers_ref.pdf")

        sc.pl.matrixplot(
            adata_ref, var_names=top_per_cluster, groupby='leiden_annot_0.1',
            show=False, use_raw=False, standard_scale='var', cmap='viridis',
            save=None
        )
        plt.savefig(os.path.join(outdir, 'matrixplot_top_markers_ref.pdf'),
                    dpi=200, bbox_inches='tight')
        plt.close()
        print("Saved: matrixplot_top_markers_ref.pdf")
except Exception as e:
    print(f"Dotplot/matrixplot failed: {e}")

# ─────────────────────────────────────────────
# PLOT 4: Per-batch breakdown of top markers
# Show mean lognorm expression of strict-filter genes
# split by ref_D2 / ref_D4 / multi_RNA
# ─────────────────────────────────────────────
import scipy.sparse as sp

all_strict_genes = list(dict.fromkeys(ttest_df_sig['gene'].tolist()))
batches_ref      = sorted(adata_ref.obs['batch'].unique())

def mean_expr(adata_sub, genes, clusters_list):
    genes_ok = [g for g in genes if g in adata_sub.var_names]
    if not genes_ok:
        return pd.DataFrame()
    sub = adata_sub[:, genes_ok]
    X   = sub.X.toarray() if sp.issparse(sub.X) else np.array(sub.X)
    df  = pd.DataFrame(X, index=adata_sub.obs_names, columns=genes_ok)
    df['leiden_annot_0.1'] = adata_sub.obs['leiden_annot_0.1'].values
    return df.groupby('leiden_annot_0.1')[genes_ok].mean().reindex(index=clusters_list, fill_value=0)

for batch in batches_ref:
    sub = adata_ref[adata_ref.obs['batch'] == batch]
    if sub.n_obs < 5:
        continue
    mat = mean_expr(sub, all_strict_genes, clusters)
    if mat.empty:
        continue

    # Use only clusters that have sig genes and are present
    plot_genes = []
    for cl in clusters:
        genes_cl = ttest_df_sig[ttest_df_sig['cell_type'] == cl]['gene'].head(10).tolist()
        plot_genes.extend([g for g in genes_cl if g in mat.columns])
    plot_genes = list(dict.fromkeys(plot_genes))

    if not plot_genes:
        continue

    plot_mat = mat[plot_genes].T   # genes × clusters
    fig_h    = max(4, len(plot_genes) * 0.25 + 2)
    fig_w    = max(6, len(clusters) * 0.6 + 2)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(
        plot_mat, ax=ax, cmap='viridis',
        linewidths=0.2, linecolor='white',
        cbar_kws={'label': 'Mean lognorm', 'shrink': 0.6},
        yticklabels=True, xticklabels=True,
    )
    ax.set_title(f'Top strict DE genes — {batch}', fontsize=10)
    ax.set_xlabel('Leiden cluster')
    ax.set_ylabel('Gene')
    ax.tick_params(axis='x', rotation=0,  labelsize=8)
    ax.tick_params(axis='y', labelsize=6)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'heatmap_strict_DE_{batch}.pdf'),
                dpi=200, bbox_inches='tight')
    plt.close()
    print(f"Saved: heatmap_strict_DE_{batch}.pdf")

print(f"\n{'='*60}")
print(f"DONE — all outputs in: {outdir}")
print(f"{'='*60}")

Loading adata_harmony_best.h5ad ...
  15920 cells × 14384 genes
  Batches: {'multi_RNA': 9912, 'ref_D2': 2545, 'ref_D4': 2335, 'scNMT_RNA': 723, 'scNMT_DCM': 405}

Reference subset: 14792 cells
batch
multi_RNA    9912
ref_D2       2545
ref_D4       2335

Leiden clusters: ['0', '1', '2', '3']

Using layers['lognorm'] for expression

Final reference subset: 14792 cells, 4 clusters

RUNNING logreg rank_genes_groups ...


/home/erlantz/.local/share/mamba/envs/Jona_env_17122025/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



RUNNING t-test rank_genes_groups ...

Genes significant in ≥1 method : 32797
Genes passing strict filter     : 2612  (pvals_adj<0.0001 & logfc>1)
Saved: kde_logreg_vs_ttest.pdf
Saved: scatter_logreg_vs_ttest.pdf
Saved: dotplot_top_markers_ref.pdf
Saved: matrixplot_top_markers_ref.pdf


/tmp/ipykernel_236665/596413555.py:249: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('leiden_annot_0.1')[genes_ok].mean().reindex(index=clusters_list, fill_value=0)


Saved: heatmap_strict_DE_multi_RNA.pdf


/tmp/ipykernel_236665/596413555.py:249: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('leiden_annot_0.1')[genes_ok].mean().reindex(index=clusters_list, fill_value=0)


Saved: heatmap_strict_DE_ref_D2.pdf


/tmp/ipykernel_236665/596413555.py:249: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('leiden_annot_0.1')[genes_ok].mean().reindex(index=clusters_list, fill_value=0)


Saved: heatmap_strict_DE_ref_D4.pdf

DONE — all outputs in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/de_reference_only


In [62]:
adata_ref

AnnData object with n_obs × n_vars = 14792 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'leiden_annot_0.2', 'leiden_annot_0.3', 'leiden_annot_0.5', 'leiden_annot_0.8', 'leiden_annot_1.0'
    var: 'feature_types', 'genome', 'pattern', 'read', 'sequence', 'mt', 'interval', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std'
    uns: '

In [68]:
ttest_df_unique['cell_type'].value_counts()

cell_type
IGs     1089
INs      567
RGCs     399
Name: count, dtype: int64

In [61]:
##############################################
# UNIQUE MARKER GENES PER CLUSTER
# Requires ttest_df from de_reference_only.py
# (run that script first or have ttest_df in memory)
##############################################

import os
import pandas as pd

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root        = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
best_outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best')
outdir      = os.path.join(best_outdir, 'de_reference_only')
os.makedirs(outdir, exist_ok=True)

# ─────────────────────────────────────────────
# FILTER: strict thresholds
# ─────────────────────────────────────────────
ttest_df_sig = ttest_df[
    (ttest_df['pvals_adj']   < 0.0001) &
    (ttest_df['logfc_ttest'] > 1)
].copy()

print("Before removing duplicates:")
print(ttest_df_sig['cell_type'].value_counts().to_string())

# ─────────────────────────────────────────────
# KEEP ONLY GENES UNIQUE TO ONE CLUSTER
# ─────────────────────────────────────────────
gene_counts  = ttest_df_sig['gene'].value_counts()
unique_genes = gene_counts[gene_counts == 1].index

ttest_df_unique = ttest_df_sig[ttest_df_sig['gene'].isin(unique_genes)].copy()

print("\nAfter removing duplicates:")
print(ttest_df_unique['cell_type'].value_counts().to_string())

# ─────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────
out_path = os.path.join(outdir, 'unique_markers_per_cluster.csv')
ttest_df_unique.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Total unique marker genes: {len(ttest_df_unique)}")

Before removing duplicates:
cell_type
3    1115
1     589
2     589
0     319

After removing duplicates:
cell_type
3    1058
1     562
2     553
0     315

Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/de_reference_only/unique_markers_per_cluster.csv
Total unique marker genes: 2488


In [63]:
adata

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'leiden_annot_0.2', 'leiden_annot_0.3', 'leiden_annot_0.5', 'leiden_annot_0.8', 'leiden_annot_1.0'
    var: 'feature_types', 'genome', 'pattern', 'read', 'sequence', 'mt', 'interval', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std'
    uns: '

In [65]:
adata_ref

AnnData object with n_obs × n_vars = 14792 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'leiden_annot_0.2', 'leiden_annot_0.3', 'leiden_annot_0.5', 'leiden_annot_0.8', 'leiden_annot_1.0'
    var: 'feature_types', 'genome', 'pattern', 'read', 'sequence', 'mt', 'interval', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std'
    uns: '

In [69]:
##############################################
# LOGREG MARKERS MATRIXPLOT
# Reference cells + DCM cells side by side
# Adapted from the notebook code to use the
# adata_harmony_best.h5ad pipeline objects.
#
# Requires in memory (from de_reference_only.py):
#   - adata_ref   (reference subset)
#   - compare_df  (merged logreg + ttest results)
#
# LEIDEN_DIM_COL: the leiden column to use for
# cell_type_broad assignment on DCM cells.
# Change to match your chosen dimension.
##############################################

import os
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# PATHS & CONFIG
# ─────────────────────────────────────────────
root        = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
best_outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best')
outdir      = os.path.join(best_outdir, 'de_reference_only')
os.makedirs(outdir, exist_ok=True)

# Leiden column in adata_harmony_best that carries the cluster annotations
# for DCM cells — change this to whatever dimension you are using
LEIDEN_DIM_COL = 'leiden_annot_0.1'

# Cell type categories expected after splitting the leiden annotation label
# e.g. '1_RGCs' → 'RGCs'. Adjust to match your actual cluster names.
CELL_TYPE_CATEGORIES = None   # set to e.g. ['RGCs', 'IN', 'IG'] to enforce order,
                               # or leave None to use whatever is present

# ─────────────────────────────────────────────
# STEP 1: SELECT TOP GENES FROM REFERENCE DE
# (same filter as the original notebook)
# ─────────────────────────────────────────────
topgenes = compare_df[
    (compare_df['score_logreg']      >= 0.15) &
    (compare_df['significant_ttest'] == True)
]['gene'].tolist()

print(f"Top genes (logreg ≥ 0.15 & ttest sig): {len(topgenes)}")

# ─────────────────────────────────────────────
# STEP 2: REFERENCE MATRIXPLOT
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("REFERENCE MATRIXPLOT")
print("=" * 60)

# adata_ref already has lognorm set as .X from de_reference_only.py — re-assign to be safe
if 'lognorm' in adata_ref.layers:
    adata_ref.X = adata_ref.layers['lognorm']

adata_ref.obs[LEIDEN_DIM_COL] = adata_ref.obs[LEIDEN_DIM_COL].astype(str)
print(f"  Reference groupby '{LEIDEN_DIM_COL}' distribution:")
print(adata_ref.obs[LEIDEN_DIM_COL].value_counts().to_string())

sc.tl.rank_genes_groups(
    adata_ref, groupby=LEIDEN_DIM_COL,
    method='wilcoxon',
    use_raw=False,
    pts=True,
    key_added='rank_genes_ref_matrixplot',
)
sc.tl.dendrogram(adata_ref, groupby=LEIDEN_DIM_COL)

# Filter topgenes to those present in adata_ref
topgenes_ref = [g for g in topgenes if g in adata_ref.var_names]
print(f"  Genes present in reference: {len(topgenes_ref)} / {len(topgenes)}")

sc.pl.rank_genes_groups_matrixplot(
    adata_ref,
    var_names=topgenes_ref,
    key='rank_genes_ref_matrixplot',
    cmap='coolwarm',
    layer='lognorm',
    standard_scale='var',
    dendrogram=False,
    show=False,
)
plt.suptitle(f'Reference cells — logreg markers ({LEIDEN_DIM_COL})', fontsize=10, y=1.01)
plt.savefig(os.path.join(outdir, 'matrixplot_logreg_ref.pdf'), dpi=200, bbox_inches='tight')
plt.close()
print("  Saved: matrixplot_logreg_ref.pdf")

# ─────────────────────────────────────────────
# STEP 3: PREPARE DCM SUBSET
# Load the full integrated adata to get the leiden
# annotation for DCM cells, then transfer to adata_dcm2.
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("PREPARING DCM SUBSET")
print("=" * 60)

# Load best adata if not already in memory

adata.obs['batch'] = adata.obs['batch'].astype(str)

if LEIDEN_DIM_COL not in adata.obs.columns:
    available = [c for c in adata.obs.columns if 'leiden' in c.lower()]
    raise KeyError(
        f"'{LEIDEN_DIM_COL}' not found in adata.obs.\n"
        f"Available leiden columns: {available}"
    )

# Subset to DCM cells
dcm_mask  = adata.obs['batch'].str.contains('DCM', case=False, na=False)
adata_dcm2 = adata[dcm_mask].copy()
print(f"  DCM cells: {adata_dcm2.n_obs}")

# Use lognorm layer
if 'lognorm' in adata_dcm2.layers:
    adata_dcm2.X = adata_dcm2.layers['lognorm']
else:
    raise KeyError("layers['lognorm'] not found in DCM subset.")

# Transfer leiden annotation and parse cell_type_broad
# e.g. '1_RGCs' → 'RGCs'  (same split logic as original notebook)
adata_dcm2.obs[LEIDEN_DIM_COL] = (
    adata_dcm2.obs[LEIDEN_DIM_COL].astype(str)
)

# Parse cell_type_broad from the leiden annotation label.
# Handles two formats:
#   '1_RGCs'  → 'RGCs'   (number_celltype — split and drop number)
#   '1'       → '1'      (plain number — keep as-is for the cluster ID)
def _parse_cell_type(label):
    label = str(label)
    if '_' in label:
        return '_'.join(label.split('_')[1:])   # drop numeric prefix
    return label   # plain cluster number — use directly

adata_dcm2.obs['cell_type_broad'] = (
    adata_dcm2.obs[LEIDEN_DIM_COL]
    .apply(_parse_cell_type)
    .replace('', 'Unknown')
    .fillna('Unknown')
)

print(f"  cell_type_broad distribution:")
print(adata_dcm2.obs['cell_type_broad'].value_counts().to_string())

# Enforce categorical order if specified
if CELL_TYPE_CATEGORIES is not None:
    adata_dcm2.obs['cell_type_broad'] = pd.Categorical(
        adata_dcm2.obs['cell_type_broad'],
        categories=CELL_TYPE_CATEGORIES,
        ordered=True,
    )
else:
    present_cats = sorted(
        adata_dcm2.obs['cell_type_broad'].dropna().unique().tolist()
    )
    adata_dcm2.obs['cell_type_broad'] = pd.Categorical(
        adata_dcm2.obs['cell_type_broad'],
        categories=present_cats,
        ordered=False,
    )

# Drop any cell_type_broad == 'Unknown' if desired
n_unknown = (adata_dcm2.obs['cell_type_broad'] == 'Unknown').sum()
if n_unknown > 0:
    print(f"  Dropping {n_unknown} cells with cell_type_broad='Unknown'")
    adata_dcm2 = adata_dcm2[adata_dcm2.obs['cell_type_broad'] != 'Unknown'].copy()

# Drop categories with fewer than 3 cells (needed for rank_genes_groups)
ct_counts   = adata_dcm2.obs['cell_type_broad'].value_counts()
valid_cats  = ct_counts[ct_counts >= 3].index.tolist()
if len(valid_cats) < len(ct_counts):
    dropped = ct_counts[ct_counts < 3].index.tolist()
    print(f"  Dropping cell types with <3 cells: {dropped}")
    adata_dcm2 = adata_dcm2[adata_dcm2.obs['cell_type_broad'].isin(valid_cats)].copy()

print(f"  Final DCM subset: {adata_dcm2.n_obs} cells, "
      f"{adata_dcm2.obs['cell_type_broad'].nunique()} cell types")

# ─────────────────────────────────────────────
# STEP 4: DCM MATRIXPLOT
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("DCM MATRIXPLOT")
print("=" * 60)

topgenes_dcm = [g for g in topgenes if g in adata_dcm2.var_names]
print(f"  Genes present in DCM: {len(topgenes_dcm)} / {len(topgenes)}")

# ── Diagnostic before rank_genes ──
print(f"  adata_dcm2 shape before DE: {adata_dcm2.shape}")
print(f"  cell_type_broad counts:\n{adata_dcm2.obs['cell_type_broad'].value_counts().to_string()}")

if adata_dcm2.n_obs == 0:
    raise ValueError(
        "DCM subset is empty after filtering.\n"
        f"Check that '{LEIDEN_DIM_COL}' is populated for DCM cells "
        "and that the cell_type_broad parsing produces valid category names."
    )
if adata_dcm2.obs['cell_type_broad'].nunique() < 2:
    raise ValueError(
        f"Only {adata_dcm2.obs['cell_type_broad'].nunique()} cell_type_broad "
        "category present — need at least 2 for rank_genes_groups.\n"
        f"Values: {adata_dcm2.obs['cell_type_broad'].value_counts().to_dict()}"
    )

sc.tl.rank_genes_groups(
    adata_dcm2, groupby='cell_type_broad',
    method='wilcoxon',
    use_raw=False,
    pts=True,
    key_added='rank_genes_dcm_matrixplot',
)
sc.tl.dendrogram(adata_dcm2, groupby='cell_type_broad')

sc.pl.rank_genes_groups_matrixplot(
    adata_dcm2,
    var_names=topgenes_dcm,
    key='rank_genes_dcm_matrixplot',
    groupby='cell_type_broad',
    cmap='coolwarm',
    layer='lognorm',
    standard_scale='var',
    dendrogram=False,
    show=False,
)
plt.suptitle(f'DCM cells — logreg markers ({LEIDEN_DIM_COL})', fontsize=10, y=1.01)
plt.savefig(os.path.join(outdir, 'matrixplot_logreg_dcm.pdf'), dpi=200, bbox_inches='tight')
plt.close()
print("  Saved: matrixplot_logreg_dcm.pdf")

# ─────────────────────────────────────────────
# STEP 5: SIDE-BY-SIDE COMPARISON
# Both matrixplots as subplots in one figure
# ─────────────────────────────────────────────
print("\nGenerating side-by-side comparison figure ...")

fig, axes = plt.subplots(1, 2, figsize=(20, max(6, len(topgenes_dcm) * 0.25 + 3)))

for ax, (adata_obj, key, title, genes) in zip(axes, [
    (adata_ref,  'rank_genes_ref_matrixplot', 'Reference cells', topgenes_ref),
    (adata_dcm2, 'rank_genes_dcm_matrixplot', f'DCM cells ({LEIDEN_DIM_COL})', topgenes_dcm),
]):
    genes_common = [g for g in topgenes if g in adata_obj.var_names]
    sc.pl.matrixplot(
        adata_obj,
        var_names=genes_common,
        groupby=LEIDEN_DIM_COL if adata_obj is adata_ref else 'cell_type_broad',
        ax=ax,
        cmap='coolwarm',
        layer='lognorm',
        standard_scale='var',
        show=False,
    )
    ax.set_title(title, fontsize=10)

plt.suptitle('logreg markers: Reference vs DCM', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'matrixplot_logreg_ref_vs_dcm.pdf'), dpi=200, bbox_inches='tight')
plt.close()
print("Saved: matrixplot_logreg_ref_vs_dcm.pdf")

print(f"\n{'='*60}")
print(f"DONE — outputs in: {outdir}")
print(f"{'='*60}")

Top genes (logreg ≥ 0.15 & ttest sig): 181

REFERENCE MATRIXPLOT
  Reference groupby 'leiden_annot_0.1' distribution:
leiden_annot_0.1
0    6699
1    4568
2    2326
3    1199
  Genes present in reference: 181 / 181
  Saved: matrixplot_logreg_ref.pdf

PREPARING DCM SUBSET
  DCM cells: 405
  cell_type_broad distribution:
cell_type_broad
1    303
0     86
3     16
  Final DCM subset: 405 cells, 3 cell types

DCM MATRIXPLOT
  Genes present in DCM: 181 / 181
  adata_dcm2 shape before DE: (405, 14384)
  cell_type_broad counts:
cell_type_broad
1    303
0     86
3     16
  Saved: matrixplot_logreg_dcm.pdf

Generating side-by-side comparison figure ...
Saved: matrixplot_logreg_ref_vs_dcm.pdf

DONE — outputs in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/de_reference_only


In [71]:
##############################################
# MODULE SCORING — UNIQUE MARKERS FROM DEA
# Uses ttest_df_unique from filter_unique_markers.py
# and adata (already in memory)
##############################################

import os
import scanpy as sc
import matplotlib.pyplot as plt

# ─────────────────────────────
# PATHS
# ─────────────────────────────
score_outdir = os.path.join(
    "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/"
    "Integration_multi_D2_D4/Python/Jona_notebook_27032026/"
    "integration/method_comparison/Harmony_best/module_score_ref"
)
os.makedirs(score_outdir, exist_ok=True)

# ─────────────────────────────
# ASSAY LABEL MAP
# ─────────────────────────────
assay_dict = {
    'Unknown':          'ref',
    'scNMT_pool1_DCM':  'scM_DCM',
    'scNMT_pool1_RNA':  'scM_RNA',
}
adata.obs['assay'] = adata.obs['orig.ident'].map(assay_dict).fillna('ref')

# ─────────────────────────────
# FIGURE SETTINGS
# ─────────────────────────────
sc.settings.figdir = score_outdir
sc.set_figure_params(dpi=300, dpi_save=300, figsize=(6, 6), fontsize=8)

LEIDEN_COL = 'leiden_annot_0.1'

# ─────────────────────────────
# SAVE leiden_annot_0.1 CLUSTERING UMAP
# ─────────────────────────────
if LEIDEN_COL in adata.obs.columns:
    sc.pl.umap(
        adata,
        color=LEIDEN_COL,
        legend_loc='right margin',
        legend_fontsize=7,
        frameon=False,
        title=LEIDEN_COL,
        show=False,
        save=f'_{LEIDEN_COL}_clustering.pdf',
    )
    print(f"Saved: _{LEIDEN_COL}_clustering.pdf")
else:
    print(f"WARNING: '{LEIDEN_COL}' not found in adata.obs — skipping clustering UMAP")

# ─────────────────────────────
# MODULE SCORING PER CELL TYPE
# ─────────────────────────────
for ct in ttest_df_unique['cell_type'].unique():
    try:
        ctgenes = (
            ttest_df_unique[
                (ttest_df_unique['cell_type']    == ct) &
                (ttest_df_unique['logfc_ttest']  >  1) &
                (ttest_df_unique['pvals_adj']     <  0.0001)
            ]
            .sort_values('pvals_adj', ascending=True)
            ['gene'].tolist()
        )
        print(f'{ct} — {len(ctgenes)} genes')
        if not ctgenes:
            print(f'  Skipping {ct}: no genes pass filter')
            continue

        score_name = f'{ct}_ttest_module'
        sc.tl.score_genes(adata, gene_list=ctgenes, score_name=score_name)

        # Single module score UMAP — PDF 300 dpi
        sc.pl.umap(
            adata,
            color=score_name,
            cmap='coolwarm',
            title=score_name,
            frameon=False,
            show=False,
            save=f'_{ct}_ttest_module_allcells.pdf',
        )

        # Paired figure: module score + leiden_annot_0.1 side by side
        if LEIDEN_COL in adata.obs.columns:
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            sc.pl.umap(adata, color=score_name, cmap='coolwarm',
                       frameon=False, title=score_name,
                       ax=axes[0], show=False)
            sc.pl.umap(adata, color=LEIDEN_COL,
                       legend_loc='right margin', legend_fontsize=7,
                       frameon=False, title=LEIDEN_COL,
                       ax=axes[1], show=False)
            plt.tight_layout()
            fig.savefig(
                os.path.join(score_outdir, f'{ct}_ttest_module_with_clusters.pdf'),
                dpi=300, bbox_inches='tight'
            )
            plt.close()

    except Exception as e:
        print(f'Error for {ct}: {e}')

Saved: _leiden_annot_0.1_clustering.pdf
0 — 315 genes
1 — 562 genes
2 — 553 genes
3 — 1058 genes


In [6]:
######################################################
# SIMPLE GENE SCORING, Z-SCORING
# Uses leiden_annot_0.1 clusters remapped to cell types:
#   0 → RGCs
#   1 → INs
#   2 → RGCs
#   3 → IGs
# Redoes ttest_df_unique with the new labels then
# runs the full scoring + z-score pipeline.
######################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import scanpy as sc
import json

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root      = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
score_dir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best', 'gene_scoring')
os.makedirs(score_dir, exist_ok=True)

# ─────────────────────────────────────────────
# CLUSTER → CELL TYPE MAP
# Edit these assignments to match your biology.
# ─────────────────────────────────────────────
CLUSTER_TO_CELLTYPE = {
    '0': 'RGCs',
    '1': 'INs',
    '2': 'RGCs',
    '3': 'IGs',
}

LEIDEN_COL = 'leiden_annot_0.1'

# ─────────────────────────────────────────────
# ASSIGN cell_type_broad ON adata
# ─────────────────────────────────────────────
adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype(str)
adata.obs['cell_type_broad'] = (
    adata.obs[LEIDEN_COL]
    .map(CLUSTER_TO_CELLTYPE)
    .fillna('Unknown')
)

print("cell_type_broad distribution:")
print(adata.obs['cell_type_broad'].value_counts().to_string())

# ─────────────────────────────────────────────
# RE-RUN DE ON REFERENCE CELLS WITH NEW LABELS
# Reference = ref_D2, ref_D4, multi_RNA
# ─────────────────────────────────────────────
print("\nRe-running t-test DE on reference cells with new cell_type_broad ...")

REF_BATCHES = ['ref_D2', 'ref_D4', 'multi_RNA']
adata.obs['batch'] = adata.obs['batch'].astype(str)
ref_mask  = adata.obs['batch'].isin(REF_BATCHES)
adata_ref = adata[ref_mask].copy()

# Use lognorm layer
if 'lognorm' in adata_ref.layers:
    adata_ref.X = adata_ref.layers['lognorm']

# Drop groups with fewer than 3 cells
ct_counts   = adata_ref.obs['cell_type_broad'].value_counts()
valid_cats  = ct_counts[ct_counts >= 3].index.tolist()
dropped     = ct_counts[ct_counts < 3].index.tolist()
if dropped:
    print(f"  Dropping cell types with <3 cells: {dropped}")
    adata_ref = adata_ref[adata_ref.obs['cell_type_broad'].isin(valid_cats)].copy()

print(f"  Reference subset: {adata_ref.n_obs} cells, groups: {sorted(valid_cats)}")

sc.tl.rank_genes_groups(
    adata_ref,
    groupby='cell_type_broad',
    method='t-test',
    use_raw=False,
    key_added='rank_genes_ttest_newlabels',
)

# Extract results
groups    = adata_ref.uns['rank_genes_ttest_newlabels']['names'].dtype.names
ttest_rows = []
for ct in groups:
    res = adata_ref.uns['rank_genes_ttest_newlabels']
    df  = pd.DataFrame({
        'gene':         res['names'][ct],
        'logfc_ttest':  res['logfoldchanges'][ct],
        'pvals_adj':    res['pvals_adj'][ct],
        'cell_type':    ct,
    })
    ttest_rows.append(df)

ttest_df = pd.concat(ttest_rows, ignore_index=True)

# ── Filter: strict thresholds ──
ttest_df_sig = ttest_df[
    (ttest_df['pvals_adj']   < 0.0001) &
    (ttest_df['logfc_ttest'] > 1.0)
].copy()

# ── Keep only genes unique to one cell type ──
gene_counts     = ttest_df_sig['gene'].value_counts()
unique_genes    = gene_counts[gene_counts == 1].index
ttest_df_unique = ttest_df_sig[ttest_df_sig['gene'].isin(unique_genes)].copy()

print("\nBefore removing duplicates:")
print(ttest_df_sig['cell_type'].value_counts().to_string())
print("\nAfter removing duplicates:")
print(ttest_df_unique['cell_type'].value_counts().to_string())

ttest_df_unique.to_csv(os.path.join(score_dir, 'ttest_df_unique_newlabels.csv'), index=False)

# ─────────────────────────────────────────────
# BUILD MARKER SETS
# ─────────────────────────────────────────────
label_order  = ['RGCs', 'INs', 'IGs', 'Unknown']
palette_dict = {
    'RGCs':    '#1f77b4',
    'INs':     '#ff7f0e',
    'IGs':     '#2ca02c',
    'Unknown': 'lightgray',
}

ttest_markers_by_ct = {}
for ct in ttest_df_unique['cell_type'].unique():
    genes = (
        ttest_df_unique[
            (ttest_df_unique['cell_type']   == ct) &
            (ttest_df_unique['logfc_ttest'] >  1) &
            (ttest_df_unique['pvals_adj']   <  0.0001)
        ]
        ['gene'].tolist()
    )
    if genes:
        ttest_markers_by_ct[ct] = genes

print(f"\nMarker sets: {len(ttest_markers_by_ct)} cell types")
for ct, g in ttest_markers_by_ct.items():
    print(f"  {ct}: {len(g)} genes")

# ─────────────────────────────────────────────
# SCORING PIPELINE
# ─────────────────────────────────────────────
def process_marker_set(marker_set_name, gene_sets_dict, output_suffix):
    print(f"\n{'='*60}")
    print(f"PROCESSING: {marker_set_name.upper()}")
    print(f"{'='*60}")

    set_dir = os.path.join(score_dir, marker_set_name)
    os.makedirs(set_dir, exist_ok=True)

    adata.X = adata.layers['lognorm']

    # Subset to scNMT cells (DCM + RNA), exclude Dox==Minus
    subset_mask = (
        adata.obs['orig.ident'].str.contains('scNMT') &
        (adata.obs['Dox'].astype(str) != 'Minus')
    )
    adata_subset = adata[subset_mask].copy()
    print(f"  scNMT subset: {adata_subset.n_obs} cells")

    # Filter gene sets to genes present in this subset
    genes_in_data      = set(adata_subset.var_names)
    filtered_gene_sets = {}
    for ct, genes in gene_sets_dict.items():
        filtered_gene_sets[ct] = [g for g in genes if g in genes_in_data]
        if not filtered_gene_sets[ct]:
            print(f"  WARNING: no matching genes for {ct}")
        else:
            print(f"  {ct}: {len(filtered_gene_sets[ct])} genes")

    # Initialise score columns
    for ct in filtered_gene_sets:
        adata_subset.obs[f'{ct}_score_raw']        = np.nan
        adata_subset.obs[f'{ct}_module_score_raw'] = np.nan

    # Compute scores WITHIN each assay
    for assay in adata_subset.obs['orig.ident'].unique():
        print(f"  Processing assay: {assay}")
        sub = adata_subset[adata_subset.obs['orig.ident'] == assay].copy()

        for ct, genes in filtered_gene_sets.items():
            if not genes:
                continue

            # Mean expression score
            sub.obs[f'{ct}_score_raw'] = (
                np.asarray(sub[:, genes].X.mean(axis=1)).ravel()
            )

            # Scanpy module score
            sc.tl.score_genes(
                sub, gene_list=genes,
                score_name=f'{ct}_module_score_raw',
                use_raw=False,
            )

            adata_subset.obs.loc[sub.obs_names, f'{ct}_score_raw']        = sub.obs[f'{ct}_score_raw']
            adata_subset.obs.loc[sub.obs_names, f'{ct}_module_score_raw'] = sub.obs[f'{ct}_module_score_raw']

    # Z-score BOTH score types per assay
    for ct in filtered_gene_sets:
        for suffix in ['_score_raw', '_module_score_raw']:
            raw_col = f'{ct}{suffix}'
            if raw_col not in adata_subset.obs:
                continue
            z_col = raw_col.replace('_raw', '_Z')
            adata_subset.obs[z_col] = np.nan
            for assay in adata_subset.obs['orig.ident'].unique():
                mask = adata_subset.obs['orig.ident'] == assay
                adata_subset.obs.loc[mask, z_col] = zscore(
                    adata_subset.obs.loc[mask, raw_col], nan_policy='omit'
                )

    # Assign max-module label per cell
    module_z_cols = [c for c in adata_subset.obs.columns if c.endswith('_module_score_Z')]
    print(f"  Module score Z columns: {module_z_cols}")

    def assign_module_label(row):
        scores    = row[module_z_cols]
        max_col   = scores.idxmax()
        max_val   = scores[max_col]
        label     = max_col.replace('_module_score_Z', '')
        return pd.Series({'module_max_label': label, 'module_max_value': max_val})

    adata_subset.obs[['module_max_label', 'module_max_value']] = (
        adata_subset.obs
        .groupby('orig.ident', group_keys=False)
        .apply(lambda df: df.apply(assign_module_label, axis=1))
    )

    CONFIDENCE_THRESHOLD = 1.65
    adata_subset.obs['module_label_confident'] = np.where(
        adata_subset.obs['module_max_value'] >= CONFIDENCE_THRESHOLD,
        adata_subset.obs['module_max_label'],
        'Unknown',
    )
    print(f"  Confidence threshold: z >= {CONFIDENCE_THRESHOLD}")

    # ── Confusion matrices ──
    results = {'marker_set_name': marker_set_name, 'confusion_matrices': {}, 'scores_summary': {}}

    for assay in adata_subset.obs['orig.ident'].unique():
        mask = adata_subset.obs['orig.ident'] == assay
        cm   = pd.crosstab(
            adata_subset.obs.loc[mask, 'cell_type_broad'],
            adata_subset.obs.loc[mask, 'module_label_confident'],
            rownames=['cell_type_broad'],
            colnames=['module_label_confident'],
        )
        results['confusion_matrices'][assay] = cm

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Cell count'})
        plt.title(f"{assay}: {marker_set_name} markers vs cell_type_broad")
        plt.tight_layout()
        plt.savefig(os.path.join(set_dir, f"{assay}_confusion_matrix_{output_suffix}.pdf"),
                    dpi=300, bbox_inches='tight')
        plt.close()

        total   = cm.values.sum()
        correct = sum(cm.loc[r, r] for r in cm.index if r in cm.columns)
        results['scores_summary'][assay] = {
            'total_cells':          int(total),
            'correct_assignments':  int(correct),
            'accuracy':             float(correct / total) if total > 0 else 0.0,
        }

    # ── Violin plots ──
    for ct in filtered_gene_sets:
        score_col = f'{ct}_score_Z'
        mod_col   = f'{ct}_module_score_Z'
        if score_col not in adata_subset.obs or mod_col not in adata_subset.obs:
            continue

        for assay in adata_subset.obs['orig.ident'].unique():
            sub_obs = adata_subset.obs[adata_subset.obs['orig.ident'] == assay]
            present_order = [l for l in label_order if l in sub_obs['cell_type_broad'].values]

            for anno_col in ['cell_type_broad', 'module_label_confident']:
                for y_col, y_label in [(score_col, 'Avg expr Z'), (mod_col, 'Module score Z')]:
                    plt.figure(figsize=(6, 5))
                    sns.violinplot(data=sub_obs, x=anno_col, y=y_col,
                                   palette=palette_dict, inner=None, order=present_order)
                    sns.stripplot(data=sub_obs, x=anno_col, y=y_col,
                                  color='black', size=2, alpha=0.6, order=present_order)
                    plt.title(f"{ct} — {assay} | {y_label} by {anno_col}")
                    plt.xticks(rotation=45)
                    plt.tight_layout()
                    tag  = 'avgExpr' if 'score_Z' in y_col else 'moduleScore'
                    path = os.path.join(set_dir, f"{assay}_{ct}_{tag}_{anno_col}_{output_suffix}.pdf")
                    plt.savefig(path, dpi=300, bbox_inches='tight')
                    plt.close()

    # ── Save AnnData ──
    for col in adata_subset.obs.columns:
        s = adata_subset.obs[col]
        if hasattr(s, 'cat'):
            adata_subset.obs[col] = s.astype(str)
        s = adata_subset.obs[col].astype(str).replace(
            {'Unknown': float('nan'), 'nan': float('nan'), 'None': float('nan')}
        )
        sn = pd.to_numeric(s, errors='coerce')
        adata_subset.obs[col] = s.fillna('Unknown').astype(str) if sn.isna().all() else sn

    save_path = os.path.join(set_dir, f'processed_data_{output_suffix}.h5ad')
    try:
        adata_subset.write_h5ad(save_path)
        print(f"  Saved AnnData: {save_path}")
    except Exception as e:
        print(f"  Could not save AnnData: {e}")

    # ── Summary JSON ──
    with open(os.path.join(set_dir, f'summary_{output_suffix}.json'), 'w') as f:
        json.dump({
            'marker_set_name':    marker_set_name,
            'cluster_map':        CLUSTER_TO_CELLTYPE,
            'cell_types':         list(filtered_gene_sets.keys()),
            'genes_per_celltype': {ct: len(g) for ct, g in filtered_gene_sets.items()},
            'confidence_threshold': CONFIDENCE_THRESHOLD,
            'scores_summary':     results['scores_summary'],
        }, f, indent=2)

    print(f"Done: {marker_set_name}  →  {set_dir}")
    return results

# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
all_results = {}
try:
    all_results['ttest'] = process_marker_set(
        'ttest', ttest_markers_by_ct, 'ttest_markers'
    )
except Exception as e:
    print(f"Error processing ttest markers: {e}")

# ─────────────────────────────────────────────
# ACCURACY SUMMARY
# ─────────────────────────────────────────────
if all_results:
    rows = []
    for ms_name, res in all_results.items():
        for assay, s in res['scores_summary'].items():
            rows.append({'marker_set': ms_name, 'assay': assay, **s})
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(os.path.join(score_dir, 'accuracy_summary.csv'), index=False)
    print("\nAccuracy summary:")
    print(summary_df[['marker_set', 'assay', 'accuracy', 'total_cells']].to_string(index=False))

print("\nDone!")

cell_type_broad distribution:
cell_type_broad
RGCs    9634
INs     4956
IGs     1330

Re-running t-test DE on reference cells with new cell_type_broad ...
  Reference subset: 14792 cells, groups: ['IGs', 'INs', 'RGCs']

Before removing duplicates:
cell_type
IGs     1115
INs      589
RGCs     403

After removing duplicates:
cell_type
IGs     1089
INs      567
RGCs     399

Marker sets: 3 cell types
  IGs: 1089 genes
  INs: 567 genes
  RGCs: 399 genes

PROCESSING: TTEST
  scNMT subset: 1128 cells
  IGs: 1089 genes
  INs: 567 genes
  RGCs: 399 genes
  Processing assay: scNMT_pool1_DCM
  Processing assay: scNMT_pool1_RNA
  Module score Z columns: ['IGs_module_score_Z', 'INs_module_score_Z', 'RGCs_module_score_Z']


/tmp/ipykernel_485362/311831497.py:235: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('orig.ident', group_keys=False)
/tmp/ipykernel_485362/311831497.py:236: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.apply(assign_module_label, axis=1))


  Confidence threshold: z >= 1.65


/tmp/ipykernel_485362/311831497.py:290: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=sub_obs, x=anno_col, y=y_col,
/tmp/ipykernel_485362/311831497.py:290: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=sub_obs, x=anno_col, y=y_col,
/tmp/ipykernel_485362/311831497.py:290: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=sub_obs, x=anno_col, y=y_col,
/tmp/ipykernel_485362/311831497.py:290: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hu

  Saved AnnData: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest/processed_data_ttest_markers.h5ad
Done: ttest  →  /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest

Accuracy summary:
marker_set           assay  accuracy  total_cells
     ttest scNMT_pool1_DCM  0.056790          405
     ttest scNMT_pool1_RNA  0.157676          723

Done!


In [8]:
ttest_df_unique = pd.read_csv("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest_df_unique_newlabels.csv")

In [9]:
ttest_df_unique

,gene,logfc_ttest,pvals_adj,cell_type
0,Fstl1,3.538600,1.382231e-303,IGs
1,Sparc,2.230833,6.860185e-283,IGs
2,Vim,1.475700,1.190468e-273,IGs
3,Lgals1,2.497915,4.470934e-255,IGs
4,Cald1,2.476768,4.587564e-219,IGs
...,...,...,...,...
2050,Aqp9,1.098003,7.164791e-05,RGCs
2051,Kcnt1,1.185933,8.028231e-05,RGCs
2052,Gm11696,1.074646,8.097981e-05,RGCs
2053,Herc6,1.024709,8.127356e-05,RGCs


In [19]:
adata.obs['cell_type_broad'].unique()

['INs', 'RGCs', 'IGs', 'Unknown']
Categories (4, object): ['RGCs' < 'INs' < 'IGs' < 'Unknown']

In [11]:
######################################################################
# GMM ANALYSIS — runs after gene_scoring_zscore.py
# Uses: adata, adata_subset (from process_marker_set),
#       ttest_markers_by_ct, score_dir
# Cell type labels: RGCs, INs, IGs, Unknown
######################################################################

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from scipy.stats import zscore
import scanpy as sc
import os
import pickle

# ─────────────────────────────────────────────
# PARAMETERS  (match gene_scoring_zscore.py)
# ─────────────────────────────────────────────
posterior_threshold  = 0.95
assay_col            = 'orig.ident'
jb_anno_col          = 'cell_type_broad'
module_score_suffix  = '_module_score_Z'

label_order  = ['RGCs', 'INs', 'IGs']
palette_dict = {
    'RGCs':      '#1f77b4',
    'INs':       '#ff7f0e',
    'IGs':       '#2ca02c',
    'ambiguous': 'lightgray',
    'Unknown':   'lightgray',   # fallback for any residual Unknown values
}

# ─────────────────────────────────────────────
# REBUILD adata_ttest FROM adata
# (mirrors the z-score pipeline in gene_scoring_zscore.py)
# ─────────────────────────────────────────────
adata.X = adata.layers['lognorm']

subset_mask = (
    adata.obs['orig.ident'].str.contains('scNMT') &
    (adata.obs['Dox'].astype(str) != 'Minus')
)
adata_ttest = adata[np.array(subset_mask)].copy()
print(f"scNMT subset: {adata_ttest.n_obs} cells")

genes_in_data    = set(adata_ttest.var_names)
filtered_gene_sets = {
    ct: [g for g in genes if g in genes_in_data]
    for ct, genes in ttest_markers_by_ct.items()
}
for ct, g in filtered_gene_sets.items():
    print(f"  {ct}: {len(g)} genes")

# Initialise score columns
for ct in filtered_gene_sets:
    adata_ttest.obs[f'{ct}_score_raw']        = np.nan
    adata_ttest.obs[f'{ct}_module_score_raw'] = np.nan

# Compute scores within each assay
for assay in adata_ttest.obs['orig.ident'].unique():
    print(f"  Scoring assay: {assay}")
    sub = adata_ttest[adata_ttest.obs['orig.ident'] == assay].copy()
    for ct, genes in filtered_gene_sets.items():
        if not genes:
            continue
        sub.obs[f'{ct}_score_raw'] = np.asarray(sub[:, genes].X.mean(axis=1)).ravel()
        sc.tl.score_genes(sub, gene_list=genes,
                          score_name=f'{ct}_module_score_raw', use_raw=False)
        adata_ttest.obs.loc[sub.obs_names, f'{ct}_score_raw']        = sub.obs[f'{ct}_score_raw']
        adata_ttest.obs.loc[sub.obs_names, f'{ct}_module_score_raw'] = sub.obs[f'{ct}_module_score_raw']

# Z-score per assay
for ct in filtered_gene_sets:
    for suffix in ['_score_raw', '_module_score_raw']:
        raw_col = f'{ct}{suffix}'
        z_col   = raw_col.replace('_raw', '_Z')
        adata_ttest.obs[z_col] = np.nan
        for assay in adata_ttest.obs['orig.ident'].unique():
            mask = adata_ttest.obs['orig.ident'] == assay
            adata_ttest.obs.loc[mask, z_col] = zscore(
                adata_ttest.obs.loc[mask, raw_col], nan_policy='omit'
            )

# ─────────────────────────────────────────────
# GMM FUNCTION
# ─────────────────────────────────────────────
def run_gmm_analysis(adata_subset, marker_set_name, output_dir,
                     posterior_threshold=0.95):

    print(f"\nRUNNING GMM FOR: {marker_set_name.upper()}")
    print("=" * 60)

    # Clean up any pre-existing GMM columns
    drop_cols = [c for c in adata_subset.obs.columns
                 if 'posterior' in c or c.endswith('_label')
                 or c == 'module_max_label']
    if drop_cols:
        adata_subset.obs = adata_subset.obs.drop(columns=drop_cols)

    module_cols = [c for c in adata_subset.obs.columns
                   if c.endswith(module_score_suffix)]
    print(f"Module score columns: {module_cols}")

    def fit_mixture_posterior(scores, n_components=2, random_state=0):
        x    = scores.values.reshape(-1, 1)
        gmm  = GaussianMixture(n_components=n_components,
                               random_state=random_state)
        gmm.fit(x)
        active  = np.argmax(gmm.means_.ravel())
        post    = gmm.predict_proba(x)[:, active]
        return post, gmm

    assay_posterior_cols = {}
    assay_label_cols     = {}
    all_assay_results    = {}

    # ── Per-assay GMM ──
    for assay in adata_subset.obs[assay_col].unique():
        print(f"\n  Assay: {assay}")
        mask       = adata_subset.obs[assay_col] == assay
        assay_data = adata_subset.obs.loc[mask]
        assay_posterior_cols[assay] = []
        assay_label_cols[assay]     = []
        all_assay_results[assay]    = {}

        for module_col in module_cols:
            module_name = module_col.replace(module_score_suffix, '')
            scores      = assay_data[module_col]
            post, gmm   = fit_mixture_posterior(scores)

            post_col  = f"{assay}_{module_col}_posterior"
            label_col = f"{assay}_{module_col}_label"

            adata_subset.obs.loc[mask, post_col]  = post
            adata_subset.obs.loc[mask, label_col] = np.where(
                post >= posterior_threshold, module_name, 'ambiguous'
            )

            assay_posterior_cols[assay].append(post_col)
            assay_label_cols[assay].append(label_col)

            n_above = (post >= posterior_threshold).sum()
            all_assay_results[assay][module_name] = {
                'gmm_means':         gmm.means_.ravel(),
                'mean_posterior':    post.mean(),
                'n_above_threshold': int(n_above),
                'pct_above_threshold': float(n_above / len(post) * 100),
            }
            print(f"    {module_name}: means={gmm.means_.ravel().round(3)}, "
                  f"cells above threshold={n_above} "
                  f"({all_assay_results[assay][module_name]['pct_above_threshold']:.1f}%)")

    # ── Assign max-posterior label per cell per assay ──
    for assay in adata_subset.obs[assay_col].unique():
        mask           = adata_subset.obs[assay_col] == assay
        post_cols_assay = assay_posterior_cols[assay]

        def assign_max(row):
            vals      = row[post_cols_assay]
            max_col   = vals.idxmax()
            max_prob  = vals.max()
            if max_prob >= posterior_threshold:
                return max_col.replace(f"{assay}_", "").replace(
                    f"{module_score_suffix}_posterior", "")
            return 'ambiguous'

        adata_subset.obs.loc[mask, f'{assay}_module_max_label'] = (
            adata_subset.obs.loc[mask].apply(assign_max, axis=1)
        )
        print(f"\n  {assay} label distribution:")
        print(adata_subset.obs.loc[mask, f'{assay}_module_max_label']
              .value_counts().to_string())

    # ── Combined label ──
    adata_subset.obs['module_max_label'] = 'ambiguous'
    for assay in adata_subset.obs[assay_col].unique():
        mask = adata_subset.obs[assay_col] == assay
        adata_subset.obs.loc[mask, 'module_max_label'] = (
            adata_subset.obs.loc[mask, f'{assay}_module_max_label']
        )

    print(f"\nCombined label distribution:")
    print(adata_subset.obs['module_max_label'].value_counts().to_string())

    # ── Confusion matrices ──
    accuracy_results = {}
    for assay in adata_subset.obs[assay_col].unique():
        mask = adata_subset.obs[assay_col] == assay
        cm   = pd.crosstab(
            adata_subset.obs.loc[mask, jb_anno_col],
            adata_subset.obs.loc[mask, 'module_max_label'],
            rownames=[jb_anno_col],
            colnames=['module_max_label'],
        )
        total   = cm.values.sum()
        correct = sum(cm.loc[r, r] for r in cm.index if r in cm.columns)
        acc     = correct / total * 100 if total > 0 else 0
        accuracy_results[assay] = acc
        print(f"\n  {assay} accuracy: {acc:.1f}%  ({correct}/{total})")

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    cbar_kws={'label': 'Cell count'})
        plt.title(f"{marker_set_name.upper()} | {assay}: GMM vs {jb_anno_col}\n"
                  f"Accuracy {acc:.1f}%  threshold={posterior_threshold}")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir,
                    f'{assay}_GMM_vs_jbANNO_{posterior_threshold}.pdf'),
                    dpi=300, bbox_inches='tight')
        plt.close()
        cm.to_csv(os.path.join(output_dir,
                  f'{assay}_confusion_matrix_{posterior_threshold}.csv'))

    # ── Posterior distribution violins ──
    # GMM uncertainty → 'ambiguous'; cell type labels from label_order
    gmm_order  = label_order + ['ambiguous']
    plot_order = [l for l in gmm_order
                  if l in adata_subset.obs['module_max_label'].values]

    for assay in adata_subset.obs[assay_col].unique():
        mask = adata_subset.obs[assay_col] == assay
        for post_col in assay_posterior_cols[assay]:
            module_name = (post_col.replace(f"{assay}_", "")
                                   .replace(f"{module_score_suffix}_posterior", ""))
            plt.figure(figsize=(8, 5))
            sns.violinplot(data=adata_subset.obs.loc[mask],
                           x='module_max_label', y=post_col,
                           hue='module_max_label', palette=palette_dict,
                           order=plot_order, inner='box', legend=False)
            sns.stripplot(data=adata_subset.obs.loc[mask],
                          x='module_max_label', y=post_col,
                          color='black', size=2, alpha=0.3, order=plot_order)
            plt.axhline(posterior_threshold, color='red', linestyle='--',
                        alpha=0.7, label=f'Threshold ({posterior_threshold})')
            plt.title(f"{marker_set_name.upper()} | {assay}: {module_name} posterior")
            plt.xlabel('Cell type'); plt.ylabel('Posterior probability')
            plt.xticks(rotation=45); plt.legend(); plt.tight_layout()
            plt.savefig(os.path.join(output_dir,
                        f'{assay}_{module_name}_posteriors_{posterior_threshold}.pdf'),
                        dpi=300, bbox_inches='tight')
            plt.close()

    # ── Module score Z violins ──
    z_cols = [c for c in adata_subset.obs.columns
              if c.endswith(module_score_suffix)]
    for assay in adata_subset.obs[assay_col].unique():
        mask = adata_subset.obs[assay_col] == assay
        for z_col in z_cols:
            module_name = z_col.replace(module_score_suffix, '')
            plt.figure(figsize=(8, 5))
            sns.violinplot(data=adata_subset.obs.loc[mask],
                           x='module_max_label', y=z_col,
                           hue='module_max_label', palette=palette_dict,
                           order=plot_order, inner='box', legend=False)
            sns.stripplot(data=adata_subset.obs.loc[mask],
                          x='module_max_label', y=z_col,
                          color='black', size=2, alpha=0.3, order=plot_order)
            plt.axhline(0,    color='gray', linestyle='--', alpha=0.5, label='Z=0')
            plt.axhline(1.65, color='red',  linestyle='--', alpha=0.7, label='Z=1.65')
            plt.title(f"{marker_set_name.upper()} | {assay}: {module_name} module score Z")
            plt.xlabel('Cell type'); plt.ylabel('Module score (Z)');
            plt.xticks(rotation=45); plt.legend(); plt.tight_layout()
            plt.savefig(os.path.join(output_dir,
                        f'{assay}_{module_name}_module_score_Z.pdf'),
                        dpi=300, bbox_inches='tight')
            plt.close()

    # ── Summary CSV ──
    summary_rows = []
    for assay in adata_subset.obs[assay_col].unique():
        mask = adata_subset.obs[assay_col] == assay
        for label in adata_subset.obs.loc[mask, 'module_max_label'].unique():
            lmask = adata_subset.obs.loc[mask, 'module_max_label'] == label
            for post_col in assay_posterior_cols[assay]:
                mn = (post_col.replace(f"{assay}_", "")
                              .replace(f"{module_score_suffix}_posterior", ""))
                vals = adata_subset.obs.loc[mask].loc[lmask, post_col]
                summary_rows.append({
                    'marker_set': marker_set_name, 'assay': assay,
                    'module_max_label': label, 'module': mn,
                    'mean_posterior': vals.mean(), 'std_posterior': vals.std(),
                    'count': len(vals), 'median_posterior': vals.median(),
                })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(output_dir,
                      f'GMM_summary_{posterior_threshold}.csv'), index=False)
    print(f"\nSummary saved: GMM_summary_{posterior_threshold}.csv")

    return {
        'adata_subset':        adata_subset,
        'summary_df':          summary_df,
        'accuracy_results':    accuracy_results,
        'assay_posterior_cols': assay_posterior_cols,
        'all_assay_results':   all_assay_results,
    }

# ─────────────────────────────────────────────
# RUN GMM
# ─────────────────────────────────────────────
ttest_dir = os.path.join(score_dir, 'ttest_gmm')
os.makedirs(ttest_dir, exist_ok=True)

ttest_gmm_results = run_gmm_analysis(
    adata_ttest, 'ttest', ttest_dir, posterior_threshold
)

# ── Save AnnData ──
for col in adata_ttest.obs.columns:
    s = adata_ttest.obs[col]
    if hasattr(s, 'cat'):
        s = s.astype(str)
    s = s.astype(str).replace({'Unknown': float('nan'), 'nan': float('nan'),
                                'None': float('nan'), '<NA>': float('nan')})
    sn = pd.to_numeric(s, errors='coerce')
    adata_ttest.obs[col] = s.fillna('Unknown').astype(str) if sn.isna().all() else sn

try:
    adata_ttest.write_h5ad(os.path.join(ttest_dir, 'gmm_processed_data.h5ad'))
    print("Saved: gmm_processed_data.h5ad")
except Exception as e:
    print(f"Could not save AnnData: {e}")

# ── Save pickle ──
try:
    with open(os.path.join(score_dir, 'all_gmm_results.pkl'), 'wb') as f:
        pickle.dump({'ttest': ttest_gmm_results}, f)
    print("Saved: all_gmm_results.pkl")
except Exception as e:
    print(f"Could not save pickle: {e}")

# ── Accuracy summary ──
print(f"\n{'='*60}")
print("GMM ACCURACY SUMMARY")
print(f"{'='*60}")
for assay, acc in ttest_gmm_results['accuracy_results'].items():
    print(f"  {assay}: {acc:.1f}%")
print(f"\nAll outputs in: {ttest_dir}")

scNMT subset: 1128 cells
  IGs: 1089 genes
  INs: 567 genes
  RGCs: 399 genes
  Scoring assay: scNMT_pool1_DCM
  Scoring assay: scNMT_pool1_RNA

RUNNING GMM FOR: TTEST
Module score columns: ['IGs_module_score_Z', 'INs_module_score_Z', 'RGCs_module_score_Z']

  Assay: scNMT_pool1_DCM
    IGs: means=[ 0.735 -0.611], cells above threshold=49 (12.1%)
    INs: means=[-0.685  0.796], cells above threshold=82 (20.2%)
    RGCs: means=[ 0.7   -0.491], cells above threshold=8 (2.0%)

  Assay: scNMT_pool1_RNA
    IGs: means=[-0.677  0.573], cells above threshold=134 (18.5%)
    INs: means=[-0.821  0.572], cells above threshold=278 (38.5%)
    RGCs: means=[ 0.925 -0.375], cells above threshold=52 (7.2%)

  scNMT_pool1_DCM label distribution:
scNMT_pool1_DCM_module_max_label
ambiguous    292
INs           74
IGs           35
RGCs           4

  scNMT_pool1_RNA label distribution:
scNMT_pool1_RNA_module_max_label
ambiguous    329
INs          269
IGs           75
RGCs          50

Combined label dis

/tmp/ipykernel_485362/3116811274.py:314: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.astype(str).replace({'Unknown': float('nan'), 'nan': float('nan'),
/tmp/ipykernel_485362/3116811274.py:314: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.astype(str).replace({'Unknown': float('nan'), 'nan': float('nan'),
/tmp/ipykernel_485362/3116811274.py:314: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=

Saved: gmm_processed_data.h5ad
Saved: all_gmm_results.pkl

GMM ACCURACY SUMMARY
  scNMT_pool1_DCM: 18.5%
  scNMT_pool1_RNA: 26.3%

All outputs in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest_gmm


In [12]:
######################################################################
# GMM FDR DIAGNOSIS + RE-RUN + POSTERIOR HEATMAPS
# Step 1: Sweep posterior thresholds to find the 4.99% FDR threshold
# Step 2: Re-run GMM at that threshold
# Step 3: Plot posterior heatmaps per assay and combined
#
# Requires in memory (from gmm_analysis.py):
#   adata_ttest, ttest_gmm_results, score_dir,
#   filtered_gene_sets, ttest_markers_by_ct,
#   label_order, palette_dict, assay_col
######################################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.mixture import GaussianMixture
from scipy.stats import zscore
from scipy.interpolate import interp1d
import scanpy as sc

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
fdr_dir = os.path.join(score_dir, 'fdr_diagnosis')
gmm_opt_dir = os.path.join(score_dir, 'ttest_gmm_optimal')
os.makedirs(fdr_dir, exist_ok=True)
os.makedirs(gmm_opt_dir, exist_ok=True)

# Cell types from the GMM run
cell_types = list(filtered_gene_sets.keys())
module_score_suffix = '_module_score_Z'

# ─────────────────────────────────────────────
# STEP 1: EXTRACT POSTERIOR COLUMNS FROM DCM CELLS
# ─────────────────────────────────────────────
print("=" * 80)
print("STEP 1: FDR THRESHOLD DIAGNOSIS")
print("=" * 80)

# Use DCM cells from adata_ttest
dcm_mask  = adata_ttest.obs['orig.ident'].str.contains('DCM', case=False, na=False)
adata_dcm = adata_ttest[dcm_mask].copy()
print(f"DCM cells: {adata_dcm.n_obs}")

# Find per-assay posterior columns created by run_gmm_analysis
# Format: {assay}_{ct}_module_score_Z_posterior
posterior_cols = [
    c for c in adata_ttest.obs.columns
    if '_posterior' in c and 'max' not in c
]
print(f"Posterior columns found: {len(posterior_cols)}")
for c in posterior_cols:
    print(f"  {c}")

if not posterior_cols:
    raise ValueError(
        "No posterior columns found — make sure gmm_analysis.py ran successfully "
        "and adata_ttest is still in memory."
    )

# For diagnosis, use per-cell max posterior across all cell types
# (same approach as the reference FDR code)
# Build a clean posterior_df: one column per cell type, aggregated across assays
# by taking the value from the cell's own assay posterior column
ct_posterior_frames = []
for ct in cell_types:
    # Find all posterior columns for this cell type
    ct_post_cols = [c for c in posterior_cols if f'_{ct}_module_score_Z_posterior' in c]
    if not ct_post_cols:
        continue
    # Each cell gets its value from its own assay's column
    series = pd.Series(np.nan, index=adata_ttest.obs_names, name=ct)
    for col in ct_post_cols:
        # Extract assay name from column prefix
        assay_prefix = col.split(f'_{ct}_module_score_Z_posterior')[0]
        assay_mask   = adata_ttest.obs['orig.ident'] == assay_prefix
        series.loc[assay_mask & series.isna()] = adata_ttest.obs.loc[assay_mask, col].values
    ct_posterior_frames.append(series)

posterior_df  = pd.concat(ct_posterior_frames, axis=1)
# Restrict to DCM cells
posterior_df  = posterior_df.loc[adata_dcm.obs_names]
max_posteriors = posterior_df.max(axis=1)
max_classes    = posterior_df.idxmax(axis=1)

print(f"\nMax posterior stats:")
print(f"  mean : {max_posteriors.mean():.3f}")
print(f"  min  : {max_posteriors.min():.3f}")
print(f"  max  : {max_posteriors.max():.3f}")

# ─────────────────────────────────────────────
# STEP 2: SWEEP THRESHOLDS → FIND 4.99% FDR
# ─────────────────────────────────────────────
thresh_sweep  = np.arange(0.50, 1.0, 0.001)
sweep_results = []
for t in thresh_sweep:
    mask = max_posteriors >= t
    n    = mask.sum()
    if n > 0:
        sweep_results.append({
            'threshold':      t,
            'n_calls':        int(n),
            'expected_error': float((1 - max_posteriors[mask]).mean() * 100),
        })

sweep_df = pd.DataFrame(sweep_results)

# Interpolate to find exact 4.99% threshold
interp_func = interp1d(
    sweep_df['expected_error'].values[::-1],
    sweep_df['threshold'].values[::-1],
    kind='linear', bounds_error=False, fill_value='extrapolate'
)
threshold_499 = float(interp_func(4.99))
threshold_499 = np.clip(threshold_499, 0.5, 0.999)

n_calls_499     = int((max_posteriors >= threshold_499).sum())
actual_error_499 = float(
    (1 - max_posteriors[max_posteriors >= threshold_499]).mean() * 100
)

print(f"\n*** 4.99% FDR threshold: {threshold_499:.4f} ***")
print(f"    Cells passing : {n_calls_499} ({n_calls_499/len(max_posteriors)*100:.1f}%)")
print(f"    Actual error  : {actual_error_499:.2f}%")
print(f"\nThreshold comparison:")
print(f"{'Threshold':<14} {'N cells':<12} {'% retained':<14} {'Exp. error %'}")
print("-" * 54)
for t in sorted({threshold_499, 0.90, 0.95, 0.99}):
    mask = max_posteriors >= t
    n    = mask.sum()
    err  = (1 - max_posteriors[mask]).mean() * 100 if n > 0 else 0
    flag = '  <-- 4.99% FDR' if abs(t - threshold_499) < 0.0011 else ''
    print(f"{t:<14.4f} {n:<12} {n/len(max_posteriors)*100:<14.1f} {err:.2f}%{flag}")

# ─────────────────────────────────────────────
# STEP 3: DIAGNOSTIC PLOTS
# ─────────────────────────────────────────────
cell_type_colors = dict(zip(
    cell_types,
    sns.color_palette('tab10', len(cell_types))
))

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Panel A: histogram of max posteriors
axes[0, 0].hist(max_posteriors, bins=50, color='steelblue',
                edgecolor='black', alpha=0.7)
axes[0, 0].axvline(threshold_499, color='green', lw=2.5,
                   label=f'4.99% FDR (t={threshold_499:.3f})')
axes[0, 0].axvline(0.95, color='red',    lw=1.5, linestyle='--', label='t=0.95')
axes[0, 0].axvline(0.90, color='orange', lw=1.5, linestyle='--', label='t=0.90')
axes[0, 0].set_xlabel('Max posterior probability')
axes[0, 0].set_ylabel('Cell count')
axes[0, 0].set_title('Distribution of max posteriors (DCM cells)')
axes[0, 0].legend(fontsize=8); axes[0, 0].grid(axis='y', alpha=0.3)

# Panel B: boxplot per cell type
bp_data, bp_labels, bp_cols = [], [], []
for ct in cell_types:
    ct_mask = max_classes == ct
    if ct_mask.sum() > 0:
        bp_data.append(max_posteriors[ct_mask].values)
        bp_labels.append(f"{ct}\n(n={ct_mask.sum()})")
        bp_cols.append(cell_type_colors[ct])
bp = axes[0, 1].boxplot(bp_data, labels=bp_labels, patch_artist=True)
for patch, col in zip(bp['boxes'], bp_cols):
    patch.set_facecolor(col); patch.set_alpha(0.7)
axes[0, 1].axhline(threshold_499, color='green', lw=2.5,
                   label=f'4.99% FDR (t={threshold_499:.3f})')
axes[0, 1].axhline(0.95, color='red', lw=1.5, linestyle='--', label='t=0.95')
axes[0, 1].set_ylabel('Max posterior'); axes[0, 1].set_xlabel('Cell type')
axes[0, 1].set_title('Max posterior per cell type')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].legend(fontsize=8); axes[0, 1].grid(axis='y', alpha=0.3)

# Panel C: error rate vs threshold trade-off
ax_twin = axes[1, 0].twinx()
axes[1, 0].plot(sweep_df['threshold'], sweep_df['expected_error'],
                color='steelblue', lw=2, label='Expected error %')
ax_twin.plot(sweep_df['threshold'], sweep_df['n_calls'],
             color='orange', lw=2, linestyle='--', label='N calls')
axes[1, 0].axhline(5,   color='red',   lw=1.5, linestyle=':', label='5% FDR ref')
axes[1, 0].axvline(threshold_499, color='green', lw=2.5,
                   label=f'4.99% FDR (t={threshold_499:.3f})')
axes[1, 0].plot(threshold_499, 4.99, 'o', color='green', ms=8, zorder=5)
axes[1, 0].set_xlabel('Posterior threshold')
axes[1, 0].set_ylabel('Expected error %', color='steelblue')
ax_twin.set_ylabel('N calls', color='orange')
axes[1, 0].set_title('Error rate vs threshold')
axes[1, 0].set_ylim(0, 10)
l1, lb1 = axes[1, 0].get_legend_handles_labels()
l2, lb2 = ax_twin.get_legend_handles_labels()
axes[1, 0].legend(l1 + l2, lb1 + lb2, fontsize=7, loc='upper left')
axes[1, 0].grid(axis='y', alpha=0.3)

# Panel D: expected error per posterior bin
cal = pd.DataFrame({'posterior': max_posteriors})
cal['bin'] = pd.cut(max_posteriors, bins=20)
cal_agg = (cal.groupby('bin')['posterior']
              .agg(mean_post='mean',
                   expected_error=lambda x: (1 - x).mean(),
                   n_cells='count')
              .reset_index())
axes[1, 1].bar(range(len(cal_agg)), cal_agg['expected_error'] * 100,
               color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 1].axhline(4.99, color='green', lw=2.5, label=f'4.99% FDR')
axes[1, 1].axhline(5,    color='red',   lw=1.5, linestyle='--', label='5% ref')
for i, row in cal_agg.iterrows():
    if row['n_cells'] > 0:
        axes[1, 1].text(i, row['expected_error'] * 100 + 0.3,
                        f"n={int(row['n_cells'])}", ha='center', fontsize=6)
axes[1, 1].set_xticks(range(len(cal_agg)))
axes[1, 1].set_xticklabels(
    [f"{b.left:.2f}" for b in cal_agg['bin']], rotation=45, ha='right', fontsize=6
)
axes[1, 1].set_xlabel('Posterior bin (left edge)')
axes[1, 1].set_ylabel('Expected error %')
axes[1, 1].set_title('Expected error per posterior bin')
axes[1, 1].legend(fontsize=8); axes[1, 1].grid(axis='y', alpha=0.3)

plt.suptitle('Posterior probability diagnostics (DCM cells)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(fdr_dir, 'posterior_diagnostics.pdf'), dpi=300, bbox_inches='tight')
plt.close()
print("Saved: posterior_diagnostics.pdf")

# Per cell type error curves
fig, axes = plt.subplots(2, 1, figsize=(14, 12))
for ct in cell_types:
    ct_mask = max_classes == ct
    if ct_mask.sum() == 0:
        continue
    ct_post = max_posteriors[ct_mask]
    ct_errs, ct_ts = [], []
    for t in thresh_sweep:
        m = ct_post >= t
        if m.sum() > 0:
            ct_errs.append((1 - ct_post[m]).mean() * 100)
            ct_ts.append(t)
    axes[0].plot(ct_ts, ct_errs, lw=2,
                 color=cell_type_colors[ct], label=ct)

axes[0].axhline(4.99, color='green', lw=2.5, linestyle='-', label='4.99% target', alpha=0.7)
axes[0].axvline(threshold_499, color='green', lw=1.5, linestyle=':', alpha=0.5)
axes[0].axvline(0.95, color='red', lw=1.5, linestyle='--', alpha=0.5, label='t=0.95')
axes[0].set_ylim(0, 15)
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Expected error %')
axes[0].set_title('Error rate per cell type')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[0].grid(alpha=0.3)

key_thresholds = [threshold_499, 0.90, 0.95, 0.99]
key_labels     = [f'{threshold_499:.3f}\n(4.99%)', '0.90', '0.95', '0.99']
ct_list        = [ct for ct in cell_types if (max_classes == ct).sum() > 0]
x_pos = np.arange(len(ct_list))
bw    = 0.2
for i, (t, lbl) in enumerate(zip(key_thresholds, key_labels)):
    vals  = [(max_posteriors[max_classes == ct] >= t).sum() for ct in ct_list]
    color = '#2ecc71' if abs(t - threshold_499) < 0.0011 else plt.cm.viridis(i / len(key_thresholds))
    axes[1].bar(x_pos + i * bw, vals, bw, label=lbl,
                color=color, edgecolor='black', lw=0.5, alpha=0.8)
axes[1].set_xticks(x_pos + bw * 1.5)
axes[1].set_xticklabels(ct_list, rotation=45, ha='right')
axes[1].set_ylabel('Cells passing threshold')
axes[1].set_xlabel('Cell type')
axes[1].set_title('Cells retained per cell type at each threshold')
axes[1].legend(title='Threshold', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fdr_dir, 'celltype_error_breakdown.pdf'), dpi=300, bbox_inches='tight')
plt.close()
print("Saved: celltype_error_breakdown.pdf")

# Save CSVs
sweep_df.to_csv(os.path.join(fdr_dir, 'threshold_sweep.csv'), index=False)
pd.DataFrame([
    {'threshold': t,
     'n_calls': (max_posteriors >= t).sum(),
     'n_calls_pct': (max_posteriors >= t).sum() / len(max_posteriors) * 100,
     'expected_error_pct': (1 - max_posteriors[max_posteriors >= t]).mean() * 100}
    for t in sorted({threshold_499, 0.90, 0.95, 0.99})
]).to_csv(os.path.join(fdr_dir, 'threshold_summary.csv'), index=False)
print("Saved CSVs")

# ─────────────────────────────────────────────
# STEP 4: RE-RUN GMM AT OPTIMAL THRESHOLD
# ─────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"STEP 4: RE-RUNNING GMM AT OPTIMAL THRESHOLD = {threshold_499:.4f}")
print(f"{'='*80}")

def fit_mixture_posterior(scores, n_components=2, random_state=0):
    x   = np.array(scores).reshape(-1, 1)
    gmm = GaussianMixture(n_components=n_components, random_state=random_state)
    gmm.fit(x)
    active = np.argmax(gmm.means_.ravel())
    return gmm.predict_proba(x)[:, active], gmm

# Clean up previous posterior columns
drop_cols = [c for c in adata_ttest.obs.columns
             if 'posterior' in c or c.endswith('_label') or c == 'module_max_label']
if drop_cols:
    adata_ttest.obs = adata_ttest.obs.drop(columns=drop_cols)

module_cols = [c for c in adata_ttest.obs.columns if c.endswith(module_score_suffix)]
assay_posterior_cols = {}

for assay in adata_ttest.obs['orig.ident'].unique():
    mask       = adata_ttest.obs['orig.ident'] == assay
    assay_data = adata_ttest.obs.loc[mask]
    assay_posterior_cols[assay] = []

    for mod_col in module_cols:
        ct_name  = mod_col.replace(module_score_suffix, '')
        post, _  = fit_mixture_posterior(assay_data[mod_col])
        post_col = f"{assay}_{mod_col}_posterior"
        adata_ttest.obs.loc[mask, post_col] = post
        adata_ttest.obs.loc[mask, f"{assay}_{mod_col}_label"] = np.where(
            post >= threshold_499, ct_name, 'ambiguous'
        )
        assay_posterior_cols[assay].append(post_col)

    # Assign max-posterior label per cell
    post_cols_assay = assay_posterior_cols[assay]
    def assign_max(row, cols=post_cols_assay, t=threshold_499, a=assay):
        vals    = row[cols]
        max_col = vals.idxmax()
        max_p   = vals.max()
        if max_p >= t:
            return max_col.replace(f"{a}_", "").replace(f"{module_score_suffix}_posterior", "")
        return 'ambiguous'
    adata_ttest.obs.loc[mask, f'{assay}_module_max_label'] = (
        adata_ttest.obs.loc[mask].apply(assign_max, axis=1)
    )

adata_ttest.obs['module_max_label'] = 'ambiguous'
for assay in adata_ttest.obs['orig.ident'].unique():
    mask = adata_ttest.obs['orig.ident'] == assay
    adata_ttest.obs.loc[mask, 'module_max_label'] = (
        adata_ttest.obs.loc[mask, f'{assay}_module_max_label']
    )

print("GMM label distribution at optimal threshold:")
print(adata_ttest.obs['module_max_label'].value_counts().to_string())

# Save re-run AnnData
try:
    adata_ttest.write_h5ad(os.path.join(gmm_opt_dir, 'gmm_optimal_threshold.h5ad'))
    print("Saved: gmm_optimal_threshold.h5ad")
except Exception as e:
    print(f"Could not save h5ad: {e}")

# ─────────────────────────────────────────────
# STEP 5: POSTERIOR HEATMAPS
# Adapted from reference posterior heatmap code.
# Columns: {assay}_{ct}_module_score_Z_posterior
# ─────────────────────────────────────────────
print(f"\n{'='*80}")
print("STEP 5: POSTERIOR HEATMAPS")
print(f"{'='*80}")

# Custom colormap: emphasise values above threshold_499
def make_cmap(threshold):
    colors_arr = plt.cm.RdBu_r(np.linspace(0, 1, 256))
    tidx = int(threshold * 256)
    for i in range(tidx, 256):
        colors_arr[i] = [0.4, 0.0, 0.1, 1.0]   # dark maroon
    return LinearSegmentedColormap.from_list('RdBu_r_highlight', colors_arr)

custom_cmap = make_cmap(threshold_499)

def plot_posterior_heatmap(adata_obj, assay_name, post_cols_map, save_path_prefix):
    """
    post_cols_map: dict {display_name: column_in_adata.obs}
    """
    mask       = adata_obj.obs['orig.ident'] == assay_name
    adata_sub  = adata_obj[mask]
    if adata_sub.n_obs == 0:
        print(f"  No cells for assay {assay_name}")
        return

    existing = {k: v for k, v in post_cols_map.items() if v in adata_sub.obs.columns}
    if not existing:
        print(f"  No posterior columns found for {assay_name}")
        return

    df = adata_sub.obs[list(existing.values())].copy()
    df.columns = list(existing.keys())
    df = df.dropna()
    if df.empty:
        print(f"  No valid cells after NaN drop for {assay_name}")
        return

    ordered_cols = [ct for ct in cell_types if ct in df.columns]
    df = df[ordered_cols]

    # Sort: group by max cell type, then descending within group
    df['_max_type'] = df.idxmax(axis=1)
    df_sorted = pd.concat([
        df[df['_max_type'] == ct].sort_values(by=ct, ascending=False)
        for ct in ordered_cols if ct in df['_max_type'].values
    ])
    cell_counts = df_sorted['_max_type'].value_counts().reindex(ordered_cols).fillna(0).astype(int)
    df_sorted   = df_sorted.drop(columns=['_max_type'])

    n_cells, n_ct = df_sorted.shape
    print(f"  {assay_name}: {n_cells} cells, {n_ct} cell types")
    for ct, cnt in cell_counts.items():
        print(f"    {ct}: {cnt}")

    fig, ax = plt.subplots(figsize=(12, max(3, n_ct * 0.6)))
    hm = sns.heatmap(
        df_sorted.T,
        cmap=custom_cmap, vmin=0, vmax=1,
        xticklabels=False, yticklabels=True,
        cbar_kws={'label': 'Posterior probability', 'shrink': 0.8},
        ax=ax, rasterized=True,
    )
    cbar = hm.collections[0].colorbar
    cbar.ax.axhline(threshold_499, color='black', lw=1.5, linestyle='--')
    cbar.ax.text(1.5, threshold_499, f'{threshold_499:.3f}',
                 va='center', ha='left', fontsize=8)

    ax.set_xlabel('Cells'); ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=10, rotation=0)

    # Vertical separators between cell type groups
    cumsum = 0
    for i, ct in enumerate(ordered_cols):
        cumsum += cell_counts.get(ct, 0)
        if i < len(ordered_cols) - 1:
            ax.axvline(x=cumsum, color='black', lw=1.5)

    # Cell type labels below
    cumsum = 0
    for ct in ordered_cols:
        cnt = cell_counts.get(ct, 0)
        if cnt > 0:
            ax.text(cumsum + cnt / 2, n_ct + 0.3, ct,
                    ha='center', va='top', fontsize=9, fontweight='bold')
            cumsum += cnt

    ax.set_title(f'Posterior probabilities — {assay_name} '
                 f'(threshold={threshold_499:.3f}, 4.99% FDR)',
                 fontsize=12, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'{save_path_prefix}_{assay_name}.pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path_prefix}_{assay_name}.pdf")


def plot_combined_heatmap(adata_obj, post_cols_map, save_path):
    existing = {k: v for k, v in post_cols_map.items() if v in adata_obj.obs.columns}
    if not existing:
        print("No posterior columns found for combined heatmap")
        return

    df = adata_obj.obs[list(existing.values()) + ['orig.ident']].copy()
    df.columns = list(existing.keys()) + ['assay']
    df = df.dropna(subset=list(existing.keys()))

    ordered_cols = [ct for ct in cell_types if ct in df.columns]
    df['_max_type'] = df[ordered_cols].idxmax(axis=1)

    df_sorted = pd.concat([
        pd.concat([
            group_df[group_df['_max_type'] == ct].sort_values(by=ct, ascending=False)
            for ct in ordered_cols
        ])
        for _, group_df in df.groupby('assay')
    ])
    df_sorted = df_sorted.drop(columns=['_max_type', 'assay'])[ordered_cols]

    n_ct = len(ordered_cols)
    fig, ax = plt.subplots(figsize=(14, max(3, n_ct * 0.6)))
    hm = sns.heatmap(
        df_sorted.T,
        cmap=custom_cmap, vmin=0, vmax=1,
        xticklabels=False, yticklabels=True,
        cbar_kws={'label': 'Posterior probability', 'shrink': 0.8},
        ax=ax, rasterized=True,
    )
    cbar = hm.collections[0].colorbar
    cbar.ax.axhline(threshold_499, color='black', lw=1.5, linestyle='--')
    cbar.ax.text(1.5, threshold_499, f'{threshold_499:.3f}',
                 va='center', ha='left', fontsize=8)
    ax.set_xlabel('Cells'); ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=10, rotation=0)
    ax.set_title(f'Posterior probabilities — all assays combined '
                 f'(threshold={threshold_499:.3f}, 4.99% FDR)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")


# Build post_cols_map: display name → actual column in adata_ttest.obs
# Format from GMM re-run: {assay}_{ct}_module_score_Z_posterior
# We want one column per cell type (using each cell's own assay column)
# so we create a merged column per cell type first
for ct in cell_types:
    merged_col = f'_posterior_merged_{ct}'
    adata_ttest.obs[merged_col] = np.nan
    for assay in adata_ttest.obs['orig.ident'].unique():
        assay_mask = adata_ttest.obs['orig.ident'] == assay
        src_col    = f"{assay}_{ct}{module_score_suffix}_posterior"
        if src_col in adata_ttest.obs.columns:
            adata_ttest.obs.loc[assay_mask, merged_col] = (
                adata_ttest.obs.loc[assay_mask, src_col].values
            )

post_cols_map = {ct: f'_posterior_merged_{ct}' for ct in cell_types
                 if f'_posterior_merged_{ct}' in adata_ttest.obs.columns}

print(f"\nPlotting posterior heatmaps for {len(post_cols_map)} cell types ...")

# Per-assay heatmaps
for assay in sorted(adata_ttest.obs['orig.ident'].unique()):
    plot_posterior_heatmap(
        adata_ttest, assay, post_cols_map,
        save_path_prefix=os.path.join(gmm_opt_dir, 'posterior_heatmap')
    )

# Combined heatmap
plot_combined_heatmap(
    adata_ttest, post_cols_map,
    save_path=os.path.join(gmm_opt_dir, 'posterior_heatmap_combined.pdf')
)

print(f"\n{'='*80}")
print(f"DONE")
print(f"  FDR diagnosis : {fdr_dir}")
print(f"  Optimal GMM   : {gmm_opt_dir}")
print(f"  4.99% FDR threshold: {threshold_499:.4f}")
print(f"{'='*80}")

STEP 1: FDR THRESHOLD DIAGNOSIS
DCM cells: 405
Posterior columns found: 6
  scNMT_pool1_DCM_IGs_module_score_Z_posterior
  scNMT_pool1_DCM_INs_module_score_Z_posterior
  scNMT_pool1_DCM_RGCs_module_score_Z_posterior
  scNMT_pool1_RNA_IGs_module_score_Z_posterior
  scNMT_pool1_RNA_INs_module_score_Z_posterior
  scNMT_pool1_RNA_RGCs_module_score_Z_posterior

Max posterior stats:
  mean : 0.659
  min  : 0.023
  max  : 1.000

*** 4.99% FDR threshold: 0.8370 ***
    Cells passing : 176 (43.5%)
    Actual error  : 4.99%

Threshold comparison:
Threshold      N cells      % retained     Exp. error %
------------------------------------------------------
0.8370         176          43.5           4.99%  <-- 4.99% FDR
0.9000         142          35.1           2.91%
0.9500         113          27.9           1.67%
0.9900         54           13.3           0.31%


/tmp/ipykernel_485362/3181764760.py:168: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[0, 1].boxplot(bp_data, labels=bp_labels, patch_artist=True)
/tmp/ipykernel_485362/3181764760.py:202: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cal_agg = (cal.groupby('bin')['posterior']


Saved: posterior_diagnostics.pdf
Saved: celltype_error_breakdown.pdf
Saved CSVs

STEP 4: RE-RUNNING GMM AT OPTIMAL THRESHOLD = 0.8370
GMM label distribution at optimal threshold:
module_max_label
ambiguous    466
INs          389
IGs          173
RGCs         100
Saved: gmm_optimal_threshold.h5ad

STEP 5: POSTERIOR HEATMAPS

Plotting posterior heatmaps for 3 cell types ...
  scNMT_pool1_DCM: 405 cells, 3 cell types
    IGs: 121
    INs: 151
    RGCs: 133
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest_gmm_optimal/posterior_heatmap_scNMT_pool1_DCM.pdf
  scNMT_pool1_RNA: 723 cells, 3 cell types
    IGs: 212
    INs: 378
    RGCs: 133
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest_gmm_op

/tmp/ipykernel_485362/3181764760.py:474: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for _, group_df in df.groupby('assay')


Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest_gmm_optimal/posterior_heatmap_combined.pdf

DONE
  FDR diagnosis : /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/fdr_diagnosis
  Optimal GMM   : /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest_gmm_optimal
  4.99% FDR threshold: 0.8370


In [13]:
##############################################
# KDE PLOTS: ACTIVE vs INACTIVE GMM GROUPS
# Requires in memory:
#   adata_ttest, threshold_499, score_dir,
#   cell_types, module_score_suffix
##############################################

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# OUTPUT DIRECTORY
# ─────────────────────────────────────────────
kde_dir = os.path.join(score_dir, 'ttest_gmm', 'kde_plots')
os.makedirs(kde_dir, exist_ok=True)

print("KDE PLOTS: ACTIVE vs INACTIVE GROUPS WITH THRESHOLD")
print("=" * 60)
print(f"Posterior threshold: {threshold_499:.4f}  (4.99% FDR)")

# ─────────────────────────────────────────────
# LOOP OVER ASSAYS AND CELL TYPES
# ─────────────────────────────────────────────
assays = sorted(adata_ttest.obs['orig.ident'].unique())
print(f"Assays: {assays}")

for assay in assays:
    print(f"\nAssay: {assay}")
    assay_mask = adata_ttest.obs['orig.ident'] == assay
    assay_data = adata_ttest.obs.loc[assay_mask]

    if assay_data.empty:
        continue

    for ct in cell_types:
        # ── Locate module score Z column ──
        score_col = None
        for candidate in [
            f'{ct}{module_score_suffix}',           # plain: RGCs_module_score_Z
            f'{assay}_{ct}{module_score_suffix}',   # assay-prefixed
        ]:
            if candidate in assay_data.columns:
                score_col = candidate
                break
        if score_col is None:
            for col in assay_data.columns:
                if ct in col and 'module_score_Z' in col and 'posterior' not in col:
                    score_col = col
                    break

        # ── Locate posterior column ──
        post_col = None
        candidate_post = f"{assay}_{ct}{module_score_suffix}_posterior"
        if candidate_post in assay_data.columns:
            post_col = candidate_post
        else:
            for col in assay_data.columns:
                if ct in col and 'posterior' in col and 'label' not in col:
                    post_col = col
                    break

        if score_col is None or post_col is None:
            print(f"  {ct}: columns not found (score={score_col}, post={post_col}) — skipping")
            continue

        scores     = assay_data[score_col].dropna()
        posteriors = assay_data[post_col].dropna()
        common_idx = scores.index.intersection(posteriors.index)
        scores     = scores.loc[common_idx]
        posteriors = posteriors.loc[common_idx]

        if len(scores) == 0:
            continue

        active_mask    = posteriors >= threshold_499
        active_scores  = scores[active_mask]
        inactive_scores = scores[~active_mask]

        print(f"  {ct}: {len(active_scores)} active, {len(inactive_scores)} inactive")

        if len(active_scores) < 3 or len(inactive_scores) < 3:
            print(f"    Not enough data for KDE — skipping")
            continue

        # ── KDE plot ──
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.kdeplot(data=inactive_scores, ax=ax,
                    label='Inactive', color='blue',
                    linewidth=2, fill=True, alpha=0.3)
        sns.kdeplot(data=active_scores, ax=ax,
                    label='Active', color='red',
                    linewidth=2, fill=True, alpha=0.3)

        # Approximate threshold: min score of the active group
        threshold_score = active_scores.min()
        ax.axvline(x=threshold_score, color='black', linestyle='--',
                   linewidth=2, alpha=0.7,
                   label=f'Threshold (posterior ≥ {threshold_499:.3f})')

        ax.set_xlabel(f'{ct} module score (Z)', fontsize=12)
        ax.set_ylabel('Density', fontsize=12)
        ax.set_title(
            f'{assay}: {ct}\n'
            f'Active n={len(active_scores)}, Inactive n={len(inactive_scores)}',
            fontsize=12
        )
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            os.path.join(kde_dir, f'{assay}_{ct}_KDE.pdf'),
            dpi=300, bbox_inches='tight'
        )
        plt.close()

        # ── Scatter: score vs posterior ──
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(scores, posteriors, alpha=0.5, s=10, color='steelblue')
        ax.axhline(y=threshold_499, color='red', linestyle='--',
                   linewidth=2, label=f'Threshold ({threshold_499:.3f})')
        ax.set_xlabel(f'{ct} module score (Z)', fontsize=12)
        ax.set_ylabel('Posterior probability', fontsize=12)
        ax.set_title(f'{assay}: {ct} — score vs posterior', fontsize=12)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            os.path.join(kde_dir, f'{assay}_{ct}_scatter.pdf'),
            dpi=300, bbox_inches='tight'
        )
        plt.close()

        print(f"    Saved: {assay}_{ct}_KDE.pdf  |  {assay}_{ct}_scatter.pdf")

print(f"\nDONE — all plots in: {kde_dir}")

KDE PLOTS: ACTIVE vs INACTIVE GROUPS WITH THRESHOLD
Posterior threshold: 0.8370  (4.99% FDR)
Assays: ['scNMT_pool1_DCM', 'scNMT_pool1_RNA']

Assay: scNMT_pool1_DCM
  IGs: 96 active, 309 inactive
    Saved: scNMT_pool1_DCM_IGs_KDE.pdf  |  scNMT_pool1_DCM_IGs_scatter.pdf
  INs: 118 active, 287 inactive
    Saved: scNMT_pool1_DCM_INs_KDE.pdf  |  scNMT_pool1_DCM_INs_scatter.pdf
  RGCs: 46 active, 359 inactive
    Saved: scNMT_pool1_DCM_RGCs_KDE.pdf  |  scNMT_pool1_DCM_RGCs_scatter.pdf

Assay: scNMT_pool1_RNA
  IGs: 243 active, 480 inactive
    Saved: scNMT_pool1_RNA_IGs_KDE.pdf  |  scNMT_pool1_RNA_IGs_scatter.pdf
  INs: 318 active, 405 inactive
    Saved: scNMT_pool1_RNA_INs_KDE.pdf  |  scNMT_pool1_RNA_INs_scatter.pdf
  RGCs: 81 active, 642 inactive
    Saved: scNMT_pool1_RNA_RGCs_KDE.pdf  |  scNMT_pool1_RNA_RGCs_scatter.pdf

DONE — all plots in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/met

In [20]:
##############################################
# SAVE GMM POSTERIORS + Z-SCORES INTO adata
# AND WRITE adata TO H5AD
#
# Requires in memory:
#   adata          — full integrated object
#   adata_ttest    — scNMT subset with GMM results
#   cell_types     — list of cell type names
#   module_score_suffix = '_module_score_Z'
#   score_dir, best_outdir
##############################################

import os
import numpy as np
import pandas as pd
import scanpy as sc

# ─────────────────────────────────────────────
# COLUMNS TO TRANSFER
# From adata_ttest.obs → adata.obs
# ─────────────────────────────────────────────

# Z-score columns (both avg-expr and module score)
zscore_cols = [
    c for c in adata_ttest.obs.columns
    if c.endswith('_score_Z') or c.endswith('_module_score_Z')
]

# Raw score columns (before z-scoring)
raw_cols = [
    c for c in adata_ttest.obs.columns
    if c.endswith('_score_raw') or c.endswith('_module_score_raw')
]

# Posterior columns (one per assay × cell type)
posterior_cols = [
    c for c in adata_ttest.obs.columns
    if '_posterior' in c and 'label' not in c
]

# GMM label columns
label_cols = [
    c for c in adata_ttest.obs.columns
    if c == 'module_max_label'
    or c.endswith('_module_max_label')
    or (c.endswith('_label') and 'module_score' in c)
]

# cell_type_broad assigned in gene_scoring_zscore.py
if 'cell_type_broad' in adata_ttest.obs.columns:
    label_cols = list(set(label_cols + ['cell_type_broad']))

all_cols_to_transfer = list(dict.fromkeys(
    zscore_cols + raw_cols + posterior_cols + label_cols
))

print(f"Columns to transfer from adata_ttest → adata:")
print(f"  Z-score cols       : {len(zscore_cols)}")
print(f"  Raw score cols     : {len(raw_cols)}")
print(f"  Posterior cols     : {len(posterior_cols)}")
print(f"  Label cols         : {len(label_cols)}")
print(f"  Total              : {len(all_cols_to_transfer)}")

# ─────────────────────────────────────────────
# TRANSFER — align on obs_names
# Cells not in adata_ttest get NaN / 'Unknown'
# ─────────────────────────────────────────────
common_cells = adata.obs_names.intersection(adata_ttest.obs_names)
print(f"\nCells in adata            : {adata.n_obs}")
print(f"Cells in adata_ttest      : {adata_ttest.n_obs}")
print(f"Common cells              : {len(common_cells)}")

for col in all_cols_to_transfer:
    src = adata_ttest.obs[col]

    # Decide fill value based on dtype
    if hasattr(src, 'cat') or src.dtype == object:
        adata.obs[col] = 'Unknown'
    else:
        adata.obs[col] = np.nan

    adata.obs.loc[common_cells, col] = adata_ttest.obs.loc[common_cells, col].values

print(f"\nTransferred {len(all_cols_to_transfer)} columns to adata.obs")

# Quick sanity check
print("\nSample of transferred columns in adata.obs:")
for col in all_cols_to_transfer[:6]:
    n_valid = adata.obs[col].notna().sum() if adata.obs[col].dtype != object else (adata.obs[col] != 'Unknown').sum()
    print(f"  {col}: {n_valid} valid values")

# ─────────────────────────────────────────────
# CLEAN .obs DTYPES BEFORE SAVING
# ─────────────────────────────────────────────
print("\nCleaning .obs dtypes ...")

for col in adata.obs.columns:
    s = adata.obs[col]
    # Strip CategoricalDtype
    if hasattr(s, 'cat'):
        s = s.astype(str)
    # Normalise sentinel strings → NaN
    s = s.astype(str).replace(
        {'Unknown': float('nan'), 'nan': float('nan'),
         'None': float('nan'), '<NA>': float('nan')}
    )
    s_num = pd.to_numeric(s, errors='coerce')
    if s_num.isna().all():
        # String column — fill NaN back to 'Unknown'
        adata.obs[col] = s.fillna('Unknown').astype(str)
    else:
        # Numeric column
        adata.obs[col] = s_num

# ─────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────
save_path = os.path.join(
    best_outdir, 'adata_harmony_best_with_gmm_scores.h5ad'
)
print(f"\nSaving adata → {save_path} ...")
adata.write_h5ad(save_path)
print("Saved.")

# ─────────────────────────────────────────────
# ALSO SAVE adata_ttest WITH ALL SCORES
# ─────────────────────────────────────────────
ttest_save_path = os.path.join(score_dir, 'ttest_gmm', 'adata_ttest_gmm_full.h5ad')
os.makedirs(os.path.dirname(ttest_save_path), exist_ok=True)

print(f"\nSaving adata_ttest → {ttest_save_path} ...")
# Clean adata_ttest obs too
for col in adata_ttest.obs.columns:
    s = adata_ttest.obs[col]
    if hasattr(s, 'cat'):
        s = s.astype(str)
    s = s.astype(str).replace(
        {'Unknown': float('nan'), 'nan': float('nan'),
         'None': float('nan'), '<NA>': float('nan')}
    )
    s_num = pd.to_numeric(s, errors='coerce')
    if s_num.isna().all():
        adata_ttest.obs[col] = s.fillna('Unknown').astype(str)
    else:
        adata_ttest.obs[col] = s_num

adata_ttest.write_h5ad(ttest_save_path)
print("Saved.")

# ─────────────────────────────────────────────
# RE-APPLY cell_type_broad FOR ALL CELLS
# The transfer above set 'Unknown' for reference cells because
# they were not in adata_ttest.  cell_type_broad is derived from
# leiden_annot_0.1 which covers ALL cells, so we re-map it here
# on the full adata to restore correct values everywhere.
# ─────────────────────────────────────────────
CLUSTER_TO_CELLTYPE = {
    '0': 'RGCs',
    '1': 'INs',
    '2': 'RGCs',
    '3': 'IGs',
}
LEIDEN_COL = 'leiden_annot_0.1'

if LEIDEN_COL in adata.obs.columns:
    adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype(str)
    mapped   = adata.obs[LEIDEN_COL].map(CLUSTER_TO_CELLTYPE)
    unmapped = mapped.isna()
    if unmapped.any():
        mapped_keys = {int(k): v for k, v in CLUSTER_TO_CELLTYPE.items() if k.isdigit()}
        def closest(label):
            try:
                n = int(label)
                return mapped_keys[min(mapped_keys.keys(), key=lambda x: abs(x - n))]
            except (ValueError, TypeError):
                return list(CLUSTER_TO_CELLTYPE.values())[0]
        mapped[unmapped] = adata.obs.loc[unmapped, LEIDEN_COL].apply(closest)
    adata.obs['cell_type_broad'] = mapped
    print("cell_type_broad re-applied on full adata:")
    print(adata.obs['cell_type_broad'].value_counts().to_string())
else:
    print(f"WARNING: '{LEIDEN_COL}' not found — cell_type_broad not updated")

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Full adata  : {save_path}")
print(f"scNMT subset: {ttest_save_path}")
print(f"Columns added to adata: {len(all_cols_to_transfer)}")
print(f"  - Z-scores   : {zscore_cols}")
print(f"  - Posteriors : {posterior_cols[:3]}{'...' if len(posterior_cols) > 3 else ''}")
print(f"  - Labels     : {label_cols}")

Columns to transfer from adata_ttest → adata:
  Z-score cols       : 6
  Raw score cols     : 6
  Posterior cols     : 9
  Label cols         : 10
  Total              : 31

Cells in adata            : 15920
Cells in adata_ttest      : 1128
Common cells              : 1128

Transferred 31 columns to adata.obs

Sample of transferred columns in adata.obs:
  IGs_score_Z: 1128 valid values
  IGs_module_score_Z: 1128 valid values
  INs_score_Z: 1128 valid values
  INs_module_score_Z: 1128 valid values
  RGCs_score_Z: 1128 valid values
  RGCs_module_score_Z: 1128 valid values

Cleaning .obs dtypes ...

Saving adata → /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/adata_harmony_best_with_gmm_scores.h5ad ...
Saved.

Saving adata_ttest → /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integ

/tmp/ipykernel_485362/3766099334.py:137: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.astype(str).replace(
/tmp/ipykernel_485362/3766099334.py:137: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.astype(str).replace(
/tmp/ipykernel_485362/3766099334.py:137: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting'

Saved.
cell_type_broad re-applied on full adata:
cell_type_broad
RGCs    9634
INs     4956
IGs     1330

SUMMARY
Full adata  : /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/adata_harmony_best_with_gmm_scores.h5ad
scNMT subset: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/ttest_gmm/adata_ttest_gmm_full.h5ad
Columns added to adata: 31
  - Z-scores   : ['IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z']
  - Posteriors : ['scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_module_score_Z_posterior', 'scNMT_pool1_DCM_RGCs_module_score_Z_posterior']...
  - Labels     : ['scNMT_pool1_RNA_INs_module_score_Z_label', 'scNMT_pool1_DCM_INs_module_score_Z_la

In [22]:
adata.obs['cell_type_broad'].unique()

array(['INs', 'RGCs', 'IGs'], dtype=object)

In [ ]:
#Paper plots

In [23]:
adata.obs['cell_type_broad']

X100_Plate_1           INs
X154_Plate_1          RGCs
X82_Plate_1            INs
X174_Plate_1          RGCs
X81_Plate_1            INs
                      ... 
TTTGTTGGTCATTACC-1    RGCs
TTTGTTGGTCATTGTC-1    RGCs
TTTGTTGGTCCCGGAA-1    RGCs
TTTGTTGGTCGCAAAC-1     IGs
TTTGTTGGTTGCTTCG-1    RGCs
Name: cell_type_broad, Length: 15920, dtype: object

In [24]:
##############################################
# UMAP PLOTS WITH CELL TYPE COLORS
# Colors from reference code:
#   RGCs → #003366  (dark blue)
#   INs  → #990000  (dark red)
#   IGs  → #00CCCC  (teal)
##############################################

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

# ─────────────────────────────────────────────
# PATHS & SETTINGS
# ─────────────────────────────────────────────
best_outdir = os.path.join(
    "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/"
    "Integration_multi_D2_D4/Python/Jona_notebook_27032026/"
    "integration/method_comparison/Harmony_best"
)
umap_dir = os.path.join(best_outdir, 'umaps_colored')
os.makedirs(umap_dir, exist_ok=True)

sc.settings.figdir = umap_dir
sc.set_figure_params(dpi=300, dpi_save=300, figsize=(6, 6), fontsize=8)

# ─────────────────────────────────────────────
# COLOR PALETTE
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs':      '#003366',
    'INs':       '#990000',
    'IGs':       '#00CCCC',
    'ambiguous': 'lightgray',
    'Unknown':   'lightgray',
}

desired_order = ['RGCs', 'INs', 'IGs', 'ambiguous', 'Unknown']

# ─────────────────────────────────────────────
# HELPER: ordered palette list for a categorical column
# ─────────────────────────────────────────────
def make_palette(series):
    cats = list(series.cat.categories) if hasattr(series, 'cat') else list(series.unique())
    # Sort by desired_order, then append anything else
    ordered = [c for c in desired_order if c in cats]
    ordered += [c for c in cats if c not in ordered]
    return ordered, [cell_type_colors.get(c, '#808080') for c in ordered]


# ─────────────────────────────────────────────
# PLOT 1: cell_type_broad coloured UMAP
# ─────────────────────────────────────────────
if 'cell_type_broad' in adata.obs.columns:
    # Ensure categorical with correct order
    cats, palette = make_palette(adata.obs['cell_type_broad'])
    adata.obs['cell_type_broad'] = pd.Categorical(
        adata.obs['cell_type_broad'], categories=cats, ordered=True
    )
    sc.pl.umap(
        adata,
        color='cell_type_broad',
        palette=palette,
        frameon=False,
        legend_loc='on data',
        show=False,
        save='_cell_type_broad.pdf',
    )
    print("Saved: _cell_type_broad.pdf")

# ─────────────────────────────────────────────
# PLOT 2: module_max_label (GMM label) coloured UMAP
# ─────────────────────────────────────────────
if 'module_max_label' in adata.obs.columns:
    cats, palette = make_palette(adata.obs['module_max_label'])
    adata.obs['module_max_label'] = pd.Categorical(
        adata.obs['module_max_label'], categories=cats, ordered=True
    )
    sc.pl.umap(
        adata,
        color='module_max_label',
        palette=palette,
        frameon=False,
        legend_loc='on data',
        show=False,
        save='_module_max_label.pdf',
    )
    print("Saved: _module_max_label.pdf")

# ─────────────────────────────────────────────
# PLOT 3: leiden_annot_0.1 coloured UMAP
# Strip numeric prefix (e.g. '1_RGCs' → 'RGCs') before plotting
# ─────────────────────────────────────────────
LEIDEN_COL = 'leiden_annot_0.1'

if LEIDEN_COL in adata.obs.columns:
    adata.obs['_leiden_clean'] = (
        adata.obs[LEIDEN_COL].astype(str)
        .apply(lambda x: re.sub(r'^\d+_', '', x))
    )
    cats, palette = make_palette(adata.obs['_leiden_clean'])
    adata.obs['_leiden_clean'] = pd.Categorical(
        adata.obs['_leiden_clean'], categories=cats, ordered=True
    )
    sc.pl.umap(
        adata,
        color='_leiden_clean',
        palette=palette,
        frameon=False,
        legend_loc='on data',
        show=False,
        save=f'_{LEIDEN_COL}_clean.pdf',
    )
    print(f"Saved: _{LEIDEN_COL}_clean.pdf")
    adata.obs.drop(columns=['_leiden_clean'], inplace=True)

# ─────────────────────────────────────────────
# PLOT 4: assay highlight
# ref → gray, scNMT_DCM → dark blue, scNMT_RNA → dark red
# ─────────────────────────────────────────────
if 'assay' in adata.obs.columns:
    umap_coords = adata.obsm['X_umap']

    fig, ax = plt.subplots(figsize=(6, 6))

    ref_mask = adata.obs['assay'].str.contains('ref', case=False, na=False) & \
               ~adata.obs['assay'].str.contains('scNMT', case=False, na=False)
    dcm_mask = adata.obs['assay'].str.contains('DCM', case=False, na=False)
    rna_mask = adata.obs['assay'].str.contains('RNA', case=False, na=False) & \
               adata.obs['assay'].str.contains('scNMT', case=False, na=False)

    ax.scatter(umap_coords[ref_mask, 0], umap_coords[ref_mask, 1],
               s=5, c='gray', alpha=0.2, label='ref', rasterized=True)
    ax.scatter(umap_coords[dcm_mask, 0], umap_coords[dcm_mask, 1],
               s=15, c='#003366', alpha=0.8, label='scNMT_DCM', rasterized=True)
    ax.scatter(umap_coords[rna_mask, 0], umap_coords[rna_mask, 1],
               s=15, c='#8B0000', alpha=0.8, label='scNMT_RNA', rasterized=True)

    ax.set_xlabel('UMAP1', fontsize=10)
    ax.set_ylabel('UMAP2', fontsize=10)
    ax.set_title('scNMT cells highlighted', fontsize=11)
    ax.legend(loc='upper right', fontsize=8, markerscale=2)
    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(umap_dir, 'umap_scNMT_highlight.pdf'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: umap_scNMT_highlight.pdf")

print(f"\nAll UMAPs saved in: {umap_dir}")

Saved: _cell_type_broad.pdf
Saved: _module_max_label.pdf
Saved: _leiden_annot_0.1_clean.pdf

All UMAPs saved in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/umaps_colored


In [ ]:
#Posterior heatmaps

In [25]:
##############################################
# POSTERIOR HEATMAP WITH FDR THRESHOLD
# Uses threshold_499 from gmm_fdr_diagnosis.py
# Plots one heatmap per assay (DCM + RNA)
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from matplotlib.colors import LinearSegmentedColormap

# ─────────────────────────────────────────────
# SETTINGS
# ─────────────────────────────────────────────
umap_dir = os.path.join(best_outdir, 'umaps_colored')
os.makedirs(umap_dir, exist_ok=True)
sc.settings.figdir = umap_dir

CELL_TYPES   = ['RGCs', 'INs', 'IGs']   # display order
MODULE_SUFFIX = '_module_score_Z_posterior'

# ─────────────────────────────────────────────
# CUSTOM COLORMAP — dark maroon above threshold_499
# ─────────────────────────────────────────────
colors_arr    = plt.cm.RdBu_r(np.linspace(0, 1, 256))
threshold_idx = int(threshold_499 * 256)
for i in range(threshold_idx, 256):
    colors_arr[i] = [0.4, 0.0, 0.1, 1.0]   # dark maroon
custom_cmap = LinearSegmentedColormap.from_list('RdBu_r_highlight', colors_arr)

# ─────────────────────────────────────────────
# PLOT FUNCTION
# ─────────────────────────────────────────────
def plot_posterior_heatmap(adata_obj, assay_name, save_suffix):
    """
    Build and save one posterior heatmap for a single assay.
    Posterior columns are expected in the format:
        {assay_name}_{ct}{MODULE_SUFFIX}
    """
    # Build column map: display name → column in adata.obs
    col_map = {}
    for ct in CELL_TYPES:
        col = f"{assay_name}_{ct}{MODULE_SUFFIX}"
        if col in adata_obj.obs.columns:
            col_map[ct] = col
        else:
            # Try merged column created by save_gmm_to_adata.py
            merged = f"_posterior_merged_{ct}"
            if merged in adata_obj.obs.columns:
                col_map[ct] = merged

    found_cts = list(col_map.keys())
    if not found_cts:
        print(f"  No posterior columns found for assay '{assay_name}' — skipping")
        return

    # Subset to this assay
    mask      = adata_obj.obs['orig.ident'] == assay_name
    df        = adata_obj.obs.loc[mask, list(col_map.values())].copy()
    df.columns = found_cts
    df_clean  = df.dropna()

    print(f"\n{assay_name}:")
    print(f"  Total cells : {len(df)}")
    print(f"  After NaN   : {len(df_clean)}")

    if df_clean.empty:
        print("  No valid cells — skipping")
        return

    # ── Sort: group by max cell type, descending within group ──
    df_clean = df_clean[found_cts].copy()
    df_clean['_max_type'] = df_clean.idxmax(axis=1)

    df_sorted = pd.concat([
        df_clean[df_clean['_max_type'] == ct]
        .sort_values(by=ct, ascending=False)
        for ct in found_cts
        if ct in df_clean['_max_type'].values
    ])

    cell_counts = (df_clean['_max_type']
                   .value_counts()
                   .reindex(found_cts)
                   .fillna(0).astype(int))
    df_sorted   = df_sorted.drop(columns=['_max_type'])

    print(f"  Cells per type:\n{cell_counts.to_string()}")
    print(f"  Data range: {df_sorted.min().min():.3f} – {df_sorted.max().max():.3f}")
    for ct in found_cts:
        n = (df_sorted[ct] > threshold_499).sum()
        print(f"  {ct} > {threshold_499:.3f}: {n} cells")

    # ── Heatmap ──
    fig, ax = plt.subplots(figsize=(10, 3))

    hm = sns.heatmap(
        df_sorted.T,
        cmap=custom_cmap,
        vmin=0, vmax=1,
        xticklabels=False,
        yticklabels=True,
        cbar_kws={'label': 'Posterior probability', 'shrink': 0.8, 'aspect': 20},
        ax=ax,
        rasterized=True,
    )

    # Colorbar threshold marker
    cbar = hm.collections[0].colorbar
    cbar.ax.axhline(y=threshold_499, color='black', linewidth=1.5, linestyle='--')
    cbar.ax.text(1.5, threshold_499, f'{threshold_499:.3f}',
                 va='center', ha='left', fontsize=9)

    ax.set_xlabel('Cells', fontsize=12)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=11, rotation=0)

    # Vertical separators between cell type groups
    cumsum = 0
    for i, ct in enumerate(found_cts):
        cumsum += cell_counts.get(ct, 0)
        if i < len(found_cts) - 1:
            ax.axvline(x=cumsum, color='black', linewidth=1.5)

    # Cell type labels below heatmap
    cumsum = 0
    n_rows = len(found_cts)
    for ct in found_cts:
        cnt = cell_counts.get(ct, 0)
        if cnt > 0:
            ax.text(cumsum + cnt / 2, n_rows + 0.3, ct,
                    ha='center', va='top', fontsize=11, fontweight='bold')
            cumsum += cnt

    ax.set_title(f'Posterior probabilities — {assay_name} '
                 f'(threshold = {threshold_499:.3f}, 4.99% FDR)',
                 fontsize=11, fontweight='bold', pad=16)

    plt.tight_layout()
    save_path = os.path.join(umap_dir, f'posterior_heatmap_{save_suffix}.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ─────────────────────────────────────────────
# RUN FOR EACH ASSAY
# ─────────────────────────────────────────────
for assay in sorted(adata_ttest.obs['orig.ident'].unique()):
    suffix = assay.replace(' ', '_').replace('/', '_')
    plot_posterior_heatmap(adata_ttest, assay, suffix)

print(f"\nAll posterior heatmaps saved in: {umap_dir}")


scNMT_pool1_DCM:
  Total cells : 405
  After NaN   : 405
  Cells per type:
_max_type
RGCs    133
INs     151
IGs     121
  Data range: 0.000 – 1.000
  RGCs > 0.837: 46 cells
  INs > 0.837: 118 cells
  IGs > 0.837: 96 cells
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/umaps_colored/posterior_heatmap_scNMT_pool1_DCM.pdf

scNMT_pool1_RNA:
  Total cells : 723
  After NaN   : 723
  Cells per type:
_max_type
RGCs    133
INs     378
IGs     212
  Data range: 0.079 – 1.000
  RGCs > 0.837: 81 cells
  INs > 0.837: 318 cells
  IGs > 0.837: 243 cells
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/umaps_colored/posterior_heatmap_scNMT_pool1_RNA.pdf

All posterior heatmaps saved in: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7

In [27]:
##############################################
# GENE HEATMAP PER CLUSTER — COMMON MARKERS
# Finds genes whose max expression is in the
# expected cluster in ALL assays, then plots
# heatmap / matrixplot / dotplot per assay.
#
# Requires in memory:
#   adata, ttest_df_unique, score_dir,
#   best_outdir, cell_types
##############################################

import os
import numpy as np
import pandas as pd
import scanpy as sc

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
plot_dir = os.path.join(best_outdir, 'marker_heatmaps')
os.makedirs(plot_dir, exist_ok=True)
sc.settings.figdir = plot_dir
sc.set_figure_params(dpi=300, dpi_save=300, fontsize=8)

# ─────────────────────────────────────────────
# ASSAY CONFIG
# Maps assay label → which obs column to use as
# cell type grouping for that modality.
# ─────────────────────────────────────────────
ASSAY_CONFIG = {
    'scNMT_DCM': {
        'batch_substr':  'scNMT_DCM',
        'cell_type_col': 'module_max_label',      # GMM label
        'exclude':       ['ambiguous', 'Unknown'],
        'extra_filter':  {'col': 'Dox', 'exclude_values': ['Minus', 'Unknown']},
    },
    'scNMT_RNA': {
        'batch_substr':  'scNMT_RNA',
        'cell_type_col': 'cell_type_broad',
        'exclude':       ['Unknown'],
        'extra_filter':  None,
    },
    'ref': {
        'batch_substr':  None,                    # None = all ref batches
        'ref_batches':   ['ref_D2', 'ref_D4', 'multi_RNA'],
        'cell_type_col': 'cell_type_broad',
        'exclude':       ['Unknown'],
        'extra_filter':  None,
    },
}

DESIRED_ORDER = cell_types   # ['RGCs', 'INs', 'IGs'] from gene_scoring_zscore.py
N_TOP_GENES   = 50           # genes to pull from ttest_df_unique per cell type
N_KEEP        = 10           # top common genes to display per cell type

# ─────────────────────────────────────────────
# STEP 1: BUILD TOP-GENE LIST FROM ttest_df_unique
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING TOP GENES FROM ttest_df_unique")
print("=" * 60)

top_genes_per_type = {}
for ct in DESIRED_ORDER:
    genes = (
        ttest_df_unique[ttest_df_unique['cell_type'] == ct]
        .sort_values('pvals_adj', ascending=True)
        ['gene'].head(N_TOP_GENES).tolist()
    )
    top_genes_per_type[ct] = genes
    print(f"  {ct}: {len(genes)} genes")

# ─────────────────────────────────────────────
# HELPER: subset adata for one assay config entry
# ─────────────────────────────────────────────
def get_assay_subset(cfg):
    adata.obs['batch'] = adata.obs['batch'].astype(str)
    if cfg.get('batch_substr'):
        mask = adata.obs['batch'].str.contains(cfg['batch_substr'], case=False, na=False)
    else:
        mask = adata.obs['batch'].isin(cfg['ref_batches'])

    sub = adata[mask].copy()

    if cfg['extra_filter']:
        ecol = cfg['extra_filter']['col']
        evals = cfg['extra_filter']['exclude_values']
        if ecol in sub.obs.columns:
            before = sub.n_obs
            sub = sub[~sub.obs[ecol].astype(str).isin(evals)].copy()
            print(f"    Extra filter on '{ecol}': removed {before - sub.n_obs} cells")

    ct_col = cfg['cell_type_col']
    if ct_col not in sub.obs.columns:
        print(f"    Column '{ct_col}' not found")
        return None

    sub.obs['_ct'] = sub.obs[ct_col].astype(str)
    excl = cfg['exclude'] + ['nan', 'ambiguous', 'Unknown']
    sub = sub[~sub.obs['_ct'].isin(excl)].copy()
    sub = sub[~sub.obs['_ct'].str.contains('nan|ambiguous', case=False, na=True)].copy()

    if sub.n_obs == 0:
        print("    No cells remaining after filtering")
        return None

    # Set lognorm as .X
    if 'lognorm' in sub.layers:
        sub.X = sub.layers['lognorm']

    # Order categories
    present = [ct for ct in DESIRED_ORDER if ct in sub.obs['_ct'].values]
    remaining = [ct for ct in sub.obs['_ct'].unique() if ct not in present]
    sub.obs['_ct'] = pd.Categorical(sub.obs['_ct'],
                                     categories=present + remaining,
                                     ordered=True)
    print(f"    {sub.n_obs} cells: {sub.obs['_ct'].value_counts().to_dict()}")
    return sub


# ─────────────────────────────────────────────
# STEP 2: FIRST PASS — find genes whose max
# expression is in the expected cluster in each assay
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: FIRST PASS — filtering genes per assay")
print("=" * 60)

filtered_genes_per_assay = {}

for assay_name, cfg in ASSAY_CONFIG.items():
    print(f"\n  {assay_name}")
    sub = get_assay_subset(cfg)
    if sub is None:
        continue

    # Mean expression per cluster
    import scipy.sparse as sp
    X = sub.X.toarray() if sp.issparse(sub.X) else np.array(sub.X)
    df_expr = pd.DataFrame(X, index=sub.obs_names, columns=sub.var_names)
    df_expr['_ct'] = sub.obs['_ct'].values
    pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types

    filtered_genes_by_ct = {}
    for ct in DESIRED_ORDER:
        if ct not in top_genes_per_type:
            continue
        passed = []
        for gene in top_genes_per_type[ct]:
            if gene not in pseudobulk.index:
                continue
            if pseudobulk.loc[gene].idxmax() == ct:
                passed.append(gene)
        filtered_genes_by_ct[ct] = passed
        print(f"    {ct}: {len(passed)} genes passed filter")

    filtered_genes_per_assay[assay_name] = filtered_genes_by_ct

# ─────────────────────────────────────────────
# STEP 3: INTERSECTION ACROSS ASSAYS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: INTERSECTION ACROSS ASSAYS")
print("=" * 60)

common_genes_per_ct = {}
for ct in DESIRED_ORDER:
    sets = [
        set(filtered_genes_per_assay[a][ct])
        for a in ASSAY_CONFIG
        if a in filtered_genes_per_assay and ct in filtered_genes_per_assay[a]
    ]
    if not sets:
        common_genes_per_ct[ct] = []
        continue
    common = sets[0]
    for s in sets[1:]:
        common = common & s
    # Preserve original rank order, keep top N_KEEP
    ordered = [g for g in top_genes_per_type[ct] if g in common][:N_KEEP]
    common_genes_per_ct[ct] = ordered
    print(f"  {ct}: {len(common)} common → keeping top {N_KEEP}: {ordered}")

all_common_genes = [g for ct in DESIRED_ORDER
                    for g in common_genes_per_ct.get(ct, [])]
print(f"\nTotal common genes to plot: {len(all_common_genes)}")

# ─────────────────────────────────────────────
# STEP 4: PLOTS PER ASSAY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: PLOTS PER ASSAY")
print("=" * 60)

for assay_name, cfg in ASSAY_CONFIG.items():
    print(f"\n  {assay_name}")
    sub = get_assay_subset(cfg)
    if sub is None:
        continue

    genes_to_plot = [g for g in all_common_genes if g in sub.var_names]
    print(f"    Genes available: {len(genes_to_plot)}")
    if not genes_to_plot:
        print("    No genes to plot — skipping")
        continue

    # Build var_names_ordered strictly in DESIRED_ORDER (RGCs → INs → IGs)
    # so columns appear left-to-right in that order in all plots.
    var_names_ordered = {}
    for ct in DESIRED_ORDER:
        genes_ct = [g for g in common_genes_per_ct.get(ct, []) if g in sub.var_names]
        if genes_ct:
            var_names_ordered[ct] = genes_ct

    # Re-enforce the categorical order on _ct so scanpy rows/groups
    # also follow DESIRED_ORDER (not alphabetical).
    present_ordered = [ct for ct in DESIRED_ORDER if ct in sub.obs['_ct'].values]
    remaining       = [ct for ct in sub.obs['_ct'].cat.categories
                       if ct not in present_ordered]
    sub.obs['_ct'] = pd.Categorical(
        sub.obs['_ct'],
        categories=present_ordered + remaining,
        ordered=True,
    )

    shared_kw = dict(
        groupby='_ct',
        standard_scale='var',
        cmap='RdBu_r',
        show=False,
    )

    # Heatmap
    try:
        sc.pl.heatmap(
            sub, var_names=var_names_ordered,
            show_gene_labels=True,
            save=f'_{assay_name}_common_markers_heatmap.pdf',
            **shared_kw,
        )
        print(f"    Saved: _{assay_name}_common_markers_heatmap.pdf")
    except Exception as e:
        print(f"    Heatmap failed: {e}")

    # Matrixplot
    try:
        sc.pl.matrixplot(
            sub, var_names=var_names_ordered,
            title=f'{assay_name}: common top {N_KEEP} marker genes',
            save=f'_{assay_name}_common_markers_matrixplot.pdf',
            **shared_kw,
        )
        print(f"    Saved: _{assay_name}_common_markers_matrixplot.pdf")
    except Exception as e:
        print(f"    Matrixplot failed: {e}")

    # Dotplot
    try:
        sc.pl.dotplot(
            sub, var_names=var_names_ordered,
            title=f'{assay_name}: common top {N_KEEP} marker genes',
            dot_max=0.5,
            save=f'_{assay_name}_common_markers_dotplot.pdf',
            **shared_kw,
        )
        print(f"    Saved: _{assay_name}_common_markers_dotplot.pdf")
    except Exception as e:
        print(f"    Dotplot failed: {e}")

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY: common genes per cell type")
print("=" * 60)
for ct in DESIRED_ORDER:
    genes = common_genes_per_ct.get(ct, [])
    print(f"  {ct} ({len(genes)}): {genes}")
print(f"\nAll plots saved in: {plot_dir}")

STEP 1: LOADING TOP GENES FROM ttest_df_unique
  IGs: 50 genes
  INs: 50 genes
  RGCs: 50 genes

STEP 2: FIRST PASS — filtering genes per assay

  scNMT_DCM
    Extra filter on 'Dox': removed 0 cells
    176 cells: {'INs': 95, 'IGs': 57, 'RGCs': 24}


/tmp/ipykernel_485362/894437180.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types


    IGs: 11 genes passed filter
    INs: 38 genes passed filter
    RGCs: 28 genes passed filter

  scNMT_RNA
    723 cells: {'RGCs': 523, 'IGs': 115, 'INs': 85}
    IGs: 50 genes passed filter
    INs: 50 genes passed filter
    RGCs: 44 genes passed filter

  ref


/tmp/ipykernel_485362/894437180.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types


    14792 cells: {'RGCs': 9025, 'INs': 4568, 'IGs': 1199}


/tmp/ipykernel_485362/894437180.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types


    IGs: 50 genes passed filter
    INs: 50 genes passed filter
    RGCs: 50 genes passed filter

STEP 3: INTERSECTION ACROSS ASSAYS
  IGs: 11 common → keeping top 10: ['Sparc', 'Nxn', 'Zfhx3', 'Tenm4', 'Anxa6', 'Mdk', 'Serpinh1', 'Igfbp4', 'Pdzrn3', 'Maml3']
  INs: 38 common → keeping top 10: ['Kcnd2', 'Sox6', 'Nav3', 'Ctnna2', 'Nav2', 'Map2', 'Nol4', 'Megf11', 'Ppp2r2b', 'Magi1']
  RGCs: 23 common → keeping top 10: ['Slc4a4', 'Slc1a3', 'Dbi', 'Ckb', 'Aldoc', 'Gstm1', 'Slc6a11', 'Sparcl1', 'Myo6', 'Nrxn1']

Total common genes to plot: 30

STEP 4: PLOTS PER ASSAY

  scNMT_DCM
    Extra filter on 'Dox': removed 0 cells
    176 cells: {'INs': 95, 'IGs': 57, 'RGCs': 24}
    Genes available: 30
    Saved: _scNMT_DCM_common_markers_heatmap.pdf
    Saved: _scNMT_DCM_common_markers_matrixplot.pdf
    Saved: _scNMT_DCM_common_markers_dotplot.pdf

  scNMT_RNA
    723 cells: {'RGCs': 523, 'IGs': 115, 'INs': 85}
    Genes available: 30
    Saved: _scNMT_RNA_common_markers_heatmap.pdf
    Saved: _

In [ ]:
#marker heatmaps

In [4]:
adata

NameError: name 'adata' is not defined

In [15]:
##############################################
# GENE HEATMAP PER CLUSTER — COMMON MARKERS
# Finds genes whose max expression is in the
# expected cluster in ALL assays, then plots
# heatmap / matrixplot / dotplot per assay.
#
# Requires in memory:
#   adata, ttest_df_unique, score_dir,
#   best_outdir, cell_types
##############################################

import os
import numpy as np
import pandas as pd
import scanpy as sc

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
plot_dir = os.path.join(best_outdir, 'marker_heatmaps')
os.makedirs(plot_dir, exist_ok=True)
sc.settings.figdir = plot_dir
sc.set_figure_params(dpi=300, dpi_save=300, fontsize=8)

# ─────────────────────────────────────────────
# ASSAY CONFIG
# Maps assay label → which obs column to use as
# cell type grouping for that modality.
# ─────────────────────────────────────────────
ASSAY_CONFIG = {
    'scNMT_DCM': {
        'batch_substr':  'scNMT_DCM',
        'cell_type_col': 'module_max_label',      # GMM label
        'exclude':       ['ambiguous', 'Unknown'],
        'extra_filter':  {'col': 'Dox', 'exclude_values': ['Minus', 'Unknown']},
    },
    'scNMT_RNA': {
        'batch_substr':  'scNMT_RNA',
        'cell_type_col': 'cell_type_broad',
        'exclude':       ['Unknown'],
        'extra_filter':  None,
    },
    'ref': {
        'batch_substr':  None,                    # None = all ref batches
        'ref_batches':   ['ref_D2', 'ref_D4', 'multi_RNA'],
        'cell_type_col': 'cell_type_broad',
        'exclude':       ['Unknown'],
        'extra_filter':  None,
    },
}

DESIRED_ORDER = cell_types   # ['RGCs', 'INs', 'IGs'] from gene_scoring_zscore.py
N_TOP_GENES   = 50           # genes to pull from ttest_df_unique per cell type
N_KEEP        = 10           # top common genes to display per cell type

# ─────────────────────────────────────────────
# STEP 1: BUILD TOP-GENE LIST FROM ttest_df_unique
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING TOP GENES FROM ttest_df_unique")
print("=" * 60)

top_genes_per_type = {}
for ct in DESIRED_ORDER:
    genes = (
        ttest_df_unique[ttest_df_unique['cell_type'] == ct]
        .sort_values('pvals_adj', ascending=True)
        ['gene'].head(N_TOP_GENES).tolist()
    )
    top_genes_per_type[ct] = genes
    print(f"  {ct}: {len(genes)} genes")

# ─────────────────────────────────────────────
# HELPER: subset adata for one assay config entry
# ─────────────────────────────────────────────
def get_assay_subset(cfg):
    adata.obs['batch'] = adata.obs['batch'].astype(str)
    if cfg.get('batch_substr'):
        mask = adata.obs['batch'].str.contains(cfg['batch_substr'], case=False, na=False)
    else:
        mask = adata.obs['batch'].isin(cfg['ref_batches'])

    sub = adata[mask].copy()

    if cfg['extra_filter']:
        ecol = cfg['extra_filter']['col']
        evals = cfg['extra_filter']['exclude_values']
        if ecol in sub.obs.columns:
            before = sub.n_obs
            sub = sub[~sub.obs[ecol].astype(str).isin(evals)].copy()
            print(f"    Extra filter on '{ecol}': removed {before - sub.n_obs} cells")

    ct_col = cfg['cell_type_col']
    if ct_col not in sub.obs.columns:
        print(f"    Column '{ct_col}' not found")
        return None

    sub.obs['_ct'] = sub.obs[ct_col].astype(str)
    excl = cfg['exclude'] + ['nan', 'ambiguous', 'Unknown']
    sub = sub[~sub.obs['_ct'].isin(excl)].copy()
    sub = sub[~sub.obs['_ct'].str.contains('nan|ambiguous', case=False, na=True)].copy()

    if sub.n_obs == 0:
        print("    No cells remaining after filtering")
        return None

    # Set lognorm as .X
    if 'lognorm' in sub.layers:
        sub.X = sub.layers['lognorm']

    # Order categories
    present = [ct for ct in DESIRED_ORDER if ct in sub.obs['_ct'].values]
    remaining = [ct for ct in sub.obs['_ct'].unique() if ct not in present]
    sub.obs['_ct'] = pd.Categorical(sub.obs['_ct'],
                                     categories=present + remaining,
                                     ordered=True)
    print(f"    {sub.n_obs} cells: {sub.obs['_ct'].value_counts().to_dict()}")
    return sub


# ─────────────────────────────────────────────
# STEP 2: FIRST PASS — find genes whose max
# expression is in the expected cluster in each assay
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: FIRST PASS — filtering genes per assay")
print("=" * 60)

filtered_genes_per_assay = {}

for assay_name, cfg in ASSAY_CONFIG.items():
    print(f"\n  {assay_name}")
    sub = get_assay_subset(cfg)
    if sub is None:
        continue

    # Mean expression per cluster
    import scipy.sparse as sp
    X = sub.X.toarray() if sp.issparse(sub.X) else np.array(sub.X)
    df_expr = pd.DataFrame(X, index=sub.obs_names, columns=sub.var_names)
    df_expr['_ct'] = sub.obs['_ct'].values
    pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types

    filtered_genes_by_ct = {}
    for ct in DESIRED_ORDER:
        if ct not in top_genes_per_type:
            continue
        passed = []
        for gene in top_genes_per_type[ct]:
            if gene not in pseudobulk.index:
                continue
            if pseudobulk.loc[gene].idxmax() == ct:
                passed.append(gene)
        filtered_genes_by_ct[ct] = passed
        print(f"    {ct}: {len(passed)} genes passed filter")

    filtered_genes_per_assay[assay_name] = filtered_genes_by_ct

# ─────────────────────────────────────────────
# STEP 3: INTERSECTION ACROSS ASSAYS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: INTERSECTION ACROSS ASSAYS")
print("=" * 60)

common_genes_per_ct = {}
for ct in DESIRED_ORDER:
    sets = [
        set(filtered_genes_per_assay[a][ct])
        for a in ASSAY_CONFIG
        if a in filtered_genes_per_assay and ct in filtered_genes_per_assay[a]
    ]
    if not sets:
        common_genes_per_ct[ct] = []
        continue
    common = sets[0]
    for s in sets[1:]:
        common = common & s
    # Preserve original rank order, keep top N_KEEP
    ordered = [g for g in top_genes_per_type[ct] if g in common][:N_KEEP]
    common_genes_per_ct[ct] = ordered
    print(f"  {ct}: {len(common)} common → keeping top {N_KEEP}: {ordered}")

all_common_genes = [g for ct in DESIRED_ORDER
                    for g in common_genes_per_ct.get(ct, [])]
print(f"\nTotal common genes to plot: {len(all_common_genes)}")

# ─────────────────────────────────────────────
# FIXED GENE LISTS (literature / curated)
# ─────────────────────────────────────────────
FIXED_GENES = {
    'RGCs': ['Slc1a3', 'Fabp7', 'Hes5', 'Aldh1l1', 'Aqp4',
             'Egfr', 'Phgdh', 'Ttyh1', 'Ndrg2', 'Fgfr3'],
    'INs':  ['Sox6', 'Dcx', 'Gad1', 'Map2', 'Erbb4',
             'Nxph1', 'Dll1', 'Mapt', 'Snap25', 'Kcnd2'],
    'IGs':  ['Fstl1', 'Col4a1', 'Lgals1', 'Dcn', 'Pdgfrb',
             'Lef1', 'Efna5', 'Postn', 'Maml3', 'Tgfbr2'],
}

# ─────────────────────────────────────────────
# STEP 4: PLOTS PER ASSAY
# Two passes per assay:
#   Pass A — fixed gene list (all genes, no filtering)
#   Pass B — fixed genes filtered to those whose max
#             expression matches the expected cell type
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: PLOTS PER ASSAY")
print("=" * 60)


def _enforce_cat_order(sub):
    """Re-enforce DESIRED_ORDER on _ct categorical."""
    present   = [ct for ct in DESIRED_ORDER if ct in sub.obs['_ct'].values]
    remaining = [ct for ct in sub.obs['_ct'].cat.categories
                 if ct not in present]
    sub.obs['_ct'] = pd.Categorical(
        sub.obs['_ct'], categories=present + remaining, ordered=True)
    return sub


def _make_plots(sub, var_names_dict, assay_name, suffix, title_suffix):
    """Run heatmap / matrixplot / dotplot for one var_names_dict."""
    import scipy.sparse as sp
    genes_available = [g for ct_genes in var_names_dict.values()
                       for g in ct_genes if g in sub.var_names]
    if not genes_available:
        print(f"    No genes available for {suffix} — skipping")
        return

    shared_kw = dict(groupby='_ct', standard_scale='var',
                     cmap='RdBu_r', show=False)

    for plot_fn, plot_name, extra_kw in [
        (sc.pl.heatmap,    'heatmap',    dict(show_gene_labels=True)),
        (sc.pl.matrixplot, 'matrixplot', dict(title=f'{assay_name} {title_suffix}')),
        (sc.pl.dotplot,    'dotplot',    dict(title=f'{assay_name} {title_suffix}',
                                              dot_max=0.5)),
    ]:
        fname = f'_{assay_name}_{suffix}_{plot_name}.pdf'
        try:
            plot_fn(sub, var_names=var_names_dict,
                    save=fname, **shared_kw, **extra_kw)
            print(f"    Saved: {fname}")
        except Exception as e:
            print(f"    {plot_name} failed: {e}")


for assay_name, cfg in ASSAY_CONFIG.items():
    print(f"\n  {assay_name}")
    sub = get_assay_subset(cfg)
    if sub is None:
        continue
    sub = _enforce_cat_order(sub)

    # ── Pseudobulk for filtering pass ──
    import scipy.sparse as sp
    X = sub.X.toarray() if sp.issparse(sub.X) else np.array(sub.X)
    df_expr = pd.DataFrame(X, index=sub.obs_names, columns=sub.var_names)
    df_expr['_ct'] = sub.obs['_ct'].values
    pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types

    # ── Pass A: ALL fixed genes (no max-expression filter) ──
    var_names_all = {}
    for ct in DESIRED_ORDER:
        avail = [g for g in FIXED_GENES.get(ct, []) if g in sub.var_names]
        if avail:
            var_names_all[ct] = avail
    print(f"    Pass A (all fixed genes): "
          f"{sum(len(v) for v in var_names_all.values())} genes")
    _make_plots(sub, var_names_all, assay_name,
                suffix='fixed_all',
                title_suffix='fixed genes (all)')

    # ── Pass B: fixed genes filtered to max-expression match ──
    var_names_filtered = {}
    for ct in DESIRED_ORDER:
        avail = [g for g in FIXED_GENES.get(ct, []) if g in sub.var_names]
        passed = [g for g in avail
                  if g in pseudobulk.index
                  and pseudobulk.loc[g].idxmax() == ct]
        if passed:
            var_names_filtered[ct] = passed
        n_all  = len(avail)
        n_pass = len(passed)
        print(f"    Pass B {ct}: {n_pass}/{n_all} fixed genes "
              f"have max expression in {ct}")
    _make_plots(sub, var_names_filtered, assay_name,
                suffix='fixed_filtered',
                title_suffix='fixed genes (max-expr filtered)')

    # ── Original common-marker plots (ttest-derived) ──
    var_names_common = {}
    for ct in DESIRED_ORDER:
        genes_ct = [g for g in common_genes_per_ct.get(ct, [])
                    if g in sub.var_names]
        if genes_ct:
            var_names_common[ct] = genes_ct
    if any(var_names_common.values()):
        print(f"    Pass C (ttest common markers): "
              f"{sum(len(v) for v in var_names_common.values())} genes")
        _make_plots(sub, var_names_common, assay_name,
                    suffix='common_markers',
                    title_suffix=f'common top {N_KEEP} marker genes')

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY: common genes per cell type")
print("=" * 60)
for ct in DESIRED_ORDER:
    genes = common_genes_per_ct.get(ct, [])
    print(f"  {ct} ({len(genes)}): {genes}")
print(f"\nAll plots saved in: {plot_dir}")

# ─────────────────────────────
# STEP 5: ULTIMATE MATRIXPLOT
# Seed  : fixed genes that passed DCM max-expr filter
# Top-up: ttest common markers consistent in
#         scNMT_RNA AND ref, until 10 per cell type
# ─────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: ULTIMATE MATRIXPLOT")
print("=" * 60)

import scipy.sparse as sp

def _get_pseudobulk(assay_name):
    cfg = ASSAY_CONFIG[assay_name]
    sub = get_assay_subset(cfg)
    if sub is None:
        return None
    X = sub.X.toarray() if sp.issparse(sub.X) else np.array(sub.X)
    df_e = pd.DataFrame(X, index=sub.obs_names, columns=sub.var_names)
    df_e['_ct'] = sub.obs['_ct'].values
    pb = df_e.groupby('_ct').mean().T
    print(f"  [{assay_name}] pseudobulk columns: {list(pb.columns)}")
    return pb

def _passes(gene, ct, *pseudobulks):
    for pb in pseudobulks:
        if pb is None or gene not in pb.index:
            return False
        if pb.loc[gene].idxmax() != ct:
            return False
    return True

pb_dcm = _get_pseudobulk('scNMT_DCM')
pb_rna = _get_pseudobulk('scNMT_RNA')
pb_ref = _get_pseudobulk('ref')

# Diagnostic: check a few fixed genes against each pseudobulk
print("\nDiagnostic — checking first fixed gene per cell type:")
for ct in DESIRED_ORDER:
    g = FIXED_GENES.get(ct, [None])[0]
    if g is None:
        continue
    for name, pb in [('DCM', pb_dcm), ('RNA', pb_rna), ('ref', pb_ref)]:
        if pb is not None and g in pb.index:
            top = pb.loc[g].idxmax()
            print(f"  {ct} | {g} | {name}: max in '{top}' (match={top==ct})")
        else:
            print(f"  {ct} | {g} | {name}: gene not in pseudobulk")

# Full ranked gene list per cell type — no cap, used as topup pool
all_ttest_genes_per_type = {
    ct: (ttest_df_unique[ttest_df_unique['cell_type'] == ct]
         .sort_values('pvals_adj', ascending=True)['gene'].tolist())
    for ct in DESIRED_ORDER
}

# Determine which pseudobulks are available (ref may be None)
available_pbs = {k: v for k, v in
                 [('DCM', pb_dcm), ('RNA', pb_rna), ('ref', pb_ref)]
                 if v is not None}
print(f"Available pseudobulks: {list(available_pbs.keys())}")

ultimate_genes = {}
for ct in DESIRED_ORDER:
    # Step 1: fixed genes that peak in the correct cell type
    #         in ALL available pseudobulks (skip ref if it has no cells)
    seed = [
        g for g in FIXED_GENES.get(ct, [])
        if _passes(g, ct, *available_pbs.values())
    ]

    # Step 2: top up with ttest markers consistent in all available pseudobulks
    fixed_set  = set(FIXED_GENES.get(ct, []))
    needed     = max(0, 10 - len(seed))
    topup_pool = [
        g for g in all_ttest_genes_per_type.get(ct, [])
        if g not in fixed_set
        and g not in seed
        and _passes(g, ct, *available_pbs.values())
    ]
    topup = topup_pool[:needed]
    final = seed + topup
    ultimate_genes[ct] = final

    print(f"  {ct}:")
    print(f"    fixed list         : {FIXED_GENES.get(ct, [])}")
    print(f"    seed               : {seed}  ({len(seed)} genes)")
    print(f"    topup pool size    : {len(topup_pool)},  taking {len(topup)}: {topup}")
    print(f"    FINAL ({len(final)}/10)  : {final}")

pd.DataFrame([
    {'cell_type': ct, 'gene': g,
     'source': 'fixed' if g in FIXED_GENES.get(ct, []) else 'ttest_topup'}
    for ct in DESIRED_ORDER for g in ultimate_genes.get(ct, [])
]).to_csv(os.path.join(plot_dir, 'ultimate_gene_list.csv'), index=False)
print(f"Gene list saved: ultimate_gene_list.csv")

for assay_name, cfg in ASSAY_CONFIG.items():
    print(f"\n  Ultimate plots: {assay_name}")
    sub = get_assay_subset(cfg)
    if sub is None:
        continue
    sub = _enforce_cat_order(sub)
    var_ult = {ct: [g for g in ultimate_genes.get(ct, []) if g in sub.var_names]
               for ct in DESIRED_ORDER}
    var_ult = {ct: gs for ct, gs in var_ult.items() if gs}
    if not var_ult:
        print("    No genes available — skipping"); continue
    shared = dict(groupby='_ct', standard_scale='var', cmap='RdBu_r', show=False)
    for fn, name, ekw in [
        (sc.pl.heatmap,    'heatmap',    dict(show_gene_labels=True)),
        (sc.pl.matrixplot, 'matrixplot', dict(title=f'{assay_name}: ultimate markers')),
        (sc.pl.dotplot,    'dotplot',    dict(title=f'{assay_name}: ultimate markers',
                                               dot_max=0.5)),
    ]:
        fname = f'_{assay_name}_ultimate_{name}.pdf'
        try:
            fn(sub, var_names=var_ult, save=fname, **shared, **ekw)
            print(f"    Saved: {fname}")
        except Exception as e:
            print(f"    {name} failed: {e}")

print(f"\nAll plots saved in: {plot_dir}")

STEP 1: LOADING TOP GENES FROM ttest_df_unique
  RGCs: 50 genes
  INs: 50 genes
  IGs: 50 genes

STEP 2: FIRST PASS — filtering genes per assay

  scNMT_DCM
    Extra filter on 'Dox': removed 0 cells
    176 cells: {'INs': 95, 'IGs': 57, 'RGCs': 24}
    RGCs: 28 genes passed filter
    INs: 38 genes passed filter
    IGs: 11 genes passed filter

  scNMT_RNA


/tmp/ipykernel_3749709/983605933.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types


    723 cells: {'RGCs': 523, 'IGs': 115, 'INs': 85}
    RGCs: 44 genes passed filter
    INs: 50 genes passed filter
    IGs: 50 genes passed filter

  ref


/tmp/ipykernel_3749709/983605933.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types


    No cells remaining after filtering

STEP 3: INTERSECTION ACROSS ASSAYS
  RGCs: 23 common → keeping top 10: ['Slc4a4', 'Slc1a3', 'Dbi', 'Ckb', 'Aldoc', 'Gstm1', 'Slc6a11', 'Sparcl1', 'Myo6', 'Nrxn1']
  INs: 38 common → keeping top 10: ['Kcnd2', 'Sox6', 'Nav3', 'Ctnna2', 'Nav2', 'Map2', 'Nol4', 'Megf11', 'Ppp2r2b', 'Magi1']
  IGs: 11 common → keeping top 10: ['Sparc', 'Nxn', 'Zfhx3', 'Tenm4', 'Anxa6', 'Mdk', 'Serpinh1', 'Igfbp4', 'Pdzrn3', 'Maml3']

Total common genes to plot: 30

STEP 4: PLOTS PER ASSAY

  scNMT_DCM
    Extra filter on 'Dox': removed 0 cells
    176 cells: {'INs': 95, 'IGs': 57, 'RGCs': 24}
    Pass A (all fixed genes): 30 genes


/tmp/ipykernel_3749709/983605933.py:261: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df_expr.groupby('_ct').mean().T   # genes × cell_types


... (3 outputs left out)

/tmp/ipykernel_3749709/983605933.py:337: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pb = df_e.groupby('_ct').mean().T


    723 cells: {'RGCs': 523, 'IGs': 115, 'INs': 85}
  [scNMT_RNA] pseudobulk columns: ['RGCs', 'INs', 'IGs']


/tmp/ipykernel_3749709/983605933.py:337: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pb = df_e.groupby('_ct').mean().T


    No cells remaining after filtering

Diagnostic — checking first fixed gene per cell type:
  RGCs | Slc1a3 | DCM: max in 'RGCs' (match=True)
  RGCs | Slc1a3 | RNA: max in 'RGCs' (match=True)
  RGCs | Slc1a3 | ref: gene not in pseudobulk
  INs | Sox6 | DCM: max in 'INs' (match=True)
  INs | Sox6 | RNA: max in 'INs' (match=True)
  INs | Sox6 | ref: gene not in pseudobulk
  IGs | Fstl1 | DCM: max in 'INs' (match=False)
  IGs | Fstl1 | RNA: max in 'IGs' (match=True)
  IGs | Fstl1 | ref: gene not in pseudobulk
Available pseudobulks: ['DCM', 'RNA']
  RGCs:
    fixed list         : ['Slc1a3', 'Fabp7', 'Hes5', 'Aldh1l1', 'Aqp4', 'Egfr', 'Phgdh', 'Ttyh1', 'Ndrg2', 'Fgfr3']
    seed               : ['Slc1a3']  (1 genes)
    topup pool size    : 101,  taking 9: ['Slc4a4', 'Dbi', 'Ckb', 'Aldoc', 'Gstm1', 'Slc6a11', 'Sparcl1', 'Myo6', 'Nrxn1']
    FINAL (10/10)  : ['Slc1a3', 'Slc4a4', 'Dbi', 'Ckb', 'Aldoc', 'Gstm1', 'Slc6a11', 'Sparcl1', 'Myo6', 'Nrxn1']
  INs:
    fixed list         : ['Sox6', 

In [30]:
##############################################
# Z-SCORE BOXPLOTS — ALL ASSAYS INCLUDING REF
# Computes module scores + z-scores for ALL
# assays (scNMT_DCM, scNMT_RNA, ref) then
# plots boxplots per cell type per assay.
#
# Mirrors the reference code logic exactly,
# using _all suffix to keep columns separate
# from the existing scNMT-only z-scores.
#
# Requires in memory:
#   adata, ttest_markers_by_ct, best_outdir,
#   cell_types (from gene_scoring_zscore.py)
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import scanpy as sc
import scipy.sparse as sp

# ─────────────────────────────────────────────
# PATHS & CONFIG
# ─────────────────────────────────────────────
plot_dir = os.path.join(best_outdir, 'zscore_boxplots')
os.makedirs(plot_dir, exist_ok=True)

DESIRED_ORDER = ['RGCs', 'INs', 'IGs']
REF_BATCHES   = ['ref_D2', 'ref_D4', 'multi_RNA']

ASSAY_COLORS = {
    'scNMT_DCM': '#003366',
    'scNMT_RNA': '#8B0000',
    'ref':       '#808080',
}

ASSAY_PLOT_CONFIG = {
    'scNMT_DCM': {
        'cell_type_col': 'module_max_label',
        'extra_filter':  {'col': 'Dox', 'exclude_values': ['Minus', 'Unknown']},
    },
    'scNMT_RNA': {
        'cell_type_col': 'cell_type_broad',
        'extra_filter':  None,
    },
    'ref': {
        'cell_type_col': 'cell_type_broad',
        'extra_filter':  None,
    },
}

# ─────────────────────────────────────────────
# STEP 1: BUILD COMBINED SUBSET
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING ALL-ASSAY SUBSET")
print("=" * 60)

adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

scnmt_mask = adata.obs['orig.ident'].str.contains('scNMT', case=False, na=False)
ref_mask   = adata.obs['batch'].isin(REF_BATCHES)
combined   = adata[scnmt_mask | ref_mask].copy()
combined.X = combined.layers['lognorm']

def assign_score_group(row):
    if 'scNMT' in str(row['orig.ident']):
        return row['orig.ident']
    elif row['batch'] in REF_BATCHES:
        return 'ref'
    return 'other'

combined.obs['_score_group'] = combined.obs.apply(assign_score_group, axis=1)

def assign_plot_assay(row):
    oi = str(row['orig.ident'])
    if 'DCM' in oi and 'scNMT' in oi:
        return 'scNMT_DCM'
    elif 'RNA' in oi and 'scNMT' in oi:
        return 'scNMT_RNA'
    elif row['batch'] in REF_BATCHES:
        return 'ref'
    return 'other'

combined.obs['_plot_assay'] = combined.obs.apply(assign_plot_assay, axis=1)

print(f"Total cells: {combined.n_obs}")
print(combined.obs['_score_group'].value_counts().to_string())

# ─────────────────────────────────────────────
# STEP 2: MODULE SCORES PER GROUP
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: COMPUTING MODULE SCORES")
print("=" * 60)

genes_in_data = set(combined.var_names)
filtered_gene_sets = {
    ct: [g for g in genes if g in genes_in_data]
    for ct, genes in ttest_markers_by_ct.items()
}
for ct, g in filtered_gene_sets.items():
    print(f"  {ct}: {len(g)} genes")

for ct in filtered_gene_sets:
    combined.obs[f'{ct}_score_raw_all']        = np.nan
    combined.obs[f'{ct}_module_score_raw_all'] = np.nan

for grp in combined.obs['_score_group'].unique():
    print(f"\n  Group: {grp}")
    mask = combined.obs['_score_group'] == grp
    sub  = combined[mask].copy()
    print(f"    {sub.n_obs} cells")

    for ct, genes in filtered_gene_sets.items():
        if not genes:
            continue
        X = sub[:, genes].X
        sub.obs[f'{ct}_score_raw_all'] = (
            np.asarray(X.toarray() if sp.issparse(X) else X).mean(axis=1)
        )
        sc.tl.score_genes(
            sub, gene_list=genes,
            score_name=f'{ct}_module_score_raw_all',
            use_raw=False,
        )
        combined.obs.loc[sub.obs_names, f'{ct}_score_raw_all']        = sub.obs[f'{ct}_score_raw_all']
        combined.obs.loc[sub.obs_names, f'{ct}_module_score_raw_all'] = sub.obs[f'{ct}_module_score_raw_all']

# ─────────────────────────────────────────────
# STEP 3: Z-SCORE PER GROUP
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Z-SCORING PER GROUP")
print("=" * 60)

MODULE_ZSCORE_COLS = []
for ct in filtered_gene_sets:
    raw_col = f'{ct}_module_score_raw_all'
    if raw_col not in combined.obs:
        continue
    z_col = f'{ct}_module_score_Z_all'
    MODULE_ZSCORE_COLS.append(z_col)
    combined.obs[z_col] = np.nan
    for grp in combined.obs['_score_group'].unique():
        mask = combined.obs['_score_group'] == grp
        combined.obs.loc[mask, z_col] = zscore(
            combined.obs.loc[mask, raw_col], nan_policy='omit'
        )

print(f"Module Z-score columns: {MODULE_ZSCORE_COLS}")

# ─────────────────────────────────────────────
# STEP 4: BUILD PLOT DATA
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: BUILDING PLOT DATA")
print("=" * 60)

all_data = []
for assay_label, cfg in ASSAY_PLOT_CONFIG.items():
    mask = combined.obs['_plot_assay'] == assay_label

    if cfg['extra_filter']:
        ecol  = cfg['extra_filter']['col']
        evals = cfg['extra_filter']['exclude_values']
        if ecol in combined.obs.columns:
            mask = mask & ~combined.obs[ecol].astype(str).isin(evals)

    ct_col = cfg['cell_type_col']
    if ct_col not in combined.obs.columns:
        print(f"  WARNING: '{ct_col}' not in obs for {assay_label}")
        continue

    sub = combined.obs[mask].copy()
    sub['_ct']    = sub[ct_col].astype(str)
    sub['_assay'] = assay_label
    sub = sub[sub['_ct'].isin(DESIRED_ORDER)]

    print(f"  {assay_label}: {len(sub)} cells — {sub['_ct'].value_counts().to_dict()}")
    if not sub.empty:
        all_data.append(sub)

plot_df = pd.concat(all_data, ignore_index=True)

avail_assays  = [a for a in ASSAY_COLORS if a in plot_df['_assay'].unique()]
plot_df['_ct']    = pd.Categorical(plot_df['_ct'],
                                    categories=DESIRED_ORDER, ordered=True)
plot_df['_assay'] = pd.Categorical(plot_df['_assay'],
                                    categories=avail_assays, ordered=True)
palette = [ASSAY_COLORS[a] for a in avail_assays]

# ─────────────────────────────────────────────
# STEP 5: INDIVIDUAL BOXPLOTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: PLOTTING")
print("=" * 60)

for z_col in MODULE_ZSCORE_COLS:
    ct_name = z_col.replace('_module_score_Z_all', '')
    data    = plot_df[['_ct', '_assay', z_col]].dropna()

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(
        data=data, x='_ct', y=z_col, hue='_assay',
        order=DESIRED_ORDER, hue_order=avail_assays,
        palette=palette, ax=ax, linewidth=1, fliersize=2,
    )
    ax.set_title(f'{ct_name} module score Z', fontsize=13, fontweight='bold')
    ax.set_xlabel('Cell type', fontsize=11)
    ax.set_ylabel('Z-score', fontsize=11)
    ax.axhline(0,    color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(1.65, color='gray',  linestyle=':',  alpha=0.4, linewidth=1,
               label='Z = 1.65')
    ax.legend(title='Assay', loc='upper right', fontsize=9)
    plt.tight_layout()
    out = os.path.join(plot_dir, f'{ct_name}_module_score_Z_boxplot.pdf')
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out}")

# ─────────────────────────────────────────────
# STEP 6: COMBINED FIGURE
# ─────────────────────────────────────────────
n = len(MODULE_ZSCORE_COLS)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), sharey=False)
if n == 1:
    axes = [axes]

for idx, z_col in enumerate(MODULE_ZSCORE_COLS):
    ax      = axes[idx]
    ct_name = z_col.replace('_module_score_Z_all', '')
    data    = plot_df[['_ct', '_assay', z_col]].dropna()

    sns.boxplot(
        data=data, x='_ct', y=z_col, hue='_assay',
        order=DESIRED_ORDER, hue_order=avail_assays,
        palette=palette, ax=ax, linewidth=1, fliersize=2,
    )
    ax.set_title(ct_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Cell type', fontsize=11)
    ax.set_ylabel('Z-score' if idx == 0 else '', fontsize=11)
    ax.axhline(0,    color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(1.65, color='gray',  linestyle=':',  alpha=0.4, linewidth=1)

    if idx == n - 1:
        ax.legend(title='Assay', loc='upper right', fontsize=9)
    elif ax.get_legend():
        ax.get_legend().remove()

plt.suptitle('Module score Z per cell type per assay',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = os.path.join(plot_dir, 'all_modules_score_Z_boxplot_combined.pdf')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved: {out}")

print(f"\nAll boxplots saved in: {plot_dir}")

STEP 1: BUILDING ALL-ASSAY SUBSET
Total cells: 15920
_score_group
ref                14792
scNMT_pool1_RNA      723
scNMT_pool1_DCM      405

STEP 2: COMPUTING MODULE SCORES
  IGs: 1089 genes
  INs: 567 genes
  RGCs: 399 genes

  Group: scNMT_pool1_DCM
    405 cells

  Group: scNMT_pool1_RNA
    723 cells

  Group: ref
    14792 cells

STEP 3: Z-SCORING PER GROUP
Module Z-score columns: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']

STEP 4: BUILDING PLOT DATA
  scNMT_DCM: 176 cells — {'INs': 95, 'IGs': 57, 'RGCs': 24}
  scNMT_RNA: 723 cells — {'RGCs': 523, 'IGs': 115, 'INs': 85}
  ref: 14792 cells — {'RGCs': 9025, 'INs': 4568, 'IGs': 1199}

STEP 5: PLOTTING
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/zscore_boxplots/IGs_module_score_Z_boxplot.pdf
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Cel

In [31]:
##############################################
# Z-SCORE BOXPLOTS WITH SIGNIFICANCE — DCM
# Uses MODULE_ZSCORE_COLS from zscore_boxplots.py
# (run that script first to have combined.obs
#  with _module_score_Z_all columns)
#
# Requires in memory:
#   combined, filtered_gene_sets,
#   MODULE_ZSCORE_COLS, best_outdir,
#   DESIRED_ORDER
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, mannwhitneyu, rankdata
from itertools import combinations

# ─────────────────────────────────────────────
# PATHS & CONFIG
# ─────────────────────────────────────────────
plot_dir = os.path.join(best_outdir, 'zscore_boxplots')
os.makedirs(plot_dir, exist_ok=True)

# Cell type colours — the target cell type gets its colour, others go gray
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}

# ─────────────────────────────────────────────
# SUBSET: DCM cells, exclude Dox == Minus
# ─────────────────────────────────────────────
dcm_mask = combined.obs['_plot_assay'] == 'scNMT_DCM'
if 'Dox' in combined.obs.columns:
    dcm_mask = dcm_mask & ~combined.obs['Dox'].astype(str).isin(['Minus', 'Unknown'])

adata_dcm = combined[dcm_mask].copy()
print(f"DCM cells (excl. Minus): {adata_dcm.n_obs}")

cell_type_col = 'module_max_label'   # GMM label column
if cell_type_col not in adata_dcm.obs.columns:
    raise KeyError(f"'{cell_type_col}' not found — run gmm_analysis.py first.")

# ─────────────────────────────────────────────
# SIGNIFICANCE HELPER
# ─────────────────────────────────────────────
def sig_label(pval):
    if pval < 0.001: return '***'
    if pval < 0.01:  return '**'
    if pval < 0.05:  return '*'
    return 'ns'

# ─────────────────────────────────────────────
# MAIN LOOP — one plot per module
# ─────────────────────────────────────────────
all_stats = []

for z_col in MODULE_ZSCORE_COLS:
    ct_name = z_col.replace('_module_score_Z_all', '')
    print(f"\n{'='*60}\n{ct_name}\n{'='*60}")

    # ── Build plot data ──
    plot_data = pd.DataFrame({
        'zscore':     adata_dcm.obs[z_col].values,
        'cell_type':  adata_dcm.obs[cell_type_col].astype(str).values,
    }).dropna()
    plot_data = plot_data[plot_data['cell_type'].isin(DESIRED_ORDER)]

    if plot_data.empty:
        print(f"  No data — skipping")
        continue

    avail = [ct for ct in DESIRED_ORDER if ct in plot_data['cell_type'].values]
    plot_data['cell_type'] = pd.Categorical(plot_data['cell_type'],
                                             categories=avail, ordered=True)

    # ── Pairwise statistics ──
    comparisons = []
    for ct1, ct2 in combinations(avail, 2):
        g1 = plot_data.loc[plot_data['cell_type'] == ct1, 'zscore'].values
        g2 = plot_data.loc[plot_data['cell_type'] == ct2, 'zscore'].values

        t_stat, t_p = ttest_ind(g1, g2)
        u_stat, u_p = mannwhitneyu(g1, g2, alternative='two-sided')

        m1, s1 = g1.mean(), g1.std()
        m2, s2 = g2.mean(), g2.std()
        pooled  = np.sqrt(((len(g1)-1)*s1**2 + (len(g2)-1)*s2**2) / (len(g1)+len(g2)-2))
        cohens_d = (m1 - m2) / pooled if pooled > 0 else 0

        row = dict(module=ct_name, group1=ct1, group2=ct2,
                   n1=len(g1), n2=len(g2),
                   mean1=m1, mean2=m2, std1=s1, std2=s2,
                   mean_diff=m1-m2,
                   t_stat=t_stat, t_pval=t_p,
                   u_stat=u_stat, u_pval=u_p,
                   cohens_d=cohens_d)
        comparisons.append(row)
        all_stats.append(row)

        print(f"  {ct1} vs {ct2}:  t-test p={t_p:.2e} {sig_label(t_p)}  "
              f"MWU p={u_p:.2e} {sig_label(u_p)}  d={cohens_d:.3f}")

    # ── Palette: target ct gets colour, others gray ──
    palette = [cell_type_colors.get(ct, '#D3D3D3') if ct == ct_name else '#D3D3D3'
               for ct in avail]

    # ── Plot ──
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.boxplot(
        data=plot_data, x='cell_type', y='zscore',
        hue='cell_type', order=avail, hue_order=avail,
        palette=palette, ax=ax, linewidth=1.5, fliersize=3, legend=False,
    )

    # Significance bars
    y_max   = plot_data['zscore'].max()
    y_range = plot_data['zscore'].max() - plot_data['zscore'].min()
    for i, comp in enumerate(comparisons):
        x1 = avail.index(comp['group1'])
        x2 = avail.index(comp['group2'])
        y  = y_max + (i + 1) * y_range * 0.12
        ax.plot([x1, x1, x2, x2],
                [y - y_range*0.02, y, y, y - y_range*0.02],
                color='black', linewidth=1)
        ax.text((x1+x2)/2, y + y_range*0.01, sig_label(comp['t_pval']),
                ha='center', va='bottom', fontsize=10)

    ax.set_ylim(plot_data['zscore'].min() - y_range*0.1,
                y_max + (len(comparisons)+1) * y_range * 0.15)
    ax.set_title(f'{ct_name} module score Z (scNMT_DCM)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Cell type', fontsize=11)
    ax.set_ylabel('Z-score', fontsize=11)
    ax.axhline(0,    color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(1.65, color='red',   linestyle=':',  alpha=0.5, linewidth=1,
               label='Z = 1.65')
    ax.legend(fontsize=9)
    plt.tight_layout()
    out = os.path.join(plot_dir, f'{ct_name}_module_score_Z_DCM_stats.pdf')
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out}")

# ─────────────────────────────────────────────
# COMBINED FIGURE
# ─────────────────────────────────────────────
n = len(MODULE_ZSCORE_COLS)
fig, axes = plt.subplots(1, n, figsize=(6*n, 6))
if n == 1:
    axes = [axes]

for idx, z_col in enumerate(MODULE_ZSCORE_COLS):
    ax      = axes[idx]
    ct_name = z_col.replace('_module_score_Z_all', '')

    plot_data = pd.DataFrame({
        'zscore':    adata_dcm.obs[z_col].values,
        'cell_type': adata_dcm.obs[cell_type_col].astype(str).values,
    }).dropna()
    plot_data = plot_data[plot_data['cell_type'].isin(DESIRED_ORDER)]
    if plot_data.empty:
        continue

    avail = [ct for ct in DESIRED_ORDER if ct in plot_data['cell_type'].values]
    plot_data['cell_type'] = pd.Categorical(plot_data['cell_type'],
                                             categories=avail, ordered=True)
    palette = [cell_type_colors.get(ct, '#D3D3D3') if ct == ct_name else '#D3D3D3'
               for ct in avail]

    sns.boxplot(
        data=plot_data, x='cell_type', y='zscore',
        hue='cell_type', order=avail, hue_order=avail,
        palette=palette, ax=ax, linewidth=1.5, fliersize=3, legend=False,
    )

    comps   = [r for r in all_stats if r['module'] == ct_name]
    y_max   = plot_data['zscore'].max()
    y_range = plot_data['zscore'].max() - plot_data['zscore'].min()
    for i, comp in enumerate(comps):
        x1 = avail.index(comp['group1'])
        x2 = avail.index(comp['group2'])
        y  = y_max + (i+1) * y_range * 0.12
        ax.plot([x1, x1, x2, x2],
                [y - y_range*0.02, y, y, y - y_range*0.02],
                color='black', linewidth=1)
        ax.text((x1+x2)/2, y + y_range*0.01, sig_label(comp['t_pval']),
                ha='center', va='bottom', fontsize=10)

    ax.set_ylim(plot_data['zscore'].min() - y_range*0.1,
                y_max + (len(comps)+1) * y_range * 0.15)
    ax.set_title(ct_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Cell type', fontsize=11)
    ax.set_ylabel('Z-score' if idx == 0 else '', fontsize=11)
    ax.axhline(0,    color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(1.65, color='red',   linestyle=':',  alpha=0.4, linewidth=1)

plt.suptitle('Module score Z — scNMT_DCM (with significance)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
out = os.path.join(plot_dir, 'all_modules_score_Z_DCM_stats_combined.pdf')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {out}")

# ─────────────────────────────────────────────
# STATISTICS TABLE + MULTIPLE TESTING CORRECTION
# ─────────────────────────────────────────────
stats_df = pd.DataFrame(all_stats)

if not stats_df.empty:
    # Bonferroni
    n_tests = len(stats_df)
    stats_df['t_pval_bonf'] = (stats_df['t_pval'] * n_tests).clip(upper=1.0)
    stats_df['u_pval_bonf'] = (stats_df['u_pval'] * n_tests).clip(upper=1.0)

    # BH-FDR
    try:
        from scipy.stats import false_discovery_control
        stats_df['t_pval_fdr'] = false_discovery_control(stats_df['t_pval'].values, method='bh')
        stats_df['u_pval_fdr'] = false_discovery_control(stats_df['u_pval'].values, method='bh')
    except Exception:
        for col, src in [('t_pval_fdr', 't_pval'), ('u_pval_fdr', 'u_pval')]:
            ranks = rankdata(stats_df[src])
            stats_df[col] = (stats_df[src] * n_tests / ranks).clip(upper=1.0)

    for src, dst in [('t_pval','t_sig'), ('t_pval_fdr','t_sig_fdr'),
                     ('u_pval','u_sig'), ('u_pval_fdr','u_sig_fdr')]:
        stats_df[dst] = stats_df[src].apply(sig_label)

    out_csv = os.path.join(plot_dir, 'module_zscore_stats_DCM.csv')
    stats_df.to_csv(out_csv, index=False)
    print(f"\nStatistics saved: {out_csv}")

    print("\nSUMMARY:")
    cols = ['module','group1','group2','mean_diff','t_pval','t_sig',
            'u_pval','u_sig','cohens_d']
    print(stats_df[cols].to_string(index=False))

print(f"\nAll outputs in: {plot_dir}")

DCM cells (excl. Minus): 405

IGs
  RGCs vs INs:  t-test p=9.73e-02 ns  MWU p=1.15e-02 *  d=-0.384
  RGCs vs IGs:  t-test p=1.94e-14 ***  MWU p=9.71e-12 ***  d=-2.302
  INs vs IGs:  t-test p=8.17e-14 ***  MWU p=2.73e-13 ***  d=-1.388
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/zscore_boxplots/IGs_module_score_Z_DCM_stats.pdf

INs
  RGCs vs INs:  t-test p=2.34e-18 ***  MWU p=5.38e-13 ***  d=-2.398
  RGCs vs IGs:  t-test p=3.50e-02 *  MWU p=6.34e-02 ns  d=-0.528
  INs vs IGs:  t-test p=1.83e-19 ***  MWU p=7.05e-17 ***  d=1.758
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/zscore_boxplots/INs_module_score_Z_DCM_stats.pdf

RGCs
  RGCs vs INs:  t-test p=2.83e-12 ***  MWU p=5.54e-12 ***  d=1.794
  RGCs vs IGs:  t-test p=4.

In [55]:
#Zscore in adata

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [35]:
##############################################
# Z-SCORE BOXPLOTS — ALL ASSAYS INCLUDING REF
# Computes module scores + z-scores for ALL
# assays (scNMT_DCM, scNMT_RNA, ref) then
# plots boxplots per cell type per assay.
#
# Mirrors the reference code logic exactly,
# using _all suffix to keep columns separate
# from the existing scNMT-only z-scores.
#
# Requires in memory:
#   adata, ttest_markers_by_ct, best_outdir,
#   cell_types (from gene_scoring_zscore.py)
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import scanpy as sc
import scipy.sparse as sp

# ─────────────────────────────────────────────
# PATHS & CONFIG
# ─────────────────────────────────────────────
plot_dir = os.path.join(best_outdir, 'zscore_boxplots')
os.makedirs(plot_dir, exist_ok=True)

DESIRED_ORDER = ['RGCs', 'INs', 'IGs']
REF_BATCHES   = ['ref_D2', 'ref_D4', 'multi_RNA']

ASSAY_COLORS = {
    'scNMT_DCM': '#003366',
    'scNMT_RNA': '#8B0000',
    'ref':       '#808080',
}

ASSAY_PLOT_CONFIG = {
    'scNMT_DCM': {
        'cell_type_col': 'module_max_label',
        'extra_filter':  {'col': 'Dox', 'exclude_values': ['Minus', 'Unknown']},
    },
    'scNMT_RNA': {
        'cell_type_col': 'cell_type_broad',
        'extra_filter':  None,
    },
    'ref': {
        'cell_type_col': 'cell_type_broad',
        'extra_filter':  None,
    },
}

# ─────────────────────────────────────────────
# STEP 1: BUILD COMBINED SUBSET
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING ALL-ASSAY SUBSET")
print("=" * 60)

adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

scnmt_mask = adata.obs['orig.ident'].str.contains('scNMT', case=False, na=False)
ref_mask   = adata.obs['batch'].isin(REF_BATCHES)
combined   = adata[scnmt_mask | ref_mask].copy()
combined.X = combined.layers['lognorm']

def assign_score_group(row):
    if 'scNMT' in str(row['orig.ident']):
        return row['orig.ident']
    elif row['batch'] in REF_BATCHES:
        return 'ref'
    return 'other'

combined.obs['_score_group'] = combined.obs.apply(assign_score_group, axis=1)

def assign_plot_assay(row):
    oi = str(row['orig.ident'])
    if 'DCM' in oi and 'scNMT' in oi:
        return 'scNMT_DCM'
    elif 'RNA' in oi and 'scNMT' in oi:
        return 'scNMT_RNA'
    elif row['batch'] in REF_BATCHES:
        return 'ref'
    return 'other'

combined.obs['_plot_assay'] = combined.obs.apply(assign_plot_assay, axis=1)

print(f"Total cells: {combined.n_obs}")
print(combined.obs['_score_group'].value_counts().to_string())

# ─────────────────────────────────────────────
# STEP 2: MODULE SCORES PER GROUP
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: COMPUTING MODULE SCORES")
print("=" * 60)

genes_in_data = set(combined.var_names)
filtered_gene_sets = {
    ct: [g for g in genes if g in genes_in_data]
    for ct, genes in ttest_markers_by_ct.items()
}
for ct, g in filtered_gene_sets.items():
    print(f"  {ct}: {len(g)} genes")

for ct in filtered_gene_sets:
    combined.obs[f'{ct}_score_raw_all']        = np.nan
    combined.obs[f'{ct}_module_score_raw_all'] = np.nan

for grp in combined.obs['_score_group'].unique():
    print(f"\n  Group: {grp}")
    mask = combined.obs['_score_group'] == grp
    sub  = combined[mask].copy()
    print(f"    {sub.n_obs} cells")

    for ct, genes in filtered_gene_sets.items():
        if not genes:
            continue
        X = sub[:, genes].X
        sub.obs[f'{ct}_score_raw_all'] = (
            np.asarray(X.toarray() if sp.issparse(X) else X).mean(axis=1)
        )
        sc.tl.score_genes(
            sub, gene_list=genes,
            score_name=f'{ct}_module_score_raw_all',
            use_raw=False,
        )
        combined.obs.loc[sub.obs_names, f'{ct}_score_raw_all']        = sub.obs[f'{ct}_score_raw_all']
        combined.obs.loc[sub.obs_names, f'{ct}_module_score_raw_all'] = sub.obs[f'{ct}_module_score_raw_all']

# ─────────────────────────────────────────────
# STEP 3: Z-SCORE PER GROUP
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Z-SCORING PER GROUP")
print("=" * 60)

MODULE_ZSCORE_COLS = []
for ct in filtered_gene_sets:
    raw_col = f'{ct}_module_score_raw_all'
    if raw_col not in combined.obs:
        continue
    z_col = f'{ct}_module_score_Z_all'
    MODULE_ZSCORE_COLS.append(z_col)
    combined.obs[z_col] = np.nan
    for grp in combined.obs['_score_group'].unique():
        mask = combined.obs['_score_group'] == grp
        combined.obs.loc[mask, z_col] = zscore(
            combined.obs.loc[mask, raw_col], nan_policy='omit'
        )

print(f"Module Z-score columns: {MODULE_ZSCORE_COLS}")

# ── Write all z-score and raw score columns back to adata.obs ──
# This means downstream scripts (umap_transitions.py etc.) can read
# directly from adata.obs without needing the combined object.
for col in MODULE_ZSCORE_COLS:
    adata.obs[col] = np.nan
    idx = adata.obs_names.intersection(combined.obs_names)
    adata.obs.loc[idx, col] = combined.obs.loc[idx, col].values

# Also write raw score columns back
for ct in filtered_gene_sets:
    for suffix in ['_module_score_raw_all', '_score_raw_all']:
        col = f'{ct}{suffix}'
        if col in combined.obs.columns:
            adata.obs[col] = np.nan
            idx = adata.obs_names.intersection(combined.obs_names)
            adata.obs.loc[idx, col] = combined.obs.loc[idx, col].values

print(f"Written to adata.obs: {MODULE_ZSCORE_COLS}")

# ─────────────────────────────────────────────
# STEP 4: BUILD PLOT DATA
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: BUILDING PLOT DATA")
print("=" * 60)

all_data = []
for assay_label, cfg in ASSAY_PLOT_CONFIG.items():
    mask = combined.obs['_plot_assay'] == assay_label

    if cfg['extra_filter']:
        ecol  = cfg['extra_filter']['col']
        evals = cfg['extra_filter']['exclude_values']
        if ecol in combined.obs.columns:
            mask = mask & ~combined.obs[ecol].astype(str).isin(evals)

    ct_col = cfg['cell_type_col']
    if ct_col not in combined.obs.columns:
        print(f"  WARNING: '{ct_col}' not in obs for {assay_label}")
        continue

    sub = combined.obs[mask].copy()
    sub['_ct']    = sub[ct_col].astype(str)
    sub['_assay'] = assay_label
    sub = sub[sub['_ct'].isin(DESIRED_ORDER)]

    print(f"  {assay_label}: {len(sub)} cells — {sub['_ct'].value_counts().to_dict()}")
    if not sub.empty:
        all_data.append(sub)

plot_df = pd.concat(all_data, ignore_index=True)

avail_assays  = [a for a in ASSAY_COLORS if a in plot_df['_assay'].unique()]
plot_df['_ct']    = pd.Categorical(plot_df['_ct'],
                                    categories=DESIRED_ORDER, ordered=True)
plot_df['_assay'] = pd.Categorical(plot_df['_assay'],
                                    categories=avail_assays, ordered=True)
palette = [ASSAY_COLORS[a] for a in avail_assays]

# ─────────────────────────────────────────────
# STEP 5: INDIVIDUAL BOXPLOTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: PLOTTING")
print("=" * 60)

for z_col in MODULE_ZSCORE_COLS:
    ct_name = z_col.replace('_module_score_Z_all', '')
    data    = plot_df[['_ct', '_assay', z_col]].dropna()

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(
        data=data, x='_ct', y=z_col, hue='_assay',
        order=DESIRED_ORDER, hue_order=avail_assays,
        palette=palette, ax=ax, linewidth=1, fliersize=2,
    )
    ax.set_title(f'{ct_name} module score Z', fontsize=13, fontweight='bold')
    ax.set_xlabel('Cell type', fontsize=11)
    ax.set_ylabel('Z-score', fontsize=11)
    ax.axhline(0,    color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(1.65, color='gray',  linestyle=':',  alpha=0.4, linewidth=1,
               label='Z = 1.65')
    ax.legend(title='Assay', loc='upper right', fontsize=9)
    plt.tight_layout()
    out = os.path.join(plot_dir, f'{ct_name}_module_score_Z_boxplot.pdf')
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out}")

# ─────────────────────────────────────────────
# STEP 6: COMBINED FIGURE
# ─────────────────────────────────────────────
n = len(MODULE_ZSCORE_COLS)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), sharey=False)
if n == 1:
    axes = [axes]

for idx, z_col in enumerate(MODULE_ZSCORE_COLS):
    ax      = axes[idx]
    ct_name = z_col.replace('_module_score_Z_all', '')
    data    = plot_df[['_ct', '_assay', z_col]].dropna()

    sns.boxplot(
        data=data, x='_ct', y=z_col, hue='_assay',
        order=DESIRED_ORDER, hue_order=avail_assays,
        palette=palette, ax=ax, linewidth=1, fliersize=2,
    )
    ax.set_title(ct_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Cell type', fontsize=11)
    ax.set_ylabel('Z-score' if idx == 0 else '', fontsize=11)
    ax.axhline(0,    color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(1.65, color='gray',  linestyle=':',  alpha=0.4, linewidth=1)

    if idx == n - 1:
        ax.legend(title='Assay', loc='upper right', fontsize=9)
    elif ax.get_legend():
        ax.get_legend().remove()

plt.suptitle('Module score Z per cell type per assay',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = os.path.join(plot_dir, 'all_modules_score_Z_boxplot_combined.pdf')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved: {out}")

print(f"\nAll boxplots saved in: {plot_dir}")

STEP 1: BUILDING ALL-ASSAY SUBSET
Total cells: 15920
_score_group
ref                14792
scNMT_pool1_RNA      723
scNMT_pool1_DCM      405

STEP 2: COMPUTING MODULE SCORES
  IGs: 1089 genes
  INs: 567 genes
  RGCs: 399 genes

  Group: scNMT_pool1_DCM
    405 cells

  Group: scNMT_pool1_RNA
    723 cells

  Group: ref
    14792 cells

STEP 3: Z-SCORING PER GROUP
Module Z-score columns: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']
Written to adata.obs: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']

STEP 4: BUILDING PLOT DATA
  scNMT_DCM: 176 cells — {'INs': 95, 'IGs': 57, 'RGCs': 24}
  scNMT_RNA: 723 cells — {'RGCs': 523, 'IGs': 115, 'INs': 85}
  ref: 0 cells — {}

STEP 5: PLOTTING
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/zscore_boxplots/IGs_module_score_Z_boxplot.pdf
  Saved: /m

In [ ]:
#UMAP specific cells

In [46]:
##############################################
# UMAP TRANSITIONS + KDE CONTOURS + HEATMAPS
# Adapted from reference code.
#
# Requires in memory:
#   adata, combined (from zscore_boxplots.py),
#   best_outdir, cell_types (DESIRED_ORDER),
#   MODULE_ZSCORE_COLS
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
trans_dir = os.path.join(best_outdir, 'umap_transitions')
heat_dir  = os.path.join(best_outdir, 'heatmap_transitions')
os.makedirs(trans_dir, exist_ok=True)
os.makedirs(heat_dir,  exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}
FALLBACK_COLOR = '#7f7f7f'
DESIRED_ORDER  = ['RGCs', 'INs', 'IGs']
REF_BATCHES    = ['ref_D2', 'ref_D4', 'multi_RNA']
BW_METHOD      = 0.5
LEVEL_FRAC     = 0.08
VCENTER_DCM    = 1.0   # vcenter for DCM panel  (e.g. 1.0)
VCENTER_RNA    = 0.0   # vcenter for RNA panel  (e.g. 0.0 or 0.5)

# ─────────────────────────────────────────────
# ASSIGN cell_type_broad ON adata
# Maps leiden cluster numbers → cell type names
# for ALL cells (scNMT + ref).
# ─────────────────────────────────────────────
CLUSTER_TO_CELLTYPE = {
    '0': 'RGCs',
    '1': 'INs',
    '2': 'RGCs',
    '3': 'IGs',
}
LEIDEN_COL = 'leiden_annot_0.1'

adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype(str)
adata.obs['cell_type_broad'] = (
    adata.obs[LEIDEN_COL]
    .str.split('_').str[-1]          # strip numeric prefix if present
    .map(CLUSTER_TO_CELLTYPE)
    .fillna(
        adata.obs[LEIDEN_COL]
        .str.split('_').str[-1]      # keep original stripped label if not in map
    )
)
print("cell_type_broad distribution after reassignment:")
print(adata.obs['cell_type_broad'].value_counts().to_string())

# ─────────────────────────────────────────────
# BUILD plot_df
# ─────────────────────────────────────────────
adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

umap    = adata.obsm['X_umap']
plot_df = pd.DataFrame({
    'UMAP1':            umap[:, 0],
    'UMAP2':            umap[:, 1],
    'batch':            adata.obs['batch'].values,
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).str.split('_').str[-1].values,
}, index=adata.obs_names)

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].astype(str).values

plot_df['_assay'] = 'other'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True), '_assay'] = 'DCM'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*RNA', case=False, na=False, regex=True), '_assay'] = 'RNA'
plot_df.loc[plot_df['batch'].isin(REF_BATCHES), '_assay'] = 'ref'

# Unified display label: DCM → module_max_label, RNA/ref → cell_type_broad
plot_df['_display_label'] = plot_df['cell_type_broad']
plot_df.loc[plot_df['_assay'] == 'DCM', '_display_label'] =     plot_df.loc[plot_df['_assay'] == 'DCM', 'module_max_label']
print("Display label value counts:")
print(plot_df['_display_label'].value_counts().head(10).to_string())

# Attach z-scores directly from adata.obs
# (computed by zscore_boxplots.py and stored on adata with _module_score_Z_all suffix)
# If the columns are only on 'combined' (a subset), copy them onto adata.obs first.
z_cols = [c for c in adata.obs.columns if c.endswith('_module_score_Z_all')]

if not z_cols and 'combined' in dir():
    # Fall back: pull from combined if zscore_boxplots.py was run on that object
    z_cols_combined = [c for c in combined.obs.columns if c.endswith('_module_score_Z_all')]
    for col in z_cols_combined:
        adata.obs[col] = np.nan
        idx = adata.obs_names.intersection(combined.obs_names)
        adata.obs.loc[idx, col] = combined.obs.loc[idx, col].values
    z_cols = z_cols_combined
    print(f"Z-score columns copied from combined → adata.obs: {z_cols}")
else:
    print(f"Z-score columns found in adata.obs: {z_cols}")

for col in z_cols:
    plot_df[col] = adata.obs[col].values

print(f"Z-score columns attached to plot_df: {z_cols}")

# ─────────────────────────────────────────────
# MATCH DCM → RNA VIA plate_well
# ─────────────────────────────────────────────
dcm_mask = plot_df['_assay'] == 'DCM'
if 'Dox' in plot_df.columns:
    dcm_mask = dcm_mask & ~plot_df['Dox'].isin(['Minus', 'Unknown'])

dcm_known = plot_df[dcm_mask & plot_df['module_max_label'].isin(DESIRED_ORDER)]
rna_cells = plot_df[plot_df['_assay'] == 'RNA']

common_pws  = set(dcm_known['plate_well']) & set(rna_cells['plate_well'])
connections = []
for pw in common_pws:
    d = dcm_known[dcm_known['plate_well'] == pw]
    r = rna_cells[rna_cells['plate_well'] == pw]
    if len(d) == 1 and len(r) == 1:
        dr, rr = d.iloc[0], r.iloc[0]
        row = {
            'plate_well': pw,
            'dcm_umap1':  dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1':  rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_label':  dr['module_max_label'],
            'rna_label':  rr['cell_type_broad'],
            'transition': f"{dr['module_max_label']} -> {rr['cell_type_broad']}",
            'Dox':        dr.get('Dox', 'Unknown'),
        }
        for col in z_cols:
            ct = col.replace('_module_score_Z_all', '')
            row[f'dcm_{ct}_zscore'] = dr.get(col, np.nan)
            row[f'rna_{ct}_zscore'] = rr.get(col, np.nan)
        connections.append(row)

conn_df = pd.DataFrame(connections)
print(f"\nConnections (all): {len(conn_df)}")

# ── Filter: keep only cells where the label matches the highest z-score ──
# DCM: dcm_label must equal the cell type with the highest dcm_*_zscore
# RNA: rna_label must equal the cell type with the highest rna_*_zscore
if not conn_df.empty:
    dcm_z_score_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'dcm_{ct}_zscore' in conn_df.columns]
    rna_z_score_cols = [f'rna_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'rna_{ct}_zscore' in conn_df.columns]

    def _argmax_label(row, z_cols, prefix):
        vals = {col.replace(prefix, '').replace('_zscore', ''): row[col]
                for col in z_cols if not np.isnan(row[col])}
        return max(vals, key=vals.get) if vals else None

    conn_df['dcm_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, dcm_z_score_cols, 'dcm_'), axis=1)
    conn_df['rna_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, rna_z_score_cols, 'rna_'), axis=1)

    dcm_match = conn_df['dcm_label'] == conn_df['dcm_max_label']
    rna_match = conn_df['rna_label'] == conn_df['rna_max_label']

    print(f"  DCM label matches highest z-score: {dcm_match.sum()}/{len(conn_df)}")
    print(f"  RNA label matches highest z-score: {rna_match.sum()}/{len(conn_df)}")
    print(f"  Both match: {(dcm_match & rna_match).sum()}/{len(conn_df)}")

    conn_df = conn_df[dcm_match & rna_match].drop(
        columns=['dcm_max_label', 'rna_max_label']
    ).reset_index(drop=True)
    print(f"Connections after z-score filtering: {len(conn_df)}")



if not conn_df.empty:
    print(conn_df['transition'].value_counts().to_string())
    print(pd.crosstab(conn_df['dcm_label'], conn_df['rna_label'], margins=True))

# ─────────────────────────────────────────────
# HELPER: background + cluster labels
# ─────────────────────────────────────────────
def draw_background(ax):
    # All cells coloured by cluster: DCM by module_max_label,
    # RNA + ref by cell_type_broad (via _display_label column)
    for ct in DESIRED_ORDER:
        sub = plot_df[plot_df['_display_label'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=6,
                       c=cell_type_colors[ct], alpha=0.2,
                       marker='o', zorder=1, rasterized=True)
    other = plot_df[~plot_df['_display_label'].isin(DESIRED_ORDER)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=6,
                   c='lightgray', alpha=0.08,
                   marker='o', zorder=1, rasterized=True)

    # Label each cluster at the centroid of its cells
    for ct in DESIRED_ORDER:
        sub = plot_df[plot_df['_display_label'] == ct]
        if len(sub) < 5:
            continue
        cx, cy = sub['UMAP1'].median(), sub['UMAP2'].median()
        ax.text(cx, cy, ct, ha='center', va='center',
                fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)

# ─────────────────────────────────────────────
# HELPER: UMAP transition plot
# Arrow colour = RNA destination
# ─────────────────────────────────────────────
# ── Arrow style config — modify here to change all arrows globally ──
ARROW_STYLE    = '-|>'   # arrowstyle: '-|>', '->', 'fancy', etc.
ARROW_MUTATION = 15      # arrowhead size
ARROW_LW       = 1.5     # linewidth
ARROW_ALPHA    = 0.75
ARROW_ZORDER   = 3


def plot_umap_transitions(subset_df, title_str, save_path):
    if subset_df.empty:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))
    draw_background(ax)
    pw_set = set(subset_df['plate_well'])

    # ── Dots: rasterized — background cells + highlighted paired cells ──
    for ct in subset_df['dcm_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'DCM') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['module_max_label'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='s', zorder=4, rasterized=True)

    for ct in subset_df['rna_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'RNA') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['cell_type_broad'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    # ── Arrows: vectorized (not rasterized — editable in Illustrator/Inkscape) ──
    for _, row in subset_df.iterrows():
        r_col = cell_type_colors.get(row['rna_label'], FALLBACK_COLOR)
        ax.add_patch(FancyArrowPatch(
            (row['dcm_umap1'], row['dcm_umap2']),
            (row['rna_umap1'], row['rna_umap2']),
            arrowstyle=ARROW_STYLE,
            mutation_scale=ARROW_MUTATION,
            color=r_col,
            alpha=ARROW_ALPHA,
            linewidth=ARROW_LW,
            zorder=ARROW_ZORDER,
        ))

    legend = []
    for ct in subset_df['dcm_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend.append(mlines.Line2D([], [], color=c, marker='s', linestyle='None',
                                     markersize=10, markeredgecolor='black',
                                     label=f'DCM ({ct})'))
    for ct in subset_df['rna_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black',
                           label=f'RNA ({ct})'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8,
                           label=f'Arrow → {ct}'),
        ]

    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")



# ─────────────────────────────
# HELPER: Z-SCORE HEATMAP
# DCM (top) + RNA (bottom), sorted by group.
# Saves: shared scale, per-column scale,
# and per-Dox split (both scales).
# ─────────────────────────────
def _add_color_strip(ax, label_series, n_cells):
    """Draw one coloured rectangle per column just above the heatmap panel.
    Each rectangle spans 1/n_cells of the axes width and is coloured by
    cell_type_colors of that cell's label."""
    for i, lbl in enumerate(label_series):
        c = cell_type_colors.get(lbl, FALLBACK_COLOR)
        ax.add_patch(plt.Rectangle(
            (i / n_cells, 1.01), 1 / n_cells, 0.05,
            transform=ax.transAxes,
            facecolor=c, edgecolor='none',
            clip_on=False, zorder=6,
        ))


def _draw_heatmap_panel(ax, matrix, row_labels, label,
                        group_col_vals, cell_label_vals, n_cells,
                        vcenter=0.0):
    """Draw one DCM or RNA heatmap panel.
    group_col_vals  : Series used for group separators + labels below.
    cell_label_vals : Series used for the per-column colour strip above
                      (typically dcm_label for DCM panel, rna_label for RNA).
    """
    mat  = matrix.copy().astype(float)
    norm = TwoSlopeNorm(vmin=vcenter - 3, vcenter=vcenter, vmax=vcenter + 3)
    cmap = plt.cm.RdBu_r

    im = ax.imshow(mat, aspect='auto', cmap=cmap, norm=norm)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_xticks([])
    ax.set_ylabel('Module', fontsize=11, fontweight='bold')
    ax.set_title(label, fontsize=12, fontweight='bold')

    # Group separators + labels below
    changes = group_col_vals.ne(group_col_vals.shift()).to_numpy().nonzero()[0]
    for x in changes:
        ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
    prev = 0
    for x in list(changes) + [n_cells]:
        mid = (prev + x) / 2
        ct  = group_col_vals.iloc[prev]
        ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                fontweight='bold', color='black')
        prev = x

    # Coloured rectangle strip above each column
    _add_color_strip(ax, cell_label_vals, n_cells)

    cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
    cbar.set_label(f'Z-score (vcenter={vcenter})', fontsize=9)
    cbar.ax.axhline(y=vcenter, color='black', lw=1, ls='--')


def plot_zscore_heatmap(subset_df, title_str, save_path,
                        group_col='dcm_label'):
    if subset_df.empty:
        print(f"  Skipping heatmap -- no cells"); return
    dcm_z_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER]
    if not any(c in subset_df.columns for c in dcm_z_cols):
        print(f"  No z-score columns -- skipping heatmap"); return

    def _sort(df):
        df = df.copy()
        df['_sort'] = df[group_col].map(
            {ct: i for i, ct in enumerate(DESIRED_ORDER)}
        ).fillna(99)
        df['_self_z'] = df.apply(
            lambda r: r.get(f'dcm_{r[group_col]}_zscore', 0)
            if r[group_col] in DESIRED_ORDER else 0, axis=1)
        return df.sort_values(['_sort', '_self_z'],
                               ascending=[True, False]).reset_index(drop=True)

    def _mats(df):
        dcm = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        rna = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        return dcm, rna

    def _save(df, path, extra=''):
        if df.empty: return
        df = _sort(df)
        n  = len(df)
        fw = max(14, n * 0.18)
        dm, rm = _mats(df)
        fig, axes = plt.subplots(2, 1, figsize=(fw, 10))
        # DCM panel: group separators by group_col, strip coloured by dcm_label
        _draw_heatmap_panel(axes[0], dm, DESIRED_ORDER, 'DCM (GMM label)',
                            df[group_col], df['dcm_label'], n, vcenter=VCENTER_DCM)
        # RNA panel: group separators by group_col, strip coloured by rna_label
        _draw_heatmap_panel(axes[1], rm, DESIRED_ORDER, 'RNA (cell_type_broad)',
                            df[group_col], df['rna_label'], n, vcenter=VCENTER_RNA)
        ttl = title_str + (f'  |  {extra}' if extra else '')
        fig.suptitle(ttl, fontsize=13, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")

    stem, ext = os.path.splitext(save_path)
    # 1. Full
    _save(subset_df, save_path)
    # 2. Per-Dox split
    if 'Dox' in subset_df.columns:
        dox_vals = sorted(subset_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub = subset_df[subset_df['Dox'] == dox]
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                _save(sub, f'{stem}_dox{ds}{ext}', extra=f'Dox={dox}')


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
if conn_df.empty:
    print("No connections found — check plate_well and assay labels.")
else:
    for transition in conn_df['transition'].unique():
        sub   = conn_df[conn_df['transition'] == transition]
        fname = transition.replace(' -> ', '_to_').replace(' ', '_')
        plot_umap_transitions(
            sub,
            title_str = f'{transition}  (n={len(sub)})\nArrow colour = RNA destination',
            save_path = os.path.join(trans_dir, f'umap_{fname}.pdf'),
        )
        plot_zscore_heatmap(
            sub,
            title_str = f'Z-score heatmap — {transition}  (n={len(sub)})',
            save_path = os.path.join(heat_dir, f'heatmap_{fname}.pdf'),
        )

    # Combined UMAP — all filtered cells
    plot_umap_transitions(
        conn_df,
        title_str = f'All DCM \u2192 RNA transitions  (n={len(conn_df)} pairs)\nArrow colour = RNA destination',
        save_path = os.path.join(trans_dir, 'umap_all_transitions.pdf'),
    )

    # ── Per-Dox UMAP — all transitions, one plot per Dox condition ──
    if 'Dox' in conn_df.columns:
        dox_vals = sorted(conn_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub_dox = conn_df[conn_df['Dox'] == dox]
                if sub_dox.empty:
                    continue
                ds = str(dox).replace(' ', '_').replace('/', '-')
                plot_umap_transitions(
                    sub_dox,
                    title_str = (f'All transitions  |  Dox={dox}  (n={len(sub_dox)} pairs)\n'
                                 f'Arrow colour = RNA destination'),
                    save_path = os.path.join(trans_dir,
                                             f'umap_all_transitions_dox{ds}.pdf'),
                )

    # ── Heatmap: RGCs/INs only, 4 transitions, RGCs+INs module scores only ──
    RGCIN_TRANSITIONS = ['RGCs -> RGCs', 'RGCs -> INs', 'INs -> INs', 'INs -> RGCs']
    RGCIN_SCORES      = ['RGCs', 'INs']   # only these two rows shown

    rgcin_sub = conn_df[conn_df['transition'].isin(RGCIN_TRANSITIONS)].copy()

    if not rgcin_sub.empty:
        # Sort by the fixed transition order, then by DCM RGCs z-score descending
        trans_order_map = {t: i for i, t in enumerate(RGCIN_TRANSITIONS)}
        rgcin_sub = rgcin_sub.copy()
        rgcin_sub['_t_order'] = rgcin_sub['transition'].map(trans_order_map).fillna(99)
        rgcin_sub['_self_z']  = rgcin_sub.apply(
            lambda r: r.get(f'dcm_{r["dcm_label"]}_zscore', 0)
            if r['dcm_label'] in RGCIN_SCORES else 0, axis=1
        )
        rgcin_sub = rgcin_sub.sort_values(
            ['_t_order', '_self_z'], ascending=[True, False]
        ).drop(columns=['_t_order', '_self_z']).reset_index(drop=True)

        def _save_rgcin(df, path, title_extra=''):
            if df.empty: return
            n  = len(df)
            fw = max(14, n * 0.18)

            dcm_mat = np.array(
                [[row.get(f'dcm_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T
            rna_mat = np.array(
                [[row.get(f'rna_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T

            fig, axes = plt.subplots(2, 1, figsize=(fw, 8))
            # DCM: group separators by transition, strip coloured by dcm_label
            _draw_heatmap_panel(axes[0], dcm_mat, RGCIN_SCORES,
                                'DCM (GMM label)', df['transition'],
                                df['dcm_label'], n, vcenter=VCENTER_DCM)
            # RNA: group separators by transition, strip coloured by rna_label
            _draw_heatmap_panel(axes[1], rna_mat, RGCIN_SCORES,
                                'RNA (cell_type_broad)', df['transition'],
                                df['rna_label'], n, vcenter=VCENTER_RNA)

            ttl = (f'Z-score heatmap — RGCs/INs transitions  (n={n})'
                   + (f'  |  {title_extra}' if title_extra else '')
                   + f'  DCM vcenter={VCENTER_DCM}, RNA vcenter={VCENTER_RNA}')
            fig.suptitle(ttl, fontsize=13, fontweight='bold')
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  Saved: {path}")

        stem = os.path.join(heat_dir, 'heatmap_RGCsINs_transitions')
        _save_rgcin(rgcin_sub, f'{stem}.pdf')
        if 'Dox' in rgcin_sub.columns:
            for dox in sorted(rgcin_sub['Dox'].dropna().unique()):
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                sub = rgcin_sub[rgcin_sub['Dox'] == dox]
                _save_rgcin(sub, f'{stem}_dox{ds}.pdf', title_extra=f'Dox={dox}')
    else:
        print("  No RGCs/INs transitions found in conn_df")

    # ── Heatmap: all transitions, grouped by DCM source label ──
    plot_zscore_heatmap(
        conn_df,
        title_str = f'Z-score heatmap — all transitions  (n={len(conn_df)})',
        save_path = os.path.join(heat_dir, 'heatmap_all_transitions.pdf'),
    )

    # ── Per DCM source: UMAP + heatmap for all RNA destinations ──
    # Equivalent of the reference plot_umap(sec_to_ent_clear, ...) +
    # plot_heatmap(sec_to_ent_clear, ...) calls in Step 7.
    for dcm_ct in DESIRED_ORDER:
        sub_dcm = conn_df[conn_df['dcm_label'] == dcm_ct]
        if sub_dcm.empty:
            continue
        n = len(sub_dcm)
        rna_dests = sorted(sub_dcm['rna_label'].unique())
        plot_umap_transitions(
            sub_dcm,
            title_str = (f'DCM source: {dcm_ct} → all RNA destinations  (n={n})\n'
                         f'RNA: {", ".join(rna_dests)} | Arrow colour = RNA destination'),
            save_path = os.path.join(trans_dir,
                                     f'umap_clear_{dcm_ct}_to_all_RNA.pdf'),
        )
        plot_zscore_heatmap(
            sub_dcm,
            title_str = (f'Z-score heatmap — DCM source: {dcm_ct}  '
                         f'(n={n}, all RNA destinations)'),
            save_path = os.path.join(heat_dir,
                                     f'heatmap_DCM_source_{dcm_ct}_all_RNA.pdf'),
            group_col = 'rna_label',
        )

    # Save tables
    conn_df.to_csv(os.path.join(trans_dir, 'DCM_RNA_connections.csv'), index=False)
    (conn_df.groupby('transition').size()
     .reset_index(name='count')
     .assign(pct=lambda d: 100 * d['count'] / d['count'].sum())
     .sort_values('count', ascending=False)
     .to_csv(os.path.join(trans_dir, 'DCM_RNA_transition_summary.csv'), index=False))

print(f"\nOutputs in:\n  {trans_dir}\n  {heat_dir}")

cell_type_broad distribution after reassignment:
cell_type_broad
RGCs    9634
INs     4956
IGs     1330
Display label value counts:
_display_label
RGCs         9572
INs          4748
IGs          1371
ambiguous     229
Z-score columns found in adata.obs: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']
Z-score columns attached to plot_df: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']

Connections (all): 174
  DCM label matches highest z-score: 151/174
  RNA label matches highest z-score: 116/174
  Both match: 97/174
Connections after z-score filtering: 97
transition
INs -> RGCs     22
INs -> INs      17
IGs -> RGCs     16
IGs -> IGs      15
RGCs -> RGCs    12
INs -> IGs       8
RGCs -> IGs      5
IGs -> INs       1
RGCs -> INs      1
rna_label  IGs  INs  RGCs  All
dcm_label                     
IGs         15    1    16   32
INs          8   17    22   47
RGCs         5    1    12   18
All         28   19    50   97
  Sav

In [49]:
##############################################
# UMAP TRANSITIONS + KDE CONTOURS + HEATMAPS
# Adapted from reference code.
#
# Requires in memory:
#   adata, combined (from zscore_boxplots.py),
#   best_outdir, cell_types (DESIRED_ORDER),
#   MODULE_ZSCORE_COLS
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
trans_dir = os.path.join(best_outdir, 'umap_transitions')
heat_dir  = os.path.join(best_outdir, 'heatmap_transitions')
os.makedirs(trans_dir, exist_ok=True)
os.makedirs(heat_dir,  exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}
FALLBACK_COLOR = '#7f7f7f'
DESIRED_ORDER  = ['RGCs', 'INs', 'IGs']
REF_BATCHES    = ['ref_D2', 'ref_D4', 'multi_RNA']
BW_METHOD      = 0.5
LEVEL_FRAC     = 0.08
VCENTER_DCM    = 1.0   # vcenter for DCM panel  (e.g. 1.0)
VCENTER_RNA    = 0.0   # vcenter for RNA panel  (e.g. 0.0 or 0.5)

# ─────────────────────────────────────────────
# ASSIGN cell_type_broad ON adata
# Maps leiden cluster numbers → cell type names
# for ALL cells (scNMT + ref).
# ─────────────────────────────────────────────
CLUSTER_TO_CELLTYPE = {
    '0': 'RGCs',
    '1': 'INs',
    '2': 'RGCs',
    '3': 'IGs',
}
LEIDEN_COL = 'leiden_annot_0.1'

adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype(str)
adata.obs['cell_type_broad'] = (
    adata.obs[LEIDEN_COL]
    .str.split('_').str[-1]          # strip numeric prefix if present
    .map(CLUSTER_TO_CELLTYPE)
    .fillna(
        adata.obs[LEIDEN_COL]
        .str.split('_').str[-1]      # keep original stripped label if not in map
    )
)
print("cell_type_broad distribution after reassignment:")
print(adata.obs['cell_type_broad'].value_counts().to_string())

# ─────────────────────────────────────────────
# BUILD plot_df
# ─────────────────────────────────────────────
adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

umap    = adata.obsm['X_umap']
plot_df = pd.DataFrame({
    'UMAP1':            umap[:, 0],
    'UMAP2':            umap[:, 1],
    'batch':            adata.obs['batch'].values,
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).str.split('_').str[-1].values,
}, index=adata.obs_names)

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].astype(str).values

plot_df['_assay'] = 'other'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True), '_assay'] = 'DCM'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*RNA', case=False, na=False, regex=True), '_assay'] = 'RNA'
plot_df.loc[plot_df['batch'].isin(REF_BATCHES), '_assay'] = 'ref'

# Unified display label: DCM → module_max_label, RNA/ref → cell_type_broad
plot_df['_display_label'] = plot_df['cell_type_broad']
plot_df.loc[plot_df['_assay'] == 'DCM', '_display_label'] =     plot_df.loc[plot_df['_assay'] == 'DCM', 'module_max_label']
print("Display label value counts:")
print(plot_df['_display_label'].value_counts().head(10).to_string())

# Attach z-scores directly from adata.obs
# (computed by zscore_boxplots.py and stored on adata with _module_score_Z_all suffix)
# If the columns are only on 'combined' (a subset), copy them onto adata.obs first.
z_cols = [c for c in adata.obs.columns if c.endswith('_module_score_Z_all')]

if not z_cols and 'combined' in dir():
    # Fall back: pull from combined if zscore_boxplots.py was run on that object
    z_cols_combined = [c for c in combined.obs.columns if c.endswith('_module_score_Z_all')]
    for col in z_cols_combined:
        adata.obs[col] = np.nan
        idx = adata.obs_names.intersection(combined.obs_names)
        adata.obs.loc[idx, col] = combined.obs.loc[idx, col].values
    z_cols = z_cols_combined
    print(f"Z-score columns copied from combined → adata.obs: {z_cols}")
else:
    print(f"Z-score columns found in adata.obs: {z_cols}")

for col in z_cols:
    plot_df[col] = adata.obs[col].values

print(f"Z-score columns attached to plot_df: {z_cols}")

# ─────────────────────────────────────────────
# MATCH DCM → RNA VIA plate_well
# ─────────────────────────────────────────────
dcm_mask = plot_df['_assay'] == 'DCM'
if 'Dox' in plot_df.columns:
    dcm_mask = dcm_mask & ~plot_df['Dox'].isin(['Minus', 'Unknown'])

dcm_known = plot_df[dcm_mask & plot_df['module_max_label'].isin(DESIRED_ORDER)]
rna_cells = plot_df[plot_df['_assay'] == 'RNA']

common_pws  = set(dcm_known['plate_well']) & set(rna_cells['plate_well'])
connections = []
for pw in common_pws:
    d = dcm_known[dcm_known['plate_well'] == pw]
    r = rna_cells[rna_cells['plate_well'] == pw]
    if len(d) == 1 and len(r) == 1:
        dr, rr = d.iloc[0], r.iloc[0]
        row = {
            'plate_well': pw,
            'dcm_umap1':  dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1':  rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_label':  dr['module_max_label'],
            'rna_label':  rr['cell_type_broad'],
            'transition': f"{dr['module_max_label']} -> {rr['cell_type_broad']}",
            'Dox':        dr.get('Dox', 'Unknown'),
        }
        for col in z_cols:
            ct = col.replace('_module_score_Z_all', '')
            row[f'dcm_{ct}_zscore'] = dr.get(col, np.nan)
            row[f'rna_{ct}_zscore'] = rr.get(col, np.nan)
        connections.append(row)

conn_df = pd.DataFrame(connections)
print(f"\nConnections (all): {len(conn_df)}")

# ── Filter: keep only cells where the label matches the highest z-score ──
# DCM: dcm_label must equal the cell type with the highest dcm_*_zscore
# RNA: rna_label must equal the cell type with the highest rna_*_zscore
if not conn_df.empty:
    dcm_z_score_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'dcm_{ct}_zscore' in conn_df.columns]
    rna_z_score_cols = [f'rna_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'rna_{ct}_zscore' in conn_df.columns]

    def _argmax_label(row, z_cols, prefix):
        vals = {col.replace(prefix, '').replace('_zscore', ''): row[col]
                for col in z_cols if not np.isnan(row[col])}
        return max(vals, key=vals.get) if vals else None

    conn_df['dcm_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, dcm_z_score_cols, 'dcm_'), axis=1)
    conn_df['rna_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, rna_z_score_cols, 'rna_'), axis=1)

    dcm_match = conn_df['dcm_label'] == conn_df['dcm_max_label']
    rna_match = conn_df['rna_label'] == conn_df['rna_max_label']

    print(f"  DCM label matches highest z-score: {dcm_match.sum()}/{len(conn_df)}")
    print(f"  RNA label matches highest z-score: {rna_match.sum()}/{len(conn_df)}")
    print(f"  Both match: {(dcm_match & rna_match).sum()}/{len(conn_df)}")

    conn_df = conn_df[dcm_match & rna_match].drop(
        columns=['dcm_max_label', 'rna_max_label']
    ).reset_index(drop=True)
    print(f"Connections after z-score filtering: {len(conn_df)}")



if not conn_df.empty:
    print(conn_df['transition'].value_counts().to_string())
    print(pd.crosstab(conn_df['dcm_label'], conn_df['rna_label'], margins=True))

# ─────────────────────────────────────────────
# HELPER: background + cluster labels
# ─────────────────────────────────────────────
def draw_background(ax):
    # All cells coloured by cluster: DCM by module_max_label,
    # RNA + ref by cell_type_broad (via _display_label column)
    for ct in DESIRED_ORDER:
        sub = plot_df[plot_df['_display_label'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=6,
                       c=cell_type_colors[ct], alpha=0.2,
                       marker='o', zorder=1, rasterized=True)
    other = plot_df[~plot_df['_display_label'].isin(DESIRED_ORDER)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=6,
                   c='lightgray', alpha=0.08,
                   marker='o', zorder=1, rasterized=True)

    # Label each cluster at the centroid of its cells
    for ct in DESIRED_ORDER:
        sub = plot_df[plot_df['_display_label'] == ct]
        if len(sub) < 5:
            continue
        cx, cy = sub['UMAP1'].median(), sub['UMAP2'].median()
        ax.text(cx, cy, ct, ha='center', va='center',
                fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)

# ─────────────────────────────────────────────
# HELPER: UMAP transition plot
# Arrow colour = RNA destination
# ─────────────────────────────────────────────
# ── Arrow style config — modify here to change all arrows globally ──
ARROW_STYLE    = '-|>'   # arrowstyle: '-|>', '->', 'fancy', etc.
ARROW_MUTATION = 15      # arrowhead size
ARROW_LW       = 1.5     # linewidth
ARROW_ALPHA    = 0.75
ARROW_ZORDER   = 3


def plot_umap_transitions(subset_df, title_str, save_path):
    if subset_df.empty:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))
    draw_background(ax)
    pw_set = set(subset_df['plate_well'])

    # ── Dots: rasterized — background cells + highlighted paired cells ──
    for ct in subset_df['dcm_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'DCM') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['module_max_label'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='s', zorder=4, rasterized=True)

    for ct in subset_df['rna_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'RNA') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['cell_type_broad'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    # ── Arrows: vectorized (not rasterized — editable in Illustrator/Inkscape) ──
    for _, row in subset_df.iterrows():
        r_col = cell_type_colors.get(row['rna_label'], FALLBACK_COLOR)
        ax.add_patch(FancyArrowPatch(
            (row['dcm_umap1'], row['dcm_umap2']),
            (row['rna_umap1'], row['rna_umap2']),
            arrowstyle=ARROW_STYLE,
            mutation_scale=ARROW_MUTATION,
            color=r_col,
            alpha=ARROW_ALPHA,
            linewidth=ARROW_LW,
            zorder=ARROW_ZORDER,
        ))

    legend = []
    for ct in subset_df['dcm_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend.append(mlines.Line2D([], [], color=c, marker='s', linestyle='None',
                                     markersize=10, markeredgecolor='black',
                                     label=f'DCM ({ct})'))
    for ct in subset_df['rna_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black',
                           label=f'RNA ({ct})'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8,
                           label=f'Arrow → {ct}'),
        ]

    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


def plot_umap_transitions_labeled(subset_df, title_str, save_path):
    """Same as plot_umap_transitions but adds plate_well cell ID labels
    next to each DCM square and RNA circle. Saved as save_path with
    '_labeled' inserted before the extension."""
    if subset_df.empty:
        print(f"  Skipping labeled UMAP — no cells"); return

    stem, ext = os.path.splitext(save_path)
    out = f'{stem}_labeled{ext}'

    fig, ax = plt.subplots(figsize=(14, 12))
    draw_background(ax)
    pw_set = set(subset_df['plate_well'])

    # Dots (rasterized)
    for ct in subset_df['dcm_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'DCM') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['module_max_label'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='s', zorder=4, rasterized=True)

    for ct in subset_df['rna_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'RNA') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['cell_type_broad'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    # Arrows (vectorized)
    for _, row in subset_df.iterrows():
        r_col = cell_type_colors.get(row['rna_label'], FALLBACK_COLOR)
        ax.add_patch(FancyArrowPatch(
            (row['dcm_umap1'], row['dcm_umap2']),
            (row['rna_umap1'], row['rna_umap2']),
            arrowstyle=ARROW_STYLE, mutation_scale=ARROW_MUTATION,
            color=r_col, alpha=ARROW_ALPHA, linewidth=ARROW_LW,
            zorder=ARROW_ZORDER,
        ))

    # Cell ID labels next to each dot
    for _, row in subset_df.iterrows():
        label = str(row['plate_well'])
        ax.annotate(label,
                    xy=(row['dcm_umap1'], row['dcm_umap2']),
                    xytext=(4, 4), textcoords='offset points',
                    fontsize=5, color='black', zorder=7,
                    va='bottom', ha='left',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white',
                              alpha=0.5, ec='none'))
        ax.annotate(label,
                    xy=(row['rna_umap1'], row['rna_umap2']),
                    xytext=(4, -8), textcoords='offset points',
                    fontsize=5, color='dimgray', zorder=7,
                    va='top', ha='left',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white',
                              alpha=0.5, ec='none'))

    legend = []
    for ct in subset_df['dcm_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend.append(mlines.Line2D([], [], color=c, marker='s', linestyle='None',
                                     markersize=10, markeredgecolor='black',
                                     label=f'DCM ({ct})'))
    for ct in subset_df['rna_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black',
                           label=f'RNA ({ct})'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8,
                           label=f'Arrow \u2192 {ct}'),
        ]
    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=9)
    ax.set_title(title_str + '  [labeled]', fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out}")


# ─────────────────────────────
# HELPER: Z-SCORE HEATMAP
# DCM (top) + RNA (bottom), sorted by group.
# Saves: shared scale, per-column scale,
# and per-Dox split (both scales).
# ─────────────────────────────
def _add_color_strip(ax, label_series, n_cells):
    """Draw one coloured rectangle per column just above the heatmap panel.
    Each rectangle spans 1/n_cells of the axes width and is coloured by
    cell_type_colors of that cell's label."""
    for i, lbl in enumerate(label_series):
        c = cell_type_colors.get(lbl, FALLBACK_COLOR)
        ax.add_patch(plt.Rectangle(
            (i / n_cells, 1.01), 1 / n_cells, 0.05,
            transform=ax.transAxes,
            facecolor=c, edgecolor='none',
            clip_on=False, zorder=6,
        ))


def _draw_heatmap_panel(ax, matrix, row_labels, label,
                        group_col_vals, cell_label_vals, n_cells,
                        vcenter=0.0):
    """Draw one DCM or RNA heatmap panel.
    group_col_vals  : Series used for group separators + labels below.
    cell_label_vals : Series used for the per-column colour strip above
                      (typically dcm_label for DCM panel, rna_label for RNA).
    """
    mat  = matrix.copy().astype(float)
    norm = TwoSlopeNorm(vmin=vcenter - 3, vcenter=vcenter, vmax=vcenter + 3)
    cmap = plt.cm.RdBu_r

    im = ax.imshow(mat, aspect='auto', cmap=cmap, norm=norm)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_xticks([])
    ax.set_ylabel('Module', fontsize=11, fontweight='bold')
    ax.set_title(label, fontsize=12, fontweight='bold')

    # Group separators + labels below
    changes = group_col_vals.ne(group_col_vals.shift()).to_numpy().nonzero()[0]
    for x in changes:
        ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
    prev = 0
    for x in list(changes) + [n_cells]:
        mid = (prev + x) / 2
        ct  = group_col_vals.iloc[prev]
        ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                fontweight='bold', color='black')
        prev = x

    # Coloured rectangle strip above each column
    _add_color_strip(ax, cell_label_vals, n_cells)

    cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
    cbar.set_label(f'Z-score (vcenter={vcenter})', fontsize=9)
    cbar.ax.axhline(y=vcenter, color='black', lw=1, ls='--')


def plot_zscore_heatmap(subset_df, title_str, save_path,
                        group_col='dcm_label'):
    if subset_df.empty:
        print(f"  Skipping heatmap -- no cells"); return
    dcm_z_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER]
    if not any(c in subset_df.columns for c in dcm_z_cols):
        print(f"  No z-score columns -- skipping heatmap"); return

    def _sort(df):
        df = df.copy()
        df['_sort'] = df[group_col].map(
            {ct: i for i, ct in enumerate(DESIRED_ORDER)}
        ).fillna(99)
        df['_self_z'] = df.apply(
            lambda r: r.get(f'dcm_{r[group_col]}_zscore', 0)
            if r[group_col] in DESIRED_ORDER else 0, axis=1)
        return df.sort_values(['_sort', '_self_z'],
                               ascending=[True, False]).reset_index(drop=True)

    def _mats(df):
        dcm = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        rna = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        return dcm, rna

    def _save(df, path, extra=''):
        if df.empty: return
        df = _sort(df)
        n  = len(df)
        fw = max(14, n * 0.18)
        dm, rm = _mats(df)
        fig, axes = plt.subplots(2, 1, figsize=(fw, 10))
        # DCM panel: group separators by group_col, strip coloured by dcm_label
        _draw_heatmap_panel(axes[0], dm, DESIRED_ORDER, 'DCM (GMM label)',
                            df[group_col], df['dcm_label'], n, vcenter=VCENTER_DCM)
        # RNA panel: group separators by group_col, strip coloured by rna_label
        _draw_heatmap_panel(axes[1], rm, DESIRED_ORDER, 'RNA (cell_type_broad)',
                            df[group_col], df['rna_label'], n, vcenter=VCENTER_RNA)
        ttl = title_str + (f'  |  {extra}' if extra else '')
        fig.suptitle(ttl, fontsize=13, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")

    stem, ext = os.path.splitext(save_path)
    # 1. Full
    _save(subset_df, save_path)
    # 2. Per-Dox split
    if 'Dox' in subset_df.columns:
        dox_vals = sorted(subset_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub = subset_df[subset_df['Dox'] == dox]
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                _save(sub, f'{stem}_dox{ds}{ext}', extra=f'Dox={dox}')


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
if conn_df.empty:
    print("No connections found — check plate_well and assay labels.")
else:
    # Cells to highlight for IGs -> RGCs specifically
    IGS_RGCS_CELLS = {'Plate_2_C22', 'Plate_2_O17'}

    for transition in conn_df['transition'].unique():
        sub   = conn_df[conn_df['transition'] == transition]
        if transition == 'IGs -> RGCs':
            sub = sub[sub['plate_well'].isin(IGS_RGCS_CELLS)]
            print(f"  IGs -> RGCs restricted to {IGS_RGCS_CELLS}: {len(sub)} cells")
        fname = transition.replace(' -> ', '_to_').replace(' ', '_')
        plot_umap_transitions(
            sub,
            title_str = f'{transition}  (n={len(sub)})\nArrow colour = RNA destination',
            save_path = os.path.join(trans_dir, f'umap_{fname}.pdf'),
        )
        plot_umap_transitions_labeled(
            sub,
            title_str = f'{transition}  (n={len(sub)})\nArrow colour = RNA destination',
            save_path = os.path.join(trans_dir, f'umap_{fname}.pdf'),
        )
        plot_zscore_heatmap(
            sub,
            title_str = f'Z-score heatmap — {transition}  (n={len(sub)})',
            save_path = os.path.join(heat_dir, f'heatmap_{fname}.pdf'),
        )

    # Combined UMAP — all filtered cells
    plot_umap_transitions(
        conn_df,
        title_str = f'All DCM \u2192 RNA transitions  (n={len(conn_df)} pairs)\nArrow colour = RNA destination',
        save_path = os.path.join(trans_dir, 'umap_all_transitions.pdf'),
    )
    plot_umap_transitions_labeled(
        conn_df,
        title_str = f'All DCM \u2192 RNA transitions  (n={len(conn_df)} pairs)\nArrow colour = RNA destination',
        save_path = os.path.join(trans_dir, 'umap_all_transitions.pdf'),
    )

    # ── Per-Dox UMAP — all transitions, one plot per Dox condition ──
    if 'Dox' in conn_df.columns:
        dox_vals = sorted(conn_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub_dox = conn_df[conn_df['Dox'] == dox]
                if sub_dox.empty:
                    continue
                ds = str(dox).replace(' ', '_').replace('/', '-')
                plot_umap_transitions(
                    sub_dox,
                    title_str = (f'All transitions  |  Dox={dox}  (n={len(sub_dox)} pairs)\n'
                                 f'Arrow colour = RNA destination'),
                    save_path = os.path.join(trans_dir,
                                             f'umap_all_transitions_dox{ds}.pdf'),
                )
                plot_umap_transitions_labeled(
                    sub_dox,
                    title_str = (f'All transitions  |  Dox={dox}  (n={len(sub_dox)} pairs)\n'
                                 f'Arrow colour = RNA destination'),
                    save_path = os.path.join(trans_dir,
                                             f'umap_all_transitions_dox{ds}.pdf'),
                )

    # ── Heatmap: RGCs/INs only, 4 transitions, RGCs+INs module scores only ──
    RGCIN_TRANSITIONS = ['RGCs -> RGCs', 'RGCs -> INs', 'INs -> INs', 'INs -> RGCs']
    RGCIN_SCORES      = ['RGCs', 'INs']   # only these two rows shown

    rgcin_sub = conn_df[conn_df['transition'].isin(RGCIN_TRANSITIONS)].copy()

    if not rgcin_sub.empty:
        # Sort by the fixed transition order, then by DCM RGCs z-score descending
        trans_order_map = {t: i for i, t in enumerate(RGCIN_TRANSITIONS)}
        rgcin_sub = rgcin_sub.copy()
        rgcin_sub['_t_order'] = rgcin_sub['transition'].map(trans_order_map).fillna(99)
        rgcin_sub['_self_z']  = rgcin_sub.apply(
            lambda r: r.get(f'dcm_{r["dcm_label"]}_zscore', 0)
            if r['dcm_label'] in RGCIN_SCORES else 0, axis=1
        )
        rgcin_sub = rgcin_sub.sort_values(
            ['_t_order', '_self_z'], ascending=[True, False]
        ).drop(columns=['_t_order', '_self_z']).reset_index(drop=True)

        def _save_rgcin(df, path, title_extra=''):
            if df.empty: return
            n  = len(df)
            fw = max(14, n * 0.18)

            dcm_mat = np.array(
                [[row.get(f'dcm_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T
            rna_mat = np.array(
                [[row.get(f'rna_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T

            fig, axes = plt.subplots(2, 1, figsize=(fw, 8))
            # DCM: group separators by transition, strip coloured by dcm_label
            _draw_heatmap_panel(axes[0], dcm_mat, RGCIN_SCORES,
                                'DCM (GMM label)', df['transition'],
                                df['dcm_label'], n, vcenter=VCENTER_DCM)
            # RNA: group separators by transition, strip coloured by rna_label
            _draw_heatmap_panel(axes[1], rna_mat, RGCIN_SCORES,
                                'RNA (cell_type_broad)', df['transition'],
                                df['rna_label'], n, vcenter=VCENTER_RNA)

            ttl = (f'Z-score heatmap — RGCs/INs transitions  (n={n})'
                   + (f'  |  {title_extra}' if title_extra else '')
                   + f'  DCM vcenter={VCENTER_DCM}, RNA vcenter={VCENTER_RNA}')
            fig.suptitle(ttl, fontsize=13, fontweight='bold')
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  Saved: {path}")

        stem = os.path.join(heat_dir, 'heatmap_RGCsINs_transitions')
        _save_rgcin(rgcin_sub, f'{stem}.pdf')
        if 'Dox' in rgcin_sub.columns:
            for dox in sorted(rgcin_sub['Dox'].dropna().unique()):
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                sub = rgcin_sub[rgcin_sub['Dox'] == dox]
                _save_rgcin(sub, f'{stem}_dox{ds}.pdf', title_extra=f'Dox={dox}')
    else:
        print("  No RGCs/INs transitions found in conn_df")

    # ── Heatmap: all transitions, grouped by DCM source label ──
    plot_zscore_heatmap(
        conn_df,
        title_str = f'Z-score heatmap — all transitions  (n={len(conn_df)})',
        save_path = os.path.join(heat_dir, 'heatmap_all_transitions.pdf'),
    )

    # ── Per DCM source: UMAP + heatmap for all RNA destinations ──
    # Equivalent of the reference plot_umap(sec_to_ent_clear, ...) +
    # plot_heatmap(sec_to_ent_clear, ...) calls in Step 7.
    for dcm_ct in DESIRED_ORDER:
        sub_dcm = conn_df[conn_df['dcm_label'] == dcm_ct]
        if sub_dcm.empty:
            continue
        n = len(sub_dcm)
        rna_dests = sorted(sub_dcm['rna_label'].unique())
        plot_umap_transitions(
            sub_dcm,
            title_str = (f'DCM source: {dcm_ct} → all RNA destinations  (n={n})\n'
                         f'RNA: {", ".join(rna_dests)} | Arrow colour = RNA destination'),
            save_path = os.path.join(trans_dir,
                                     f'umap_clear_{dcm_ct}_to_all_RNA.pdf'),
        )
        plot_zscore_heatmap(
            sub_dcm,
            title_str = (f'Z-score heatmap — DCM source: {dcm_ct}  '
                         f'(n={n}, all RNA destinations)'),
            save_path = os.path.join(heat_dir,
                                     f'heatmap_DCM_source_{dcm_ct}_all_RNA.pdf'),
            group_col = 'rna_label',
        )

    # Save tables
    conn_df.to_csv(os.path.join(trans_dir, 'DCM_RNA_connections.csv'), index=False)
    (conn_df.groupby('transition').size()
     .reset_index(name='count')
     .assign(pct=lambda d: 100 * d['count'] / d['count'].sum())
     .sort_values('count', ascending=False)
     .to_csv(os.path.join(trans_dir, 'DCM_RNA_transition_summary.csv'), index=False))

print(f"\nOutputs in:\n  {trans_dir}\n  {heat_dir}")

cell_type_broad distribution after reassignment:
cell_type_broad
RGCs    9634
INs     4956
IGs     1330
Display label value counts:
_display_label
RGCs         9572
INs          4748
IGs          1371
ambiguous     229
Z-score columns found in adata.obs: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']
Z-score columns attached to plot_df: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']

Connections (all): 174
  DCM label matches highest z-score: 151/174
  RNA label matches highest z-score: 116/174
  Both match: 97/174
Connections after z-score filtering: 97
transition
INs -> RGCs     22
INs -> INs      17
IGs -> RGCs     16
IGs -> IGs      15
RGCs -> RGCs    12
INs -> IGs       8
RGCs -> IGs      5
IGs -> INs       1
RGCs -> INs      1
rna_label  IGs  INs  RGCs  All
dcm_label                     
IGs         15    1    16   32
INs          8   17    22   47
RGCs         5    1    12   18
All         28   19    50   97
  Sav

In [50]:
adata.write_h5ad("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/adata_harmony_09042026.h5ad")

In [5]:
adata = sc.read_h5ad("/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/adata_harmony_09042026.h5ad")

In [8]:
adata

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [9]:
import scanpy as sc
import matplotlib.pyplot as plt
import os

root      = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
score_dir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best', 'gene_scoring')
os.makedirs(score_dir, exist_ok=True)

gene_groups = {
    'NPCs_RGCs':               ['Nes', 'Hes5', 'Slc1a3'],
    'Immature_dorsal_neurons':  ['Gria2', 'Tubb3', 'Dcx', 'Sox6'],
    'Other':                    ['Col3a1', 'Fn1', 'Prrx1', 'Twist1'],
}

from matplotlib.colors import LinearSegmentedColormap

emerald_cmap = LinearSegmentedColormap.from_list(
    'white_emerald', ['#ffffff', '#50C878']
)
batches = sorted(adata.obs['batch'].unique())

for group_name, genes in gene_groups.items():
    genes_present = [g for g in genes if g in adata.raw.var_names]
    genes_missing = [g for g in genes if g not in adata.raw.var_names]
    if genes_missing:
        print(f"[{group_name}] Missing: {genes_missing}")
    if not genes_present:
        print(f"[{group_name}] No genes found, skipping.")
        continue

    n = len(genes_present)

    # ── All cells ────────────────────────────────────────────────
    fig = sc.pl.umap(
        adata,
        color      = genes_present,
        use_raw    = True,
        cmap       = emerald_cmap,
        frameon    = False,
        show       = False,
        return_fig = True,
        ncols      = n,
        title      = [f'{g} — ALL' for g in genes_present],
    )
    fig.suptitle(group_name.replace('_', ' '), fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_path = os.path.join(score_dir, f'umap_{group_name}_ALL.pdf')
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[{group_name}] Saved: {save_path}")

    # ── Per batch ────────────────────────────────────────────────
    for batch in batches:
        adata_sub = adata[adata.obs['batch'] == batch].copy()

        # Skip if none of the genes have any expression in this batch
        if adata_sub.n_obs == 0:
            print(f"  [{batch}] No cells, skipping.")
            continue

        fig = sc.pl.umap(
            adata_sub,
            color      = genes_present,
            use_raw    = True,
            cmap       = emerald_cmap,
            frameon    = False,
            show       = False,
            return_fig = True,
            ncols      = n,
            title      = [f'{g} — {batch}' for g in genes_present],
        )
        fig.suptitle(
            f"{group_name.replace('_', ' ')} | {batch}",
            fontsize=13, fontweight='bold', y=1.02
        )
        plt.tight_layout()

        # Sanitize batch name for filename
        batch_tag = batch.replace('/', '-').replace(' ', '_')
        save_path = os.path.join(score_dir, f'umap_{group_name}_{batch_tag}.pdf')
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"  [{batch}] Saved: {save_path}")

/tmp/ipykernel_657669/83015939.py:46: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


[NPCs_RGCs] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/umap_NPCs_RGCs_ALL.pdf


/tmp/ipykernel_657669/83015939.py:76: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  [multi_RNA] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/umap_NPCs_RGCs_multi_RNA.pdf


/tmp/ipykernel_657669/83015939.py:76: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  [ref_D2] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/umap_NPCs_RGCs_ref_D2.pdf


... (26 outputs left out)

/tmp/ipykernel_657669/83015939.py:76: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  [scNMT_DCM] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/umap_Other_scNMT_DCM.pdf


/tmp/ipykernel_657669/83015939.py:76: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  [scNMT_RNA] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/umap_Other_scNMT_RNA.pdf


In [12]:
adata

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [16]:
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from matplotlib.colors import LinearSegmentedColormap

root      = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
score_dir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best', 'gene_scoring')
os.makedirs(score_dir, exist_ok=True)

gene_groups = {
    'NPCs_RGCs':               ['Nes', 'Hes5', 'Slc1a3'],
    'Immature_dorsal_neurons':  ['Gria2', 'Tubb3', 'Dcx', 'Sox6'],
    'Other':                    ['Col3a1', 'Fn1', 'Prrx1', 'Twist1'],
}

batches    = sorted(adata.obs['batch'].unique())
celltypes  = sorted(adata.obs['module_max_label'].unique())

batch_colors    = dict(zip(batches,   sns.color_palette('tab10',  len(batches))))
celltype_colors = dict(zip(celltypes, sns.color_palette('Set2',   len(celltypes))))

# ── Extract log-normalised expression from .raw ──────────────────
raw_X   = adata.layers['lognorm_scaled']
raw_var = adata.raw.var_names.tolist()
if hasattr(raw_X, 'toarray'):
    raw_X = raw_X.toarray()

expr_df = pd.DataFrame(raw_X, index=adata.obs_names, columns=raw_var)
expr_df['batch']           = adata.obs['batch'].values
expr_df['cell_type_broad'] = adata.obs['module_max_label'].values

for group_name, genes in gene_groups.items():
    genes_present = [g for g in genes if g in raw_var]
    genes_missing = [g for g in genes if g not in raw_var]
    if genes_missing:
        print(f"[{group_name}] Missing: {genes_missing}")
    if not genes_present:
        print(f"[{group_name}] No genes found, skipping.")
        continue

    n = len(genes_present)

    # ── Plot 1: split by batch ────────────────────────────────────
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, gene in zip(axes, genes_present):
        plot_df = expr_df[['batch', gene]].rename(columns={gene: 'expression'})
        sns.boxplot(
            data       = plot_df,
            x          = 'batch',
            y          = 'expression',
            hue        = 'batch',
            palette    = batch_colors,
            width      = 0.5,
            legend     = False,
            flierprops = dict(marker='o', markersize=2, alpha=0.3),
            ax         = ax,
        )
        ax.set_title(gene, fontsize=11, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Log-normalised expression', fontsize=9)
        ax.tick_params(axis='x', rotation=35, labelsize=8)

    fig.suptitle(f"{group_name.replace('_', ' ')} — per batch",
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    path = os.path.join(score_dir, f'boxplot_{group_name}_per_batch.pdf')
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[{group_name}] Saved: {path}")

    # ── Plot 2: split by cell_type_broad ─────────────────────────
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, gene in zip(axes, genes_present):
        plot_df = expr_df[['cell_type_broad', gene]].rename(columns={gene: 'expression'})
        sns.boxplot(
            data       = plot_df,
            x          = 'cell_type_broad',
            y          = 'expression',
            hue        = 'cell_type_broad',
            palette    = celltype_colors,
            width      = 0.5,
            legend     = False,
            flierprops = dict(marker='o', markersize=2, alpha=0.3),
            ax         = ax,
        )
        ax.set_title(gene, fontsize=11, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Log-normalised expression', fontsize=9)
        ax.tick_params(axis='x', rotation=35, labelsize=8)

    fig.suptitle(f"{group_name.replace('_', ' ')} — per cell type",
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    path = os.path.join(score_dir, f'boxplot_{group_name}_per_celltype.pdf')
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[{group_name}] Saved: {path}")

    # ── Plot 3: split by both (batch columns, cell_type hue) ──────
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, gene in zip(axes, genes_present):
        plot_df = expr_df[['batch', 'cell_type_broad', gene]].rename(
            columns={gene: 'expression'})
        sns.boxplot(
            data       = plot_df,
            x          = 'batch',
            y          = 'expression',
            hue        = 'cell_type_broad',
            palette    = celltype_colors,
            width      = 0.6,
            flierprops = dict(marker='o', markersize=2, alpha=0.3),
            ax         = ax,
        )
        ax.set_title(gene, fontsize=11, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Log-normalised expression', fontsize=9)
        ax.tick_params(axis='x', rotation=35, labelsize=8)
        ax.legend(title='Cell type', bbox_to_anchor=(1.01, 1),
                  loc='upper left', fontsize=7, frameon=True)

    fig.suptitle(f"{group_name.replace('_', ' ')} — batch × cell type",
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    path = os.path.join(score_dir, f'boxplot_{group_name}_batch_x_celltype.pdf')
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[{group_name}] Saved: {path}")

[NPCs_RGCs] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/boxplot_NPCs_RGCs_per_batch.pdf
[NPCs_RGCs] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/boxplot_NPCs_RGCs_per_celltype.pdf
[NPCs_RGCs] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/boxplot_NPCs_RGCs_batch_x_celltype.pdf
[Immature_dorsal_neurons] Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/gene_scoring/boxplot_Immature_dorsal_neurons_per_batch.pdf
[Immature_dors

In [57]:
##############################################
# UMAP TRANSITIONS + KDE CONTOURS + HEATMAPS
# Adapted from reference code.
#
# Requires in memory:
#   adata, combined (from zscore_boxplots.py),
#   best_outdir, cell_types (DESIRED_ORDER),
#   MODULE_ZSCORE_COLS
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import gaussian_kde

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
trans_dir = os.path.join(best_outdir, 'umap_transitions')
heat_dir  = os.path.join(best_outdir, 'heatmap_transitions')
os.makedirs(trans_dir, exist_ok=True)
os.makedirs(heat_dir,  exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}
FALLBACK_COLOR = '#7f7f7f'
DESIRED_ORDER  = ['RGCs', 'INs', 'IGs']
REF_BATCHES    = ['ref_D2', 'ref_D4', 'multi_RNA']
BW_METHOD      = 0.5
LEVEL_FRAC     = 0.08

# ─────────────────────────────────────────────
# BUILD plot_df
# ─────────────────────────────────────────────
adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

umap    = adata.obsm['X_umap']
plot_df = pd.DataFrame({
    'UMAP1':            umap[:, 0],
    'UMAP2':            umap[:, 1],
    'batch':            adata.obs['batch'].values,
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).values,
}, index=adata.obs_names)

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].astype(str).values

plot_df['_assay'] = 'other'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True), '_assay'] = 'DCM'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*RNA', case=False, na=False, regex=True), '_assay'] = 'RNA'
plot_df.loc[plot_df['batch'].isin(REF_BATCHES), '_assay'] = 'ref'

# Attach z-scores from combined
z_cols = [c for c in combined.obs.columns if c.endswith('_module_score_Z_all')]
for col in z_cols:
    plot_df[col] = np.nan
    idx = plot_df.index.intersection(combined.obs_names)
    plot_df.loc[idx, col] = combined.obs.loc[idx, col].values

print(f"Z-score columns: {z_cols}")

# ─────────────────────────────────────────────
# MATCH DCM → RNA VIA plate_well
# ─────────────────────────────────────────────
dcm_mask = plot_df['_assay'] == 'DCM'
if 'Dox' in plot_df.columns:
    dcm_mask = dcm_mask & ~plot_df['Dox'].isin(['Minus', 'Unknown'])

dcm_known = plot_df[dcm_mask & plot_df['module_max_label'].isin(DESIRED_ORDER)]
rna_cells = plot_df[plot_df['_assay'] == 'RNA']

common_pws  = set(dcm_known['plate_well']) & set(rna_cells['plate_well'])
connections = []
for pw in common_pws:
    d = dcm_known[dcm_known['plate_well'] == pw]
    r = rna_cells[rna_cells['plate_well'] == pw]
    if len(d) == 1 and len(r) == 1:
        dr, rr = d.iloc[0], r.iloc[0]
        row = {
            'plate_well': pw,
            'dcm_umap1':  dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1':  rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_label':  dr['module_max_label'],
            'rna_label':  rr['cell_type_broad'],
            'transition': f"{dr['module_max_label']} -> {rr['cell_type_broad']}",
            'Dox':        dr.get('Dox', 'Unknown'),
        }
        for col in z_cols:
            ct = col.replace('_module_score_Z_all', '')
            row[f'dcm_{ct}_zscore'] = dr.get(col, np.nan)
            row[f'rna_{ct}_zscore'] = rr.get(col, np.nan)
        connections.append(row)

conn_df = pd.DataFrame(connections)
print(f"\nConnections: {len(conn_df)}")
if not conn_df.empty:
    print(conn_df['transition'].value_counts().to_string())
    print(pd.crosstab(conn_df['dcm_label'], conn_df['rna_label'], margins=True))

# ─────────────────────────────────────────────
# KDE CONTOUR STORE (ref cells)
# ─────────────────────────────────────────────
ref_df = plot_df[plot_df['_assay'] == 'ref']

def build_kde(df, ct):
    sub = df[df['cell_type_broad'] == ct]
    if len(sub) < 10:
        return None
    return gaussian_kde(
        np.vstack([sub['UMAP1'].values, sub['UMAP2'].values]),
        bw_method=BW_METHOD,
    )

kde_store = {ct: build_kde(ref_df, ct) for ct in DESIRED_ORDER}

# ─────────────────────────────────────────────
# HELPER: ref background + KDE contours + labels
# ─────────────────────────────────────────────
def draw_background(ax):
    for ct in DESIRED_ORDER:
        sub = ref_df[ref_df['cell_type_broad'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=8,
                       c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)
    other = ref_df[~ref_df['cell_type_broad'].isin(DESIRED_ORDER)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=8,
                   c='lightgray', alpha=0.05,
                   marker='o', zorder=1, rasterized=True)

    for ct in DESIRED_ORDER:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = ref_df[ref_df['cell_type_broad'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
        yg = np.linspace(y.min() - 0.5, y.max() + 0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct,
                ha='center', va='center', fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)

# ─────────────────────────────────────────────
# HELPER: UMAP transition plot
# Arrow colour = RNA destination
# ─────────────────────────────────────────────
def plot_umap_transitions(subset_df, title_str, save_path):
    if subset_df.empty:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))
    draw_background(ax)
    pw_set = set(subset_df['plate_well'])

    for _, row in subset_df.iterrows():
        r_col = cell_type_colors.get(row['rna_label'], FALLBACK_COLOR)
        ax.add_patch(FancyArrowPatch(
            (row['dcm_umap1'], row['dcm_umap2']),
            (row['rna_umap1'], row['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=r_col, alpha=0.75, linewidth=1.5, zorder=3,
        ))

    for ct in subset_df['dcm_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'DCM') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['module_max_label'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='s', zorder=4, rasterized=True)

    for ct in subset_df['rna_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'RNA') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['cell_type_broad'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    legend = []
    for ct in subset_df['dcm_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend.append(mlines.Line2D([], [], color=c, marker='s', linestyle='None',
                                     markersize=10, markeredgecolor='black',
                                     label=f'DCM ({ct})'))
    for ct in subset_df['rna_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black',
                           label=f'RNA ({ct})'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8,
                           label=f'Arrow \u2192 {ct}'),
        ]

    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ─────────────────────────────
# HELPER: Z-SCORE HEATMAP
# DCM (top) + RNA (bottom), sorted by group.
# Saves: shared scale, per-column scale,
# and per-Dox split (both scales).
# ─────────────────────────────
def _draw_heatmap_panel(ax, matrix, row_labels, label,
                        group_col_vals, n_cells, scale_per_col=False):
    cmap = plt.cm.RdBu_r
    if scale_per_col:
        mat = matrix.copy().astype(float)
        for i in range(mat.shape[0]):
            row = mat[i, :]
            rmin, rmax = np.nanmin(row), np.nanmax(row)
            if rmax > rmin:
                mat[i, :] = (row - rmin) / (rmax - rmin) * 2 - 1
        norm = plt.Normalize(vmin=-1, vmax=1)
        cbar_label, vline_val = 'Z-score (col-scaled)', 0.0
    else:
        mat  = matrix.copy().astype(float)
        norm = TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
        cbar_label, vline_val = 'Z-score', 1.65
    im = ax.imshow(mat, aspect='auto', cmap=cmap, norm=norm)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_xticks([])
    ax.set_ylabel('Module', fontsize=11, fontweight='bold')
    ax.set_title(label, fontsize=12, fontweight='bold')
    changes = group_col_vals.ne(group_col_vals.shift()).to_numpy().nonzero()[0]
    for x in changes:
        ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
    prev = 0
    for x in list(changes) + [n_cells]:
        mid = (prev + x) / 2
        ct  = group_col_vals.iloc[prev]
        ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                fontweight='bold', color=cell_type_colors.get(ct, 'black'))
        prev = x
    cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
    cbar.set_label(cbar_label, fontsize=9)
    cbar.ax.axhline(y=vline_val, color='black', lw=1, ls='--')


def plot_zscore_heatmap(subset_df, title_str, save_path,
                        group_col='dcm_label', scale_per_col=False):
    if subset_df.empty:
        print(f"  Skipping heatmap -- no cells"); return
    dcm_z_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER]
    if not any(c in subset_df.columns for c in dcm_z_cols):
        print(f"  No z-score columns -- skipping heatmap"); return

    def _sort(df):
        df = df.copy()
        df['_sort'] = df[group_col].map(
            {ct: i for i, ct in enumerate(DESIRED_ORDER)}
        ).fillna(99)
        df['_self_z'] = df.apply(
            lambda r: r.get(f'dcm_{r[group_col]}_zscore', 0)
            if r[group_col] in DESIRED_ORDER else 0, axis=1)
        return df.sort_values(['_sort', '_self_z'],
                               ascending=[True, False]).reset_index(drop=True)

    def _mats(df):
        dcm = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        rna = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        return dcm, rna

    def _save(df, path, extra='', scaled=False):
        if df.empty: return
        df = _sort(df)
        n  = len(df)
        fw = max(14, n * 0.18)
        dm, rm = _mats(df)
        fig, axes = plt.subplots(2, 1, figsize=(fw, 10))
        for ax, mat, lbl in [(axes[0], dm, 'DCM (GMM label)'),
                              (axes[1], rm, 'RNA (cell_type_broad)')]:
            _draw_heatmap_panel(ax, mat, DESIRED_ORDER, lbl,
                                df[group_col], n, scale_per_col=scaled)
        ttl = title_str + (f'  |  {extra}' if extra else '') + ('  [col-scaled]' if scaled else '')
        fig.suptitle(ttl, fontsize=13, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")

    stem, ext = os.path.splitext(save_path)
    # 1. Full -- shared scale
    _save(subset_df, save_path, scaled=False)
    # 2. Full -- per-column scale
    _save(subset_df, f'{stem}_colscale{ext}', scaled=True)
    # 3. Per-Dox split (both scales)
    if 'Dox' in subset_df.columns:
        dox_vals = sorted(subset_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub = subset_df[subset_df['Dox'] == dox]
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                _save(sub, f'{stem}_dox{ds}{ext}',         extra=f'Dox={dox}', scaled=False)
                _save(sub, f'{stem}_dox{ds}_colscale{ext}', extra=f'Dox={dox}', scaled=True)


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
if conn_df.empty:
    print("No connections found — check plate_well and assay labels.")
else:
    for transition in conn_df['transition'].unique():
        sub   = conn_df[conn_df['transition'] == transition]
        fname = transition.replace(' -> ', '_to_').replace(' ', '_')
        plot_umap_transitions(
            sub,
            title_str = f'{transition}  (n={len(sub)})\nArrow colour = RNA destination',
            save_path = os.path.join(trans_dir, f'umap_{fname}.pdf'),
        )
        plot_zscore_heatmap(
            sub,
            title_str = f'Z-score heatmap — {transition}  (n={len(sub)})',
            save_path = os.path.join(heat_dir, f'heatmap_{fname}.pdf'),
        )

    # Combined UMAP
    plot_umap_transitions(
        conn_df,
        title_str = f'All DCM \u2192 RNA transitions  (n={len(conn_df)} pairs)\nArrow colour = RNA destination',
        save_path = os.path.join(trans_dir, 'umap_all_transitions.pdf'),
    )

    # ── Heatmap: all transitions, grouped by DCM source label ──
    # Mirrors the reference plot_heatmap(sec_to_ent_clear, ...) call:
    # one column per cell, sorted by DCM cell type then self-z-score,
    # white separators at group boundaries, cell type labels below.
    plot_zscore_heatmap(
        conn_df,
        title_str = f'Z-score heatmap — all transitions  (n={len(conn_df)})',
        save_path = os.path.join(heat_dir, 'heatmap_all_transitions.pdf'),
    )

    # ── Per DCM source: UMAP + heatmap for all RNA destinations ──
    # Equivalent of the reference plot_umap(sec_to_ent_clear, ...) +
    # plot_heatmap(sec_to_ent_clear, ...) calls in Step 7.
    for dcm_ct in DESIRED_ORDER:
        sub_dcm = conn_df[conn_df['dcm_label'] == dcm_ct]
        if sub_dcm.empty:
            continue
        n = len(sub_dcm)
        rna_dests = sorted(sub_dcm['rna_label'].unique())
        plot_umap_transitions(
            sub_dcm,
            title_str = (f'DCM source: {dcm_ct} → all RNA destinations  (n={n})\n'
                         f'RNA: {", ".join(rna_dests)} | Arrow colour = RNA destination'),
            save_path = os.path.join(trans_dir,
                                     f'umap_clear_{dcm_ct}_to_all_RNA.pdf'),
        )
        plot_zscore_heatmap(
            sub_dcm,
            title_str = (f'Z-score heatmap — DCM source: {dcm_ct}  '
                         f'(n={n}, all RNA destinations)'),
            save_path = os.path.join(heat_dir,
                                     f'heatmap_DCM_source_{dcm_ct}_all_RNA.pdf'),
            group_col = 'rna_label',
        )

    # Save tables
    conn_df.to_csv(os.path.join(trans_dir, 'DCM_RNA_connections.csv'), index=False)
    (conn_df.groupby('transition').size()
     .reset_index(name='count')
     .assign(pct=lambda d: 100 * d['count'] / d['count'].sum())
     .sort_values('count', ascending=False)
     .to_csv(os.path.join(trans_dir, 'DCM_RNA_transition_summary.csv'), index=False))

print(f"\nOutputs in:\n  {trans_dir}\n  {heat_dir}")

Z-score columns: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']

Connections: 174
transition
INs -> RGCs     66
IGs -> RGCs     38
RGCs -> RGCs    18
INs -> INs      18
IGs -> IGs      17
INs -> IGs       9
RGCs -> IGs      5
IGs -> INs       2
RGCs -> INs      1
rna_label  IGs  INs  RGCs  All
dcm_label                     
IGs         17    2    38   57
INs          9   18    66   93
RGCs         5    1    18   24
All         31   21   122  174
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/umap_transitions/umap_INs_to_RGCs.pdf
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/heatmap_transitions/heatmap_INs_to_RGCs.pdf
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/Cel

In [26]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import gaussian_kde

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
root        = "/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026"
best_outdir = os.path.join(root, 'integration', 'method_comparison', 'Harmony_best')
trans_dir   = os.path.join(best_outdir, 'umap_transitions3')
heat_dir    = os.path.join(best_outdir, 'heatmap_transitions3')
os.makedirs(trans_dir, exist_ok=True)
os.makedirs(heat_dir,  exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}
FALLBACK_COLOR = '#7f7f7f'
DESIRED_ORDER  = ['RGCs', 'INs', 'IGs']
REF_BATCHES    = ['ref_D2', 'ref_D4', 'multi_RNA']
BW_METHOD      = 0.5
LEVEL_FRAC     = 0.08
VCENTER_DCM    = 0.5
VCENTER_RNA    = 0.0

# ─────────────────────────────────────────────
# BUILD plot_df
# ─────────────────────────────────────────────
adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

umap    = adata.obsm['X_umap']
plot_df = pd.DataFrame({
    'UMAP1':            umap[:, 0],
    'UMAP2':            umap[:, 1],
    'batch':            adata.obs['batch'].values,
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).values,
}, index=adata.obs_names)

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].astype(str).values

plot_df['_assay'] = 'other'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True), '_assay'] = 'DCM'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*RNA', case=False, na=False, regex=True), '_assay'] = 'RNA'
plot_df.loc[plot_df['batch'].isin(REF_BATCHES), '_assay'] = 'ref'

# Attach z-scores from combined
z_cols = [c for c in combined.obs.columns if c.endswith('_module_score_Z_all')]
for col in z_cols:
    plot_df[col] = np.nan
    idx = plot_df.index.intersection(combined.obs_names)
    plot_df.loc[idx, col] = combined.obs.loc[idx, col].values

print(f"Z-score columns: {z_cols}")

# ─────────────────────────────────────────────
# MATCH DCM → RNA VIA plate_well
# ─────────────────────────────────────────────
dcm_mask = plot_df['_assay'] == 'DCM'
if 'Dox' in plot_df.columns:
    dcm_mask = dcm_mask & ~plot_df['Dox'].isin(['Minus', 'Unknown'])

dcm_known = plot_df[dcm_mask & plot_df['module_max_label'].isin(DESIRED_ORDER)]
rna_cells = plot_df[plot_df['_assay'] == 'RNA']

common_pws  = set(dcm_known['plate_well']) & set(rna_cells['plate_well'])
connections = []
for pw in common_pws:
    d = dcm_known[dcm_known['plate_well'] == pw]
    r = rna_cells[rna_cells['plate_well'] == pw]
    if len(d) == 1 and len(r) == 1:
        dr, rr = d.iloc[0], r.iloc[0]
        row = {
            'plate_well': pw,
            'dcm_umap1':  dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1':  rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_label':  dr['module_max_label'],
            'rna_label':  rr['cell_type_broad'],
            'transition': f"{dr['module_max_label']} -> {rr['cell_type_broad']}",
            'Dox':        dr.get('Dox', 'Unknown'),
        }
        for col in z_cols:
            ct = col.replace('_module_score_Z_all', '')
            row[f'dcm_{ct}_zscore'] = dr.get(col, np.nan)
            row[f'rna_{ct}_zscore'] = rr.get(col, np.nan)
        connections.append(row)

conn_df = pd.DataFrame(connections)
print(f"\nConnections (all): {len(conn_df)}")

# Map plate_well → is_hybrid from adata.obs
hybrid_map = (
    pd.DataFrame({
        'plate_well': adata.obs['plate_well'].astype(str).values,
        'is_hybrid':  adata.obs['is_hybrid'].values,
    }, index=adata.obs_names)
    [adata.obs['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True)]
    .drop_duplicates('plate_well')
    .set_index('plate_well')['is_hybrid']
    .to_dict()
)

conn_df['is_hybrid'] = conn_df['plate_well'].map(hybrid_map).fillna(False).astype(bool)
print(f"Hybrid cells in conn_df: {conn_df['is_hybrid'].sum()}")

# ── Filter: keep only cells where label matches highest z-score ──
if not conn_df.empty:
    dcm_z_score_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'dcm_{ct}_zscore' in conn_df.columns]
    rna_z_score_cols = [f'rna_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'rna_{ct}_zscore' in conn_df.columns]

    def _argmax_label(row, z_cols, prefix):
        vals = {col.replace(prefix, '').replace('_zscore', ''): row[col]
                for col in z_cols if not np.isnan(row[col])}
        return max(vals, key=vals.get) if vals else None

    conn_df['dcm_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, dcm_z_score_cols, 'dcm_'), axis=1)
    conn_df['rna_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, rna_z_score_cols, 'rna_'), axis=1)

    dcm_match = conn_df['dcm_label'] == conn_df['dcm_max_label']
    rna_match = conn_df['rna_label'] == conn_df['rna_max_label']

    print(f"  DCM label matches highest z-score: {dcm_match.sum()}/{len(conn_df)}")
    print(f"  RNA label matches highest z-score: {rna_match.sum()}/{len(conn_df)}")
    print(f"  Both match: {(dcm_match & rna_match).sum()}/{len(conn_df)}")

    conn_df = conn_df[dcm_match & rna_match].drop(
        columns=['dcm_max_label', 'rna_max_label']
    ).reset_index(drop=True)
    print(f"Connections after filtering: {len(conn_df)}")

if not conn_df.empty:
    print(conn_df['transition'].value_counts().to_string())
    print(pd.crosstab(conn_df['dcm_label'], conn_df['rna_label'], margins=True))

# ─────────────────────────────────────────────
# KDE CONTOUR STORE (ref cells)
# ─────────────────────────────────────────────
ref_df = plot_df[plot_df['_assay'] == 'ref']

def build_kde(df, ct):
    sub = df[df['cell_type_broad'] == ct]
    if len(sub) < 10:
        return None
    return gaussian_kde(
        np.vstack([sub['UMAP1'].values, sub['UMAP2'].values]),
        bw_method=BW_METHOD,
    )

kde_store = {ct: build_kde(ref_df, ct) for ct in DESIRED_ORDER}

# ─────────────────────────────────────────────
# HELPER: ref background + KDE contours + labels
# ─────────────────────────────────────────────
def draw_background(ax):
    for ct in DESIRED_ORDER:
        sub = ref_df[ref_df['cell_type_broad'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=8,
                       c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)
    other = ref_df[~ref_df['cell_type_broad'].isin(DESIRED_ORDER)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=8,
                   c='lightgray', alpha=0.05,
                   marker='o', zorder=1, rasterized=True)

    for ct in DESIRED_ORDER:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = ref_df[ref_df['cell_type_broad'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
        yg = np.linspace(y.min() - 0.5, y.max() + 0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct,
                ha='center', va='center', fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)

# ─────────────────────────────────────────────
# HELPER: UMAP transition plot
# ─────────────────────────────────────────────
def plot_umap_transitions(subset_df, title_str, save_path):
    if subset_df.empty:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))
    draw_background(ax)
    pw_set = set(subset_df['plate_well'])

    for _, row in subset_df.iterrows():
        r_col = cell_type_colors.get(row['rna_label'], FALLBACK_COLOR)
        ax.add_patch(FancyArrowPatch(
            (row['dcm_umap1'], row['dcm_umap2']),
            (row['rna_umap1'], row['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=r_col, alpha=0.75, linewidth=1.5, zorder=3,
        ))

    for ct in subset_df['dcm_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'DCM') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['module_max_label'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='s', zorder=4, rasterized=True)

    for ct in subset_df['rna_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'RNA') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['cell_type_broad'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    legend = []
    for ct in subset_df['dcm_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend.append(mlines.Line2D([], [], color=c, marker='s', linestyle='None',
                                    markersize=10, markeredgecolor='black',
                                    label=f'DCM ({ct})'))
    for ct in subset_df['rna_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                          markersize=10, markeredgecolor='black',
                          label=f'RNA ({ct})'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                          alpha=0.8, marker='>', markersize=8,
                          label=f'Arrow \u2192 {ct}'),
        ]

    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

# ─────────────────────────────────────────────
# HELPER: color strip + hybrid * marker
# ─────────────────────────────────────────────
def _add_color_strip(ax, label_series, n_cells, hybrid_series=None):
    for i, lbl in enumerate(label_series):
        c = cell_type_colors.get(lbl, FALLBACK_COLOR)
        ax.add_patch(plt.Rectangle(
            (i / n_cells, 1.01), 1 / n_cells, 0.05,
            transform=ax.transAxes,
            facecolor=c, edgecolor='none',
            clip_on=False, zorder=6,
        ))
        if hybrid_series is not None and hybrid_series.iloc[i]:
            ax.text(
                (i + 0.5) / n_cells, 1.07, '*',
                transform=ax.transAxes,
                ha='center', va='bottom',
                fontsize=8, fontweight='bold',
                color='black', clip_on=False, zorder=7,
            )

# ─────────────────────────────────────────────
# HELPER: single heatmap panel
# ─────────────────────────────────────────────
def _draw_heatmap_panel(ax, matrix, row_labels, label,
                        group_col_vals, cell_label_vals, n_cells,
                        vcenter=0.0, hybrid_series=None):
    mat  = matrix.copy().astype(float)
    norm = TwoSlopeNorm(vmin=vcenter - 3, vcenter=vcenter, vmax=vcenter + 3)
    cmap = plt.cm.RdBu_r

    im = ax.imshow(mat, aspect='auto', cmap=cmap, norm=norm)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_xticks([])
    ax.set_ylabel('Module', fontsize=11, fontweight='bold')
    ax.set_title(label, fontsize=12, fontweight='bold')

    changes = group_col_vals.ne(group_col_vals.shift()).to_numpy().nonzero()[0]
    for x in changes:
        ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
    prev = 0
    for x in list(changes) + [n_cells]:
        mid = (prev + x) / 2
        ct  = group_col_vals.iloc[prev]
        ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                fontweight='bold', color='black')
        prev = x

    _add_color_strip(ax, cell_label_vals, n_cells, hybrid_series=hybrid_series)

    cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
    cbar.set_label(f'Z-score (vcenter={vcenter})', fontsize=9)
    cbar.ax.axhline(y=vcenter, color='black', lw=1, ls='--')

# ─────────────────────────────────────────────
# HELPER: Z-SCORE HEATMAP
# ─────────────────────────────────────────────
def plot_zscore_heatmap(subset_df, title_str, save_path,
                        group_col='dcm_label'):
    if subset_df.empty:
        print(f"  Skipping heatmap -- no cells"); return
    dcm_z_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER]
    if not any(c in subset_df.columns for c in dcm_z_cols):
        print(f"  No z-score columns -- skipping heatmap"); return

    def _sort(df):
        df = df.copy()
        df['_dcm_order'] = df[group_col].map(
            {ct: i for i, ct in enumerate(DESIRED_ORDER)}
        ).fillna(99)
        df['_rna_order'] = df['rna_label'].map(
            {ct: i for i, ct in enumerate(DESIRED_ORDER)}
        ).fillna(99)
        df['_self_z'] = df.apply(
            lambda r: r.get(f'dcm_{r[group_col]}_zscore', 0)
            if r[group_col] in DESIRED_ORDER else 0, axis=1)
        return df.sort_values(
            ['_dcm_order', '_rna_order', '_self_z'],
            ascending=[True, True, False]
        ).reset_index(drop=True)

    def _mats(df):
        dcm = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                        for _, row in df.iterrows()]).T
        rna = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                        for _, row in df.iterrows()]).T
        return dcm, rna

    def _save(df, path, extra=''):
        if df.empty: return
        df = _sort(df)
        n  = len(df)
        fw = max(14, n * 0.18)
        dm, rm = _mats(df)

        hybrid_s = df['is_hybrid'].reset_index(drop=True) if 'is_hybrid' in df.columns else None

        fig, axes = plt.subplots(2, 1, figsize=(fw, 10))

        _draw_heatmap_panel(axes[0], dm, DESIRED_ORDER, 'DCM (GMM label)',
                            df[group_col].reset_index(drop=True),
                            df['dcm_label'].reset_index(drop=True),
                            n, vcenter=VCENTER_DCM,
                            hybrid_series=hybrid_s)

        _draw_heatmap_panel(axes[1], rm, DESIRED_ORDER, 'RNA (cell_type_broad)',
                            df['rna_label'].reset_index(drop=True),
                            df['rna_label'].reset_index(drop=True),
                            n, vcenter=VCENTER_RNA,
                            hybrid_series=hybrid_s)

        dcm_changes = df[group_col].ne(df[group_col].shift()).to_numpy().nonzero()[0]
        for x in dcm_changes:
            axes[1].axvline(x - 0.5, color='black', lw=2.0, alpha=0.9)

        ttl = title_str + (f'  |  {extra}' if extra else '')
        fig.suptitle(ttl, fontsize=13, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")

    stem, ext = os.path.splitext(save_path)
    _save(subset_df, save_path)
    if 'Dox' in subset_df.columns:
        dox_vals = sorted(subset_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub = subset_df[subset_df['Dox'] == dox]
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                _save(sub, f'{stem}_dox{ds}{ext}', extra=f'Dox={dox}')

# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
if conn_df.empty:
    print("No connections found — check plate_well and assay labels.")
else:
    for transition in conn_df['transition'].unique():
        sub   = conn_df[conn_df['transition'] == transition]
        fname = transition.replace(' -> ', '_to_').replace(' ', '_')
        plot_umap_transitions(
            sub,
            title_str = f'{transition}  (n={len(sub)})\nArrow colour = RNA destination',
            save_path = os.path.join(trans_dir, f'umap_{fname}.pdf'),
        )
        plot_zscore_heatmap(
            sub,
            title_str = f'Z-score heatmap — {transition}  (n={len(sub)})',
            save_path = os.path.join(heat_dir, f'heatmap_{fname}.pdf'),
        )

    # Combined UMAP — all filtered cells
    plot_umap_transitions(
        conn_df,
        title_str = f'All DCM \u2192 RNA transitions  (n={len(conn_df)} pairs)\nArrow colour = RNA destination',
        save_path = os.path.join(trans_dir, 'umap_all_transitions.pdf'),
    )

    # Per-Dox UMAP
    if 'Dox' in conn_df.columns:
        dox_vals = sorted(conn_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub_dox = conn_df[conn_df['Dox'] == dox]
                if sub_dox.empty:
                    continue
                ds = str(dox).replace(' ', '_').replace('/', '-')
                plot_umap_transitions(
                    sub_dox,
                    title_str = (f'All transitions  |  Dox={dox}  (n={len(sub_dox)} pairs)\n'
                                 f'Arrow colour = RNA destination'),
                    save_path = os.path.join(trans_dir,
                                             f'umap_all_transitions_dox{ds}.pdf'),
                )

    # ── Heatmap: RGCs/INs only ──
    RGCIN_TRANSITIONS = ['RGCs -> RGCs', 'RGCs -> INs', 'INs -> INs', 'INs -> RGCs']
    RGCIN_SCORES      = ['RGCs', 'INs']

    rgcin_sub = conn_df[conn_df['transition'].isin(RGCIN_TRANSITIONS)].copy()

    if not rgcin_sub.empty:
        trans_order_map = {t: i for i, t in enumerate(RGCIN_TRANSITIONS)}
        rgcin_sub['_t_order'] = rgcin_sub['transition'].map(trans_order_map).fillna(99)
        rgcin_sub['_self_z']  = rgcin_sub.apply(
            lambda r: r.get(f'dcm_{r["dcm_label"]}_zscore', 0)
            if r['dcm_label'] in RGCIN_SCORES else 0, axis=1
        )
        rgcin_sub = rgcin_sub.sort_values(
            ['_t_order', '_self_z'], ascending=[True, False]
        ).drop(columns=['_t_order', '_self_z']).reset_index(drop=True)

        def _save_rgcin(df, path, title_extra=''):
            if df.empty: return
            n  = len(df)
            fw = max(14, n * 0.18)

            dcm_mat = np.array(
                [[row.get(f'dcm_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T
            rna_mat = np.array(
                [[row.get(f'rna_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T

            hybrid_s = df['is_hybrid'].reset_index(drop=True) if 'is_hybrid' in df.columns else None

            fig, axes = plt.subplots(2, 1, figsize=(fw, 8))

            _draw_heatmap_panel(axes[0], dcm_mat, RGCIN_SCORES,
                                'DCM (GMM label)',
                                df['transition'].reset_index(drop=True),
                                df['dcm_label'].reset_index(drop=True),
                                n, vcenter=VCENTER_DCM,
                                hybrid_series=hybrid_s)

            _draw_heatmap_panel(axes[1], rna_mat, RGCIN_SCORES,
                                'RNA (cell_type_broad)',
                                df['transition'].reset_index(drop=True),
                                df['rna_label'].reset_index(drop=True),
                                n, vcenter=VCENTER_RNA,
                                hybrid_series=hybrid_s)

            ttl = (f'Z-score heatmap — RGCs/INs transitions  (n={n})'
                   + (f'  |  {title_extra}' if title_extra else '')
                   + f'  DCM vcenter={VCENTER_DCM}, RNA vcenter={VCENTER_RNA}')
            fig.suptitle(ttl, fontsize=13, fontweight='bold')
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  Saved: {path}")

        stem = os.path.join(heat_dir, 'heatmap_RGCsINs_transitions')
        _save_rgcin(rgcin_sub, f'{stem}.pdf')
        if 'Dox' in rgcin_sub.columns:
            for dox in sorted(rgcin_sub['Dox'].dropna().unique()):
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                sub = rgcin_sub[rgcin_sub['Dox'] == dox]
                _save_rgcin(sub, f'{stem}_dox{ds}.pdf', title_extra=f'Dox={dox}')

    else:
        print("  No RGCs/INs transitions found in conn_df")

    # ── Heatmap: all transitions ──
    plot_zscore_heatmap(
        conn_df,
        title_str = f'Z-score heatmap — all transitions  (n={len(conn_df)})',
        save_path = os.path.join(heat_dir, 'heatmap_all_transitions.pdf'),
    )

    # ── Per DCM source: UMAP + heatmap ──
    for dcm_ct in DESIRED_ORDER:
        sub_dcm = conn_df[conn_df['dcm_label'] == dcm_ct]
        if sub_dcm.empty:
            continue
        n = len(sub_dcm)
        rna_dests = sorted(sub_dcm['rna_label'].unique())
        plot_umap_transitions(
            sub_dcm,
            title_str = (f'DCM source: {dcm_ct} → all RNA destinations  (n={n})\n'
                         f'RNA: {", ".join(rna_dests)} | Arrow colour = RNA destination'),
            save_path = os.path.join(trans_dir,
                                     f'umap_clear_{dcm_ct}_to_all_RNA.pdf'),
        )
        plot_zscore_heatmap(
            sub_dcm,
            title_str = (f'Z-score heatmap — DCM source: {dcm_ct}  '
                         f'(n={n}, all RNA destinations)'),
            save_path = os.path.join(heat_dir,
                                     f'heatmap_DCM_source_{dcm_ct}_all_RNA.pdf'),
            group_col = 'rna_label',
        )

    # Save tables
    conn_df.to_csv(os.path.join(trans_dir, 'DCM_RNA_connections.csv'), index=False)
    (conn_df.groupby('transition').size()
     .reset_index(name='count')
     .assign(pct=lambda d: 100 * d['count'] / d['count'].sum())
     .sort_values('count', ascending=False)
     .to_csv(os.path.join(trans_dir, 'DCM_RNA_transition_summary.csv'), index=False))

print(f"\nOutputs in:\n  {trans_dir}\n  {heat_dir}")

Z-score columns: ['IGs_module_score_Z_all', 'INs_module_score_Z_all', 'RGCs_module_score_Z_all']

Connections (all): 174
Hybrid cells in conn_df: 72
  DCM label matches highest z-score: 151/174
  RNA label matches highest z-score: 116/174
  Both match: 97/174
Connections after filtering: 97
transition
INs -> RGCs     22
INs -> INs      17
IGs -> RGCs     16
IGs -> IGs      15
RGCs -> RGCs    12
INs -> IGs       8
RGCs -> IGs      5
IGs -> INs       1
RGCs -> INs      1
rna_label  IGs  INs  RGCs  All
dcm_label                     
IGs         15    1    16   32
INs          8   17    22   47
RGCs         5    1    12   18
All         28   19    50   97
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/umap_transitions3/umap_INs_to_INs.pdf
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona

In [ ]:
#hybrids

In [9]:
THRESHOLD = 0.837
posterior_cols = {
    'IGs': '_posterior_merged_IGs',
    'INs': '_posterior_merged_INs',
    'RGCs': '_posterior_merged_RGCs',
}

# Boolean mask per cell type: posterior > threshold
post_df = pd.DataFrame(
    {ct: adata.obs[col].values for ct, col in posterior_cols.items()},
    index=adata.obs_names
)

above = post_df > THRESHOLD

# is_hybrid: more than one cell type above threshold
adata.obs['is_hybrid'] = (above.sum(axis=1) > 1).astype(bool)

# hybrid_cell_types: which cell types are above threshold (empty string if not hybrid)
adata.obs['hybrid_cell_types'] = above.apply(
    lambda row: '+'.join([ct for ct in posterior_cols if row[ct]])
    if row.sum() > 1 else '',
    axis=1
)

print(f"Hybrid cells: {adata.obs['is_hybrid'].sum()}")
print(adata.obs[adata.obs['is_hybrid']][['hybrid_cell_types']].value_counts().to_string())

Hybrid cells: 225
hybrid_cell_types
IGs+INs              182
IGs+INs+RGCs          15
INs+RGCs              15
IGs+RGCs              13


In [14]:
adata.obs['is_hybrid']

X100_Plate_1          False
X154_Plate_1          False
X82_Plate_1           False
X174_Plate_1          False
X81_Plate_1           False
                      ...  
TTTGTTGGTCATTACC-1    False
TTTGTTGGTCATTGTC-1    False
TTTGTTGGTCCCGGAA-1    False
TTTGTTGGTCGCAAAC-1    False
TTTGTTGGTTGCTTCG-1    False
Name: is_hybrid, Length: 15920, dtype: bool

In [12]:
# Cells where exactly one cell type is above threshold
above_count = above.sum(axis=1)
single_pos = above[above_count == 1]
print(single_pos.apply(lambda row: row[row].index[0], axis=1).value_counts().to_string())
print(f"\nTotal single cell type cells: {(above_count == 1).sum()}")
print(f"Total no cell type above threshold: {(above_count == 0).sum()}")
print(f"Total hybrid: {(above_count > 1).sum()}")

INs     224
IGs     129
RGCs     84

Total single cell type cells: 437
Total no cell type above threshold: 15258
Total hybrid: 225


In [ ]:
##filter cells that do notmatch Zscore with corresponding cell types

In [16]:
# ─────────────────────────────────────────────
# STEP 1: BUILD COMBINED SUBSET
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING ALL-ASSAY SUBSET")
print("=" * 60)

adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

scnmt_mask = adata.obs['orig.ident'].str.contains('scNMT', case=False, na=False)
ref_mask   = adata.obs['batch'].isin(REF_BATCHES)
combined   = adata[scnmt_mask | ref_mask].copy()
combined.X = combined.layers['lognorm']

def assign_score_group(row):
    if 'scNMT' in str(row['orig.ident']):
        return row['orig.ident']
    elif row['batch'] in REF_BATCHES:
        return 'ref'
    return 'other'

combined.obs['_score_group'] = combined.obs.apply(assign_score_group, axis=1)

STEP 1: BUILDING ALL-ASSAY SUBSET


In [33]:
combined

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [34]:

# ─────────────────────────────────────────────
# BUILD MARKER SETS
# ─────────────────────────────────────────────
label_order  = ['RGCs', 'INs', 'IGs', 'Unknown']
palette_dict = {
    'RGCs':    '#1f77b4',
    'INs':     '#ff7f0e',
    'IGs':     '#2ca02c',
    'Unknown': 'lightgray',
}

ttest_markers_by_ct = {}
for ct in ttest_df_unique['cell_type'].unique():
    genes = (
        ttest_df_unique[
            (ttest_df_unique['cell_type']   == ct) &
            (ttest_df_unique['logfc_ttest'] >  1) &
            (ttest_df_unique['pvals_adj']   <  0.0001)
        ]
        ['gene'].tolist()
    )
    if genes:
        ttest_markers_by_ct[ct] = genes

print(f"\nMarker sets: {len(ttest_markers_by_ct)} cell types")
for ct, g in ttest_markers_by_ct.items():
    print(f"  {ct}: {len(g)} genes")




DESIRED_ORDER = ['RGCs', 'INs', 'IGs']
REF_BATCHES   = ['ref_D2', 'ref_D4', 'multi_RNA']

ASSAY_COLORS = {
    'scNMT_DCM': '#003366',
    'scNMT_RNA': '#8B0000',
    'ref':       '#808080',
}

ASSAY_PLOT_CONFIG = {
    'scNMT_DCM': {
        'cell_type_col': 'module_max_label',
        'extra_filter':  {'col': 'Dox', 'exclude_values': ['Minus', 'Unknown']},
    },
    'scNMT_RNA': {
        'cell_type_col': 'cell_type_broad',
        'extra_filter':  None,
    },
    'ref': {
        'cell_type_col': 'cell_type_broad',
        'extra_filter':  None,
    },
}



# ─────────────────────────────────────────────
# STEP 2: MODULE SCORES PER GROUP
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: COMPUTING MODULE SCORES")
print("=" * 60)

genes_in_data = set(combined.var_names)
filtered_gene_sets = {
    ct: [g for g in genes if g in genes_in_data]
    for ct, genes in ttest_markers_by_ct.items()
}
for ct, g in filtered_gene_sets.items():
    print(f"  {ct}: {len(g)} genes")

for ct in filtered_gene_sets:
    combined.obs[f'{ct}_score_raw_all']        = np.nan
    combined.obs[f'{ct}_module_score_raw_all'] = np.nan

for grp in combined.obs['_score_group'].unique():
    print(f"\n  Group: {grp}")
    mask = combined.obs['_score_group'] == grp
    sub  = combined[mask].copy()
    print(f"    {sub.n_obs} cells")

    for ct, genes in filtered_gene_sets.items():
        if not genes:
            continue
        X = sub[:, genes].X
        sub.obs[f'{ct}_score_raw_all'] = (
            np.asarray(X.toarray() if sp.issparse(X) else X).mean(axis=1)
        )
        sc.tl.score_genes(
            sub, gene_list=genes,
            score_name=f'{ct}_module_score_raw_all',
            use_raw=False,
        )
        combined.obs.loc[sub.obs_names, f'{ct}_score_raw_all']        = sub.obs[f'{ct}_score_raw_all']
        combined.obs.loc[sub.obs_names, f'{ct}_module_score_raw_all'] = sub.obs[f'{ct}_module_score_raw_all']

# ─────────────────────────────────────────────
# STEP 3: Z-SCORE PER GROUP
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Z-SCORING PER GROUP")
print("=" * 60)

MODULE_ZSCORE_COLS = []
for ct in filtered_gene_sets:
    raw_col = f'{ct}_module_score_raw_all'
    if raw_col not in combined.obs:
        continue
    z_col = f'{ct}_module_score_Z_all'
    MODULE_ZSCORE_COLS.append(z_col)
    combined.obs[z_col] = np.nan
    for grp in combined.obs['_score_group'].unique():
        mask = combined.obs['_score_group'] == grp
        combined.obs.loc[mask, z_col] = zscore(
            combined.obs.loc[mask, raw_col], nan_policy='omit'
        )

print(f"Module Z-score columns: {MODULE_ZSCORE_COLS}")

# ─────────────────────────────────────────────
# STEP 4: BUILD PLOT DATA
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: BUILDING PLOT DATA")
print("=" * 60)

all_data = []
for assay_label, cfg in ASSAY_PLOT_CONFIG.items():
    mask = combined.obs['_plot_assay'] == assay_label

    if cfg['extra_filter']:
        ecol  = cfg['extra_filter']['col']
        evals = cfg['extra_filter']['exclude_values']
        if ecol in combined.obs.columns:
            mask = mask & ~combined.obs[ecol].astype(str).isin(evals)

    ct_col = cfg['cell_type_col']
    if ct_col not in combined.obs.columns:
        print(f"  WARNING: '{ct_col}' not in obs for {assay_label}")
        continue

    sub = combined.obs[mask].copy()
    sub['_ct']    = sub[ct_col].astype(str)
    sub['_assay'] = assay_label
    sub = sub[sub['_ct'].isin(DESIRED_ORDER)]

    print(f"  {assay_label}: {len(sub)} cells — {sub['_ct'].value_counts().to_dict()}")
    if not sub.empty:
        all_data.append(sub)

plot_df = pd.concat(all_data, ignore_index=True)

avail_assays  = [a for a in ASSAY_COLORS if a in plot_df['_assay'].unique()]
plot_df['_ct']    = pd.Categorical(plot_df['_ct'],
                                    categories=DESIRED_ORDER, ordered=True)
plot_df['_assay'] = pd.Categorical(plot_df['_assay'],
                                    categories=avail_assays, ordered=True)
palette = [ASSAY_COLORS[a] for a in avail_assays]



Marker sets: 3 cell types
  IGs: 1089 genes
  INs: 567 genes
  RGCs: 399 genes

STEP 2: COMPUTING MODULE SCORES
  IGs: 1089 genes
  INs: 567 genes
  RGCs: 399 genes

  Group: scNMT_pool1_DCM
    405 cells

  Group: scNMT_pool1_RNA
    723 cells

  Group: ref
    14792 cells

STEP 3: Z-SCORING PER GROUP


NameError: name 'zscore' is not defined

In [20]:
combined

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [19]:
##############################################
# UMAP TRANSITIONS + KDE CONTOURS + HEATMAPS
# Adapted from reference code.
#
# Requires in memory:
#   adata, combined (from zscore_boxplots.py),
#   best_outdir, cell_types (DESIRED_ORDER),
#   MODULE_ZSCORE_COLS
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import gaussian_kde

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
trans_dir = os.path.join(best_outdir, 'umap_transitions')
heat_dir  = os.path.join(best_outdir, 'heatmap_transitions')
os.makedirs(trans_dir, exist_ok=True)
os.makedirs(heat_dir,  exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}
FALLBACK_COLOR = '#7f7f7f'
DESIRED_ORDER  = ['RGCs', 'INs', 'IGs']
REF_BATCHES    = ['ref_D2', 'ref_D4', 'multi_RNA']
BW_METHOD      = 0.5
LEVEL_FRAC     = 0.08
VCENTER        = 0.75   # change to e.g. 1.0 or 0.5

# ─────────────────────────────────────────────
# BUILD plot_df
# ─────────────────────────────────────────────
adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

umap    = adata.obsm['X_umap']
plot_df = pd.DataFrame({
    'UMAP1':            umap[:, 0],
    'UMAP2':            umap[:, 1],
    'batch':            adata.obs['batch'].values,
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).values,
}, index=adata.obs_names)

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].astype(str).values

plot_df['_assay'] = 'other'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True), '_assay'] = 'DCM'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*RNA', case=False, na=False, regex=True), '_assay'] = 'RNA'
plot_df.loc[plot_df['batch'].isin(REF_BATCHES), '_assay'] = 'ref'

# Attach z-scores from combined
z_cols = [c for c in combined.obs.columns if c.endswith('_module_score_Z_all')]
for col in z_cols:
    plot_df[col] = np.nan
    idx = plot_df.index.intersection(combined.obs_names)
    plot_df.loc[idx, col] = combined.obs.loc[idx, col].values

print(f"Z-score columns: {z_cols}")

# ─────────────────────────────────────────────
# MATCH DCM → RNA VIA plate_well
# ─────────────────────────────────────────────
dcm_mask = plot_df['_assay'] == 'DCM'
if 'Dox' in plot_df.columns:
    dcm_mask = dcm_mask & ~plot_df['Dox'].isin(['Minus', 'Unknown'])

dcm_known = plot_df[dcm_mask & plot_df['module_max_label'].isin(DESIRED_ORDER)]
rna_cells = plot_df[plot_df['_assay'] == 'RNA']

common_pws  = set(dcm_known['plate_well']) & set(rna_cells['plate_well'])
connections = []
for pw in common_pws:
    d = dcm_known[dcm_known['plate_well'] == pw]
    r = rna_cells[rna_cells['plate_well'] == pw]
    if len(d) == 1 and len(r) == 1:
        dr, rr = d.iloc[0], r.iloc[0]
        row = {
            'plate_well': pw,
            'dcm_umap1':  dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1':  rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_label':  dr['module_max_label'],
            'rna_label':  rr['cell_type_broad'],
            'transition': f"{dr['module_max_label']} -> {rr['cell_type_broad']}",
            'Dox':        dr.get('Dox', 'Unknown'),
        }
        for col in z_cols:
            ct = col.replace('_module_score_Z_all', '')
            row[f'dcm_{ct}_zscore'] = dr.get(col, np.nan)
            row[f'rna_{ct}_zscore'] = rr.get(col, np.nan)
        connections.append(row)

conn_df = pd.DataFrame(connections)
print(f"\nConnections (all): {len(conn_df)}")

# ── Filter: keep only cells where the label matches the highest z-score ──
# DCM: dcm_label must equal the cell type with the highest dcm_*_zscore
# RNA: rna_label must equal the cell type with the highest rna_*_zscore
if not conn_df.empty:
    dcm_z_score_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'dcm_{ct}_zscore' in conn_df.columns]
    rna_z_score_cols = [f'rna_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'rna_{ct}_zscore' in conn_df.columns]

    def _argmax_label(row, z_cols, prefix):
        vals = {col.replace(prefix, '').replace('_zscore', ''): row[col]
                for col in z_cols if not np.isnan(row[col])}
        return max(vals, key=vals.get) if vals else None

    conn_df['dcm_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, dcm_z_score_cols, 'dcm_'), axis=1)
    conn_df['rna_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, rna_z_score_cols, 'rna_'), axis=1)

    dcm_match = conn_df['dcm_label'] == conn_df['dcm_max_label']
    rna_match = conn_df['rna_label'] == conn_df['rna_max_label']

    print(f"  DCM label matches highest z-score: {dcm_match.sum()}/{len(conn_df)}")
    print(f"  RNA label matches highest z-score: {rna_match.sum()}/{len(conn_df)}")
    print(f"  Both match: {(dcm_match & rna_match).sum()}/{len(conn_df)}")

    conn_df = conn_df[dcm_match & rna_match].drop(
        columns=['dcm_max_label', 'rna_max_label']
    ).reset_index(drop=True)
    print(f"Connections after filtering: {len(conn_df)}")

if not conn_df.empty:
    print(conn_df['transition'].value_counts().to_string())
    print(pd.crosstab(conn_df['dcm_label'], conn_df['rna_label'], margins=True))

# ─────────────────────────────────────────────
# KDE CONTOUR STORE (ref cells)
# ─────────────────────────────────────────────
ref_df = plot_df[plot_df['_assay'] == 'ref']

def build_kde(df, ct):
    sub = df[df['cell_type_broad'] == ct]
    if len(sub) < 10:
        return None
    return gaussian_kde(
        np.vstack([sub['UMAP1'].values, sub['UMAP2'].values]),
        bw_method=BW_METHOD,
    )

kde_store = {ct: build_kde(ref_df, ct) for ct in DESIRED_ORDER}

# ─────────────────────────────────────────────
# HELPER: ref background + KDE contours + labels
# ─────────────────────────────────────────────
def draw_background(ax):
    for ct in DESIRED_ORDER:
        sub = ref_df[ref_df['cell_type_broad'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=8,
                       c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)
    other = ref_df[~ref_df['cell_type_broad'].isin(DESIRED_ORDER)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=8,
                   c='lightgray', alpha=0.05,
                   marker='o', zorder=1, rasterized=True)

    for ct in DESIRED_ORDER:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = ref_df[ref_df['cell_type_broad'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
        yg = np.linspace(y.min() - 0.5, y.max() + 0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct,
                ha='center', va='center', fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)

# ─────────────────────────────────────────────
# HELPER: UMAP transition plot
# Arrow colour = RNA destination
# ─────────────────────────────────────────────
def plot_umap_transitions(subset_df, title_str, save_path):
    if subset_df.empty:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))
    draw_background(ax)
    pw_set = set(subset_df['plate_well'])

    for _, row in subset_df.iterrows():
        r_col = cell_type_colors.get(row['rna_label'], FALLBACK_COLOR)
        ax.add_patch(FancyArrowPatch(
            (row['dcm_umap1'], row['dcm_umap2']),
            (row['rna_umap1'], row['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=r_col, alpha=0.75, linewidth=1.5, zorder=3,
        ))

    for ct in subset_df['dcm_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'DCM') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['module_max_label'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='s', zorder=4, rasterized=True)

    for ct in subset_df['rna_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'RNA') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['cell_type_broad'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    legend = []
    for ct in subset_df['dcm_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend.append(mlines.Line2D([], [], color=c, marker='s', linestyle='None',
                                     markersize=10, markeredgecolor='black',
                                     label=f'DCM ({ct})'))
    for ct in subset_df['rna_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black',
                           label=f'RNA ({ct})'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8,
                           label=f'Arrow \u2192 {ct}'),
        ]

    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ─────────────────────────────
# HELPER: Z-SCORE HEATMAP
# DCM (top) + RNA (bottom), sorted by group.
# Saves: shared scale, per-column scale,
# and per-Dox split (both scales).
# ─────────────────────────────
def _add_color_strip(ax, label_series, n_cells):
    """Draw one coloured rectangle per column just above the heatmap panel.
    Each rectangle spans 1/n_cells of the axes width and is coloured by
    cell_type_colors of that cell's label."""
    for i, lbl in enumerate(label_series):
        c = cell_type_colors.get(lbl, FALLBACK_COLOR)
        ax.add_patch(plt.Rectangle(
            (i / n_cells, 1.01), 1 / n_cells, 0.05,
            transform=ax.transAxes,
            facecolor=c, edgecolor='none',
            clip_on=False, zorder=6,
        ))


def _draw_heatmap_panel(ax, matrix, row_labels, label,
                        group_col_vals, cell_label_vals, n_cells,
                        vcenter=0.0):
    """Draw one DCM or RNA heatmap panel.
    group_col_vals  : Series used for group separators + labels below.
    cell_label_vals : Series used for the per-column colour strip above
                      (typically dcm_label for DCM panel, rna_label for RNA).
    """
    mat  = matrix.copy().astype(float)
    norm = TwoSlopeNorm(vmin=vcenter - 3, vcenter=vcenter, vmax=vcenter + 3)
    cmap = plt.cm.RdBu_r

    im = ax.imshow(mat, aspect='auto', cmap=cmap, norm=norm)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_xticks([])
    ax.set_ylabel('Module', fontsize=11, fontweight='bold')
    ax.set_title(label, fontsize=12, fontweight='bold')

    # Group separators + labels below
    changes = group_col_vals.ne(group_col_vals.shift()).to_numpy().nonzero()[0]
    for x in changes:
        ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
    prev = 0
    for x in list(changes) + [n_cells]:
        mid = (prev + x) / 2
        ct  = group_col_vals.iloc[prev]
        ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                fontweight='bold', color='black')
        prev = x

    # Coloured rectangle strip above each column
    _add_color_strip(ax, cell_label_vals, n_cells)

    cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
    cbar.set_label(f'Z-score (vcenter={vcenter})', fontsize=9)
    cbar.ax.axhline(y=vcenter, color='black', lw=1, ls='--')


def plot_zscore_heatmap(subset_df, title_str, save_path,
                        group_col='dcm_label'):
    if subset_df.empty:
        print(f"  Skipping heatmap -- no cells"); return
    dcm_z_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER]
    if not any(c in subset_df.columns for c in dcm_z_cols):
        print(f"  No z-score columns -- skipping heatmap"); return

    def _sort(df):
        df = df.copy()
        df['_sort'] = df[group_col].map(
            {ct: i for i, ct in enumerate(DESIRED_ORDER)}
        ).fillna(99)
        df['_self_z'] = df.apply(
            lambda r: r.get(f'dcm_{r[group_col]}_zscore', 0)
            if r[group_col] in DESIRED_ORDER else 0, axis=1)
        return df.sort_values(['_sort', '_self_z'],
                               ascending=[True, False]).reset_index(drop=True)

    def _mats(df):
        dcm = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        rna = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        return dcm, rna

    def _save(df, path, extra=''):
        if df.empty: return
        df = _sort(df)
        n  = len(df)
        fw = max(14, n * 0.18)
        dm, rm = _mats(df)
        fig, axes = plt.subplots(2, 1, figsize=(fw, 10))
        # DCM panel: group separators by group_col, strip coloured by dcm_label
        _draw_heatmap_panel(axes[0], dm, DESIRED_ORDER, 'DCM (GMM label)',
                            df[group_col], df['dcm_label'], n, vcenter=VCENTER)
        # RNA panel: group separators by group_col, strip coloured by rna_label
        _draw_heatmap_panel(axes[1], rm, DESIRED_ORDER, 'RNA (cell_type_broad)',
                            df[group_col], df['rna_label'], n, vcenter=VCENTER)
        ttl = title_str + (f'  |  {extra}' if extra else '')
        fig.suptitle(ttl, fontsize=13, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")

    stem, ext = os.path.splitext(save_path)
    # 1. Full
    _save(subset_df, save_path)
    # 2. Per-Dox split
    if 'Dox' in subset_df.columns:
        dox_vals = sorted(subset_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub = subset_df[subset_df['Dox'] == dox]
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                _save(sub, f'{stem}_dox{ds}{ext}', extra=f'Dox={dox}')


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
if conn_df.empty:
    print("No connections found — check plate_well and assay labels.")
else:
    for transition in conn_df['transition'].unique():
        sub   = conn_df[conn_df['transition'] == transition]
        fname = transition.replace(' -> ', '_to_').replace(' ', '_')
        plot_umap_transitions(
            sub,
            title_str = f'{transition}  (n={len(sub)})\nArrow colour = RNA destination',
            save_path = os.path.join(trans_dir, f'umap_{fname}.pdf'),
        )
        plot_zscore_heatmap(
            sub,
            title_str = f'Z-score heatmap — {transition}  (n={len(sub)})',
            save_path = os.path.join(heat_dir, f'heatmap_{fname}.pdf'),
        )

    # Combined UMAP — all filtered cells
    plot_umap_transitions(
        conn_df,
        title_str = f'All DCM \u2192 RNA transitions  (n={len(conn_df)} pairs)\nArrow colour = RNA destination',
        save_path = os.path.join(trans_dir, 'umap_all_transitions.pdf'),
    )

    # ── Per-Dox UMAP — all transitions, one plot per Dox condition ──
    if 'Dox' in conn_df.columns:
        dox_vals = sorted(conn_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub_dox = conn_df[conn_df['Dox'] == dox]
                if sub_dox.empty:
                    continue
                ds = str(dox).replace(' ', '_').replace('/', '-')
                plot_umap_transitions(
                    sub_dox,
                    title_str = (f'All transitions  |  Dox={dox}  (n={len(sub_dox)} pairs)\n'
                                 f'Arrow colour = RNA destination'),
                    save_path = os.path.join(trans_dir,
                                             f'umap_all_transitions_dox{ds}.pdf'),
                )

    # ── Heatmap: RGCs/INs only, 4 transitions, RGCs+INs module scores only ──
    RGCIN_TRANSITIONS = ['RGCs -> RGCs', 'RGCs -> INs', 'INs -> INs', 'INs -> RGCs']
    RGCIN_SCORES      = ['RGCs', 'INs']   # only these two rows shown

    rgcin_sub = conn_df[conn_df['transition'].isin(RGCIN_TRANSITIONS)].copy()

    if not rgcin_sub.empty:
        # Sort by the fixed transition order, then by DCM RGCs z-score descending
        trans_order_map = {t: i for i, t in enumerate(RGCIN_TRANSITIONS)}
        rgcin_sub = rgcin_sub.copy()
        rgcin_sub['_t_order'] = rgcin_sub['transition'].map(trans_order_map).fillna(99)
        rgcin_sub['_self_z']  = rgcin_sub.apply(
            lambda r: r.get(f'dcm_{r["dcm_label"]}_zscore', 0)
            if r['dcm_label'] in RGCIN_SCORES else 0, axis=1
        )
        rgcin_sub = rgcin_sub.sort_values(
            ['_t_order', '_self_z'], ascending=[True, False]
        ).drop(columns=['_t_order', '_self_z']).reset_index(drop=True)

        def _save_rgcin(df, path, title_extra=''):
            if df.empty: return
            n  = len(df)
            fw = max(14, n * 0.18)

            dcm_mat = np.array(
                [[row.get(f'dcm_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T
            rna_mat = np.array(
                [[row.get(f'rna_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T

            fig, axes = plt.subplots(2, 1, figsize=(fw, 8))
            # DCM: group separators by transition, strip coloured by dcm_label
            _draw_heatmap_panel(axes[0], dcm_mat, RGCIN_SCORES,
                                'DCM (GMM label)', df['transition'],
                                df['dcm_label'], n, vcenter=VCENTER)
            # RNA: group separators by transition, strip coloured by rna_label
            _draw_heatmap_panel(axes[1], rna_mat, RGCIN_SCORES,
                                'RNA (cell_type_broad)', df['transition'],
                                df['rna_label'], n, vcenter=VCENTER)

            ttl = (f'Z-score heatmap — RGCs/INs transitions  (n={n})'
                   + (f'  |  {title_extra}' if title_extra else '')
                   + f'  vcenter={VCENTER}')
            fig.suptitle(ttl, fontsize=13, fontweight='bold')
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  Saved: {path}")

        stem = os.path.join(heat_dir, 'heatmap_RGCsINs_transitions')
        _save_rgcin(rgcin_sub, f'{stem}.pdf')
        if 'Dox' in rgcin_sub.columns:
            for dox in sorted(rgcin_sub['Dox'].dropna().unique()):
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                sub = rgcin_sub[rgcin_sub['Dox'] == dox]
                _save_rgcin(sub, f'{stem}_dox{ds}.pdf', title_extra=f'Dox={dox}')
    else:
        print("  No RGCs/INs transitions found in conn_df")

    # ── Heatmap: all transitions, grouped by DCM source label ──
    plot_zscore_heatmap(
        conn_df,
        title_str = f'Z-score heatmap — all transitions  (n={len(conn_df)})',
        save_path = os.path.join(heat_dir, 'heatmap_all_transitions.pdf'),
    )

    # ── Per DCM source: UMAP + heatmap for all RNA destinations ──
    # Equivalent of the reference plot_umap(sec_to_ent_clear, ...) +
    # plot_heatmap(sec_to_ent_clear, ...) calls in Step 7.
    for dcm_ct in DESIRED_ORDER:
        sub_dcm = conn_df[conn_df['dcm_label'] == dcm_ct]
        if sub_dcm.empty:
            continue
        n = len(sub_dcm)
        rna_dests = sorted(sub_dcm['rna_label'].unique())
        plot_umap_transitions(
            sub_dcm,
            title_str = (f'DCM source: {dcm_ct} → all RNA destinations  (n={n})\n'
                         f'RNA: {", ".join(rna_dests)} | Arrow colour = RNA destination'),
            save_path = os.path.join(trans_dir,
                                     f'umap_clear_{dcm_ct}_to_all_RNA.pdf'),
        )
        plot_zscore_heatmap(
            sub_dcm,
            title_str = (f'Z-score heatmap — DCM source: {dcm_ct}  '
                         f'(n={n}, all RNA destinations)'),
            save_path = os.path.join(heat_dir,
                                     f'heatmap_DCM_source_{dcm_ct}_all_RNA.pdf'),
            group_col = 'rna_label',
        )

    # Save tables
    conn_df.to_csv(os.path.join(trans_dir, 'DCM_RNA_connections.csv'), index=False)
    (conn_df.groupby('transition').size()
     .reset_index(name='count')
     .assign(pct=lambda d: 100 * d['count'] / d['count'].sum())
     .sort_values('count', ascending=False)
     .to_csv(os.path.join(trans_dir, 'DCM_RNA_transition_summary.csv'), index=False))

print(f"\nOutputs in:\n  {trans_dir}\n  {heat_dir}")

Z-score columns: []

Connections (all): 174
  DCM label matches highest z-score: 0/174
  RNA label matches highest z-score: 0/174
  Both match: 0/174
Connections after filtering: 0
No connections found — check plate_well and assay labels.

Outputs in:
  /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/umap_transitions
  /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/heatmap_transitions


In [ ]:
#umap transitions showing only the specific ones

In [17]:
adata

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [16]:
##############################################
# UMAP TRANSITIONS + KDE CONTOURS + HEATMAPS
# Adapted from reference code.
#
# Requires in memory:
#   adata, combined (from zscore_boxplots.py),
#   best_outdir, cell_types (DESIRED_ORDER),
#   MODULE_ZSCORE_COLS
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import gaussian_kde

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
trans_dir = os.path.join(best_outdir, 'umap_transitions')
heat_dir  = os.path.join(best_outdir, 'heatmap_transitions')
os.makedirs(trans_dir, exist_ok=True)
os.makedirs(heat_dir,  exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}
FALLBACK_COLOR = '#7f7f7f'
DESIRED_ORDER  = ['RGCs', 'INs', 'IGs']
REF_BATCHES    = ['ref_D2', 'ref_D4', 'multi_RNA']
BW_METHOD      = 0.5
LEVEL_FRAC     = 0.08
VCENTER_DCM    = 1.0   # vcenter for DCM panel  (e.g. 1.0)
VCENTER_RNA    = 0.0   # vcenter for RNA panel  (e.g. 0.0 or 0.5)

# ─────────────────────────────────────────────
# BUILD plot_df
# ─────────────────────────────────────────────
adata.obs['batch']      = adata.obs['batch'].astype(str)
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)

umap    = adata.obsm['X_umap']
plot_df = pd.DataFrame({
    'UMAP1':            umap[:, 0],
    'UMAP2':            umap[:, 1],
    'batch':            adata.obs['batch'].values,
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).values,
}, index=adata.obs_names)

if 'Dox' in adata.obs.columns:
    plot_df['Dox'] = adata.obs['Dox'].astype(str).values

plot_df['_assay'] = 'other'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True), '_assay'] = 'DCM'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*RNA', case=False, na=False, regex=True), '_assay'] = 'RNA'
plot_df.loc[plot_df['batch'].isin(REF_BATCHES), '_assay'] = 'ref'

# Attach z-scores from combined
z_cols = [c for c in combined.obs.columns if c.endswith('_module_score_Z_all')]
for col in z_cols:
    plot_df[col] = np.nan
    idx = plot_df.index.intersection(combined.obs_names)
    plot_df.loc[idx, col] = combined.obs.loc[idx, col].values

print(f"Z-score columns: {z_cols}")

# ─────────────────────────────────────────────
# MATCH DCM → RNA VIA plate_well
# ─────────────────────────────────────────────
dcm_mask = plot_df['_assay'] == 'DCM'
if 'Dox' in plot_df.columns:
    dcm_mask = dcm_mask & ~plot_df['Dox'].isin(['Minus', 'Unknown'])

dcm_known = plot_df[dcm_mask & plot_df['module_max_label'].isin(DESIRED_ORDER)]
rna_cells = plot_df[plot_df['_assay'] == 'RNA']

common_pws  = set(dcm_known['plate_well']) & set(rna_cells['plate_well'])
connections = []
for pw in common_pws:
    d = dcm_known[dcm_known['plate_well'] == pw]
    r = rna_cells[rna_cells['plate_well'] == pw]
    if len(d) == 1 and len(r) == 1:
        dr, rr = d.iloc[0], r.iloc[0]
        row = {
            'plate_well': pw,
            'dcm_umap1':  dr['UMAP1'], 'dcm_umap2': dr['UMAP2'],
            'rna_umap1':  rr['UMAP1'], 'rna_umap2': rr['UMAP2'],
            'dcm_label':  dr['module_max_label'],
            'rna_label':  rr['cell_type_broad'],
            'transition': f"{dr['module_max_label']} -> {rr['cell_type_broad']}",
            'Dox':        dr.get('Dox', 'Unknown'),
        }
        for col in z_cols:
            ct = col.replace('_module_score_Z_all', '')
            row[f'dcm_{ct}_zscore'] = dr.get(col, np.nan)
            row[f'rna_{ct}_zscore'] = rr.get(col, np.nan)
        connections.append(row)

conn_df = pd.DataFrame(connections)
print(f"\nConnections (all): {len(conn_df)}")

# ─────────────────────────────────────────────
# SPATIAL FILTER for cross-lineage transitions
# IGs -> RGCs : DCM must be in IGs KDE region,
#               RNA must be in RGCs KDE region
# RGCs -> IGs : DCM must be in RGCs KDE region,
#               RNA must be in IGs KDE region
# Uses the same KDE store built below — we
# evaluate the KDE at each cell's UMAP position
# and check it exceeds the LEVEL_FRAC threshold.
# ─────────────────────────────────────────────
SPATIAL_FILTER_TRANSITIONS = {'IGs -> RGCs', 'RGCs -> IGs'}

def _in_kde_region(x, y, ct):
    """True if (x, y) falls within the KDE contour for ct."""
    kde = kde_store.get(ct)
    if kde is None:
        return True   # can't filter if no KDE — keep cell
    density = kde(np.array([[x], [y]]))[0]
    # threshold: same fraction of peak used for contour drawing
    sub = ref_df[ref_df['cell_type_broad'] == ct]
    if len(sub) < 10:
        return True
    coords = np.vstack([sub['UMAP1'].values, sub['UMAP2'].values])
    peak_density = kde(coords).max()
    return density >= peak_density * LEVEL_FRAC

# ── Filter: keep only cells where the label matches the highest z-score ──
# DCM: dcm_label must equal the cell type with the highest dcm_*_zscore
# RNA: rna_label must equal the cell type with the highest rna_*_zscore
if not conn_df.empty:
    dcm_z_score_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'dcm_{ct}_zscore' in conn_df.columns]
    rna_z_score_cols = [f'rna_{ct}_zscore' for ct in DESIRED_ORDER
                        if f'rna_{ct}_zscore' in conn_df.columns]

    def _argmax_label(row, z_cols, prefix):
        vals = {col.replace(prefix, '').replace('_zscore', ''): row[col]
                for col in z_cols if not np.isnan(row[col])}
        return max(vals, key=vals.get) if vals else None

    conn_df['dcm_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, dcm_z_score_cols, 'dcm_'), axis=1)
    conn_df['rna_max_label'] = conn_df.apply(
        lambda r: _argmax_label(r, rna_z_score_cols, 'rna_'), axis=1)

    dcm_match = conn_df['dcm_label'] == conn_df['dcm_max_label']
    rna_match = conn_df['rna_label'] == conn_df['rna_max_label']

    print(f"  DCM label matches highest z-score: {dcm_match.sum()}/{len(conn_df)}")
    print(f"  RNA label matches highest z-score: {rna_match.sum()}/{len(conn_df)}")
    print(f"  Both match: {(dcm_match & rna_match).sum()}/{len(conn_df)}")

    conn_df = conn_df[dcm_match & rna_match].drop(
        columns=['dcm_max_label', 'rna_max_label']
    ).reset_index(drop=True)
    print(f"Connections after z-score filtering: {len(conn_df)}")

    # ── Spatial filter for cross-lineage transitions ──
    # For IGs->RGCs and RGCs->IGs, additionally require
    # DCM cell sits in its source KDE region and RNA cell
    # sits in its destination KDE region.
    if not conn_df.empty:
        cross_mask = conn_df['transition'].isin(SPATIAL_FILTER_TRANSITIONS)
        if cross_mask.any():
            def _passes_spatial(row):
                if row['transition'] not in SPATIAL_FILTER_TRANSITIONS:
                    return True
                dcm_ok = _in_kde_region(
                    row['dcm_umap1'], row['dcm_umap2'], row['dcm_label'])
                rna_ok = _in_kde_region(
                    row['rna_umap1'], row['rna_umap2'], row['rna_label'])
                return dcm_ok and rna_ok

            spatial_keep = conn_df.apply(_passes_spatial, axis=1)
            n_before = cross_mask.sum()
            n_after  = (cross_mask & spatial_keep).sum()
            print(f"  Spatial filter on cross-lineage transitions:")
            for t in SPATIAL_FILTER_TRANSITIONS:
                t_mask = conn_df['transition'] == t
                kept   = (t_mask & spatial_keep).sum()
                print(f"    {t}: {kept}/{t_mask.sum()} kept")
            conn_df = conn_df[~cross_mask | spatial_keep].reset_index(drop=True)
            print(f"Connections after spatial filtering: {len(conn_df)}")

if not conn_df.empty:
    print(conn_df['transition'].value_counts().to_string())
    print(pd.crosstab(conn_df['dcm_label'], conn_df['rna_label'], margins=True))

# ─────────────────────────────────────────────
# KDE CONTOUR STORE (ref cells)
# ─────────────────────────────────────────────
ref_df = plot_df[plot_df['_assay'] == 'ref']

def build_kde(df, ct):
    sub = df[df['cell_type_broad'] == ct]
    if len(sub) < 10:
        return None
    return gaussian_kde(
        np.vstack([sub['UMAP1'].values, sub['UMAP2'].values]),
        bw_method=BW_METHOD,
    )

kde_store = {ct: build_kde(ref_df, ct) for ct in DESIRED_ORDER}

# ─────────────────────────────────────────────
# HELPER: ref background + KDE contours + labels
# ─────────────────────────────────────────────
def draw_background(ax):
    for ct in DESIRED_ORDER:
        sub = ref_df[ref_df['cell_type_broad'] == ct]
        if len(sub):
            ax.scatter(sub['UMAP1'], sub['UMAP2'], s=8,
                       c=cell_type_colors[ct], alpha=0.15,
                       marker='o', zorder=1, rasterized=True)
    other = ref_df[~ref_df['cell_type_broad'].isin(DESIRED_ORDER)]
    if len(other):
        ax.scatter(other['UMAP1'], other['UMAP2'], s=8,
                   c='lightgray', alpha=0.05,
                   marker='o', zorder=1, rasterized=True)

    for ct in DESIRED_ORDER:
        kde = kde_store.get(ct)
        if kde is None:
            continue
        sub = ref_df[ref_df['cell_type_broad'] == ct]
        x, y = sub['UMAP1'].values, sub['UMAP2'].values
        xg = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
        yg = np.linspace(y.min() - 0.5, y.max() + 0.5, 200)
        xx, yy = np.meshgrid(xg, yg)
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contour(xx, yy, zz, levels=[zz.max() * LEVEL_FRAC],
                   colors=[cell_type_colors[ct]],
                   linewidths=2, linestyles='--', alpha=0.8, zorder=2)
        peak = np.unravel_index(zz.argmax(), zz.shape)
        ax.text(xx[peak], yy[peak], ct,
                ha='center', va='center', fontsize=10, fontweight='bold',
                color=cell_type_colors[ct], zorder=5)

# ─────────────────────────────────────────────
# HELPER: UMAP transition plot
# Arrow colour = RNA destination
# ─────────────────────────────────────────────
def plot_umap_transitions(subset_df, title_str, save_path):
    if subset_df.empty:
        print(f"  Skipping UMAP — no cells"); return

    fig, ax = plt.subplots(figsize=(12, 10))
    draw_background(ax)
    pw_set = set(subset_df['plate_well'])

    for _, row in subset_df.iterrows():
        r_col = cell_type_colors.get(row['rna_label'], FALLBACK_COLOR)
        ax.add_patch(FancyArrowPatch(
            (row['dcm_umap1'], row['dcm_umap2']),
            (row['rna_umap1'], row['rna_umap2']),
            arrowstyle='-|>', mutation_scale=15,
            color=r_col, alpha=0.75, linewidth=1.5, zorder=3,
        ))

    for ct in subset_df['dcm_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'DCM') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['module_max_label'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='s', zorder=4, rasterized=True)

    for ct in subset_df['rna_label'].unique():
        pts = plot_df[(plot_df['_assay'] == 'RNA') &
                      (plot_df['plate_well'].isin(pw_set)) &
                      (plot_df['cell_type_broad'] == ct)]
        if len(pts):
            ax.scatter(pts['UMAP1'], pts['UMAP2'], s=100,
                       c=cell_type_colors.get(ct, FALLBACK_COLOR),
                       alpha=1.0, edgecolors='black', linewidths=1.5,
                       marker='o', zorder=4, rasterized=True)

    legend = []
    for ct in subset_df['dcm_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend.append(mlines.Line2D([], [], color=c, marker='s', linestyle='None',
                                     markersize=10, markeredgecolor='black',
                                     label=f'DCM ({ct})'))
    for ct in subset_df['rna_label'].unique():
        c = cell_type_colors.get(ct, FALLBACK_COLOR)
        legend += [
            mlines.Line2D([], [], color=c, marker='o', linestyle='None',
                           markersize=10, markeredgecolor='black',
                           label=f'RNA ({ct})'),
            mlines.Line2D([], [], color=c, linestyle='-', linewidth=2,
                           alpha=0.8, marker='>', markersize=8,
                           label=f'Arrow \u2192 {ct}'),
        ]

    ax.legend(handles=legend, loc='center left',
              bbox_to_anchor=(1, 0.5), frameon=True, fontsize=9)
    ax.set_title(title_str, fontsize=13, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ─────────────────────────────
# HELPER: Z-SCORE HEATMAP
# DCM (top) + RNA (bottom), sorted by group.
# Saves: shared scale, per-column scale,
# and per-Dox split (both scales).
# ─────────────────────────────
def _add_color_strip(ax, label_series, n_cells):
    """Draw one coloured rectangle per column just above the heatmap panel.
    Each rectangle spans 1/n_cells of the axes width and is coloured by
    cell_type_colors of that cell's label."""
    for i, lbl in enumerate(label_series):
        c = cell_type_colors.get(lbl, FALLBACK_COLOR)
        ax.add_patch(plt.Rectangle(
            (i / n_cells, 1.01), 1 / n_cells, 0.05,
            transform=ax.transAxes,
            facecolor=c, edgecolor='none',
            clip_on=False, zorder=6,
        ))


def _draw_heatmap_panel(ax, matrix, row_labels, label,
                        group_col_vals, cell_label_vals, n_cells,
                        vcenter=0.0):
    """Draw one DCM or RNA heatmap panel.
    group_col_vals  : Series used for group separators + labels below.
    cell_label_vals : Series used for the per-column colour strip above
                      (typically dcm_label for DCM panel, rna_label for RNA).
    """
    mat  = matrix.copy().astype(float)
    norm = TwoSlopeNorm(vmin=vcenter - 3, vcenter=vcenter, vmax=vcenter + 3)
    cmap = plt.cm.RdBu_r

    im = ax.imshow(mat, aspect='auto', cmap=cmap, norm=norm)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_xticks([])
    ax.set_ylabel('Module', fontsize=11, fontweight='bold')
    ax.set_title(label, fontsize=12, fontweight='bold')

    # Group separators + labels below
    changes = group_col_vals.ne(group_col_vals.shift()).to_numpy().nonzero()[0]
    for x in changes:
        ax.axvline(x - 0.5, color='white', lw=1.5, alpha=0.8)
    prev = 0
    for x in list(changes) + [n_cells]:
        mid = (prev + x) / 2
        ct  = group_col_vals.iloc[prev]
        ax.text(mid, -0.8, ct, ha='center', va='top', fontsize=7,
                fontweight='bold', color='black')
        prev = x

    # Coloured rectangle strip above each column
    _add_color_strip(ax, cell_label_vals, n_cells)

    cbar = plt.colorbar(im, ax=ax, fraction=0.01, pad=0.01)
    cbar.set_label(f'Z-score (vcenter={vcenter})', fontsize=9)
    cbar.ax.axhline(y=vcenter, color='black', lw=1, ls='--')


def plot_zscore_heatmap(subset_df, title_str, save_path,
                        group_col='dcm_label'):
    if subset_df.empty:
        print(f"  Skipping heatmap -- no cells"); return
    dcm_z_cols = [f'dcm_{ct}_zscore' for ct in DESIRED_ORDER]
    if not any(c in subset_df.columns for c in dcm_z_cols):
        print(f"  No z-score columns -- skipping heatmap"); return

    def _sort(df):
        df = df.copy()
        df['_sort'] = df[group_col].map(
            {ct: i for i, ct in enumerate(DESIRED_ORDER)}
        ).fillna(99)
        df['_self_z'] = df.apply(
            lambda r: r.get(f'dcm_{r[group_col]}_zscore', 0)
            if r[group_col] in DESIRED_ORDER else 0, axis=1)
        return df.sort_values(['_sort', '_self_z'],
                               ascending=[True, False]).reset_index(drop=True)

    def _mats(df):
        dcm = np.array([[row.get(f'dcm_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        rna = np.array([[row.get(f'rna_{ct}_zscore', np.nan) for ct in DESIRED_ORDER]
                         for _, row in df.iterrows()]).T
        return dcm, rna

    def _save(df, path, extra=''):
        if df.empty: return
        df = _sort(df)
        n  = len(df)
        fw = max(14, n * 0.18)
        dm, rm = _mats(df)
        fig, axes = plt.subplots(2, 1, figsize=(fw, 10))
        # DCM panel: group separators by group_col, strip coloured by dcm_label
        _draw_heatmap_panel(axes[0], dm, DESIRED_ORDER, 'DCM (GMM label)',
                            df[group_col], df['dcm_label'], n, vcenter=VCENTER_DCM)
        # RNA panel: group separators by group_col, strip coloured by rna_label
        _draw_heatmap_panel(axes[1], rm, DESIRED_ORDER, 'RNA (cell_type_broad)',
                            df[group_col], df['rna_label'], n, vcenter=VCENTER_RNA)
        ttl = title_str + (f'  |  {extra}' if extra else '')
        fig.suptitle(ttl, fontsize=13, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")

    stem, ext = os.path.splitext(save_path)
    # 1. Full
    _save(subset_df, save_path)
    # 2. Per-Dox split
    if 'Dox' in subset_df.columns:
        dox_vals = sorted(subset_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub = subset_df[subset_df['Dox'] == dox]
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                _save(sub, f'{stem}_dox{ds}{ext}', extra=f'Dox={dox}')


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
if conn_df.empty:
    print("No connections found — check plate_well and assay labels.")
else:
    for transition in conn_df['transition'].unique():
        sub   = conn_df[conn_df['transition'] == transition]
        fname = transition.replace(' -> ', '_to_').replace(' ', '_')
        plot_umap_transitions(
            sub,
            title_str = f'{transition}  (n={len(sub)})\nArrow colour = RNA destination',
            save_path = os.path.join(trans_dir, f'umap_{fname}2.pdf'),
        )
        plot_zscore_heatmap(
            sub,
            title_str = f'Z-score heatmap — {transition}  (n={len(sub)})',
            save_path = os.path.join(heat_dir, f'heatmap_{fname}2.pdf'),
        )

    # Combined UMAP — all filtered cells
    plot_umap_transitions(
        conn_df,
        title_str = f'All DCM \u2192 RNA transitions  (n={len(conn_df)} pairs)\nArrow colour = RNA destination',
        save_path = os.path.join(trans_dir, 'umap_all_transitions2.pdf'),
    )

    # ── Per-Dox UMAP — all transitions, one plot per Dox condition ──
    if 'Dox' in conn_df.columns:
        dox_vals = sorted(conn_df['Dox'].dropna().unique())
        if len(dox_vals) > 1:
            for dox in dox_vals:
                sub_dox = conn_df[conn_df['Dox'] == dox]
                if sub_dox.empty:
                    continue
                ds = str(dox).replace(' ', '_').replace('/', '-')
                plot_umap_transitions(
                    sub_dox,
                    title_str = (f'All transitions  |  Dox={dox}  (n={len(sub_dox)} pairs)\n'
                                 f'Arrow colour = RNA destination'),
                    save_path = os.path.join(trans_dir,
                                             f'umap_all_transitions_dox{ds}.pdf'),
                )

    # ── Heatmap: RGCs/INs only, 4 transitions, RGCs+INs module scores only ──
    RGCIN_TRANSITIONS = ['RGCs -> RGCs', 'RGCs -> INs', 'INs -> INs', 'INs -> RGCs']
    RGCIN_SCORES      = ['RGCs', 'INs']   # only these two rows shown

    rgcin_sub = conn_df[conn_df['transition'].isin(RGCIN_TRANSITIONS)].copy()

    if not rgcin_sub.empty:
        # Sort by the fixed transition order, then by DCM RGCs z-score descending
        trans_order_map = {t: i for i, t in enumerate(RGCIN_TRANSITIONS)}
        rgcin_sub = rgcin_sub.copy()
        rgcin_sub['_t_order'] = rgcin_sub['transition'].map(trans_order_map).fillna(99)
        rgcin_sub['_self_z']  = rgcin_sub.apply(
            lambda r: r.get(f'dcm_{r["dcm_label"]}_zscore', 0)
            if r['dcm_label'] in RGCIN_SCORES else 0, axis=1
        )
        rgcin_sub = rgcin_sub.sort_values(
            ['_t_order', '_self_z'], ascending=[True, False]
        ).drop(columns=['_t_order', '_self_z']).reset_index(drop=True)

        def _save_rgcin(df, path, title_extra=''):
            if df.empty: return
            n  = len(df)
            fw = max(14, n * 0.18)

            dcm_mat = np.array(
                [[row.get(f'dcm_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T
            rna_mat = np.array(
                [[row.get(f'rna_{ct}_zscore', np.nan) for ct in RGCIN_SCORES]
                 for _, row in df.iterrows()]
            ).T

            fig, axes = plt.subplots(2, 1, figsize=(fw, 8))
            # DCM: group separators by transition, strip coloured by dcm_label
            _draw_heatmap_panel(axes[0], dcm_mat, RGCIN_SCORES,
                                'DCM (GMM label)', df['transition'],
                                df['dcm_label'], n, vcenter=VCENTER_DCM)
            # RNA: group separators by transition, strip coloured by rna_label
            _draw_heatmap_panel(axes[1], rna_mat, RGCIN_SCORES,
                                'RNA (cell_type_broad)', df['transition'],
                                df['rna_label'], n, vcenter=VCENTER_RNA)

            ttl = (f'Z-score heatmap — RGCs/INs transitions  (n={n})'
                   + (f'  |  {title_extra}' if title_extra else '')
                   + f'  DCM vcenter={VCENTER_DCM}, RNA vcenter={VCENTER_RNA}')
            fig.suptitle(ttl, fontsize=13, fontweight='bold')
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  Saved: {path}")

        stem = os.path.join(heat_dir, 'heatmap_RGCsINs_transitions2')
        _save_rgcin(rgcin_sub, f'{stem}.pdf')
        if 'Dox' in rgcin_sub.columns:
            for dox in sorted(rgcin_sub['Dox'].dropna().unique()):
                ds  = str(dox).replace(' ', '_').replace('/', '-')
                sub = rgcin_sub[rgcin_sub['Dox'] == dox]
                _save_rgcin(sub, f'{stem}_dox{ds}.pdf', title_extra=f'Dox={dox}')
    else:
        print("  No RGCs/INs transitions found in conn_df")

    # ── Heatmap: all transitions, grouped by DCM source label ──
    plot_zscore_heatmap(
        conn_df,
        title_str = f'Z-score heatmap — all transitions  (n={len(conn_df)})',
        save_path = os.path.join(heat_dir, 'heatmap_all_transitions2.pdf'),
    )

    # ── Per DCM source: UMAP + heatmap for all RNA destinations ──
    # Equivalent of the reference plot_umap(sec_to_ent_clear, ...) +
    # plot_heatmap(sec_to_ent_clear, ...) calls in Step 7.
    for dcm_ct in DESIRED_ORDER:
        sub_dcm = conn_df[conn_df['dcm_label'] == dcm_ct]
        if sub_dcm.empty:
            continue
        n = len(sub_dcm)
        rna_dests = sorted(sub_dcm['rna_label'].unique())
        plot_umap_transitions(
            sub_dcm,
            title_str = (f'DCM source: {dcm_ct} → all RNA destinations  (n={n})\n'
                         f'RNA: {", ".join(rna_dests)} | Arrow colour = RNA destination'),
            save_path = os.path.join(trans_dir,
                                     f'umap_clear_{dcm_ct}_to_all_RNA2.pdf'),
        )
        plot_zscore_heatmap(
            sub_dcm,
            title_str = (f'Z-score heatmap — DCM source: {dcm_ct}  '
                         f'(n={n}, all RNA destinations)'),
            save_path = os.path.join(heat_dir,
                                     f'heatmap_DCM_source_{dcm_ct}_all_RNA2.pdf'),
            group_col = 'rna_label',
        )

    # Save tables
    conn_df.to_csv(os.path.join(trans_dir, 'DCM_RNA_connections2.csv'), index=False)
    (conn_df.groupby('transition').size()
     .reset_index(name='count')
     .assign(pct=lambda d: 100 * d['count'] / d['count'].sum())
     .sort_values('count', ascending=False)
     .to_csv(os.path.join(trans_dir, 'DCM_RNA_transition_summary2.csv'), index=False))

print(f"\nOutputs in:\n  {trans_dir}\n  {heat_dir}")

NameError: name 'combined' is not defined

In [16]:
adata.obs['Dox']

X100_Plate_1          16h_24h_dtag
X154_Plate_1          16h_24h_dtag
X82_Plate_1           16h_24h_dtag
X174_Plate_1          16h_24h_dtag
X81_Plate_1           16h_24h_dtag
                          ...     
TTTGTTGGTCATTACC-1         Unknown
TTTGTTGGTCATTGTC-1         Unknown
TTTGTTGGTCCCGGAA-1         Unknown
TTTGTTGGTCGCAAAC-1         Unknown
TTTGTTGGTTGCTTCG-1         Unknown
Name: Dox, Length: 15920, dtype: category
Categories (3, object): ['16h', '16h_24h_dtag', 'Unknown']

In [49]:
##############################################
# TRANSITION COUNT HEATMAPS PER DOX
# Heatmap 1 (AFTER GMM):  DCM module_max_label → RNA cell_type_broad
# Heatmap 2 (BEFORE GMM): DCM cell_type_broad  → RNA cell_type_broad
# Separated by Dox condition
#
# Requires in memory: adata, best_outdir, DESIRED_ORDER
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
heat_dir = os.path.join(best_outdir, 'transition_heatmaps')
os.makedirs(heat_dir, exist_ok=True)

DESIRED_ORDER = ['RGCs', 'INs', 'IGs']

# ─────────────────────────────────────────────
# BUILD PLOT_DF
# ─────────────────────────────────────────────
adata.obs['batch'] = adata.obs['batch'].astype(str)

plot_df = pd.DataFrame({
    'batch':            adata.obs['batch'].values,
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).values,
    'Dox':              adata.obs['Dox'].astype(str).values
                        if 'Dox' in adata.obs.columns else 'Unknown',
}, index=adata.obs_names)

plot_df['_assay'] = 'other'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True), '_assay'] = 'DCM'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*RNA', case=False, na=False, regex=True), '_assay'] = 'RNA'

dox_conditions = sorted([
    d for d in plot_df['Dox'].unique()
    if d not in ('nan', 'Unknown', 'Minus')
])
print(f"Dox conditions: {dox_conditions}")

# ─────────────────────────────────────────────
# HELPER: match plate_wells 1:1 and return conn df
# ─────────────────────────────────────────────
def match_cells(dcm_df, rna_df, dcm_col, rna_col):
    common_pws = set(dcm_df['plate_well']) & set(rna_df['plate_well'])
    rows = []
    for pw in common_pws:
        d = dcm_df[dcm_df['plate_well'] == pw]
        r = rna_df[rna_df['plate_well'] == pw]
        if len(d) == 1 and len(r) == 1:
            rows.append({
                'dcm_label': d.iloc[0][dcm_col],
                'rna_label': r.iloc[0][rna_col],
            })
    return pd.DataFrame(rows)


def make_crosstab(conn_df, row_order, col_order):
    ct = pd.crosstab(conn_df['dcm_label'], conn_df['rna_label'], margins=True)
    rows = [r for r in row_order if r in ct.index] + (['All'] if 'All' in ct.index else [])
    cols = [c for c in col_order if c in ct.columns] + (['All'] if 'All' in ct.columns else [])
    return ct.reindex(index=rows, columns=cols, fill_value=0)


def plot_heatmap(crosstab_no_margins, title, save_path):
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(
        crosstab_no_margins,
        annot=True, fmt='d', cmap='RdBu_r',
        cbar_kws={'label': 'Cell count'},
        ax=ax, linewidths=0.5, linecolor='white',
    )
    ax.set_xlabel('RNA (destination)', fontsize=11)
    ax.set_ylabel('DCM (source)',       fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


# ─────────────────────────────────────────────
# HEATMAP 1 — AFTER GMM
# DCM module_max_label → RNA cell_type_broad
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("HEATMAP 1: AFTER GMM  (DCM module_max_label → RNA cell_type_broad)")
print("=" * 60)

for dox in dox_conditions:
    print(f"\n  Dox = {dox}")

    dcm = plot_df[
        (plot_df['_assay'] == 'DCM') &
        (plot_df['module_max_label'].isin(DESIRED_ORDER)) &
        (plot_df['Dox'] == dox)
    ]
    rna = plot_df[
        (plot_df['_assay'] == 'RNA') &
        (plot_df['Dox'] == dox)
    ]
    print(f"    DCM: {len(dcm)}  RNA: {len(rna)}")

    if dcm.empty or rna.empty:
        print("    Skipping — no data"); continue

    conn = match_cells(dcm, rna, 'module_max_label', 'cell_type_broad')
    if conn.empty:
        print("    No connections found"); continue
    print(f"    Connections: {len(conn)}")

    ct = make_crosstab(conn, DESIRED_ORDER, DESIRED_ORDER)
    ct.to_csv(os.path.join(heat_dir,
              f'crosstab_after_GMM_Dox_{dox}.csv'))

    ct_plot = ct.drop('All', axis=0, errors='ignore').drop('All', axis=1, errors='ignore')
    plot_heatmap(
        ct_plot,
        title     = f'DCM → RNA  AFTER GMM  |  Dox = {dox}\n(n = {len(conn)} pairs)',
        save_path = os.path.join(heat_dir, f'heatmap_after_GMM_Dox_{dox}.pdf'),
    )
    print(ct.to_string())

# ─────────────────────────────────────────────
# HEATMAP 2 — BEFORE GMM
# DCM cell_type_broad → RNA cell_type_broad
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("HEATMAP 2: BEFORE GMM  (DCM cell_type_broad → RNA cell_type_broad)")
print("=" * 60)

for dox in dox_conditions:
    print(f"\n  Dox = {dox}")

    dcm = plot_df[
        (plot_df['_assay'] == 'DCM') &
        (plot_df['cell_type_broad'].isin(DESIRED_ORDER)) &
        (plot_df['Dox'] == dox)
    ]
    rna = plot_df[
        (plot_df['_assay'] == 'RNA') &
        (plot_df['cell_type_broad'].isin(DESIRED_ORDER)) &
        (plot_df['Dox'] == dox)
    ]
    print(f"    DCM: {len(dcm)}  RNA: {len(rna)}")

    if dcm.empty or rna.empty:
        print("    Skipping — no data"); continue

    conn = match_cells(dcm, rna, 'cell_type_broad', 'cell_type_broad')
    if conn.empty:
        print("    No connections found"); continue
    print(f"    Connections: {len(conn)}")

    ct = make_crosstab(conn, DESIRED_ORDER, DESIRED_ORDER)
    ct.to_csv(os.path.join(heat_dir,
              f'crosstab_before_GMM_Dox_{dox}.csv'))

    ct_plot = ct.drop('All', axis=0, errors='ignore').drop('All', axis=1, errors='ignore')
    plot_heatmap(
        ct_plot,
        title     = f'DCM → RNA  BEFORE GMM  |  Dox = {dox}\n(n = {len(conn)} pairs)',
        save_path = os.path.join(heat_dir, f'heatmap_before_GMM_Dox_{dox}.pdf'),
    )
    print(ct.to_string())

# ─────────────────────────────────────────────
# COMPARISON SUMMARY — matching % per Dox
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("COMPARISON SUMMARY: BEFORE vs AFTER GMM")
print("=" * 60)

summary_rows = []
for dox in dox_conditions:
    row = {'Dox': dox}

    # AFTER GMM
    dcm_a = plot_df[(plot_df['_assay'] == 'DCM') &
                    (plot_df['module_max_label'].isin(DESIRED_ORDER)) &
                    (plot_df['Dox'] == dox)]
    rna_a = plot_df[(plot_df['_assay'] == 'RNA') & (plot_df['Dox'] == dox)]
    conn_a = match_cells(dcm_a, rna_a, 'module_max_label', 'cell_type_broad')
    if not conn_a.empty:
        match_a = (conn_a['dcm_label'] == conn_a['rna_label']).sum()
        row['n_after']        = len(conn_a)
        row['n_match_after']  = int(match_a)
        row['acc_after_pct']  = round(100 * match_a / len(conn_a), 1)
    else:
        row.update({'n_after': 0, 'n_match_after': 0, 'acc_after_pct': None})

    # BEFORE GMM
    dcm_b = plot_df[(plot_df['_assay'] == 'DCM') &
                    (plot_df['cell_type_broad'].isin(DESIRED_ORDER)) &
                    (plot_df['Dox'] == dox)]
    rna_b = plot_df[(plot_df['_assay'] == 'RNA') &
                    (plot_df['cell_type_broad'].isin(DESIRED_ORDER)) &
                    (plot_df['Dox'] == dox)]
    conn_b = match_cells(dcm_b, rna_b, 'cell_type_broad', 'cell_type_broad')
    if not conn_b.empty:
        match_b = (conn_b['dcm_label'] == conn_b['rna_label']).sum()
        row['n_before']       = len(conn_b)
        row['n_match_before'] = int(match_b)
        row['acc_before_pct'] = round(100 * match_b / len(conn_b), 1)
    else:
        row.update({'n_before': 0, 'n_match_before': 0, 'acc_before_pct': None})

    summary_rows.append(row)
    print(f"\n  Dox {dox}:")
    print(f"    AFTER  GMM: {row.get('n_match_after',0)}/{row.get('n_after',0)} matching  ({row.get('acc_after_pct','n/a')}%)")
    print(f"    BEFORE GMM: {row.get('n_match_before',0)}/{row.get('n_before',0)} matching  ({row.get('acc_before_pct','n/a')}%)")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(heat_dir, 'comparison_before_after_GMM.csv'), index=False)
print(f"\nSummary saved: {os.path.join(heat_dir, 'comparison_before_after_GMM.csv')}")
print(f"\nAll outputs in: {heat_dir}")

Dox conditions: ['16h', '16h_24h_dtag']

HEATMAP 1: AFTER GMM  (DCM module_max_label → RNA cell_type_broad)

  Dox = 16h
    DCM: 61  RNA: 233
    Connections: 59
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/transition_heatmaps/heatmap_after_GMM_Dox_16h.pdf
rna_label  RGCs  INs  IGs  All
dcm_label                     
RGCs          5    0    0    5
INs          24    8    3   35
IGs          13    1    5   19
All          42    9    8   59

  Dox = 16h_24h_dtag
    DCM: 115  RNA: 490
    Connections: 115
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/transition_heatmaps/heatmap_after_GMM_Dox_16h_24h_dtag.pdf
rna_label  RGCs  INs  IGs  All
dcm_label                     
RGCs         13    1    5   19
INs          42   1

In [50]:
##############################################
# PAIRED DCM→RNA Z-SCORE HEATMAPS
# Per-cell row-normalised z-scores for all
# cell types, split by transition and Dox.
# DCM on top, RNA on bottom per combined plot.
#
# Requires in memory:
#   adata, best_outdir, DESIRED_ORDER,
#   MODULE_ZSCORE_COLS (from zscore_boxplots.py)
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
heat_dir = os.path.join(best_outdir, 'transition_heatmaps')
os.makedirs(heat_dir, exist_ok=True)

DESIRED_ORDER = ['RGCs', 'INs', 'IGs']

# ─────────────────────────────────────────────
# FIND MODULE SCORE Z COLUMNS IN adata.obs
# We use the per-assay z-scored columns computed
# in gene_scoring_zscore.py (_module_score_Z suffix)
# ─────────────────────────────────────────────
module_z_cols = [c for c in adata.obs.columns
                 if c.endswith('_module_score_Z') and
                 not c.endswith('_module_score_Z_all')]

# Fall back to _all suffix if needed
if not module_z_cols:
    module_z_cols = [c for c in adata.obs.columns
                     if c.endswith('_module_score_Z_all')]

print(f"Module Z-score columns: {module_z_cols}")

# Map cell type name → column
ct_to_col = {}
for ct in DESIRED_ORDER:
    match = [c for c in module_z_cols if c.startswith(ct)]
    if match:
        ct_to_col[ct] = match[0]

print(f"Cell type → column map: {ct_to_col}")
if len(ct_to_col) < 2:
    raise ValueError("Need at least 2 cell type z-score columns. "
                     "Run gene_scoring_zscore.py first.")

# ─────────────────────────────────────────────
# BUILD plot_df
# ─────────────────────────────────────────────
adata.obs['batch'] = adata.obs['batch'].astype(str)

plot_df = pd.DataFrame({
    'batch':            adata.obs['batch'].values,
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).values,
    'Dox':              adata.obs['Dox'].astype(str).values
                        if 'Dox' in adata.obs.columns else 'Unknown',
}, index=adata.obs_names)

for ct, col in ct_to_col.items():
    plot_df[col] = adata.obs[col].values

plot_df['_assay'] = 'other'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*DCM', case=False, na=False, regex=True), '_assay'] = 'DCM'
plot_df.loc[plot_df['batch'].str.contains('scNMT.*RNA', case=False, na=False, regex=True), '_assay'] = 'RNA'

dox_conditions = sorted([
    d for d in plot_df['Dox'].unique()
    if d not in ('nan', 'Unknown', 'Minus')
])
print(f"Dox conditions: {dox_conditions}")

# ─────────────────────────────────────────────
# TRANSITIONS OF INTEREST
# All pairwise combinations of DESIRED_ORDER
# ─────────────────────────────────────────────
from itertools import product as iterproduct

transitions_of_interest = [
    f"{d} -> {r}"
    for d in DESIRED_ORDER
    for r in DESIRED_ORDER
]
print(f"Transitions to check: {transitions_of_interest}")

# ─────────────────────────────────────────────
# HELPER: per-cell row normalisation
# ─────────────────────────────────────────────
def row_scale(matrix):
    scaled = np.zeros_like(matrix, dtype=float)
    for i in range(matrix.shape[0]):
        row = matrix[i, :]
        mu, sigma = row.mean(), row.std()
        scaled[i, :] = (row - mu) / sigma if sigma > 0 else np.zeros_like(row)
    return scaled


# ─────────────────────────────────────────────
# HELPER: transition boundary positions + labels
# ─────────────────────────────────────────────
def get_boundaries(transitions_series, order):
    # Sort by transition order
    order_map  = {t: i for i, t in enumerate(order)}
    sorted_idx = transitions_series.map(order_map).fillna(99).argsort()
    transitions_sorted = transitions_series.iloc[sorted_idx].reset_index(drop=True)

    boundaries, labels, centers = [], [], []
    current, start = None, 0
    for i, t in enumerate(transitions_sorted):
        if t != current:
            if current is not None:
                boundaries.append(i)
                labels.append(current)
                centers.append((start + i - 1) / 2)
            current, start = t, i
    labels.append(current)
    centers.append((start + len(transitions_sorted) - 1) / 2)
    return boundaries, labels, centers, sorted_idx


# ─────────────────────────────────────────────
# HELPER: draw one heatmap panel
# ─────────────────────────────────────────────
def draw_panel(ax, data, row_labels, boundaries, ylabel, title=None):
    sns.heatmap(
        data,
        cmap='RdBu_r', center=0,
        annot=False,
        xticklabels=False, yticklabels=row_labels,
        cbar_kws={'label': 'Scaled Z-score'},
        ax=ax,
        linewidths=0.1, linecolor='white',
        vmin=-2, vmax=2,
    )
    for b in boundaries:
        ax.axvline(x=b, color='black', linewidth=2)
    ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold')


# ─────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────
ct_display = list(ct_to_col.keys())   # ordered cell type names for heatmap rows
score_cols  = [ct_to_col[ct] for ct in ct_display]

for dox in dox_conditions:
    print(f"\n{'='*60}\nDox = {dox}\n{'='*60}")

    dcm = plot_df[
        (plot_df['_assay'] == 'DCM') &
        (plot_df['module_max_label'].isin(DESIRED_ORDER)) &
        (plot_df['Dox'] == dox)
    ]
    rna = plot_df[
        (plot_df['_assay'] == 'RNA') &
        (plot_df['cell_type_broad'].isin(DESIRED_ORDER)) &
        (plot_df['Dox'] == dox)
    ]
    print(f"  DCM: {len(dcm)}  RNA: {len(rna)}")

    if dcm.empty or rna.empty:
        print("  Skipping — no data"); continue

    # Match 1:1
    common_pws = set(dcm['plate_well']) & set(rna['plate_well'])
    paired = []
    for pw in common_pws:
        d = dcm[dcm['plate_well'] == pw]
        r = rna[rna['plate_well'] == pw]
        if len(d) == 1 and len(r) == 1:
            dr, rr = d.iloc[0], r.iloc[0]
            trans = f"{dr['module_max_label']} -> {rr['cell_type_broad']}"
            if trans not in transitions_of_interest:
                continue
            row = {
                'plate_well':  pw,
                'dcm_label':   dr['module_max_label'],
                'rna_label':   rr['cell_type_broad'],
                'transition':  trans,
            }
            for col in score_cols:
                row[f'dcm_{col}'] = dr.get(col, np.nan)
                row[f'rna_{col}'] = rr.get(col, np.nan)
            paired.append(row)

    if not paired:
        print("  No paired data"); continue

    paired_df = pd.DataFrame(paired)
    print(f"  Pairs: {len(paired_df)}")
    print(paired_df['transition'].value_counts().to_string())

    # Sort by transition order then by first cell-type score descending
    boundaries, labels, centers, sorted_idx = get_boundaries(
        paired_df['transition'], transitions_of_interest
    )
    paired_df = paired_df.iloc[sorted_idx].reset_index(drop=True)

    # Build matrices (n_cells × n_celltypes)
    dcm_mat = paired_df[[f'dcm_{c}' for c in score_cols]].values.astype(float)
    rna_mat = paired_df[[f'rna_{c}' for c in score_cols]].values.astype(float)

    dcm_scaled = row_scale(dcm_mat)
    rna_scaled = row_scale(rna_mat)

    # Transpose: rows = cell types, cols = cells
    dcm_hm = pd.DataFrame(dcm_scaled.T, index=ct_display)
    rna_hm = pd.DataFrame(rna_scaled.T, index=ct_display)

    n_cells   = len(paired_df)
    fig_width = max(10, n_cells * 0.3)
    n_rows    = len(ct_display)

    # ── DCM only ──
    fig, ax = plt.subplots(figsize=(fig_width, 1.5 * n_rows))
    draw_panel(ax, dcm_hm, ct_display, boundaries, 'Cell type',
               title=f'DCM module scores — scaled per cell  |  Dox={dox}  (n={n_cells})')
    for lbl, ctr in zip(labels, centers):
        ax.text(ctr + 0.5, n_rows + 0.2, lbl, ha='center', va='top',
                fontsize=9, fontweight='bold')
    ax.set_xlabel('Cells', fontsize=11)
    plt.tight_layout()
    out = os.path.join(heat_dir, f'heatmap_DCM_scaled_Dox_{dox}.pdf')
    plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {out}")

    # ── RNA only ──
    fig, ax = plt.subplots(figsize=(fig_width, 1.5 * n_rows))
    draw_panel(ax, rna_hm, ct_display, boundaries, 'Cell type',
               title=f'RNA module scores — scaled per cell  |  Dox={dox}  (n={n_cells})')
    for lbl, ctr in zip(labels, centers):
        ax.text(ctr + 0.5, n_rows + 0.2, lbl, ha='center', va='top',
                fontsize=9, fontweight='bold')
    ax.set_xlabel('Cells', fontsize=11)
    plt.tight_layout()
    out = os.path.join(heat_dir, f'heatmap_RNA_scaled_Dox_{dox}.pdf')
    plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {out}")

    # ── Combined DCM + RNA ──
    fig, axes = plt.subplots(2, 1, figsize=(fig_width, 3 * n_rows))
    draw_panel(axes[0], dcm_hm, ct_display, boundaries, 'DCM',
               title=f'Scaled module scores  |  Dox={dox}  (n={n_cells})')
    draw_panel(axes[1], rna_hm, ct_display, boundaries, 'RNA')
    for lbl, ctr in zip(labels, centers):
        axes[1].text(ctr + 0.5, n_rows + 0.2, lbl, ha='center', va='top',
                     fontsize=9, fontweight='bold')
    axes[1].set_xlabel('Cells', fontsize=11)
    plt.tight_layout()
    out = os.path.join(heat_dir, f'heatmap_DCM_RNA_combined_scaled_Dox_{dox}.pdf')
    plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved: {out}")

    # ── Save paired CSV ──
    for ct, col in zip(ct_display, score_cols):
        paired_df[f'dcm_{ct}_scaled'] = dcm_scaled[:, ct_display.index(ct)]
        paired_df[f'rna_{ct}_scaled'] = rna_scaled[:, ct_display.index(ct)]

    paired_df.to_csv(
        os.path.join(heat_dir, f'paired_DCM_RNA_scaled_Dox_{dox}.csv'),
        index=False,
    )
    print(f"  Saved: paired_DCM_RNA_scaled_Dox_{dox}.csv")

print(f"\nAll outputs in: {heat_dir}")

Module Z-score columns: ['IGs_module_score_Z', 'INs_module_score_Z', 'RGCs_module_score_Z']
Cell type → column map: {'RGCs': 'RGCs_module_score_Z', 'INs': 'INs_module_score_Z', 'IGs': 'IGs_module_score_Z'}
Dox conditions: ['16h', '16h_24h_dtag']
Transitions to check: ['RGCs -> RGCs', 'RGCs -> INs', 'RGCs -> IGs', 'INs -> RGCs', 'INs -> INs', 'INs -> IGs', 'IGs -> RGCs', 'IGs -> INs', 'IGs -> IGs']

Dox = 16h
  DCM: 61  RNA: 233
  Pairs: 59
transition
INs -> RGCs     24
IGs -> RGCs     13
INs -> INs       8
RGCs -> RGCs     5
IGs -> IGs       5
INs -> IGs       3
IGs -> INs       1
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/transition_heatmaps/heatmap_DCM_scaled_Dox_16h.pdf
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_be

In [ ]:
# Module scores

In [10]:
# Check that module_max_label matches the column with the highest value (when max >= 0.8370)
score_cols = [
    'scNMT_pool1_DCM_IGs_module_score_Z_posterior',
    'scNMT_pool1_DCM_INs_module_score_Z_posterior',
    'scNMT_pool1_DCM_RGCs_module_score_Z_posterior'
]

# Map column names to expected labels
col_to_label = {
    'scNMT_pool1_DCM_IGs_module_score_Z_posterior': 'IGs',
    'scNMT_pool1_DCM_INs_module_score_Z_posterior': 'INs',
    'scNMT_pool1_DCM_RGCs_module_score_Z_posterior': 'RGCs'
}

scores = adata.obs[score_cols]
max_vals = scores.max(axis=1)
max_col = scores.idxmax(axis=1)
expected_label = max_col.map(col_to_label)

mask = max_vals >= 0.8370

matches = adata.obs.loc[mask, 'module_max_label'] == expected_label[mask]

print(f"Cells with max >= 0.8370: {mask.sum()}")
print(f"Matching: {matches.sum()}")
print(f"Mismatching: {(~matches).sum()}")

if (~matches).any():
    mis = adata.obs.loc[mask & ~matches, ['module_max_label'] + score_cols]INs_score_raw_all
    print("\nMismatched cells:")
    print(mis)
else:
    print("\nAll labels match correctly!")

Cells with max >= 0.8370: 176
Matching: 176
Mismatching: 0

All labels match correctly!


/tmp/ipykernel_3523769/3108663731.py:17: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  max_col = scores.idxmax(axis=1)


In [4]:
adata

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [6]:
import seaborn as sns
stable_IGs = paired[(paired['module_max_label'] == 'IGs') & (paired['transition'] == 'Stable')]
stable_INs = paired[(paired['module_max_label'] == 'INs') & (paired['transition'] == 'Stable')]

print("Stable IGs — IGs raw score:", stable_IGs['IGs_score_raw_all'].median())
print("Stable INs — IGs raw score:", stable_INs['IGs_score_raw_all'].median())
print("Stable INs — INs raw score:", stable_INs['INs_score_raw_all'].median())
print("Stable IGs — INs raw score:", stable_IGs['INs_score_raw_all'].median())

Stable IGs — IGs raw score: 0.27453123053188877
Stable INs — IGs raw score: 0.30550458249346296
Stable INs — INs raw score: 0.5149144026118038
Stable IGs — INs raw score: 0.40865410039531247


In [9]:
import seaborn as sns
stable_IGs = paired[(paired['module_max_label'] == 'IGs') & (paired['transition'] == 'Stable')]
stable_INs = paired[(paired['module_max_label'] == 'INs') & (paired['transition'] == 'Stable')]

print("Stable IGs — IGs raw score:", stable_IGs['IGs_module_score_Z_all'].median())
print("Stable INs — IGs raw score:", stable_INs['IGs_module_score_Z_all'].median())
print("Stable INs — INs raw score:", stable_INs['INs_module_score_Z_all'].median())
print("Stable IGs — INs raw score:", stable_IGs['INs_module_score_Z_all'].median())

Stable IGs — IGs raw score: 1.7148434559373724
Stable INs — IGs raw score: 0.3923823307599535
Stable INs — INs raw score: 1.4440730153555315
Stable IGs — INs raw score: 0.5889324787900931


In [3]:
##############################################
# STABLE vs REPROGRAMMED-TO-RGC ANALYSIS
# NPC data
#
# For each DCM cell type (INs / IGs),
# cells are split into:
#   Stable     — DCM label == RNA label
#   To_RGC     — DCM label != RNA label AND RNA == RGCs
#
# The module score for that cell type is plotted per group, per Dox,
# and combined (ALL).
#
# NEGATIVE CONTROLS:
#   per_celltype_merged plots:
#     INs subplot → NegCtrl = Stable IGs scored on INs module
#     IGs subplot → NegCtrl = Stable INs scored on IGs module
#   stemta_stable_vs_reversed plots:
#     NegCtrl = Stable INs + Stable IGs pooled, scored on RGCs module
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import binomtest

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
score_dir = '/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/MS_stable_transition'
os.makedirs(score_dir, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}

GROUP_COLORS = {
    'Stable':              '#2ecc71',
    'To_RGC':              '#003366',
    'NegCtrl':             '#AAAAAA',
    'Stable_RGCs':         '#2ecc71',
    'Reversed_to_RGCs':    '#003366',
}

label_order = ['RGCs', 'INs', 'IGs']

# "Progenitor" = RGCs (like Stem in the intestinal code)
PROGENITOR_TYPES = ['RGCs']
# "Mature" = INs, IGs (like Enterocyte + Secretory)
MATURE_TYPES     = ['INs', 'IGs']
ANALYSIS_TYPES   = ['INs', 'IGs']   # DCM cell types whose stability we assess

# Module score columns
ttest_cols = [
    'RGCs_module_score_Z_all',
    'INs_module_score_Z_all',
    'IGs_module_score_Z_all',
]

ct_from_col = {
    'RGCs_module_score_Z_all': 'RGCs',
    'INs_module_score_Z_all':  'INs',
    'IGs_module_score_Z_all':  'IGs',
}
col_from_ct = {v: k for k, v in ct_from_col.items()}

DOX_COLORS = {
    '16h':          '#4A90D9',
    '16h_24h_dtag': '#E87722',
}
DOX_ORDER  = ['16h', '16h_24h_dtag']
DOX_LABELS = {'16h': 'Pulse (16h)', '16h_24h_dtag': 'Chase (16h_24h_dtag)'}

# ─────────────────────────────────────────────
# STEP 1: BUILD DATAFRAME
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING DATAFRAME")
print("=" * 60)

base = pd.DataFrame({
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'batch':            adata.obs['batch'].astype(str).values,
    'Dox':              adata.obs['Dox'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).values,
}, index=adata.obs_names)

for col in ttest_cols:
    base[col] = adata.obs[col].values

print(f"Total cells : {len(base)}")
print(f"Batch values: {sorted(base['batch'].unique())}")
print(f"Dox values  : {sorted(base['Dox'].unique())}")

DOX_VALUES = sorted(base['Dox'].dropna().unique())

# ─────────────────────────────────────────────
# STEP 2: PAIR DCM ↔ RNA  (1:1 via plate_well)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: PAIRING DCM ↔ RNA")
print("=" * 60)

dcm_df = base[base['batch'] == 'scNMT_DCM'].copy()
rna_df = (base[base['batch'] == 'scNMT_RNA']
          [['plate_well', 'cell_type_broad']]
          .copy()
          .rename(columns={'cell_type_broad': 'rna_celltype'}))

def match_one_to_one(df_a, df_b):
    common = set(df_a['plate_well']) & set(df_b['plate_well'])
    a_s = set(df_a[df_a['plate_well'].isin(common)]
              ['plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    b_s = set(df_b[df_b['plate_well'].isin(common)]
              ['plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    return a_s & b_s

matched_wells = match_one_to_one(dcm_df, rna_df)
print(f"1:1 matched wells: {len(matched_wells)}")

paired = (dcm_df[dcm_df['plate_well'].isin(matched_wells)]
          .merge(rna_df[rna_df['plate_well'].isin(matched_wells)],
                 on='plate_well', how='inner'))
print(f"Paired cells: {len(paired)}")

# ─────────────────────────────────────────────
# STEP 3: FILTER AND CLASSIFY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: CLASSIFYING")
print("=" * 60)

paired = paired[paired['module_max_label'].isin(label_order)].copy()
paired = paired[paired['module_max_label'].isin(ANALYSIS_TYPES)].copy()
print(f"Cells with DCM in ANALYSIS_TYPES: {len(paired)}")

def classify(row):
    dcm = row['module_max_label']
    rna = row['rna_celltype']
    if dcm == rna:
        return 'Stable'
    if rna == 'RGCs':
        return 'To_RGC'
    return 'Other'

paired['transition'] = paired.apply(classify, axis=1)
# No combined needed since there's only one progenitor destination,
# but keep column for code compatibility
paired['transition_combined'] = paired['transition']

print("\nTransition counts (ALL Dox):")
print(paired['transition'].value_counts().to_string())
print("\nPer DCM cell type:")
print(paired.groupby(['module_max_label', 'transition']).size()
           .unstack(fill_value=0).to_string())

def get_self_module(row):
    col = col_from_ct.get(row['module_max_label'])
    return row[col] if (col is not None and col in row.index) else np.nan

paired['self_module_score'] = paired.apply(get_self_module, axis=1)

# ─────────────────────────────────────────────
# BUILD RGCs REFERENCE DATAFRAME
# ─────────────────────────────────────────────
_all_paired_raw = (dcm_df[dcm_df['plate_well'].isin(matched_wells)]
                   .merge(rna_df[rna_df['plate_well'].isin(matched_wells)],
                          on='plate_well', how='inner'))

rows_rgc = []
dest_ct = 'RGCs'
mod_col = col_from_ct.get(dest_ct)

if mod_col is not None:
    stable_sub = _all_paired_raw[
        (_all_paired_raw['module_max_label'] == dest_ct) &
        (_all_paired_raw['rna_celltype']     == dest_ct)
    ].copy()
    stable_sub['stemta_group']  = 'Stable_RGCs'
    stable_sub['stemta_dest']   = dest_ct
    stable_sub['stemta_score']  = stable_sub[mod_col]

    reversed_sub = paired[paired['rna_celltype'] == dest_ct].copy()
    reversed_sub['stemta_group']  = 'Reversed_to_RGCs'
    reversed_sub['stemta_dest']   = dest_ct
    reversed_sub['stemta_score']  = reversed_sub[mod_col]

    rows_rgc.append(stable_sub)
    rows_rgc.append(reversed_sub)

paired_stemta = pd.concat(rows_rgc, ignore_index=True)

print(f"\nRGCs comparison dataframe: {len(paired_stemta)} rows")
print(paired_stemta.groupby(['stemta_dest', 'stemta_group']).size().to_string())

# ─────────────────────────────────────────────
# NEGATIVE CONTROL HELPERS
# ─────────────────────────────────────────────

def build_neg_ctrl_for_celltype(paired_all, ct, dox_subset=None):
    """
    For focal cell type `ct`, extract Stable cells of the OTHER mature type
    and return their score on the `ct` module.
    INs subplot → NegCtrl = Stable IGs
    IGs subplot → NegCtrl = Stable INs
    """
    mod_col = col_from_ct.get(ct)
    if mod_col is None:
        return pd.DataFrame()

    other_types = [t for t in MATURE_TYPES if t != ct]
    ctrl_label = f'NegCtrl ({"+".join(other_types)})'

    src = paired_all if dox_subset is None else paired_all[
        paired_all['Dox'].isin(dox_subset)
    ]
    ctrl = src[
        (src['module_max_label'].isin(other_types)) &
        (src['transition'] == 'Stable')
    ].copy()

    if ctrl.empty:
        return pd.DataFrame()

    ctrl['group'] = ctrl_label
    ctrl['score'] = ctrl[mod_col]
    return ctrl[['group', 'score', 'module_max_label']]


def build_neg_ctrl_stemta(paired_all_raw, dest_ct, dox_subset=None):
    """
    NegCtrl for RGCs module: Stable INs + Stable IGs pooled,
    scored on RGCs module.
    """
    mod_col = col_from_ct.get(dest_ct)
    if mod_col is None:
        return pd.DataFrame()

    src = paired_all_raw if dox_subset is None else paired_all_raw[
        paired_all_raw['Dox'].isin(dox_subset)
    ]

    ctrl = src[
        (src['module_max_label'].isin(MATURE_TYPES)) &
        (src['rna_celltype'] == src['module_max_label'])   # Stable
    ].copy()

    if ctrl.empty:
        return pd.DataFrame()

    ctrl['stemta_group'] = 'NegCtrl'
    ctrl['stemta_dest']  = dest_ct
    ctrl['stemta_score'] = ctrl[mod_col]
    return ctrl[['stemta_group', 'stemta_dest', 'stemta_score',
                 'module_max_label', 'Dox']]


# ─────────────────────────────────────────────
# STATS HELPERS
# ─────────────────────────────────────────────
def get_sig(p):
    if pd.isna(p):  return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

def ttest_vs_ref(df, score_col, group_col, test_group, ref_group='Stable'):
    s = df[df[group_col] == ref_group  ][score_col].dropna()
    u = df[df[group_col] == test_group ][score_col].dropna()
    if len(s) > 1 and len(u) > 1:
        return stats.ttest_ind(u, s)[1]
    return np.nan

def ttest_vs_stable(df, score_col, group_col, test_group):
    return ttest_vs_ref(df, score_col, group_col, test_group, ref_group='Stable')

def compute_stats(df, score_col, group_col, dox_label, ct, context):
    rows = []
    for grp in df[group_col].unique():
        vals = df[df[group_col] == grp][score_col].dropna()
        pval = ttest_vs_stable(df, score_col, group_col, grp) if grp != 'Stable' else np.nan
        rows.append({
            'context':              context,
            'cell_type':            ct,
            'Dox':                  dox_label,
            'group':                grp,
            'score_col':            score_col,
            'N':                    len(vals),
            'mean':                 vals.mean()   if len(vals) else np.nan,
            'median':               vals.median() if len(vals) else np.nan,
            'std':                  vals.std()    if len(vals) else np.nan,
            'ttest_vs_Stable_pval': pval,
            'significance':         get_sig(pval),
        })
    return pd.DataFrame(rows)

# ─────────────────────────────────────────────
# PLOT HELPERS
# ─────────────────────────────────────────────
def draw_bracket(ax, group_order, i_test, i_ref, y_lvl, stars):
    ax.annotate('', xy=(i_test, y_lvl), xytext=(i_ref, y_lvl),
                arrowprops=dict(arrowstyle='-', color='#333333', lw=0.9))
    x_mid = (i_test + i_ref) / 2
    ax.text(x_mid, y_lvl, stars,
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            color='black' if stars not in ('ns', '') else '#888888')


def make_per_celltype_plot(df, group_col, group_order_full,
                           palette, dox_label, save_path, context):
    ct_present = []
    for ct in ANALYSIS_TYPES:
        sub = df[df['module_max_label'] == ct]
        grps = sub[group_col].unique()
        if 'Stable' in grps and len(grps) > 1:
            ct_present.append(ct)

    if not ct_present:
        print(f"  Skipping per-celltype Dox={dox_label} — no data")
        return pd.DataFrame()

    n_ct  = len(ct_present)
    fig, axes = plt.subplots(1, n_ct, figsize=(4.5 * n_ct, 6), sharey=False)
    if n_ct == 1:
        axes = [axes]

    all_stats = []

    for ax, ct in zip(axes, ct_present):
        col = col_from_ct.get(ct)
        if col is None:
            ax.set_visible(False)
            continue

        sub = df[df['module_max_label'] == ct].copy()
        present = [g for g in group_order_full if g in sub[group_col].values]

        plot_sub = sub[[col, group_col]].rename(columns={col: 'score'})

        sns.boxplot(
            data=plot_sub, x=group_col, y='score',
            hue=group_col, order=present,
            palette=palette, legend=False, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        for i, grp in enumerate(present):
            n = len(plot_sub[plot_sub[group_col] == grp])
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        if 'Stable' in present:
            ref_i    = present.index('Stable')
            y_levels = [y_top * 1.09, y_top * 1.17]
            y_max    = y_levels[0]
            k = 0
            for grp in present:
                if grp == 'Stable':
                    continue
                pval  = ttest_vs_stable(plot_sub, 'score', group_col, grp)
                stars = get_sig(pval)
                y_lvl = y_levels[k % 2]
                y_max = max(y_max, y_lvl)
                draw_bracket(ax, present, present.index(grp), ref_i, y_lvl, stars)
                k += 1
            ax.set_ylim(ylim[0], y_max * 1.12)

        ax.set_title(ct, fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{ct} module score', fontsize=9)
        ax.tick_params(axis='x', rotation=20, labelsize=9)

        all_stats.append(compute_stats(sub, col, group_col,
                                       dox_label, ct, context))

    plt.suptitle(
        f'Module Scores — Stable vs Reprogrammed to RGCs | Dox = {dox_label}\n'
        f't-tests vs Stable within each cell type',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()


def make_per_celltype_merged_plot_with_negctrl(
        df, paired_full, group_col, group_order_full,
        palette, dox_label, save_path, context):
    ct_present = []
    for ct in ANALYSIS_TYPES:
        sub = df[df['module_max_label'] == ct]
        grps = sub[group_col].unique()
        if 'Stable' in grps and len(grps) > 1:
            ct_present.append(ct)

    if not ct_present:
        print(f"  Skipping per-celltype-merged+negctrl Dox={dox_label} — no data")
        return pd.DataFrame()

    active_dox = list(df['Dox'].unique()) if dox_label != 'ALL' else None

    n_ct  = len(ct_present)
    fig, axes = plt.subplots(1, n_ct, figsize=(5 * n_ct, 6), sharey=False)
    if n_ct == 1:
        axes = [axes]

    all_stats = []
    NEGCTRL_COLOR = '#BBBBBB'

    for ax, ct in zip(axes, ct_present):
        col = col_from_ct.get(ct)
        if col is None:
            ax.set_visible(False)
            continue

        sub = df[df['module_max_label'] == ct].copy()
        present = [g for g in group_order_full if g in sub[group_col].values]

        nc_df = build_neg_ctrl_for_celltype(paired_full, ct,
                                             dox_subset=active_dox)
        neg_ctrl_label = nc_df['group'].iloc[0] if not nc_df.empty else None

        main_plot = sub[[col, group_col]].rename(
            columns={col: 'score', group_col: 'group'})
        main_plot['is_negctrl'] = False

        if neg_ctrl_label is not None:
            nc_plot = nc_df[['score', 'group']].copy()
            nc_plot['is_negctrl'] = True
            all_data = pd.concat([main_plot, nc_plot], ignore_index=True)
        else:
            all_data = main_plot

        full_order  = present + ([neg_ctrl_label] if neg_ctrl_label else [])
        full_palette = {g: palette.get(g, NEGCTRL_COLOR) for g in full_order}
        if neg_ctrl_label:
            full_palette[neg_ctrl_label] = NEGCTRL_COLOR

        sns.boxplot(
            data=all_data, x='group', y='score',
            hue='group', order=full_order,
            palette=full_palette, legend=False, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.4),
        )

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        for i, grp in enumerate(full_order):
            n = int((all_data['group'] == grp).sum())
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        y_max_bracket = y_top
        if 'Stable' in present:
            ref_i    = full_order.index('Stable')
            y_levels = [y_top * 1.09, y_top * 1.17]
            y_max_bracket = y_levels[0]
            k = 0
            for grp in present:
                if grp == 'Stable':
                    continue
                pval  = ttest_vs_ref(all_data, 'score', 'group', grp,
                                     ref_group='Stable')
                stars = get_sig(pval)
                y_lvl = y_levels[k % 2]
                y_max_bracket = max(y_max_bracket, y_lvl)
                draw_bracket(ax, full_order, full_order.index(grp),
                             ref_i, y_lvl, stars)
                k += 1

        if neg_ctrl_label and len(present) > 0:
            sep_x = len(present) - 0.5
            ax.axvline(sep_x, color='#666666', lw=1.2,
                       linestyle='--', alpha=0.7)
            pval_nc = ttest_vs_ref(all_data, 'score', 'group',
                                   neg_ctrl_label, ref_group='Stable')
            stars_nc = get_sig(pval_nc)
            ax.text(len(full_order) - 1, y_max_bracket * 1.05,
                    f'vs Stable:\n{stars_nc}',
                    ha='center', va='bottom', fontsize=7,
                    color='#555555', style='italic')

        ax.set_ylim(ylim[0], max(y_max_bracket, y_top) * 1.22)

        ax.set_title(ct, fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{ct} module Z-score', fontsize=9)
        ax.tick_params(axis='x', rotation=25, labelsize=8)

        all_stats.append(compute_stats(
            all_data, 'score', 'group', dox_label, ct, context))

    plt.suptitle(
        f'Module Scores — Stable vs To_RGC | Dox = {dox_label}\n'
        f'Grey box = negative control (opposite mature type, Stable cells)\n'
        f't-tests vs Stable; dashed line separates neg ctrl',
        fontsize=12, fontweight='bold', y=1.03
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()


def make_combined_plot(df, group_col, group_order_full,
                       palette, dox_label, save_path, context,
                       paired_full=None):
    sub = df[df['self_module_score'].notna()].copy()
    present = [g for g in group_order_full if g in sub[group_col].values]

    if len(present) < 2 or 'Stable' not in present:
        print(f"  Skipping combined Dox={dox_label} — insufficient groups")
        return pd.DataFrame()

    # ── Build NegCtrl: cross-scores of Stable cells ──────────────────────
    # Stable INs → scored on IGs module
    # Stable IGs → scored on INs module
    nc_rows = []
    if paired_full is not None:
        active_dox = list(df['Dox'].unique()) if dox_label != 'ALL' else None
        src = paired_full if active_dox is None else paired_full[
            paired_full['Dox'].isin(active_dox)
        ]
        stable_src = src[src['transition'] == 'Stable']

        for focal_ct, cross_ct in [('INs', 'IGs'), ('IGs', 'INs')]:
            cross_col = col_from_ct.get(cross_ct)
            if cross_col is None:
                continue
            cross_cells = stable_src[
                stable_src['module_max_label'] == focal_ct
            ][[cross_col, 'module_max_label']].copy()
            cross_cells = cross_cells.rename(columns={cross_col: 'self_module_score'})
            cross_cells[group_col] = 'NegCtrl'
            nc_rows.append(cross_cells)

    neg_ctrl_label = None
    if nc_rows:
        nc_df = pd.concat(nc_rows, ignore_index=True)
        neg_ctrl_label = 'NegCtrl'
        plot_sub = pd.concat([
            sub[[group_col, 'self_module_score', 'module_max_label']],
            nc_df[[group_col, 'self_module_score', 'module_max_label']],
        ], ignore_index=True)
    else:
        plot_sub = sub[[group_col, 'self_module_score', 'module_max_label']].copy()

    full_order   = present + ([neg_ctrl_label] if neg_ctrl_label else [])
    NEGCTRL_COLOR = '#BBBBBB'
    full_palette  = {g: palette.get(g, NEGCTRL_COLOR) for g in full_order}
    if neg_ctrl_label:
        full_palette[neg_ctrl_label] = NEGCTRL_COLOR

    fig, ax = plt.subplots(figsize=(max(5, 2.2 * len(full_order)), 6))

    sns.boxplot(
        data=plot_sub, x=group_col, y='self_module_score',
        hue=group_col, order=full_order,
        palette=full_palette, legend=False, ax=ax,
        flierprops=dict(marker='o', markersize=2, alpha=0.4),
        width=0.5,
    )
    sns.stripplot(
        data=plot_sub, x=group_col, y='self_module_score',
        hue='module_max_label', order=full_order,
        palette=cell_type_colors, dodge=False,
        alpha=0.5, size=3, jitter=True, ax=ax,
    )
    h, l = ax.get_legend_handles_labels()
    ax.legend(h, l, title='Cell type',
              bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=8, frameon=True)

    ylim  = ax.get_ylim()
    y_top = ylim[1]

    for i, grp in enumerate(full_order):
        n = int((plot_sub[group_col] == grp).sum())
        ax.text(i, y_top * 1.01, f'n={n}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    y_max_bracket = y_top
    if 'Stable' in present:
        ref_i    = full_order.index('Stable')
        y_levels = [y_top * 1.09, y_top * 1.17]
        y_max_bracket = y_levels[0]
        k = 0
        for grp in present:
            if grp == 'Stable':
                continue
            pval  = ttest_vs_stable(plot_sub, 'self_module_score', group_col, grp)
            stars = get_sig(pval)
            y_lvl = y_levels[k % 2]
            y_max_bracket = max(y_max_bracket, y_lvl)
            draw_bracket(ax, full_order, full_order.index(grp), ref_i, y_lvl, stars)
            k += 1

    # ── NegCtrl separator + annotation ───────────────────────────────────
    if neg_ctrl_label:
        sep_x = len(present) - 0.5
        ax.axvline(sep_x, color='#666666', lw=1.2, linestyle='--', alpha=0.7)

        pval_nc = ttest_vs_ref(plot_sub, 'self_module_score', group_col,
                               neg_ctrl_label, ref_group='Stable')
        stars_nc = get_sig(pval_nc)
        ax.text(full_order.index(neg_ctrl_label),
                y_max_bracket * 1.05,
                f'vs Stable:\n{stars_nc}',
                ha='center', va='bottom', fontsize=7,
                color='#555555', style='italic')

    ax.set_ylim(ylim[0], max(y_max_bracket, y_top) * 1.25)

    ax.set_xlabel('')
    ax.set_ylabel('Self-module score (own DCM cell-type module)', fontsize=10)
    ax.set_title(
        f'All cell types combined — Stable vs Reprogrammed to RGCs\n'
        f'Grey = Stable INs cross-scored on IGs module + Stable IGs cross-scored on INs module\n'
        f'Dox = {dox_label}',
        fontsize=11, fontweight='bold'
    )
    ax.tick_params(axis='x', rotation=20, labelsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return compute_stats(plot_sub, 'self_module_score', group_col,
                         dox_label, 'ALL_combined', context)


# ─────────────────────────────────────────────
# RGCs COMPARISON PLOT WITH NEGATIVE CONTROLS
# ─────────────────────────────────────────────
def make_rgc_plot(df, dox_label, save_path, paired_all_raw=None):
    dest_ct = 'RGCs'
    mod_col = col_from_ct.get(dest_ct)

    fig, ax = plt.subplots(figsize=(7, 6))
    all_stats = []

    active_dox = list(df['Dox'].unique()) if dox_label != 'ALL' else None

    sub = df[df['stemta_dest'] == dest_ct][
        ['stemta_group', 'stemta_score', 'module_max_label', 'Dox']
    ].copy()

    stbl = 'Stable_RGCs'
    rev  = 'Reversed_to_RGCs'
    main_order = [g for g in [stbl, rev] if g in sub['stemta_group'].values]

    if len(main_order) < 2:
        print(f"  Skipping RGC plot Dox={dox_label} — insufficient groups")
        plt.close()
        return pd.DataFrame()

    nc_combined = pd.DataFrame()
    if paired_all_raw is not None:
        nc_df = build_neg_ctrl_stemta(paired_all_raw, dest_ct,
                                      dox_subset=active_dox)
        if not nc_df.empty:
            nc_combined = nc_df[['stemta_group', 'stemta_score',
                                 'module_max_label']].copy()

    nc_labels = ['NegCtrl'] if not nc_combined.empty else []
    full_order = main_order + nc_labels

    pal = {stbl: GROUP_COLORS[stbl], rev: GROUP_COLORS[rev]}
    if nc_labels:
        pal['NegCtrl'] = '#AAAAAA'

    plot_df = sub[['stemta_group', 'stemta_score', 'module_max_label']].copy()
    if not nc_combined.empty:
        plot_df = pd.concat([plot_df, nc_combined], ignore_index=True)

    sns.boxplot(
        data=plot_df, x='stemta_group', y='stemta_score',
        hue='stemta_group', order=full_order,
        palette=pal, legend=False, ax=ax,
        flierprops=dict(marker='o', markersize=2, alpha=0.4),
    )
    sns.stripplot(
        data=plot_df, x='stemta_group', y='stemta_score',
        hue='module_max_label', order=full_order,
        palette=cell_type_colors, dodge=False,
        alpha=0.5, size=3, jitter=True, ax=ax,
    )
    h, l = ax.get_legend_handles_labels()
    ax.legend(h, l, title='DCM cell type',
              bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=7, frameon=True)

    ylim  = ax.get_ylim()
    y_top = ylim[1]

    for i, grp in enumerate(full_order):
        n = int((plot_df['stemta_group'] == grp).sum())
        ax.text(i, y_top * 1.01, f'n={n}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')

    s_vals = plot_df[plot_df['stemta_group'] == stbl]['stemta_score'].dropna()
    r_vals = plot_df[plot_df['stemta_group'] == rev ]['stemta_score'].dropna()
    y_brac = y_top
    pval = np.nan
    if len(s_vals) > 1 and len(r_vals) > 1:
        _, pval = stats.ttest_ind(r_vals, s_vals)
        stars   = get_sig(pval)
        y_ann   = y_top * 1.09
        ax.annotate('', xy=(full_order.index(rev), y_ann),
                    xytext=(full_order.index(stbl), y_ann),
                    arrowprops=dict(arrowstyle='-', color='#333333', lw=1.0))
        ax.text((full_order.index(stbl) + full_order.index(rev)) / 2,
                y_ann, stars,
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                color='black' if stars not in ('ns', '') else '#888888')
        y_brac = y_ann

    if nc_labels:
        sep_x = len(main_order) - 0.5
        ax.axvline(sep_x, color='#666666', lw=1.2,
                   linestyle='--', alpha=0.7)
        ax.text(sep_x + 0.02, ylim[0] + (ylim[1]-ylim[0])*0.02,
                'Neg. controls', fontsize=7, color='#555555',
                rotation=90, va='bottom', ha='left', style='italic')

        nc_i = full_order.index('NegCtrl')
        nc_v = plot_df[plot_df['stemta_group'] == 'NegCtrl']['stemta_score'].dropna()

        pval_nc_vs_stable = np.nan
        if len(s_vals) > 1 and len(nc_v) > 1:
            _, pval_nc_vs_stable = stats.ttest_ind(nc_v, s_vals)
        stars_nc_vs_stable = get_sig(pval_nc_vs_stable)
        ax.text(nc_i, y_brac * 1.05,
                f'vs {stbl}:\n{stars_nc_vs_stable}',
                ha='center', va='bottom', fontsize=7,
                color='#444444', style='italic')

        pval_nc = np.nan
        rev_i = full_order.index(rev)
        if len(r_vals) > 1 and len(nc_v) > 1:
            _, pval_nc = stats.ttest_ind(r_vals, nc_v)
        stars_rev_nc = get_sig(pval_nc)
        y_rev_nc = max(y_brac, y_top) * 1.17
        ax.annotate('', xy=(nc_i, y_rev_nc), xytext=(rev_i, y_rev_nc),
                    arrowprops=dict(arrowstyle='-', color='#333333', lw=0.9))
        ax.text((rev_i + nc_i) / 2, y_rev_nc, stars_rev_nc,
                ha='center', va='bottom', fontsize=10, fontweight='bold',
                color='black' if stars_rev_nc not in ('ns', '') else '#888888')
        ax.set_ylim(ylim[0], y_rev_nc * 1.15)

    ax.set_ylim(ylim[0], max(y_brac, y_top) * 1.25)

    ax.set_title(f'RGCs module Z-score: Stable vs Reversed from Mature | Dox = {dox_label}\n'
                 f'Grey = Stable INs + IGs pooled (Neg Ctrl)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('RGCs module score', fontsize=9)
    ax.tick_params(axis='x', rotation=20, labelsize=8)

    # Stats
    for grp in main_order:
        vals = plot_df[plot_df['stemta_group'] == grp]['stemta_score'].dropna()
        pv   = pval if grp == rev else np.nan
        all_stats.append({
            'context':              'RGC_StableVsReversed',
            'cell_type':            dest_ct,
            'Dox':                  dox_label,
            'group':                grp,
            'score_col':            mod_col,
            'N':                    len(vals),
            'mean':                 vals.mean()   if len(vals) else np.nan,
            'median':               vals.median() if len(vals) else np.nan,
            'std':                  vals.std()    if len(vals) else np.nan,
            'ttest_vs_Stable_pval': pv,
            'significance':         get_sig(pv) if grp == rev else '',
        })
    if nc_labels:
        nc_v = plot_df[plot_df['stemta_group'] == 'NegCtrl']['stemta_score'].dropna()
        pval_nc_vs_stable = np.nan
        if len(s_vals) > 1 and len(nc_v) > 1:
            _, pval_nc_vs_stable = stats.ttest_ind(nc_v, s_vals)
        pval_rev_vs_nc = np.nan
        if len(r_vals) > 1 and len(nc_v) > 1:
            _, pval_rev_vs_nc = stats.ttest_ind(r_vals, nc_v)
        all_stats.append({
            'context':                    'RGC_NegCtrl',
            'cell_type':                  dest_ct,
            'Dox':                        dox_label,
            'group':                      'NegCtrl (INs+IGs)',
            'score_col':                  mod_col,
            'N':                          len(nc_v),
            'mean':                       nc_v.mean()   if len(nc_v) else np.nan,
            'median':                     nc_v.median() if len(nc_v) else np.nan,
            'std':                        nc_v.std()    if len(nc_v) else np.nan,
            'ttest_vs_Stable_pval':       pval_nc_vs_stable,
            'significance_vs_Stable':     get_sig(pval_nc_vs_stable),
            'ttest_Reversed_vs_NegCtrl':  pval_rev_vs_nc,
            'significance_Rev_vs_NC':     get_sig(pval_rev_vs_nc),
        })

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return pd.DataFrame(all_stats) if all_stats else pd.DataFrame()


# ─────────────────────────────────────────────
# BARPLOT — Pulse vs Chase comparison
# ─────────────────────────────────────────────
def make_combined_dox_barplot(df, group_col, groups, colors,
                               dox_present, title_str, save_path,
                               per_celltype=False):
    if len(dox_present) < 2:
        print(f"  Skipping — fewer than 2 Dox conditions")
        return pd.DataFrame()

    d1, d2 = dox_present[0], dox_present[1]

    if not per_celltype:
        fig, ax = plt.subplots(figsize=(max(6, 2.2 * len(groups)), 6))

        group_width = 0.35
        gap         = 0.15
        all_rows    = []

        for g_idx, (grp, color) in enumerate(zip(groups, colors)):
            sub   = df[df[group_col] == grp]
            n1    = int((sub['Dox'] == d1).sum())
            n2    = int((sub['Dox'] == d2).sum())
            tot1  = int((df['Dox'] == d1).sum())
            tot2  = int((df['Dox'] == d2).sum())
            pval  = binomtest(n1, n1 + n2, p=0.5, alternative='two-sided').pvalue if (n1 + n2) > 0 else np.nan
            stars = get_sig(pval)

            x_centre = g_idx * (2 * group_width + gap + 0.1)
            x1 = x_centre - group_width / 2
            x2 = x_centre + group_width / 2

            b1 = ax.bar(x1, n1, group_width,
                        color=DOX_COLORS.get(d1, color),
                        edgecolor='black', linewidth=0.8,
                        label=d1 if g_idx == 0 else '')
            b2 = ax.bar(x2, n2, group_width,
                        color=DOX_COLORS.get(d2, color),
                        edgecolor='black', linewidth=0.8,
                        hatch='//', label=d2 if g_idx == 0 else '')

            for bar, val in [(b1, n1), (b2, n2)]:
                ax.text(bar[0].get_x() + bar[0].get_width() / 2,
                        bar[0].get_height() + max(n1, n2, 1) * 0.04,
                        str(val), ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

            y_max  = max(n1, n2, 1)
            y_brac = y_max * 1.18
            ax.plot([x1, x1, x2, x2],
                    [y_brac - y_max * 0.04, y_brac,
                     y_brac, y_brac - y_max * 0.04],
                    color='black', lw=1.2)
            pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
            ax.text(x_centre, y_brac + y_max * 0.04,
                    f'{stars}\n{pval_txt}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold',
                    color='black' if stars != 'ns' else '#888888')

            ax.text(x_centre, -max(n1, n2, 1) * 0.08, grp,
                    ha='center', va='top', fontsize=9, fontweight='bold',
                    color=color)

            for dox_val, cnt, total in [(d1, n1, tot1), (d2, n2, tot2)]:
                all_rows.append({
                    'group':          grp,
                    'cell_type':      'ALL',
                    'Dox':            dox_val,
                    'count':          cnt,
                    'total_in_dox':   total,
                    'binom_p_pulse_vs_chase': pval,
                    'significance':   stars,
                })

        ax.set_xticks([])
        ax.set_ylabel('Number of cells', fontsize=11, fontweight='bold')
        ax.set_title(title_str, fontsize=12, fontweight='bold')
        handles = [
            plt.Rectangle((0,0), 1, 1,
                           facecolor=DOX_COLORS.get(d1, '#4A90D9'),
                           edgecolor='black', label=DOX_LABELS.get(d1, d1)),
            plt.Rectangle((0,0), 1, 1,
                           facecolor=DOX_COLORS.get(d2, '#E87722'),
                           edgecolor='black', hatch='//',
                           label=DOX_LABELS.get(d2, d2)),
        ]
        ax.legend(handles=handles, loc='upper right', fontsize=9)
        ax.set_ylim(bottom=0)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")
        return pd.DataFrame(all_rows)

    else:
        ct_present = [ct for ct in label_order
                      if ct in df['module_max_label'].unique()]
        if not ct_present:
            return pd.DataFrame()

        n_ct      = len(ct_present)
        fig, axes = plt.subplots(1, n_ct, figsize=(3.5 * n_ct, 6), sharey=False)
        if n_ct == 1:
            axes = [axes]

        all_rows  = []
        group_width = 0.3
        gap         = 0.1

        for ax, ct in zip(axes, ct_present):
            df_ct = df[df['module_max_label'] == ct]

            for g_idx, (grp, color) in enumerate(zip(groups, colors)):
                sub  = df_ct[df_ct[group_col] == grp]
                tot1 = int((df_ct['Dox'] == d1).sum())
                tot2 = int((df_ct['Dox'] == d2).sum())
                n1   = int((sub['Dox'] == d1).sum())
                n2   = int((sub['Dox'] == d2).sum())
                pval = binomtest(n1, n1 + n2, p=0.5, alternative='two-sided').pvalue if (n1 + n2) > 0 else np.nan
                stars= get_sig(pval)

                x_centre = g_idx * (2 * group_width + gap + 0.08)
                x1 = x_centre - group_width / 2
                x2 = x_centre + group_width / 2

                b1 = ax.bar(x1, n1, group_width,
                            color=DOX_COLORS.get(d1, color),
                            edgecolor='black', linewidth=0.7)
                b2 = ax.bar(x2, n2, group_width,
                            color=DOX_COLORS.get(d2, color),
                            edgecolor='black', linewidth=0.7, hatch='//')

                y_max  = max(n1, n2, 1)
                for bar, val in [(b1, n1), (b2, n2)]:
                    ax.text(bar[0].get_x() + bar[0].get_width() / 2,
                            bar[0].get_height() + y_max * 0.05,
                            str(val), ha='center', va='bottom', fontsize=7)

                y_brac = y_max * 1.20
                ax.plot([x1, x1, x2, x2],
                        [y_brac - y_max*0.05, y_brac,
                         y_brac, y_brac - y_max*0.05],
                        color='black', lw=1.0)
                pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
                ax.text(x_centre, y_brac + y_max * 0.05,
                        f'{stars}\n{pval_txt}',
                        ha='center', va='bottom', fontsize=7,
                        color='black' if stars != 'ns' else '#888888')

                ax.text(x_centre, -y_max * 0.10, grp,
                        ha='center', va='top', fontsize=7,
                        color=color, fontweight='bold')

                for dox_val, cnt, total in [(d1, n1, tot1), (d2, n2, tot2)]:
                    all_rows.append({
                        'group':        grp,
                        'cell_type':    ct,
                        'Dox':          dox_val,
                        'count':        cnt,
                        'total_in_dox': total,
                        'binom_p_pulse_vs_chase': pval,
                        'significance': stars,
                    })

            ax.set_xticks([])
            ax.set_title(ct, fontsize=10, fontweight='bold',
                         color=cell_type_colors.get(ct, 'black'))
            ax.set_ylabel('Cell count', fontsize=8)
            ax.set_ylim(bottom=0)

        plt.suptitle(title_str, fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")

        return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────
# STEP 4: GENERATE PLOTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: GENERATING PLOTS")
print("=" * 60)

ORDER_MAIN    = ['Stable', 'To_RGC']
PALETTE_MAIN  = {k: GROUP_COLORS[k] for k in ORDER_MAIN}

collected_stats = []

def dox_tag(d):
    return str(d).replace(' ', '_').replace('/', '-')

for dox_label, subset in (
    [('ALL', paired)] +
    [(str(d), paired[paired['Dox'] == str(d)]) for d in DOX_VALUES]
):
    tag = dox_tag(dox_label)
    print(f"\n  Dox = {dox_label}  (n={len(subset)})")

    # ── A: Per cell type (Stable vs To_RGC) ──────────
    _s = make_per_celltype_plot(
        subset[subset['transition'].isin(ORDER_MAIN)],
        group_col        = 'transition',
        group_order_full = ORDER_MAIN,
        palette          = PALETTE_MAIN,
        dox_label        = dox_label,
        save_path        = os.path.join(score_dir,
                           f'per_celltype_Dox_{tag}.pdf'),
        context          = 'PerCelltype',
    )
    collected_stats.append(_s)

    # ── B: Per cell type WITH NEG CTRL ────────
    _s = make_per_celltype_merged_plot_with_negctrl(
        df           = subset[subset['transition'].isin(ORDER_MAIN)],
        paired_full  = paired,
        group_col    = 'transition',
        group_order_full = ORDER_MAIN,
        palette      = PALETTE_MAIN,
        dox_label    = dox_label,
        save_path    = os.path.join(score_dir,
                       f'per_celltype_negctrl_Dox_{tag}.pdf'),
        context      = 'PerCelltype_NegCtrl',
    )
    collected_stats.append(_s)

    # ── C: Combined (all cell types pooled) ──────────
    _s = make_combined_plot(
        subset[subset['transition'].isin(ORDER_MAIN)],
        group_col        = 'transition',
        group_order_full = ORDER_MAIN,
        palette          = PALETTE_MAIN,
        dox_label        = dox_label,
        save_path        = os.path.join(score_dir,
                           f'combined_Dox_{tag}.pdf'),
        context          = 'Combined',
        paired_full      = paired,          # <-- added
    )

    # ── D: RGCs stable vs reversed WITH NEG CTRL ────────
    stemta_subset = paired_stemta if dox_label == 'ALL' else (
        paired_stemta[paired_stemta['Dox'] == str(dox_label)]
    )
    _s = make_rgc_plot(
        df              = stemta_subset,
        dox_label       = dox_label,
        save_path       = os.path.join(score_dir,
                          f'rgc_stable_vs_reversed_negctrl_Dox_{tag}.pdf'),
        paired_all_raw  = _all_paired_raw,
    )
    collected_stats.append(_s)


# ─────────────────────────────────────────────
# STEP 5: BARPLOTS — Pulse vs Chase cell counts
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: BARPLOTS — PULSE vs CHASE CELL COUNTS")
print("=" * 60)

barplot_stats = []

paired_dox        = paired[paired['Dox'].isin(DOX_ORDER)].copy()
paired_stemta_dox = paired_stemta[paired_stemta['Dox'].isin(DOX_ORDER)].copy()
dox_present       = [d for d in DOX_ORDER if d in paired_dox['Dox'].unique()]

# ── A: Stable + To_RGC  ─────────────────────
barplot_stats.append(make_combined_dox_barplot(
    df=paired_dox, group_col='transition',
    groups=['Stable', 'To_RGC'],
    colors=['#2ecc71', '#003366'],
    dox_present=dox_present,
    title_str='Stable / To_RGC — Pulse vs Chase (binomial test)',
    save_path=os.path.join(score_dir, 'barplot_stable_toRGC_overall.pdf'),
    per_celltype=False,
))
barplot_stats.append(make_combined_dox_barplot(
    df=paired_dox, group_col='transition',
    groups=['Stable', 'To_RGC'],
    colors=['#2ecc71', '#003366'],
    dox_present=dox_present,
    title_str='Stable / To_RGC per cell type — Pulse vs Chase',
    save_path=os.path.join(score_dir, 'barplot_stable_toRGC_per_celltype.pdf'),
    per_celltype=True,
))

# ── B: Stable_RGCs vs Reversed_to_RGCs ───────
sub_rgc = paired_stemta_dox[paired_stemta_dox['stemta_dest'] == 'RGCs'].copy()
sub_rgc['transition']      = sub_rgc['stemta_group']
barplot_stats.append(make_combined_dox_barplot(
    df=sub_rgc, group_col='transition',
    groups=['Stable_RGCs', 'Reversed_to_RGCs'],
    colors=['#2ecc71', '#003366'],
    dox_present=dox_present,
    title_str='Stable_RGCs vs Reversed_to_RGCs — Pulse vs Chase',
    save_path=os.path.join(score_dir, 'barplot_RGCs_stable_vs_reversed_overall.pdf'),
    per_celltype=False,
))
barplot_stats.append(make_combined_dox_barplot(
    df=sub_rgc, group_col='transition',
    groups=['Stable_RGCs', 'Reversed_to_RGCs'],
    colors=['#2ecc71', '#003366'],
    dox_present=dox_present,
    title_str='Stable_RGCs vs Reversed_to_RGCs per cell type — Pulse vs Chase',
    save_path=os.path.join(score_dir, 'barplot_RGCs_stable_vs_reversed_per_celltype.pdf'),
    per_celltype=True,
))

bp_stats_df  = pd.concat(
    [s for s in barplot_stats if s is not None and not s.empty],
    ignore_index=True
)
bp_stats_csv = os.path.join(score_dir, 'pulse_vs_chase_count_stats.csv')
bp_stats_df.to_csv(bp_stats_csv, index=False)
print(f"\nBarplot stats CSV: {bp_stats_csv}")
print(bp_stats_df.to_string(index=False))


# ─────────────────────────────────────────────
# STEP 6: CSV OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: SAVING CSV OUTPUTS")
print("=" * 60)

stats_df  = pd.concat([s for s in collected_stats if s is not None and not s.empty],
                      ignore_index=True)
stats_csv = os.path.join(score_dir, 'stable_vs_reprogrammed_stats.csv')
stats_df.to_csv(stats_csv, index=False)
print(f"Stats CSV : {stats_csv}")
print(stats_df.to_string(index=False))

out_cols = (['plate_well', 'Dox', 'module_max_label',
             'rna_celltype', 'transition', 'transition_combined',
             'self_module_score'] + ttest_cols)
data_csv = os.path.join(score_dir, 'stable_vs_reprogrammed_data.csv')
paired[out_cols].to_csv(data_csv, index=False)
print(f"Data CSV  : {data_csv}")

print("\nDone!")
print(f"Outputs in: {score_dir}")

STEP 1: BUILDING DATAFRAME
Total cells : 15920
Batch values: ['multi_RNA', 'ref_D2', 'ref_D4', 'scNMT_DCM', 'scNMT_RNA']
Dox values  : ['16h', '16h_24h_dtag', 'Unknown']

STEP 2: PAIRING DCM ↔ RNA
1:1 matched wells: 399
Paired cells: 399

STEP 3: CLASSIFYING
Cells with DCM in ANALYSIS_TYPES: 150

Transition counts (ALL Dox):
transition
To_RGC    104
Stable     35
Other      11

Per DCM cell type:
transition        Other  Stable  To_RGC
module_max_label                       
IGs                   2      17      38
INs                   9      18      66

RGCs comparison dataframe: 122 rows
stemta_dest  stemta_group    
RGCs         Reversed_to_RGCs    104
             Stable_RGCs          18

STEP 4: GENERATING PLOTS

  Dox = ALL  (n=150)
  Saved: /media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/MS_stable_transition/per_celltype_Dox_ALL.pdf
  Saved: /media/erlantz

In [ ]:
#TO IGs

In [4]:
##############################################
# STABLE vs REPROGRAMMED ANALYSIS — NPC data
#
# Destinations analysed:
#   To_RGC  — mature cell (INs/IGs) whose RNA label == RGCs
#   To_IG   — INs cell whose RNA label == IGs
#
# Negative controls:
#   per_celltype / combined plots:
#     INs subplot → NegCtrl = Stable IGs scored on INs module
#     IGs subplot → NegCtrl = Stable INs scored on IGs module
#   dest plots (make_dest_plot):
#     RGCs dest → NegCtrl = Stable INs + Stable IGs pooled, scored on RGCs module
#     IGs  dest → NegCtrl = Stable INs scored on IGs module
##############################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import binomtest

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
score_dir = '/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best/MS_stable_transition/To_IG'
os.makedirs(score_dir, exist_ok=True)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
cell_type_colors = {
    'RGCs': '#003366',
    'INs':  '#990000',
    'IGs':  '#00CCCC',
}

GROUP_COLORS = {
    'Stable':             '#2ecc71',
    'To_RGC':             '#003366',
    'To_IG':              '#00CCCC',
    'NegCtrl':            '#AAAAAA',
    'Stable_RGCs':        '#2ecc71',
    'Reversed_to_RGCs':   '#003366',
    'Stable_IGs':         '#2ecc71',
    'Reversed_to_IGs':    '#00CCCC',
}

label_order    = ['RGCs', 'INs', 'IGs']
PROGENITOR_TYPES = ['RGCs']
MATURE_TYPES     = ['INs', 'IGs']
ANALYSIS_TYPES   = ['INs', 'IGs']   # DCM cell types whose stability we assess
DEST_TYPES       = ['RGCs', 'IGs']  # destinations to analyse

# Module score columns
ttest_cols = [
    'RGCs_module_score_Z_all',
    'INs_module_score_Z_all',
    'IGs_module_score_Z_all',
]
ct_from_col = {
    'RGCs_module_score_Z_all': 'RGCs',
    'INs_module_score_Z_all':  'INs',
    'IGs_module_score_Z_all':  'IGs',
}
col_from_ct = {v: k for k, v in ct_from_col.items()}

DOX_COLORS  = {'16h': '#4A90D9', '16h_24h_dtag': '#E87722'}
DOX_ORDER   = ['16h', '16h_24h_dtag']
DOX_LABELS  = {'16h': 'Pulse (16h)', '16h_24h_dtag': 'Chase (16h_24h_dtag)'}
NEGCTRL_COLOR = '#BBBBBB'

# ─────────────────────────────────────────────
# STEP 1: BUILD DATAFRAME
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: BUILDING DATAFRAME")
print("=" * 60)

base = pd.DataFrame({
    'plate_well':       adata.obs['plate_well'].astype(str).values,
    'batch':            adata.obs['batch'].astype(str).values,
    'Dox':              adata.obs['Dox'].astype(str).values,
    'cell_type_broad':  adata.obs['cell_type_broad'].astype(str).values,
    'module_max_label': adata.obs['module_max_label'].astype(str).values,
}, index=adata.obs_names)

for col in ttest_cols:
    base[col] = adata.obs[col].values

print(f"Total cells : {len(base)}")
print(f"Batch values: {sorted(base['batch'].unique())}")
print(f"Dox values  : {sorted(base['Dox'].unique())}")

DOX_VALUES = sorted(base['Dox'].dropna().unique())

# ─────────────────────────────────────────────
# STEP 2: PAIR DCM ↔ RNA  (1:1 via plate_well)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: PAIRING DCM ↔ RNA")
print("=" * 60)

dcm_df = base[base['batch'] == 'scNMT_DCM'].copy()
rna_df = (base[base['batch'] == 'scNMT_RNA']
          [['plate_well', 'cell_type_broad']]
          .copy()
          .rename(columns={'cell_type_broad': 'rna_celltype'}))

def match_one_to_one(df_a, df_b):
    common = set(df_a['plate_well']) & set(df_b['plate_well'])
    a_s = set(df_a[df_a['plate_well'].isin(common)]
              ['plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    b_s = set(df_b[df_b['plate_well'].isin(common)]
              ['plate_well'].value_counts()
              .pipe(lambda s: s[s == 1].index))
    return a_s & b_s

matched_wells = match_one_to_one(dcm_df, rna_df)
print(f"1:1 matched wells: {len(matched_wells)}")

paired = (dcm_df[dcm_df['plate_well'].isin(matched_wells)]
          .merge(rna_df[rna_df['plate_well'].isin(matched_wells)],
                 on='plate_well', how='inner'))
print(f"Paired cells: {len(paired)}")

# ─────────────────────────────────────────────
# STEP 3: FILTER AND CLASSIFY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: CLASSIFYING")
print("=" * 60)

paired = paired[paired['module_max_label'].isin(label_order)].copy()
paired = paired[paired['module_max_label'].isin(ANALYSIS_TYPES)].copy()
print(f"Cells with DCM in ANALYSIS_TYPES: {len(paired)}")

def classify(row):
    dcm = row['module_max_label']
    rna = row['rna_celltype']
    if dcm == rna:
        return 'Stable'
    if rna == 'RGCs':
        return 'To_RGC'
    if rna == 'IGs':
        return 'To_IG'
    return 'Other'

paired['transition'] = paired.apply(classify, axis=1)

print("\nTransition counts (ALL Dox):")
print(paired['transition'].value_counts().to_string())
print("\nPer DCM cell type:")
print(paired.groupby(['module_max_label', 'transition']).size()
           .unstack(fill_value=0).to_string())

def get_self_module(row):
    col = col_from_ct.get(row['module_max_label'])
    return row[col] if (col is not None and col in row.index) else np.nan

paired['self_module_score'] = paired.apply(get_self_module, axis=1)

# ─────────────────────────────────────────────
# STEP 4: BUILD DESTINATION REFERENCE DATAFRAMES
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: BUILDING DESTINATION DATAFRAMES")
print("=" * 60)

_all_paired_raw = (dcm_df[dcm_df['plate_well'].isin(matched_wells)]
                   .merge(rna_df[rna_df['plate_well'].isin(matched_wells)],
                          on='plate_well', how='inner'))
_all_paired_raw['transition'] = _all_paired_raw.apply(classify, axis=1)

paired_stemta_by_dest = {}

for dest_ct in DEST_TYPES:
    mod_col        = col_from_ct.get(dest_ct)
    stable_label   = f'Stable_{dest_ct}'
    reversed_label = f'Reversed_to_{dest_ct}'
    trans_label    = 'To_RGC' if dest_ct == 'RGCs' else f'To_{dest_ct}'

    stable_sub = _all_paired_raw[
        (_all_paired_raw['module_max_label'] == dest_ct) &
        (_all_paired_raw['rna_celltype']     == dest_ct)
    ].copy()
    stable_sub['stemta_group'] = stable_label
    stable_sub['stemta_dest']  = dest_ct
    stable_sub['stemta_score'] = stable_sub[mod_col]

    reversed_sub = paired[paired['transition'] == trans_label].copy()
    reversed_sub['stemta_group'] = reversed_label
    reversed_sub['stemta_dest']  = dest_ct
    reversed_sub['stemta_score'] = reversed_sub[mod_col]

    df_dest = pd.concat([stable_sub, reversed_sub], ignore_index=True)
    paired_stemta_by_dest[dest_ct] = df_dest

    print(f"\n{dest_ct} dataframe: {len(df_dest)} rows")
    print(df_dest.groupby(['stemta_dest', 'stemta_group']).size().to_string())

# ─────────────────────────────────────────────
# STATS HELPERS
# ─────────────────────────────────────────────
def get_sig(p):
    if pd.isna(p): return ''
    if p < 0.001:  return '***'
    if p < 0.01:   return '**'
    if p < 0.05:   return '*'
    return 'ns'

def ttest_vs_ref(df, score_col, group_col, test_group, ref_group='Stable'):
    s = df[df[group_col] == ref_group ][score_col].dropna()
    u = df[df[group_col] == test_group][score_col].dropna()
    if len(s) > 1 and len(u) > 1:
        return stats.ttest_ind(u, s)[1]
    return np.nan

def ttest_vs_stable(df, score_col, group_col, test_group):
    return ttest_vs_ref(df, score_col, group_col, test_group, ref_group='Stable')

def compute_stats(df, score_col, group_col, dox_label, ct, context):
    rows = []
    for grp in df[group_col].unique():
        vals = df[df[group_col] == grp][score_col].dropna()
        pval = ttest_vs_stable(df, score_col, group_col, grp) if grp != 'Stable' else np.nan
        rows.append({
            'context':              context,
            'cell_type':            ct,
            'Dox':                  dox_label,
            'group':                grp,
            'score_col':            score_col,
            'N':                    len(vals),
            'mean':                 vals.mean()   if len(vals) else np.nan,
            'median':               vals.median() if len(vals) else np.nan,
            'std':                  vals.std()    if len(vals) else np.nan,
            'ttest_vs_Stable_pval': pval,
            'significance':         get_sig(pval),
        })
    return pd.DataFrame(rows)

# ─────────────────────────────────────────────
# NEGATIVE CONTROL HELPERS
# ─────────────────────────────────────────────
def build_neg_ctrl_for_celltype(paired_all, ct, dox_subset=None):
    """
    Cross-score NegCtrl for per-celltype / combined plots.
    INs subplot → Stable IGs scored on INs module
    IGs subplot → Stable INs scored on IGs module
    """
    mod_col     = col_from_ct.get(ct)
    other_types = [t for t in MATURE_TYPES if t != ct]
    if mod_col is None or not other_types:
        return pd.DataFrame()

    src = paired_all if dox_subset is None else paired_all[
        paired_all['Dox'].isin(dox_subset)]

    ctrl = src[
        (src['module_max_label'].isin(other_types)) &
        (src['transition'] == 'Stable')
    ].copy()

    if ctrl.empty:
        return pd.DataFrame()

    ctrl_label = f'NegCtrl ({"+".join(other_types)})'
    ctrl['group'] = ctrl_label
    ctrl['score'] = ctrl[mod_col]
    return ctrl[['group', 'score', 'module_max_label']]


def build_neg_ctrl_stemta(paired_all_raw, dest_ct, dox_subset=None):
    """
    NegCtrl for dest_plot:
    - RGCs dest → Stable INs + Stable IGs pooled, scored on RGCs module
    - IGs  dest → Stable INs scored on IGs module
    """
    mod_col = col_from_ct.get(dest_ct)
    if mod_col is None:
        return pd.DataFrame()

    ctrl_types = MATURE_TYPES if dest_ct in PROGENITOR_TYPES else [
        t for t in MATURE_TYPES if t != dest_ct]

    src = paired_all_raw if dox_subset is None else paired_all_raw[
        paired_all_raw['Dox'].isin(dox_subset)]

    ctrl = src[
        (src['module_max_label'].isin(ctrl_types)) &
        (src['rna_celltype'] == src['module_max_label'])
    ].copy()

    if ctrl.empty:
        return pd.DataFrame()

    ctrl['stemta_group'] = 'NegCtrl'
    ctrl['stemta_dest']  = dest_ct
    ctrl['stemta_score'] = ctrl[mod_col]
    return ctrl[['stemta_group', 'stemta_dest', 'stemta_score',
                 'module_max_label', 'Dox']]

# ─────────────────────────────────────────────
# PLOT HELPERS
# ─────────────────────────────────────────────
def draw_bracket(ax, full_order, i_test, i_ref, y_lvl, stars):
    ax.annotate('', xy=(i_test, y_lvl), xytext=(i_ref, y_lvl),
                arrowprops=dict(arrowstyle='-', color='#333333', lw=0.9))
    x_mid = (i_test + i_ref) / 2
    ax.text(x_mid, y_lvl, stars,
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            color='black' if stars not in ('ns', '') else '#888888')


def _add_negctrl_annotation(ax, full_order, main_order,
                             plot_df, score_col, group_col,
                             ref_label, y_max_bracket, ylim):
    """Shared logic: dashed separator + NegCtrl vs Stable annotation."""
    sep_x = len(main_order) - 0.5
    ax.axvline(sep_x, color='#666666', lw=1.2, linestyle='--', alpha=0.7)

    pval_nc = ttest_vs_ref(plot_df, score_col, group_col,
                           'NegCtrl', ref_group=ref_label)
    stars_nc = get_sig(pval_nc)
    nc_i = full_order.index('NegCtrl')
    ax.text(nc_i, y_max_bracket * 1.05,
            f'vs {ref_label}:\n{stars_nc}',
            ha='center', va='bottom', fontsize=7,
            color='#555555', style='italic')
    return pval_nc


# ── A: Per cell type (Stable vs To_<dest>) ──────────────────────────────
def make_per_celltype_plot(df, group_col, group_order_full,
                           palette, dox_label, save_path, context):
    ct_present = [ct for ct in ANALYSIS_TYPES
                  if set(['Stable']) < set(df[df['module_max_label'] == ct]
                                           [group_col].unique())]
    if not ct_present:
        print(f"  Skipping per-celltype Dox={dox_label} — no data")
        return pd.DataFrame()

    fig, axes = plt.subplots(1, len(ct_present),
                             figsize=(4.5 * len(ct_present), 6), sharey=False)
    if len(ct_present) == 1:
        axes = [axes]

    all_stats = []
    for ax, ct in zip(axes, ct_present):
        col = col_from_ct.get(ct)
        if col is None:
            ax.set_visible(False)
            continue

        sub     = df[df['module_max_label'] == ct].copy()
        present = [g for g in group_order_full if g in sub[group_col].values]
        plot_sub = sub[[col, group_col]].rename(columns={col: 'score'})

        sns.boxplot(data=plot_sub, x=group_col, y='score',
                    hue=group_col, order=present,
                    palette=palette, legend=False, ax=ax,
                    flierprops=dict(marker='o', markersize=2, alpha=0.4))

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        for i, grp in enumerate(present):
            n = int((plot_sub[group_col] == grp).sum())
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        y_max = y_top
        if 'Stable' in present:
            ref_i    = present.index('Stable')
            y_levels = [y_top * 1.09, y_top * 1.17]
            for k, grp in enumerate([g for g in present if g != 'Stable']):
                pval  = ttest_vs_stable(plot_sub, 'score', group_col, grp)
                y_lvl = y_levels[k % 2]
                y_max = max(y_max, y_lvl)
                draw_bracket(ax, present, present.index(grp),
                             ref_i, y_lvl, get_sig(pval))
            ax.set_ylim(ylim[0], y_max * 1.12)

        ax.set_title(ct, fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{ct} module score', fontsize=9)
        ax.tick_params(axis='x', rotation=20, labelsize=9)
        all_stats.append(compute_stats(sub, col, group_col,
                                       dox_label, ct, context))

    plt.suptitle(
        f'Module Scores — Stable vs Reprogrammed | Dox = {dox_label}\n'
        f't-tests vs Stable within each cell type',
        fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()


# ── B: Per cell type WITH NegCtrl ───────────────────────────────────────
def make_per_celltype_negctrl_plot(df, paired_full, group_col,
                                    group_order_full, palette,
                                    dox_label, save_path, context):
    ct_present = [ct for ct in ANALYSIS_TYPES
                  if set(['Stable']) < set(df[df['module_max_label'] == ct]
                                           [group_col].unique())]
    if not ct_present:
        print(f"  Skipping per-celltype-negctrl Dox={dox_label} — no data")
        return pd.DataFrame()

    active_dox = list(df['Dox'].unique()) if dox_label != 'ALL' else None
    fig, axes  = plt.subplots(1, len(ct_present),
                              figsize=(5 * len(ct_present), 6), sharey=False)
    if len(ct_present) == 1:
        axes = [axes]

    all_stats = []
    for ax, ct in zip(axes, ct_present):
        col = col_from_ct.get(ct)
        if col is None:
            ax.set_visible(False)
            continue

        sub     = df[df['module_max_label'] == ct].copy()
        present = [g for g in group_order_full if g in sub[group_col].values]

        nc_df        = build_neg_ctrl_for_celltype(paired_full, ct, active_dox)
        neg_ctrl_lbl = nc_df['group'].iloc[0] if not nc_df.empty else None

        main_plot = sub[[col, group_col]].rename(
            columns={col: 'score', group_col: 'group'})
        main_plot['is_negctrl'] = False

        if neg_ctrl_lbl:
            nc_plot = nc_df[['score', 'group']].copy()
            nc_plot['is_negctrl'] = True
            all_data = pd.concat([main_plot, nc_plot], ignore_index=True)
        else:
            all_data = main_plot.copy()

        full_order   = present + ([neg_ctrl_lbl] if neg_ctrl_lbl else [])
        full_palette = {g: palette.get(g, NEGCTRL_COLOR) for g in full_order}
        if neg_ctrl_lbl:
            full_palette[neg_ctrl_lbl] = NEGCTRL_COLOR

        sns.boxplot(data=all_data, x='group', y='score',
                    hue='group', order=full_order,
                    palette=full_palette, legend=False, ax=ax,
                    flierprops=dict(marker='o', markersize=2, alpha=0.4))

        ylim  = ax.get_ylim()
        y_top = ylim[1]

        for i, grp in enumerate(full_order):
            n = int((all_data['group'] == grp).sum())
            ax.text(i, y_top * 1.01, f'n={n}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

        y_max = y_top
        if 'Stable' in present:
            ref_i    = full_order.index('Stable')
            y_levels = [y_top * 1.09, y_top * 1.17]
            for k, grp in enumerate([g for g in present if g != 'Stable']):
                pval  = ttest_vs_ref(all_data, 'score', 'group', grp,
                                     ref_group='Stable')
                y_lvl = y_levels[k % 2]
                y_max = max(y_max, y_lvl)
                draw_bracket(ax, full_order, full_order.index(grp),
                             ref_i, y_lvl, get_sig(pval))

        if neg_ctrl_lbl:
            sep_x = len(present) - 0.5
            ax.axvline(sep_x, color='#666666', lw=1.2,
                       linestyle='--', alpha=0.7)
            pval_nc  = ttest_vs_ref(all_data, 'score', 'group',
                                    neg_ctrl_lbl, ref_group='Stable')
            stars_nc = get_sig(pval_nc)
            ax.text(len(full_order) - 1, y_max * 1.05,
                    f'vs Stable:\n{stars_nc}',
                    ha='center', va='bottom', fontsize=7,
                    color='#555555', style='italic')

        ax.set_ylim(ylim[0], max(y_max, y_top) * 1.22)
        ax.set_title(ct, fontsize=12, fontweight='bold',
                     color=cell_type_colors.get(ct, 'black'))
        ax.set_xlabel('')
        ax.set_ylabel(f'{ct} module Z-score', fontsize=9)
        ax.tick_params(axis='x', rotation=25, labelsize=8)
        all_stats.append(compute_stats(all_data, 'score', 'group',
                                       dox_label, ct, context))

    plt.suptitle(
        f'Module Scores — Stable vs Reprogrammed | Dox = {dox_label}\n'
        f'Grey = NegCtrl (opposite mature type, Stable cells)\n'
        f't-tests vs Stable; dashed line separates neg ctrl',
        fontsize=12, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()


# ── C: Combined (all cell types pooled, self-module score) ──────────────
def make_combined_plot(df, group_col, group_order_full,
                       palette, dox_label, save_path, context,
                       paired_full=None):
    sub = df[df['self_module_score'].notna()].copy()
    present = [g for g in group_order_full if g in sub[group_col].values]

    if len(present) < 2 or 'Stable' not in present:
        print(f"  Skipping combined Dox={dox_label} — insufficient groups")
        return pd.DataFrame()

    # Build NegCtrl: cross-scores of Stable cells
    # Stable INs → IGs module score; Stable IGs → INs module score
    nc_rows = []
    if paired_full is not None:
        active_dox = list(df['Dox'].unique()) if dox_label != 'ALL' else None
        src = paired_full if active_dox is None else paired_full[
            paired_full['Dox'].isin(active_dox)]
        stable_src = src[src['transition'] == 'Stable']

        for focal_ct, cross_ct in [('INs', 'IGs'), ('IGs', 'INs')]:
            cross_col = col_from_ct.get(cross_ct)
            if cross_col is None:
                continue
            cross_cells = stable_src[
                stable_src['module_max_label'] == focal_ct
            ][[cross_col, 'module_max_label']].copy()
            cross_cells = cross_cells.rename(
                columns={cross_col: 'self_module_score'})
            cross_cells[group_col] = 'NegCtrl'
            nc_rows.append(cross_cells)

    neg_ctrl_lbl = None
    if nc_rows:
        nc_df        = pd.concat(nc_rows, ignore_index=True)
        neg_ctrl_lbl = 'NegCtrl'
        plot_sub = pd.concat([
            sub[[group_col, 'self_module_score', 'module_max_label']],
            nc_df[[group_col, 'self_module_score', 'module_max_label']],
        ], ignore_index=True)
    else:
        plot_sub = sub[[group_col, 'self_module_score',
                        'module_max_label']].copy()

    full_order   = present + ([neg_ctrl_lbl] if neg_ctrl_lbl else [])
    full_palette = {g: palette.get(g, NEGCTRL_COLOR) for g in full_order}
    if neg_ctrl_lbl:
        full_palette[neg_ctrl_lbl] = NEGCTRL_COLOR

    fig, ax = plt.subplots(figsize=(max(5, 2.2 * len(full_order)), 6))

    sns.boxplot(data=plot_sub, x=group_col, y='self_module_score',
                hue=group_col, order=full_order,
                palette=full_palette, legend=False, ax=ax,
                flierprops=dict(marker='o', markersize=2, alpha=0.4),
                width=0.5)
    sns.stripplot(data=plot_sub, x=group_col, y='self_module_score',
                  hue='module_max_label', order=full_order,
                  palette=cell_type_colors, dodge=False,
                  alpha=0.5, size=3, jitter=True, ax=ax)
    h, l = ax.get_legend_handles_labels()
    ax.legend(h, l, title='Cell type',
              bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=8, frameon=True)

    ylim  = ax.get_ylim()
    y_top = ylim[1]

    for i, grp in enumerate(full_order):
        n = int((plot_sub[group_col] == grp).sum())
        ax.text(i, y_top * 1.01, f'n={n}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    y_max = y_top
    if 'Stable' in present:
        ref_i    = full_order.index('Stable')
        y_levels = [y_top * 1.09, y_top * 1.17]
        for k, grp in enumerate([g for g in present if g != 'Stable']):
            pval  = ttest_vs_stable(plot_sub, 'self_module_score',
                                    group_col, grp)
            y_lvl = y_levels[k % 2]
            y_max = max(y_max, y_lvl)
            draw_bracket(ax, full_order, full_order.index(grp),
                         ref_i, y_lvl, get_sig(pval))

    if neg_ctrl_lbl:
        sep_x = len(present) - 0.5
        ax.axvline(sep_x, color='#666666', lw=1.2, linestyle='--', alpha=0.7)
        pval_nc  = ttest_vs_ref(plot_sub, 'self_module_score', group_col,
                                neg_ctrl_lbl, ref_group='Stable')
        stars_nc = get_sig(pval_nc)
        ax.text(full_order.index(neg_ctrl_lbl), y_max * 1.05,
                f'vs Stable:\n{stars_nc}',
                ha='center', va='bottom', fontsize=7,
                color='#555555', style='italic')

    ax.set_ylim(ylim[0], max(y_max, y_top) * 1.25)
    ax.set_xlabel('')
    ax.set_ylabel('Self-module score (own DCM cell-type module)', fontsize=10)
    ax.set_title(
        f'All cell types combined — Stable vs Reprogrammed\n'
        f'Grey = Stable INs→IGs + Stable IGs→INs cross-scores (NegCtrl)\n'
        f'Dox = {dox_label}',
        fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=20, labelsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
    return compute_stats(plot_sub, 'self_module_score', group_col,
                         dox_label, 'ALL_combined', context)


# ── D: Destination plot (Stable_<dest> vs Reversed_to_<dest>) ───────────
def make_dest_plot(df, dest_ct, dox_label, save_path, paired_all_raw=None):
    mod_col      = col_from_ct.get(dest_ct)
    stable_label = f'Stable_{dest_ct}'
    rev_label    = f'Reversed_to_{dest_ct}'

    sub = df[df['stemta_dest'] == dest_ct][
        ['stemta_group', 'stemta_score', 'module_max_label', 'Dox']
    ].copy()

    main_order = [g for g in [stable_label, rev_label]
                  if g in sub['stemta_group'].values]

    if len(main_order) < 2:
        print(f"  Skipping {dest_ct} dest plot Dox={dox_label} — insufficient groups")
        return pd.DataFrame()

    active_dox  = list(df['Dox'].unique()) if dox_label != 'ALL' else None
    nc_combined = pd.DataFrame()
    if paired_all_raw is not None:
        nc_df = build_neg_ctrl_stemta(paired_all_raw, dest_ct, active_dox)
        if not nc_df.empty:
            nc_combined = nc_df[['stemta_group', 'stemta_score',
                                 'module_max_label']].copy()

    nc_labels  = ['NegCtrl'] if not nc_combined.empty else []
    full_order = main_order + nc_labels

    stbl_color = GROUP_COLORS.get(stable_label, '#2ecc71')
    rev_color  = GROUP_COLORS.get(rev_label,    cell_type_colors.get(dest_ct, '#333333'))
    pal = {stable_label: stbl_color, rev_label: rev_color}
    if nc_labels:
        pal['NegCtrl'] = NEGCTRL_COLOR

    plot_df = sub[['stemta_group', 'stemta_score', 'module_max_label']].copy()
    if not nc_combined.empty:
        plot_df = pd.concat([plot_df, nc_combined], ignore_index=True)

    fig, ax = plt.subplots(figsize=(7, 6))

    sns.boxplot(data=plot_df, x='stemta_group', y='stemta_score',
                hue='stemta_group', order=full_order,
                palette=pal, legend=False, ax=ax,
                flierprops=dict(marker='o', markersize=2, alpha=0.4))
    sns.stripplot(data=plot_df, x='stemta_group', y='stemta_score',
                  hue='module_max_label', order=full_order,
                  palette=cell_type_colors, dodge=False,
                  alpha=0.5, size=3, jitter=True, ax=ax)
    h, l = ax.get_legend_handles_labels()
    ax.legend(h, l, title='DCM cell type',
              bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=7, frameon=True)

    ylim  = ax.get_ylim()
    y_top = ylim[1]

    for i, grp in enumerate(full_order):
        n = int((plot_df['stemta_group'] == grp).sum())
        ax.text(i, y_top * 1.01, f'n={n}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')

    s_vals = plot_df[plot_df['stemta_group'] == stable_label]['stemta_score'].dropna()
    r_vals = plot_df[plot_df['stemta_group'] == rev_label   ]['stemta_score'].dropna()

    y_brac = y_top
    pval   = np.nan
    if len(s_vals) > 1 and len(r_vals) > 1:
        _, pval = stats.ttest_ind(r_vals, s_vals)
        stars   = get_sig(pval)
        y_ann   = y_top * 1.09
        ax.annotate('', xy=(full_order.index(rev_label), y_ann),
                    xytext=(full_order.index(stable_label), y_ann),
                    arrowprops=dict(arrowstyle='-', color='#333333', lw=1.0))
        ax.text((full_order.index(stable_label) + full_order.index(rev_label)) / 2,
                y_ann, stars,
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                color='black' if stars not in ('ns', '') else '#888888')
        y_brac = y_ann

    if nc_labels:
        sep_x = len(main_order) - 0.5
        ax.axvline(sep_x, color='#666666', lw=1.2, linestyle='--', alpha=0.7)
        ax.text(sep_x + 0.02, ylim[0] + (ylim[1] - ylim[0]) * 0.02,
                'Neg. controls', fontsize=7, color='#555555',
                rotation=90, va='bottom', ha='left', style='italic')

        nc_i = full_order.index('NegCtrl')
        nc_v = plot_df[plot_df['stemta_group'] == 'NegCtrl']['stemta_score'].dropna()

        pval_nc_vs_stable = np.nan
        if len(s_vals) > 1 and len(nc_v) > 1:
            _, pval_nc_vs_stable = stats.ttest_ind(nc_v, s_vals)
        ax.text(nc_i, y_brac * 1.05,
                f'vs {stable_label}:\n{get_sig(pval_nc_vs_stable)}',
                ha='center', va='bottom', fontsize=7,
                color='#444444', style='italic')

        pval_rev_vs_nc = np.nan
        rev_i = full_order.index(rev_label)
        if len(r_vals) > 1 and len(nc_v) > 1:
            _, pval_rev_vs_nc = stats.ttest_ind(r_vals, nc_v)
        stars_rev_nc = get_sig(pval_rev_vs_nc)
        y_rev_nc = max(y_brac, y_top) * 1.17
        ax.annotate('', xy=(nc_i, y_rev_nc), xytext=(rev_i, y_rev_nc),
                    arrowprops=dict(arrowstyle='-', color='#333333', lw=0.9))
        ax.text((rev_i + nc_i) / 2, y_rev_nc, stars_rev_nc,
                ha='center', va='bottom', fontsize=10, fontweight='bold',
                color='black' if stars_rev_nc not in ('ns', '') else '#888888')
        ax.set_ylim(ylim[0], y_rev_nc * 1.15)

    ax.set_ylim(ylim[0], max(y_brac, y_top) * 1.25)

    ctrl_desc = ('+'.join(MATURE_TYPES) if dest_ct in PROGENITOR_TYPES
                 else '+'.join(t for t in MATURE_TYPES if t != dest_ct))
    ax.set_title(
        f'{dest_ct} module Z-score: {stable_label} vs {rev_label}\n'
        f'Grey = Stable {ctrl_desc} (NegCtrl) | Dox = {dox_label}',
        fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(f'{dest_ct} module score', fontsize=9)
    ax.tick_params(axis='x', rotation=20, labelsize=8)

    all_stats = []
    for grp, pv in [(stable_label, np.nan), (rev_label, pval)]:
        vals = plot_df[plot_df['stemta_group'] == grp]['stemta_score'].dropna()
        all_stats.append({
            'context':              f'{dest_ct}_StableVsReversed',
            'cell_type':            dest_ct,
            'Dox':                  dox_label,
            'group':                grp,
            'score_col':            mod_col,
            'N':                    len(vals),
            'mean':                 vals.mean()   if len(vals) else np.nan,
            'median':               vals.median() if len(vals) else np.nan,
            'std':                  vals.std()    if len(vals) else np.nan,
            'ttest_vs_Stable_pval': pv,
            'significance':         get_sig(pv) if grp == rev_label else '',
        })
    if nc_labels:
        nc_v = plot_df[plot_df['stemta_group'] == 'NegCtrl']['stemta_score'].dropna()
        pval_nc_vs_stable, pval_rev_vs_nc = np.nan, np.nan
        if len(s_vals) > 1 and len(nc_v) > 1:
            _, pval_nc_vs_stable = stats.ttest_ind(nc_v, s_vals)
        if len(r_vals) > 1 and len(nc_v) > 1:
            _, pval_rev_vs_nc = stats.ttest_ind(r_vals, nc_v)
        all_stats.append({
            'context':                   f'{dest_ct}_NegCtrl',
            'cell_type':                 dest_ct,
            'Dox':                       dox_label,
            'group':                     'NegCtrl',
            'score_col':                 mod_col,
            'N':                         len(nc_v),
            'mean':                      nc_v.mean()   if len(nc_v) else np.nan,
            'median':                    nc_v.median() if len(nc_v) else np.nan,
            'std':                       nc_v.std()    if len(nc_v) else np.nan,
            'ttest_vs_Stable_pval':      pval_nc_vs_stable,
            'significance_vs_Stable':    get_sig(pval_nc_vs_stable),
            'ttest_Reversed_vs_NegCtrl': pval_rev_vs_nc,
            'significance_Rev_vs_NC':    get_sig(pval_rev_vs_nc),
        })

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
    return pd.DataFrame(all_stats)


# ── E: Barplot — Pulse vs Chase cell counts ─────────────────────────────
def make_combined_dox_barplot(df, group_col, groups, colors,
                               dox_present, title_str, save_path,
                               per_celltype=False):
    if len(dox_present) < 2:
        print(f"  Skipping barplot — fewer than 2 Dox conditions")
        return pd.DataFrame()

    d1, d2    = dox_present[0], dox_present[1]
    all_rows  = []

    if not per_celltype:
        fig, ax     = plt.subplots(figsize=(max(6, 2.2 * len(groups)), 6))
        group_width = 0.35
        gap         = 0.15

        for g_idx, (grp, color) in enumerate(zip(groups, colors)):
            sub  = df[df[group_col] == grp]
            n1   = int((sub['Dox'] == d1).sum())
            n2   = int((sub['Dox'] == d2).sum())
            tot1 = int((df['Dox'] == d1).sum())
            tot2 = int((df['Dox'] == d2).sum())
            pval = (binomtest(n1, n1 + n2, p=0.5, alternative='two-sided').pvalue
                    if (n1 + n2) > 0 else np.nan)
            stars = get_sig(pval)

            x_centre = g_idx * (2 * group_width + gap + 0.1)
            x1, x2   = x_centre - group_width / 2, x_centre + group_width / 2

            b1 = ax.bar(x1, n1, group_width,
                        color=DOX_COLORS.get(d1, color),
                        edgecolor='black', linewidth=0.8,
                        label=d1 if g_idx == 0 else '')
            b2 = ax.bar(x2, n2, group_width,
                        color=DOX_COLORS.get(d2, color),
                        edgecolor='black', linewidth=0.8, hatch='//',
                        label=d2 if g_idx == 0 else '')

            y_max = max(n1, n2, 1)
            for bar, val in [(b1, n1), (b2, n2)]:
                ax.text(bar[0].get_x() + bar[0].get_width() / 2,
                        bar[0].get_height() + y_max * 0.04,
                        str(val), ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

            y_brac   = y_max * 1.18
            pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
            ax.plot([x1, x1, x2, x2],
                    [y_brac - y_max*0.04, y_brac, y_brac, y_brac - y_max*0.04],
                    color='black', lw=1.2)
            ax.text(x_centre, y_brac + y_max * 0.04,
                    f'{stars}\n{pval_txt}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold',
                    color='black' if stars != 'ns' else '#888888')
            ax.text(x_centre, -y_max * 0.08, grp,
                    ha='center', va='top', fontsize=9,
                    fontweight='bold', color=color)

            for dox_val, cnt, total in [(d1, n1, tot1), (d2, n2, tot2)]:
                all_rows.append({
                    'group': grp, 'cell_type': 'ALL', 'Dox': dox_val,
                    'count': cnt, 'total_in_dox': total,
                    'binom_p_pulse_vs_chase': pval, 'significance': stars,
                })

        ax.set_xticks([])
        ax.set_ylabel('Number of cells', fontsize=11, fontweight='bold')
        ax.set_title(title_str, fontsize=12, fontweight='bold')
        handles = [
            plt.Rectangle((0,0), 1, 1,
                           facecolor=DOX_COLORS.get(d1, '#4A90D9'),
                           edgecolor='black',
                           label=DOX_LABELS.get(d1, d1)),
            plt.Rectangle((0,0), 1, 1,
                           facecolor=DOX_COLORS.get(d2, '#E87722'),
                           edgecolor='black', hatch='//',
                           label=DOX_LABELS.get(d2, d2)),
        ]
        ax.legend(handles=handles, loc='upper right', fontsize=9)
        ax.set_ylim(bottom=0)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")
        return pd.DataFrame(all_rows)

    else:
        ct_present = [ct for ct in label_order
                      if ct in df['module_max_label'].unique()]
        if not ct_present:
            return pd.DataFrame()

        fig, axes   = plt.subplots(1, len(ct_present),
                                   figsize=(3.5 * len(ct_present), 6),
                                   sharey=False)
        if len(ct_present) == 1:
            axes = [axes]
        group_width = 0.3
        gap         = 0.1

        for ax, ct in zip(axes, ct_present):
            df_ct = df[df['module_max_label'] == ct]

            for g_idx, (grp, color) in enumerate(zip(groups, colors)):
                sub  = df_ct[df_ct[group_col] == grp]
                n1   = int((sub['Dox'] == d1).sum())
                n2   = int((sub['Dox'] == d2).sum())
                tot1 = int((df_ct['Dox'] == d1).sum())
                tot2 = int((df_ct['Dox'] == d2).sum())
                pval = (binomtest(n1, n1 + n2, p=0.5, alternative='two-sided').pvalue
                        if (n1 + n2) > 0 else np.nan)
                stars = get_sig(pval)

                x_centre = g_idx * (2 * group_width + gap + 0.08)
                x1, x2   = x_centre - group_width / 2, x_centre + group_width / 2

                b1 = ax.bar(x1, n1, group_width,
                            color=DOX_COLORS.get(d1, color),
                            edgecolor='black', linewidth=0.7)
                b2 = ax.bar(x2, n2, group_width,
                            color=DOX_COLORS.get(d2, color),
                            edgecolor='black', linewidth=0.7, hatch='//')

                y_max = max(n1, n2, 1)
                for bar, val in [(b1, n1), (b2, n2)]:
                    ax.text(bar[0].get_x() + bar[0].get_width() / 2,
                            bar[0].get_height() + y_max * 0.05,
                            str(val), ha='center', va='bottom', fontsize=7)

                y_brac   = y_max * 1.20
                pval_txt = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
                ax.plot([x1, x1, x2, x2],
                        [y_brac - y_max*0.05, y_brac,
                         y_brac, y_brac - y_max*0.05],
                        color='black', lw=1.0)
                ax.text(x_centre, y_brac + y_max * 0.05,
                        f'{stars}\n{pval_txt}',
                        ha='center', va='bottom', fontsize=7,
                        color='black' if stars != 'ns' else '#888888')
                ax.text(x_centre, -y_max * 0.10, grp,
                        ha='center', va='top', fontsize=7,
                        fontweight='bold', color=color)

                for dox_val, cnt, total in [(d1, n1, tot1), (d2, n2, tot2)]:
                    all_rows.append({
                        'group': grp, 'cell_type': ct, 'Dox': dox_val,
                        'count': cnt, 'total_in_dox': total,
                        'binom_p_pulse_vs_chase': pval, 'significance': stars,
                    })

            ax.set_xticks([])
            ax.set_title(ct, fontsize=10, fontweight='bold',
                         color=cell_type_colors.get(ct, 'black'))
            ax.set_ylabel('Cell count', fontsize=8)
            ax.set_ylim(bottom=0)

        plt.suptitle(title_str, fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {save_path}")
        return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────
# STEP 5: GENERATE PLOTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: GENERATING PLOTS")
print("=" * 60)

# Per-destination transition orders and palettes
DEST_CONFIG = {
    'RGCs': {
        'trans_label':  'To_RGC',
        'order':        ['Stable', 'To_RGC'],
        'palette':      {'Stable': GROUP_COLORS['Stable'],
                         'To_RGC': GROUP_COLORS['To_RGC']},
    },
    'IGs': {
        'trans_label':  'To_IG',
        'order':        ['Stable', 'To_IG'],
        'palette':      {'Stable': GROUP_COLORS['Stable'],
                         'To_IG':  GROUP_COLORS['To_IG']},
    },
}

collected_stats = []

def dox_tag(d):
    return str(d).replace(' ', '_').replace('/', '-')

for dox_label, subset in (
    [('ALL', paired)] +
    [(str(d), paired[paired['Dox'] == str(d)]) for d in DOX_VALUES]
):
    tag = dox_tag(dox_label)
    print(f"\n{'─'*60}")
    print(f"  Dox = {dox_label}  (n={len(subset)})")

    for dest_ct, cfg in DEST_CONFIG.items():
        trans_label = cfg['trans_label']
        order       = cfg['order']
        palette     = cfg['palette']

        sub_dest = subset[subset['transition'].isin(order)].copy()

        # ── A: Per cell type ──────────────────────────────────────────
        collected_stats.append(make_per_celltype_plot(
            df               = sub_dest,
            group_col        = 'transition',
            group_order_full = order,
            palette          = palette,
            dox_label        = dox_label,
            save_path        = os.path.join(score_dir,
                               f'per_celltype_{dest_ct}_Dox_{tag}.pdf'),
            context          = f'PerCelltype_{dest_ct}',
        ))

        # ── B: Per cell type WITH NegCtrl ────────────────────────────
        collected_stats.append(make_per_celltype_negctrl_plot(
            df               = sub_dest,
            paired_full      = paired,
            group_col        = 'transition',
            group_order_full = order,
            palette          = palette,
            dox_label        = dox_label,
            save_path        = os.path.join(score_dir,
                               f'per_celltype_negctrl_{dest_ct}_Dox_{tag}.pdf'),
            context          = f'PerCelltype_NegCtrl_{dest_ct}',
        ))

        # ── C: Combined (self-module, all cell types pooled) ─────────
        collected_stats.append(make_combined_plot(
            df               = sub_dest,
            group_col        = 'transition',
            group_order_full = order,
            palette          = palette,
            dox_label        = dox_label,
            save_path        = os.path.join(score_dir,
                               f'combined_{dest_ct}_Dox_{tag}.pdf'),
            context          = f'Combined_{dest_ct}',
            paired_full      = paired,
        ))

        # ── D: Destination plot (Stable_<dest> vs Reversed_to_<dest>) ─
        dest_df   = paired_stemta_by_dest.get(dest_ct, pd.DataFrame())
        dest_sub  = dest_df if dox_label == 'ALL' else (
            dest_df[dest_df['Dox'] == str(dox_label)]
        )
        collected_stats.append(make_dest_plot(
            df             = dest_sub,
            dest_ct        = dest_ct,
            dox_label      = dox_label,
            save_path      = os.path.join(score_dir,
                             f'{dest_ct.lower()}_stable_vs_reversed_negctrl_Dox_{tag}.pdf'),
            paired_all_raw = _all_paired_raw,
        ))

# ─────────────────────────────────────────────
# STEP 6: BARPLOTS — Pulse vs Chase cell counts
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: BARPLOTS — PULSE vs CHASE CELL COUNTS")
print("=" * 60)

barplot_stats = []
paired_dox    = paired[paired['Dox'].isin(DOX_ORDER)].copy()
dox_present   = [d for d in DOX_ORDER if d in paired_dox['Dox'].unique()]

# ── Stable / To_<dest> overall and per cell type ─────────────────────────
for dest_ct, cfg in DEST_CONFIG.items():
    trans_label = cfg['trans_label']
    dest_color  = cell_type_colors.get(dest_ct, '#333333')
    sub_dox     = paired_dox[paired_dox['transition'].isin(
                      ['Stable', trans_label])].copy()

    barplot_stats.append(make_combined_dox_barplot(
        df=sub_dox, group_col='transition',
        groups=['Stable', trans_label],
        colors=['#2ecc71', dest_color],
        dox_present=dox_present,
        title_str=f'Stable / {trans_label} — Pulse vs Chase (binomial test)',
        save_path=os.path.join(score_dir,
                  f'barplot_stable_{trans_label}_overall.pdf'),
        per_celltype=False,
    ))
    barplot_stats.append(make_combined_dox_barplot(
        df=sub_dox, group_col='transition',
        groups=['Stable', trans_label],
        colors=['#2ecc71', dest_color],
        dox_present=dox_present,
        title_str=f'Stable / {trans_label} per cell type — Pulse vs Chase',
        save_path=os.path.join(score_dir,
                  f'barplot_stable_{trans_label}_per_celltype.pdf'),
        per_celltype=True,
    ))

# ── Stable_<dest> vs Reversed_to_<dest> overall and per cell type ────────
for dest_ct in DEST_TYPES:
    stable_label = f'Stable_{dest_ct}'
    rev_label    = f'Reversed_to_{dest_ct}'
    dest_color   = cell_type_colors.get(dest_ct, '#333333')

    dest_df     = paired_stemta_by_dest.get(dest_ct, pd.DataFrame())
    dest_df_dox = dest_df[dest_df['Dox'].isin(DOX_ORDER)].copy()
    dest_df_dox['transition'] = dest_df_dox['stemta_group']

    barplot_stats.append(make_combined_dox_barplot(
        df=dest_df_dox, group_col='transition',
        groups=[stable_label, rev_label],
        colors=['#2ecc71', dest_color],
        dox_present=dox_present,
        title_str=f'{stable_label} vs {rev_label} — Pulse vs Chase',
        save_path=os.path.join(score_dir,
                  f'barplot_{dest_ct}_stable_vs_reversed_overall.pdf'),
        per_celltype=False,
    ))
    barplot_stats.append(make_combined_dox_barplot(
        df=dest_df_dox, group_col='transition',
        groups=[stable_label, rev_label],
        colors=['#2ecc71', dest_color],
        dox_present=dox_present,
        title_str=f'{stable_label} vs {rev_label} per cell type — Pulse vs Chase',
        save_path=os.path.join(score_dir,
                  f'barplot_{dest_ct}_stable_vs_reversed_per_celltype.pdf'),
        per_celltype=True,
    ))

bp_stats_df  = pd.concat(
    [s for s in barplot_stats if s is not None and not s.empty],
    ignore_index=True)
bp_stats_csv = os.path.join(score_dir, 'pulse_vs_chase_count_stats.csv')
bp_stats_df.to_csv(bp_stats_csv, index=False)
print(f"\nBarplot stats CSV: {bp_stats_csv}")
print(bp_stats_df.to_string(index=False))

# ─────────────────────────────────────────────
# STEP 7: CSV OUTPUTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: SAVING CSV OUTPUTS")
print("=" * 60)

stats_df  = pd.concat(
    [s for s in collected_stats if s is not None and not s.empty],
    ignore_index=True)
stats_csv = os.path.join(score_dir, 'stable_vs_reprogrammed_stats.csv')
stats_df.to_csv(stats_csv, index=False)
print(f"Stats CSV : {stats_csv}")
print(stats_df.to_string(index=False))

out_cols = (['plate_well', 'Dox', 'module_max_label',
             'rna_celltype', 'transition',
             'self_module_score'] + ttest_cols)
data_csv = os.path.join(score_dir, 'stable_vs_reprogrammed_data.csv')
paired[out_cols].to_csv(data_csv, index=False)
print(f"Data CSV  : {data_csv}")

print("\nDone!")
print(f"Outputs in: {score_dir}")

STEP 1: BUILDING DATAFRAME
Total cells : 15920
Batch values: ['multi_RNA', 'ref_D2', 'ref_D4', 'scNMT_DCM', 'scNMT_RNA']
Dox values  : ['16h', '16h_24h_dtag', 'Unknown']

STEP 2: PAIRING DCM ↔ RNA
1:1 matched wells: 399
Paired cells: 399

STEP 3: CLASSIFYING
Cells with DCM in ANALYSIS_TYPES: 150

Transition counts (ALL Dox):
transition
To_RGC    104
Stable     35
To_IG       9
Other       2

Per DCM cell type:
transition        Other  Stable  To_IG  To_RGC
module_max_label                              
IGs                   2      17      0      38
INs                   0      18      9      66

STEP 4: BUILDING DESTINATION DATAFRAMES

RGCs dataframe: 122 rows
stemta_dest  stemta_group    
RGCs         Reversed_to_RGCs    104
             Stable_RGCs          18

IGs dataframe: 17 rows
stemta_dest  stemta_group
IGs          Stable_IGs      17

STEP 5: GENERATING PLOTS

────────────────────────────────────────────────────────────
  Dox = ALL  (n=150)
  Saved: /media/erlantz/71DE167A75E9

In [ ]:
#CellRank metadata UMAP and PCs

In [23]:
import pandas as pd
import os

out_dir = '/media/erlantz/71DE167A75E99BB4/Multiome_NPC_Run7/10x_data/CellRank/figures/Integration_multi_D2_D4/Python/Jona_notebook_27032026/integration/method_comparison/Harmony_best'

# PCA Harmony
pca_df = pd.DataFrame(
    adata.obsm['X_pca_harmony'],
    index=adata.obs_names,
    columns=[f'PC{i+1}' for i in range(adata.obsm['X_pca_harmony'].shape[1])]
)
pca_df.to_csv(os.path.join(out_dir, 'X_pca_harmony_adataNPC_08062026.csv'))

# UMAP
umap_df = pd.DataFrame(
    adata.obsm['X_umap'],
    index=adata.obs_names,
    columns=['UMAP1', 'UMAP2']
)
umap_df.to_csv(os.path.join(out_dir, 'X_umap_adataNPC_08062026.csv'))

# Metadata
adata.obs.to_csv(os.path.join(out_dir, 'obs_metadata_adataNPC_08062026.csv'))

print("Saved all three CSVs.")

Saved all three CSVs.


In [20]:
adata

AnnData object with n_obs × n_vars = 15920 × 14384
    obs: 'n_genes', 'Sample_RNA', 'Plate', 'PE_raw', 'PE_aligned', 'Assigned', 'UMI', 'Unique_Gene_Count', 'Aligned_percentage', 'Assigned_percentage', 'Unique_percentage', 'Barcode_seq', 'Barcode', 'Well', 'Day', 'Dox', 'Plate.y', 'Annotated', 'Aligned', 'CpG', 'DCM', 'CpG_unique', 'DCM_unique', 'DCM_UMI_genebodies', 'Genes', 'DCM_UMI_wholegene', 'Genes_wholegene', 'Aligned_P', 'CpG_unique_P', 'DCM_unique_P', 'DCM.CpG', 'pool', 'plate_well', 'batch', 'orig.ident', 'total_counts', 'total_counts_mt', 'DCM_matrixID', 'pct_counts_mt', 'n_genes_by_counts', 'assay_day', 'leiden', 'leiden_annot', 'leiden_annot_0.1', 'cell_type_broad', 'IGs_score_Z', 'IGs_module_score_Z', 'INs_score_Z', 'INs_module_score_Z', 'RGCs_score_Z', 'RGCs_module_score_Z', 'IGs_score_raw', 'IGs_module_score_raw', 'INs_score_raw', 'INs_module_score_raw', 'RGCs_score_raw', 'RGCs_module_score_raw', 'scNMT_pool1_DCM_IGs_module_score_Z_posterior', 'scNMT_pool1_DCM_INs_modul

In [ ]:
adata.obs

In [19]:
pd.DataFrame(index=adata.obs_names,
             UMAP=adata.obsm['X_umap'])

array([[-10.820651  ,  -4.951078  , -16.381208  , ...,  -1.7697773 ,
          0.14121899,   3.0176768 ],
       [  8.131812  ,  -2.916734  ,  -9.103461  , ...,  -0.5768331 ,
          1.1610408 ,  -2.3301377 ],
       [ -9.698967  ,   2.6849048 ,  -7.2186866 , ...,   2.0726438 ,
         -1.7812636 ,  -1.1624042 ],
       ...,
       [ 13.120082  ,  -5.795705  ,   2.5240319 , ...,  -0.02467123,
         -1.585274  ,  -2.0890317 ],
       [  3.2579143 , -10.757147  ,  -1.6470402 , ...,  -4.588636  ,
          0.1099861 ,  -0.80926037],
       [  8.104546  ,  -1.3677636 ,   1.8891808 , ...,   0.9987882 ,
         -0.3286534 ,   1.1095651 ]], shape=(15920, 20), dtype=float32)